# Figures and tables of the paper

Set the run in the first cell. Every cell below reads and writes that run's folder. The notebook runs from top to bottom: the two preparation cells (the projections, and the data, the uncertainty budget and the global mean that most figures and tables use), then the figures and tables, each with its name in the paper in the heading. Diagnostics (printouts for checks and numbers for the text) sit next to the figure whose variables they use. The last cell copies every display item under the name the manuscript uses (Fig, EDFig, EDTable, SuppFig, SuppTable). The README lists for each display item the cell that draws it.

## Run

In [ ]:
# Which run all cells below use (data, figures and tables go to that run's folder data/<run>/):
#   'u74'    standard: K1, K2 Lueker et al. (2000), total boron Uppstroem (1974)
#   'l10'    sensitivity: total boron Lee et al. (2010)
#   'm10'    sensitivity: K1, K2 Millero (2010)
#   'm10l10' sensitivity: both
# The notebook runs from top to bottom: the preparation cell computes the projections once (cached in the run
# folder, projection.py); every cell that uses them calls projection.py, so the cells can also run in any order.
import os, sys, importlib, warnings
warnings.filterwarnings('ignore', message='Mean of empty slice', category=RuntimeWarning)   # averages over cells without data
os.environ['OA_RUN'] = 'u74'
os.environ.setdefault('OA_WEIGHTS', 'ocean')      # area weights: 'ocean' (cell area x ocean fraction) or 'cell'
CODE_DIR = os.path.abspath('.')    # the code folder: open the notebook from code/
sys.path.insert(0, CODE_DIR)
import regional_common as rc
importlib.reload(rc)
import oa_config
print(f'run {rc.RUN}: data from {rc.ROOT}, figures to {rc.FIGDIR}; area weights: {oa_config.WEIGHTS_TEXT}')
# the product files record the weights they were made with: they must be the ones of this notebook session
import glob as _glob, netCDF4 as _nc
for _p in _glob.glob(f'{rc.ROOT}/product/carbonate_system_*.nc')[:1]:
    with _nc.Dataset(_p) as _f:
        assert getattr(_f, 'area_weights', oa_config.WEIGHTS_TEXT) == oa_config.WEIGHTS_TEXT, \
            f'{_p} was made with other area weights ({_f.area_weights}); set OA_WEIGHTS accordingly'

## Preparation

The projections the figures use (global and per cell), computed once per run and cached in the run folder (`projection.py`; delete `fig_crossing_projection.pkl` or `fig_regional_crossing_projection.pkl` there to recompute). Slow the first time, instant afterwards. No figure.

In [ ]:
# Preparation: the projections of every product, global and per cell (no figure)
import importlib
import projection
importlib.reload(projection)
P_GLOBAL = projection.global_projection()
P_REGIONAL = projection.regional_projection()
print(f'run {rc.RUN}: projections ready for {sorted(P_GLOBAL)} (global) and {sorted(P_REGIONAL)} (per cell)')

## Preparation: the data of Fig. 3, the uncertainty budget and the global mean

Observed values, constrained 1750 values, the 2024 estimators and the full uncertainty budget (Extended Data Table 2), and `decline_crossing`, the year and CO2 at which the forced decline reaches a given level (10 %, 20 %). Used by Figs. 1, 3, and 4, the tables, Extended Data Figs. 1, 2, and 6, Supplementary Fig. 3, and the numbers for the text. Draws nothing.

In [ ]:
# Notebook cell (preparation): the data of Fig. 3 and the uncertainty budget, used by Figs. 1, 3, and 4, Extended Data Tables 1 to 3,
# Extended Data Figs. 1 and 6, and Supplementary Fig. 3 and the numbers for the text. Draws nothing; the Fig. 3 cell draws the figure from these variables.
# Its variables are kept in FIG4, so that the Fig. 3 cell and the tables find them even if other cells reuse a name.
# Fig. 3, drawn in its own cell from these variables: a, annual global mean surface Omega_arag of each product; b, the
# decline from 1750; c, from the Planetary Health Check to this study for UExP-FNN-U, one change per bar (EDT3_A).
# Observed values: quadratic fit to the annual area-weighted means, evaluated in 2024 (as in Extended Data Table 1).
# Needs data/constraint_results.pkl (with 'states'), the product files (rc.OBS) and regional_common.py.
import sys
import pickle
import importlib
import numpy as np
import netCDF4
import matplotlib.pyplot as plt
from matplotlib.gridspec import GridSpec, GridSpecFromSubplotSpec
from matplotlib.lines import Line2D

sys.path.insert(0, CODE_DIR)
import regional_common as rc
importlib.reload(rc)
import oa_config                      # the area weights of the run (OA_WEIGHTS)

# total boron of the run: 'l10' Lee et al. (2010); 'u74' (the paper) and the others Uppstroem (1974). The run (data,
# product files, figures) is chosen with os.environ['OA_RUN'] at the top of the notebook (regional_common)
BORON = 'l10' if rc.RUN == 'l10' else 'u74'               # total boron of this run ('m10' uses Uppstroem as well)
ROOT = rc.ROOT
OBSF = dict(rc.OBS)
print(f'run {rc.RUN} (total boron {BORON}): {ROOT}')
LAB = rc.LAB
YEAR = 2024
OMEGA = r'$\Omega_{\mathrm{arag}}$'
# Planetary Health Check (2026), ocean acidification section: observed 2.85 ± 0.01 (2σ) in 2024 from UExP-FNN-U on its
# full area; preindustrial 3.57 = Jiang et al. (2023, Table S5) for 1750; boundary 2.86 (80 % of 3.57, as published)
PHC_OBS, PHC_PI, PHC_B = 2.85, 3.57, 2.86
DETAILED = False                      # panel c: False = the three consistency steps, True = every single change
STEP_REF = 'UExP'                     # product of the Planetary Health Check
BCOL = 'k'                            # boundary

with open(f'{ROOT}/constraint_results.pkl', 'rb') as f:
    PK = pickle.load(f)
RCOM, ROWN = PK['results_common'], PK['results']
refs = [q for q in ('OceanSODA', 'UExP', 'JMA', 'CMEMS') if q in RCOM]


def read_obs(q):
    r = RCOM[q]
    with netCDF4.Dataset(OBSF[q]) as f:
        return (np.asarray(f['year'][:]), np.asarray(r['lat'], 'f8'), np.asarray(r['lon'], 'f8'),
                np.ma.filled(f['omega_arag'][:].astype('f8'), np.nan))


def gm_series(yo, om, lat, lon, mask):
    """annual area-weighted means over mask (the weights of the run)"""
    wm = oa_config.area_weights(lat, lon) * mask
    return np.array([np.nansum(np.where(np.isfinite(o), o, 0) * wm) / wm[np.isfinite(o)].sum() for o in om])


def fit_now(yo, gm):
    """quadratic fit to the annual means, evaluated in YEAR (as Extended Data Table 1)"""
    ok = np.isfinite(gm) & (yo <= rc.LAST_YEAR)
    return np.polyval(np.polyfit(yo[ok] - 2000.0, gm[ok], 2), YEAR - 2000.0)


# ---------------------------------------------------------------- coverage and observed values per product
cov, V, TS = {}, {}, {}
for q in refs:
    yo, lat, lon, om = read_obs(q)
    full = np.isfinite(om[yo <= rc.LAST_YEAR]).all(0)          # complete: every year (and so every month) up to 2024
    own = np.asarray(ROWN[q]['mask']).astype(bool)
    com = np.asarray(RCOM[q]['mask']).astype(bool)
    g = {k: gm_series(yo, om, lat, lon, m_) for k, m_ in (('full', full), ('own', own), ('com', com))}
    keep = yo <= rc.LAST_YEAR
    TS[q] = dict(year=yo[keep], own=g['own'][keep], com=g['com'][keep])
    V[q] = {k: fit_now(yo, v) for k, v in g.items()}
    # one map grid for all products: longitudes -180..180 (products differ in convention), latitudes ascending
    lon180, full180 = rc.to_180(lon, full)
    com180 = rc.to_180(lon, com)[1]
    own180 = rc.to_180(lon, own)[1]
    if lat[0] > lat[-1]:
        lat, full180, com180, own180 = lat[::-1], full180[::-1], com180[::-1], own180[::-1]
    cov[q] = dict(lat=lat, lon=lon180, full=full180, com=com180, own=own180)
lat, lon = cov[refs[0]]['lat'], cov[refs[0]]['lon']
for q in refs[1:]:
    assert np.allclose(cov[q]['lat'], lat) and np.allclose(cov[q]['lon'], lon), 'products on different grids'
n_prod = sum(cov[q]['full'].astype(int) for q in refs)
common = cov[refs[0]]['com']
w = oa_config.area_weights(lat, lon)
ocean = rc.ocean_on(lat, lon)          # one ocean definition for every percentage (regional_common.ocean_mask)
pct = lambda m: 100 * w[m & ocean].sum() / w[ocean].sum()
AREA_C = pct(common & ocean)

# preindustrial (constrained, 1750) and decline, own area and common area
dec = lambda o, p: 100 * (1 - o / p)
PI = {}
for q in refs:
    yo_, so_ = ROWN[q]['constrained'][LAB][:2]
    yc_, sc_ = RCOM[q]['constrained'][LAB][:2]
    PI[q] = dict(own=yo_, sown=so_, com=yc_, scom=sc_,
                 dec_own=dec(V[q]['own'], yo_), sdec_own=100 * V[q]['own'] / yo_ ** 2 * so_,
                 dec_com=dec(V[q]['com'], yc_), sdec_com=100 * V[q]['com'] / yc_ ** 2 * sc_)
    print(f"{q:10s} 2024: full area {V[q]['full']:.3f}, own ({pct(cov[q]['own'] & ocean):.1f} %) {V[q]['own']:.3f}, "
          f"common ({AREA_C:.1f} %) {V[q]['com']:.3f}; 1750: own {yo_:.3f}, common {yc_:.3f}; decline: own "
          f"{PI[q]['dec_own']:.2f} %, common {PI[q]['dec_com']:.2f} % (own minus common {PI[q]['dec_own'] - PI[q]['dec_com']:+.2f} "
          f"points)")
yc_m = np.mean([PI[q]['com'] for q in refs])
B = 0.8 * yc_m
B_sig = 0.8 * np.sqrt(np.mean([PI[q]['scom'] ** 2 for q in refs]))

# ---------------------------------------------------------------- steps from the Planetary Health Check to this study
S = RCOM[STEP_REF]['states']
now_c = V[STEP_REF]['com']
now_m = np.mean([V[q]['com'] for q in refs])
dec_q = np.array([PI[q]['dec_com'] for q in refs])
dec_m = dec_q.mean()
# ---------------------------------------------------------------- uncertainty of the decline (1 sigma, percentage points)
# all components, combined in quadrature; per-product components are averaged as RMS (largely common to all products)
import os
_ddec = lambda q, s_om: 100 * V[q]['com'] / PI[q]['com'] ** 2 * s_om      # d(decline) for an error s_om in 1750


# ---------------------------------------------------------------- estimate of the 2024 state
# main estimator: quadratic fit in time (as in constraint_refs.py). Alternatives: quadratic fit against atmospheric CO2
# (NOAA global annual means) and Enting (1987) smoothing splines in time and in CO2 with cutoffs of 15, 20 and 25 years
# (20 as in Silvy et al. 2024 and Terhaar et al. 2025: removes most interannual variability, keeps the trend; below 15
# years the 2023/24 El Nino warmth enters the end of the spline, above 25 years the spline approaches a straight line,
# which misses the curvature against CO2, and the cutoff approaches the length of the records). All are linear in
# the annual means y (estimate = w . y); the uncertainty of the main estimator is sqrt(w' C w) with C the covariance of
# the residuals around it, modelled as AR(1) (variance s2, lag-1 autocorrelation r), since interannual variability such
# as ENSO is not independent from year to year. The estimator term is half the range of the alternatives.
ENT_CUTOFFS = (15.0, 20.0, 25.0)    # years; in CO2 converted with the mean growth rate of the fit period
ENT_MAIN = 20.0                      # the spline drawn in bold in Extended Data Fig. 1
import projection                    # the NOAA record of the run folder (co2_annmean_gl.txt), as the projections
importlib.reload(projection)
XCO2 = projection.noaa_co2()


def _series(q):
    r = RCOM[q]
    yo_, gm_ = np.asarray(r['year'], 'f8'), np.asarray(r['ogm'], 'f8')
    ok = np.isfinite(gm_) & (yo_ <= rc.LAST_YEAR)
    assert np.all(np.diff(yo_[ok]) == 1), f'{q}: annual series with gaps'
    return yo_[ok], gm_[ok]


def _w_poly(x, x0, deg):
    """weights of a polynomial fit of degree deg in x, evaluated at x0, and the hat matrix (all fitted values)"""
    X = np.column_stack([x ** k for k in range(deg, -1, -1)])
    H = np.linalg.solve(X.T @ X, X.T)
    v0 = np.array([x0 ** k for k in range(deg, -1, -1)])
    return v0 @ H, X @ H


def _smoother(x, cutoff):
    """Enting (1987) smoothing spline on (possibly uneven) x: penalised least squares with second divided differences,
    lambda chosen for 50 % response at the cutoff (in units of x) for the mean spacing; returns the smoother matrix"""
    n_ = x.size
    h = np.diff(x)
    D = np.zeros((n_ - 2, n_))
    for i in range(n_ - 2):
        D[i, i], D[i, i + 2] = 2 / (h[i] * (h[i] + h[i + 1])), 2 / (h[i + 1] * (h[i] + h[i + 1]))
        D[i, i + 1] = -D[i, i] - D[i, i + 2]
    hm = h.mean()
    lam = hm ** 4 / (2.0 - 2.0 * np.cos(2.0 * np.pi * hm / cutoff)) ** 2
    return np.linalg.inv(np.eye(n_) + lam * D.T @ D)


_AR1_CACHE = {}


def _ar1_calibration(n, wq, r_obs, n_sim=20000, seed=1):
    """The lag-1 autocorrelation of the residuals of a quadratic fit to n annual values is biased low, and so
    is the AR(1) standard error built from it. Monte Carlo with AR(1) noise: rho_c, the autocorrelation whose residuals
    give r_obs as median estimate, and the factor by which the standard error the chain computes (residual variance
    and r from the residuals) underestimates the true error of the fit value (weights wq) at rho_c"""
    key = (n, round(float(r_obs), 3), np.round(wq, 6).tobytes(), n_sim, seed)
    if key in _AR1_CACHE:
        return _AR1_CACHE[key]
    rng_ = np.random.default_rng(seed)
    t_ = np.arange(n, dtype='f8')
    X_ = np.vander(t_, 3)
    H_ = X_ @ np.linalg.pinv(X_)
    lag = np.abs(np.subtract.outer(np.arange(n), np.arange(n)))

    def sim(rho):
        e_ = rng_.standard_normal((n_sim, n))
        x_ = np.empty_like(e_)
        x_[:, 0] = e_[:, 0]
        for i_ in range(1, n):
            x_[:, i_] = rho * x_[:, i_ - 1] + np.sqrt(1 - rho ** 2) * e_[:, i_]
        r_ = x_ - x_ @ H_.T
        a_, b_ = r_[:, :-1] - r_[:, :-1].mean(1, keepdims=True), r_[:, 1:] - r_[:, 1:].mean(1, keepdims=True)
        rh = np.clip((a_ * b_).sum(1) / np.sqrt((a_ * a_).sum(1) * (b_ * b_).sum(1)), 0.0, 0.99)
        return x_, r_, rh

    grid = np.linspace(0.0, 0.8, 33)
    med = [np.median(sim(rho)[2]) for rho in grid]
    rho_c = float(np.interp(r_obs, med, grid))
    x_, r_, rh = sim(rho_c)
    s2_ = (r_ ** 2).sum(1) / (n - 3)
    k_ = slice(0, 2000)                               # the chain's estimate on a subsample (memory)
    se_ = np.sqrt(s2_[k_] * np.einsum('i,kij,j->k', wq, rh[k_, None, None] ** lag[None], wq))
    _AR1_CACHE[key] = (rho_c, float((x_ @ wq).std() / np.median(se_)))
    return _AR1_CACHE[key]


def _trend(q, series=None, year=YEAR):
    """the estimators of the state of `year` from annual means (default: the product on the common area, _series),
    their spread and the AR(1) standard error of the quadratic fit; also used by Extended Data Table 1 for every area and year"""
    yrs, y_ = _series(q) if series is None else series
    t = yrs - 2000.0
    xc = np.array([XCO2[int(v)] for v in yrs])
    gr = (xc[-1] - xc[0]) / (yrs[-1] - yrs[0])                  # ppm per year over the fit period
    i24 = list(yrs).index(year)
    e = dict(yrs=yrs, y=y_, xco2=xc)
    wq, Hq = _w_poly(t, year - 2000.0, 2)
    e['time quadratic'], e['fit time quadratic'] = float(wq @ y_), Hq @ y_
    wc, Hc = _w_poly(xc - 400.0, XCO2[year] - 400.0, 2)
    e['CO2 quadratic'], e['fit CO2 quadratic'] = float(wc @ y_), Hc @ y_
    for co in ENT_CUTOFFS:
        St, Sc = _smoother(yrs, co), _smoother(xc, co * gr)
        e[f'fit time Enting {co:.0f} yr'], e[f'fit CO2 Enting {co:.0f} yr'] = St @ y_, Sc @ y_
        e[f'time Enting {co:.0f} yr'] = float((St @ y_)[i24])
        e[f'CO2 Enting {co * gr:.0f} ppm'] = float((Sc @ y_)[i24])
    res = y_ - e['fit time quadratic']
    s2 = res @ res / (yrs.size - 3)
    r1 = float(np.clip(np.corrcoef(res[:-1], res[1:])[0, 1], 0.0, 0.99))
    C = s2 * r1 ** np.abs(np.subtract.outer(np.arange(yrs.size), np.arange(yrs.size)))
    alt = [v for k, v in e.items() if isinstance(v, float) and k[:4] in ('time', 'CO2 ')]
    rho_c, k_se = _ar1_calibration(yrs.size, wq, r1)
    e['se_uncorrected'] = float(np.sqrt(wq @ C @ wq))
    e['se'], e['r1'], e['rho_c'], e['k_se'] = e['se_uncorrected'] * k_se, r1, rho_c, k_se
    e['gr'], e['range'] = gr, max(alt) - min(alt)
    return e


EST = {q: _trend(q) for q in refs}
print(f'  2024 state by estimator (quadratic fits and Enting splines with {", ".join(f"{c:.0f}" for c in ENT_CUTOFFS)} yr '
      'cutoff):')
for q in refs:
    e = EST[q]
    print(f'    {q:10s} {int(e["yrs"][0])}-{int(e["yrs"][-1])} (CO2 +{e["gr"]:.2f} ppm/yr): '
          + ', '.join(f'{k} {v:.3f}' for k, v in e.items() if isinstance(v, float) and k[:4] in ('time', 'CO2 '))
          + f'; quadratic +- {e["se"]:.4f} (AR(1), r = {e["r1"]:.2f}, bias-corrected {e["rho_c"]:.2f}, factor '
            f'{e["k_se"]:.2f} on {e["se_uncorrected"]:.4f}), range of alternatives {e["range"]:.3f}')
    print(f'    {"":10s} annual mean {YEAR} {e["y"][-1]:.3f}, i.e. {e["y"][-1] - e["time quadratic"]:+.3f} from the quadratic fit '
          f'({100 * (e["y"][-1] - e["time quadratic"]) / e["time quadratic"]:+.2f} %, '
          f'{-100 * (e["y"][-1] - e["time quadratic"]) / PI[q]["com"]:+.2f} points of decline)')


def _co2_sig(q):
    """1750 Omega change for +-1 sigma of the 1750 CO2 (AR6, 278.3 +- 1.8 ppm), from the states 1750_lo/_hi"""
    st = RCOM[q]['states']
    if '1750_lo' not in st or '1750_hi' not in st:
        return np.nan
    return PI[q]['com'] * abs(st['1750_lo']['ratio'] - st['1750_hi']['ratio']) / 2


def _run_means(path):
    """mean decline, 2024 state and 1750 state over the products in a run (fit in YEAR, constrained 1750, common mask)"""
    with open(path, 'rb') as f:
        rr = pickle.load(f)['results_common']
    now_, pi_ = np.array([rc.fit_now(rr[q]) for q in refs]), np.array([rr[q]['constrained'][LAB][0] for q in refs])
    return np.mean(100 * (1 - now_ / pi_)), now_.mean(), pi_.mean()


rms = lambda a: float(np.sqrt(np.nanmean(np.square(a))))
K_ESM, K_CO2, K_PROD_D = '1750 state estimated from the ESMs (after the emergent constraint)', 'atmospheric CO$_2$ in 1750 (AR6, 278.3 ± 1.8 ppm)', 'differences between the four fCO$_2$-products'
K_FIT, K_METH, K_B, K_K12 = 'forced 2024 state: statistical uncertainty of the fit (natural variability)', 'forced 2024 state: choice of fitting method', 'total boron (Uppström 1974 or Lee et al. 2010)', 'K$_1$, K$_2$ (Lueker et al. 2000 or Millero 2010)'
SIG = {K_ESM: rms([PI[q]['sdec_com'] for q in refs]),
       K_CO2: rms([_ddec(q, _co2_sig(q)) for q in refs]),
       K_PROD_D: float(dec_q.std(ddof=1)),
       K_FIT: rms([100 * EST[q]['se'] / PI[q]['com'] for q in refs]),
       K_METH: rms([100 * EST[q]['range'] / 2 / PI[q]['com'] for q in refs])}
# the same components in absolute Omega_arag (1 sigma): (1750 state, 2024 state); nan = does not affect that state
_v24 = np.array([V[q]['com'] for q in refs])
_v50 = np.array([PI[q]['com'] for q in refs])
ABS = {K_ESM: (rms([PI[q]['scom'] for q in refs]), np.nan),
       K_CO2: (rms([_co2_sig(q) for q in refs]), np.nan),
       K_PROD_D: (float(_v50.std(ddof=1)), float(_v24.std(ddof=1))),
       K_FIT: (np.nan, rms([EST[q]['se'] for q in refs])),
       K_METH: (np.nan, rms([EST[q]['range'] / 2 for q in refs]))}
# equilibrium constants: difference of the decline between two formulations taken as 2 sigma (as Orr et al., 2018, for
# total boron); from the sensitivity runs, if available
_here = _run_means(f'{ROOT}/constraint_results.pkl')
_other = rc.RUNS['u74' if BORON == 'l10' else 'l10'] + '/constraint_results.pkl'
_m10 = rc.RUNS['m10'] + '/constraint_results.pkl'
for lab_, path_, needs in ((K_B, _other, rc.RUN == 'u74'), (K_K12, _m10, rc.RUN == 'u74')):
    try:
        if not needs:
            raise FileNotFoundError("only evaluated for the standard run (OA_RUN = 'u74')")
        _o = _run_means(path_)
        SIG[lab_] = abs(_o[0] - _here[0]) / 2
        ABS[lab_] = (abs(_o[2] - _here[2]) / 2, abs(_o[1] - _here[1]) / 2)
    except Exception as e_:
        if needs and oa_config.WEIGHTS == 'ocean':                 # the paper's run: boron and K1/K2 are required
            raise FileNotFoundError(f'{path_}: the sensitivity run is needed for {lab_} ({e_})') from e_
        SIG[lab_], ABS[lab_] = np.nan, (np.nan, np.nan)
        print(f'  {lab_}: not included ({e_})')
# Equilibrium constants within each formulation (Orr et al. 2018: pK1 0.0075, pK2 0.015, pKb 0.01, pKw 0.01,
# pKspa 0.02, total boron 2 %), propagated with mocsy.merrors on the observed 2024 state (UExP-FNN-U v2026-1) and on
# the 1750 states of the 18 ESMs (product_constants_uncertainty.py). The same constants apply everywhere, so the
# errors are fully correlated; the aragonite solubility product (96 % of the absolute value) cancels in the decline.
K_CONST = 'equilibrium constants and total boron within each formulation (Orr et al. 2018)'
_fc = f'{ROOT}/product_uncertainty/constants_UExP_2024.npz'
if not os.path.exists(_fc):
    raise FileNotFoundError(f'{_fc} missing: run ./run_checks.sh {rc.RUN} (product_constants_uncertainty.py) first')
_c = np.load(_fc)
_rel17, _rel24, _cd = float(_c['rel17']), float(_c['rel24']), float(_c['totd'])
SIG[K_CONST] = _cd
ABS[K_CONST] = (_rel17 * float(_v50.mean()), _rel24 * float(_v24.mean()))

# Mapping uncertainty of a product's global mean: spatially correlated errors (decorrelation lengths of Ford et al.
# 2024), same pattern in all twelve months (C2, conservative), per product (product_uncertainty_mc_C.py), and the same
# error fields in the observed CUC, the predictor (cuc_mapping_uncertainty.py: sigma of CUC, its correlation rho with
# the Omega_arag error, both from the product's window state; beta = coefficient of CUC in the constraint). One mapping
# error d shifts the 2024 value by d and the 1750 value by d (bias adjustment) + beta dCUC (constraint), so
#   1750:      sigma^2 = s^2 + beta^2 sC^2 + 2 beta rho s sC
#   2024:      sigma = s
#   decline:   100 [d (now - pi) + beta dCUC now] / pi^2       (the remainder of the level error and the CUC part)
#   distance:  0.2 d - 0.8 beta dCUC
# RMS over the four products.
K_MAP = 'mapping uncertainty of a fCO$_2$-product (spatially correlated errors)'
MAP_SIG, MAP_CUC = {}, {}
for p_ in ('UExP', 'OceanSODA', 'JMA', 'CMEMS'):
    _fm = f'{ROOT}/product_uncertainty/mcC_{p_}_2024_L1_r{oa_config.L_RANGE:g}_n100.npz'   # lengths drawn within a factor 2
    _fc_ = f'{ROOT}/product_uncertainty/cuc_mapping_{p_}.npz'
    for _f in (_fm, _fc_):
        if not os.path.exists(_f):
            raise FileNotFoundError(f'{_f} missing: run ./run_checks.sh {rc.RUN} (product_uncertainty_mc_C.py) and '
                                    'cuc_mapping_uncertainty.py first')
    MAP_SIG[p_] = float(np.load(_fm)['C2_global'].std())
    _z = np.load(_fc_)
    MAP_CUC[p_] = {a_: (float(_z[f'{a_}_sigma_cuc_C2']), float(_z[f'{a_}_rho_C2']), float(_z[f'{a_}_beta_cuc']))
                   for a_ in ('own', 'common') if f'{a_}_sigma_cuc_C2' in _z.files}


def map_terms(s, cuc, now, pi):
    """1 sigma of the 1750 value, the 2024 value, the decline (percentage points) and the distance from one product's
    mapping error: s (Omega_arag, 2024), cuc = (sigma of CUC, correlation with the Omega_arag error, beta_CUC)"""
    sC, rho, beta = cuc
    A, B = 100 * (now - pi) / pi ** 2, 100 * beta * now / pi ** 2
    v = lambda a, b: float(np.sqrt(max((a * s) ** 2 + (b * sC) ** 2 + 2 * a * b * rho * s * sC, 0.0)))
    return dict(pi=v(1.0, beta), now=s, dec=v(A, B), dist=v(0.2, -0.8 * beta))


_mt = {q: map_terms(MAP_SIG[q], MAP_CUC[q]['common'], V[q]['com'], PI[q]['com']) for q in refs}
_mp = [MAP_SIG[q] for q in refs]
_mm = rms(_mp)
SIG[K_MAP] = rms([_mt[q]['dec'] for q in refs])
ABS[K_MAP] = (rms([_mt[q]['pi'] for q in refs]), rms([_mt[q]['now'] for q in refs]))
for q in refs:
    print(f'  mapping {q:10s}: 2024 {MAP_SIG[q]:.4f}, CUC {MAP_CUC[q]["common"][0]:.5f} (rho {MAP_CUC[q]["common"][1]:+.2f}, '
          f'beta {MAP_CUC[q]["common"][2]:.2f}) -> 1750 {_mt[q]["pi"]:.4f}, decline {_mt[q]["dec"]:.3f} pp (without CUC '
          f'{100 * MAP_SIG[q] * (PI[q]["com"] - V[q]["com"]) / PI[q]["com"] ** 2:.3f}), distance {_mt[q]["dist"]:.4f}')
CONST_REL = (_rel17, _rel24, _cd)                                            # for Extended Data Table 1
print(f'  constants: 1 sigma 1750 {ABS[K_CONST][0]:.3f}, 2024 {ABS[K_CONST][1]:.3f}, decline {_cd:.3f} pp; '
      f'mapping (C2, {len(_mp)} products, RMS): 2024 {_mm:.4f}, 1750 {ABS[K_MAP][0]:.4f}, decline {SIG[K_MAP]:.3f} pp')

sdec_m = float(np.sqrt(np.nansum(np.square(list(SIG.values())))))
# the part of it shared by all products (without their spread): the uncertainty of the crossing of the boundary
# (Figs. 1 and 4, Extended Data Fig. 6, Supplementary Fig. 3, and the numbers for the text; the spread of the products is added there from their own crossings)
SDEC_SHARED = float(np.sqrt(sdec_m ** 2 - SIG[K_PROD_D] ** 2))
print('uncertainty of the decline (1 sigma, percentage points):')
for k_, v_ in SIG.items():
    print(f'  {k_.replace("$", ""):55s} {v_:.2f}')
print(f'  {"total (quadrature)":55s} {sdec_m:.2f}')
# without the emergent constraint (for the text): the ESM term from the spread of the bias-adjusted 1750
# states across the 18 ESMs (s.d.) instead of the constrained sigma, all other terms unchanged; common area
_s_unc = {q: float(np.std(np.asarray(RCOM[q]['y1750'], 'f8'), ddof=1)) for q in refs}
_s_con = rms([PI[q]['scom'] for q in refs])
_esm_unc = rms([_ddec(q, _s_unc[q]) for q in refs])
SDEC_UNCONSTRAINED = float(np.sqrt(sdec_m ** 2 - SIG[K_ESM] ** 2 + _esm_unc ** 2))
print(f'  without the emergent constraint: sigma of the 1750 state {rms(list(_s_unc.values())):.4f} instead of {_s_con:.4f} '
      f'({100 * (1 - _s_con / rms(list(_s_unc.values()))):.0f} % smaller with it); ESM term {_esm_unc:.2f} instead of '
      f'{SIG[K_ESM]:.2f}; total {SDEC_UNCONSTRAINED:.2f} instead of {sdec_m:.2f} percentage points '
      f'(+{SDEC_UNCONSTRAINED - sdec_m:.2f})')
X_J, X_1750 = S['jiang']['xco2'], S['1750']['xco2']
# observed side for STEP_REF, from the product as published to our value on the common area
yo_s, lat_s, lon_s, om_s = read_obs(STEP_REF)
with netCDF4.Dataset(OBSF[STEP_REF]) as f:
    om_p = np.ma.filled(f['omega_arag_product'][:].astype('f8'), np.nan)    # the product's own Omega (mean of months)
    om_mm = np.ma.filled(f['omega_arag_monthly_mean'][:].astype('f8'), np.nan)   # mocsy, mean of the monthly values
i24 = list(yo_s).index(YEAR)
mean24 = lambda fld, m_: gm_series(yo_s[i24:i24 + 1], fld[i24:i24 + 1], lat_s, lon_s, m_)[0]
own_s = np.asarray(ROWN[STEP_REF]['mask']).astype(bool)
OBS_STEPS = dict(prod=mean24(om_p, np.isfinite(om_p[i24])), mocsy=mean24(om_s, np.isfinite(om_s[i24])),
                 mocsy_mm=mean24(om_mm, np.isfinite(om_mm[i24])),
                 own=mean24(om_s, own_s), own_fit=V[STEP_REF]['own'], com_fit=now_c,
                 com24=mean24(om_s, np.asarray(RCOM[STEP_REF]['mask']).astype(bool)))
print(f'{STEP_REF} 2024: product Omega {OBS_STEPS["prod"]:.4f} (mean of months), mocsy mean of months '
      f'{OBS_STEPS["mocsy_mm"]:.4f}, mocsy annual-mean state {OBS_STEPS["mocsy"]:.4f}, own area {OBS_STEPS["own"]:.3f}, '
      f'fit {OBS_STEPS["own_fit"]:.3f}, common area (fit) {OBS_STEPS["com_fit"]:.3f}')
# Planetary Health Check (2026), reproduced: the pre-release UExP-FNN-U v2026-pre1 (Zenodo 19605928) in 2024, the
# product's own Omega_arag averaged over the 12 months, weighted with the file's own weights, cell area on the ellipsoid
# ('area') times the ocean proportion from ESA Land Cover CCI ('mask_sfc'; the file: "Multiply area and mask_sfc to get
# true area"). The published 2.85 is this value rounded. The same for the released v2026-1 and with cos(latitude)
# separates the weighting from the version.
PHC_FILES = {'v2026-pre1': f'{rc.BASE}/inputs/UExP-FNN-U_v2026-pre1/Fordetal_UExP-FNN-U_surface-carbonate-system_v2026-pre1.nc',
             'v2026-1': str(oa_config.OBS_DIR / 'UExP-FNN-U/Fordetal_UExP-FNN-U_surface-carbonate-system_v2026-1.nc')}


def phc_mean(path, weights):
    """2024 global mean of saturation_aragonite (mean of the 12 months, cells with all 12) with the file's weights
    ('file': area x mask_sfc) or cos(latitude) ('cos')"""
    with netCDF4.Dataset(path) as f_:
        v_ = f_['saturation_aragonite']
        d_ = np.datetime64('1970-01-15') + np.asarray(f_['time'][:], 'i8').astype('timedelta64[D]')
        it_ = np.where(d_.astype('datetime64[Y]').astype(int) + 1970 == YEAR)[0]
        assert it_.size == 12 and v_.dimensions == ('lon', 'lat', 'time'), (path, it_.size, v_.dimensions)
        a_ = np.ma.filled(v_[:, :, it_[0]:it_[-1] + 1].astype('f8'), np.nan).mean(axis=2)     # NaN unless 12 months
        lat_ = np.asarray(f_['lat'][:], 'f8')
        if weights == 'file':
            w_ = (np.ma.filled(f_['area'][:].astype('f8'), np.nan) * np.ma.filled(f_['mask_sfc'][:].astype('f8'), np.nan))
            assert f_['area'].dimensions == f_['mask_sfc'].dimensions == ('lon', 'lat')
        else:
            w_ = np.cos(np.deg2rad(lat_))[None, :] * np.ones(a_.shape)      # explicit whole-cell weighting of this comparison (not oa_config.WEIGHTS)
    m_ = np.isfinite(a_) & np.isfinite(w_)
    return float(np.sum(a_[m_] * w_[m_]) / w_[m_].sum())


PHC_REPRO = {(v_, w_): phc_mean(p_, w_) for v_, p_ in PHC_FILES.items() for w_ in ('file', 'cos')}
PHC_OBS_R = PHC_REPRO[('v2026-pre1', 'file')]
assert round(PHC_OBS_R, 2) == PHC_OBS, f'Planetary Health Check not reproduced: {PHC_OBS_R:.4f}'
print('Planetary Health Check reproduced (UExP-FNN-U 2024, decline from 3.57): '
      + '; '.join(f'{v_} {"area x ocean fraction" if w_ == "file" else "cos(latitude)"} {x_:.4f} '
                  f'({dec(x_, PHC_PI):.2f} %)' for (v_, w_), x_ in PHC_REPRO.items())
      + f'; published {PHC_OBS} ({dec(PHC_OBS, PHC_PI):.2f} %); ours from the obs file {OBS_STEPS["prod"]:.4f}')
# the weights of the file (area x mask_sfc of v2026-1) on the grid of our observation file, so that the steps before the
# coverage step keep the averaging of the Planetary Health Check; the coverage step goes to the common area with the
# weights of this run (oa_config.WEIGHTS)
with netCDF4.Dataset(PHC_FILES['v2026-1']) as f_:
    _fw = (np.ma.filled(f_['area'][:].astype('f8'), np.nan) * np.ma.filled(f_['mask_sfc'][:].astype('f8'), np.nan)).T
    _flat, _flon = np.asarray(f_['lat'][:], 'f8'), np.asarray(f_['lon'][:], 'f8')
_, _, lon_s, _ = read_obs(STEP_REF)
_ls = np.where(lon_s > 180, lon_s - 360, lon_s)
W_PHC = _fw[np.ix_(np.abs(lat_s[:, None] - _flat[None, :]).argmin(1),
                   np.abs(((_ls[:, None] - _flon[None, :]) + 180) % 360 - 180).argmin(1))]       # (lat, lon) of lat_s, lon_s


def mean24_phc(fld):
    """2024 mean of fld (years, lat, lon of our observation file) over all its cells, with the weights of the file"""
    a_ = fld[i24]
    m_ = np.isfinite(a_) & np.isfinite(W_PHC)
    return float(np.sum(a_[m_] * W_PHC[m_]) / W_PHC[m_].sum())


OBS_STEPS.update(prod_f=mean24_phc(om_p), mocsy_mm_f=mean24_phc(om_mm), mocsy_f=mean24_phc(om_s))
print(f'{STEP_REF} 2024 with the weights of the file (area x ocean fraction): product Omega {OBS_STEPS["prod_f"]:.4f} '
      f'(direct from the file {PHC_REPRO[("v2026-1", "file")]:.4f}), mocsy mean of months {OBS_STEPS["mocsy_mm_f"]:.4f}, '
      f'mocsy annual-mean state {OBS_STEPS["mocsy_f"]:.4f}')
assert abs(OBS_STEPS['prod_f'] - PHC_REPRO[('v2026-1', 'file')]) < 0.002, 'weights of the file not reproduced on our grid'
# per step: label, present value, its area, chemistry, estimate, preindustrial value, its source, its CO2, product
# Jiang et al. (2023) 1750 field (NCEI 0259391, median of 14 ESMs, CO2System.jl with the same constants as mocsy here)
# evaluated on our common area: separates the area effect from the observed baseline the models are anchored to
JFILE = f'{ROOT}/jiang2023/Aragonite_median_historical.nc'
with netCDF4.Dataset(JFILE) as f:
    _jy = np.asarray(f['time'][:]).astype(int)
    _jlat = np.asarray(f['latitude'][:], 'f8')[:, 0]
    _jlon = (np.asarray(f['longitude'][:], 'f8')[0, :] + 180.0) % 360.0 - 180.0
    _J = np.ma.filled(f['aragonite'][:].astype('f8'), np.nan)
_J[~np.isfinite(_J) | (_J <= 0) | (_J > 10)] = np.nan
_o = np.argsort(_jlon)
_jlon, _J = _jlon[_o], _J[:, :, _o]
_wj = oa_config.area_weights(_jlat, _jlon)


def jiang_mask(lat_, lon_, mask_):
    """mask_ (product grid) on the grid of Jiang et al."""
    lon180, m_ = rc.to_180(np.asarray(lon_, 'f8'), np.asarray(mask_).astype(bool))
    return m_[np.ix_(np.abs(np.asarray(lat_)[None, :] - _jlat[:, None]).argmin(1),
                     np.abs(lon180[None, :] - _jlon[:, None]).argmin(1))]


def jiang_mean(field, lat_, lon_, mask_):
    m_ = jiang_mask(lat_, lon_, mask_) & np.isfinite(field)
    return np.sum(field[m_] * _wj[m_]) / _wj[m_].sum()


_rc = RCOM[STEP_REF]
PI_J_OWN = np.sum(np.where(np.isfinite(_J[0]), _J[0], 0) * _wj) / _wj[np.isfinite(_J[0])].sum()
PI_J_COM = jiang_mean(_J[list(_jy).index(1750)], _rc['lat'], _rc['lon'], _rc['mask'])
J10_COM = jiang_mean(_J[list(_jy).index(2010)], _rc['lat'], _rc['lon'], _rc['mask'])
# The 1750 side of the "same coverage" steps uses the cells of the common area where Jiang et al. have a
# value, the present-day side all cells of the common area; the share of the common area without a Jiang value and
# the effect of evaluating the present day on the same cells
_mj = jiang_mask(_rc['lat'], _rc['lon'], _rc['mask'])
_jcov = 100 * _wj[_mj & np.isfinite(_J[list(_jy).index(1750)])].sum() / _wj[_mj].sum()
_obs_j = np.asarray(_rc['mask']).astype(bool).copy()
_lon180_r, _ = rc.to_180(np.asarray(_rc['lon'], 'f8'), _obs_j)
_jfin = np.isfinite(_J[list(_jy).index(1750)])
_jy_i = np.abs(_jlat[None, :] - np.asarray(_rc['lat'], 'f8')[:, None]).argmin(1)
_jx_i = np.abs(_jlon[None, :] - np.where(np.asarray(_rc['lon'], 'f8') > 180, np.asarray(_rc['lon'], 'f8') - 360,
                                         np.asarray(_rc['lon'], 'f8'))[:, None]).argmin(1)
_obs_j &= _jfin[np.ix_(_jy_i, _jx_i)]
_com24_j = mean24(om_s, _obs_j)
_com24 = mean24(om_s, np.asarray(_rc['mask']).astype(bool))
print(f'Jiang et al. have a 1750 value in {_jcov:.1f} % of the common area; {STEP_REF} 2024 on the common area '
      f'{_com24:.4f}, on its cells with a Jiang value {_com24_j:.4f} (decline effect '
      f'{100 * (_com24 - _com24_j) / PI_J_COM:+.3f} points)')
_o10 = np.mean([np.nanmean(np.asarray(RCOM[q]['ogm'])[(np.asarray(RCOM[q]['year']) >= 2005)
                                                       & (np.asarray(RCOM[q]['year']) <= 2014)]) for q in refs])
_pi272 = np.mean([RCOM[q]['states']['jiang']['y_raw'] for q in refs])
print(f'Jiang et al. 1750: own coverage {PI_J_OWN:.3f}, common area {PI_J_COM:.3f} (ours at the same CO2 {_pi272:.3f}); '
      f'2010s on the common area: Jiang {J10_COM:.3f}, products {_o10:.3f}; decline 1750-2010s: Jiang '
      f'{dec(J10_COM, PI_J_COM):.1f} %, ours {dec(_o10, _pi272):.1f} %')
# 1750 from our 18 ESMs, but relative to the observations of Jiang et al.: their 2010s state on the common coverage
# times our modelled ratio 1750 / 2010s for STEP_REF (approximation; separates the models from the observations)
_y_ = np.asarray(RCOM[STEP_REF]['year'])
U10_COM = np.nanmean(np.asarray(RCOM[STEP_REF]['ogm'])[(_y_ >= 2005) & (_y_ <= 2014)])
# constants: this study uses mocsy with the constants of the run (oa_config: total boron OPTB, K1/K2 OPTK1K2); Jiang et
# al. (2023) use K1, K2 of Lueker et al. (2000) and total boron of Lee et al. (2010). Their states are defined by fCO2
# and TA, so their published values are converted with the ratio of their states recomputed with mocsy, DIC solved
# for their fCO2 with our constants and with theirs (jiang_chem.py; the same step as for our preindustrial states):
#   1750:  their 1850 fields with surface fCO2 lowered by the ice-core difference 1852 - 1752 (their method)
#   2010s: their 2010 fields
# The same recomputation, with their constants, gives their 1750 and 1850 states at the AR6 CO2 (R1750_AR6,
# R1850_AR6: relative to their 1750 and 1850; Fig. S_findlay c).
# Needs data/jiang2023/{fCO2,TA,DIC,Temperature,Salinity}_median_historical.nc.
import jiang_chem
importlib.reload(jiang_chem)
import pi_states
J_BORON = jiang_chem.JIANG['optb']
assert np.allclose(jiang_chem.grid()[0], _jlat) and np.allclose(jiang_chem.grid()[1], _jlon), 'Jiang grids differ'
_own = lambda fld: np.nansum(np.where(np.isfinite(fld), fld, 0) * _wj) / _wj[np.isfinite(fld)].sum()
_com = lambda fld: jiang_mean(fld, _rc['lat'], _rc['lon'], _rc['mask'])


def _ratio(a_, b_, mean_):
    """mean of a_ over mean of b_ on the cells where both are valid"""
    m_ = np.isfinite(a_) & np.isfinite(b_)
    return mean_(np.where(m_, a_, np.nan)) / mean_(np.where(m_, b_, np.nan))


_jd = jiang_chem.X_J1750 - jiang_chem.X1850_CMIP6                  # their 1750: -12.18 ppm from their 1850
_j1750 = {k_: jiang_chem.omega(1850, _jd, **c_) for k_, c_ in (('ours', jiang_chem.OURS), ('theirs', jiang_chem.JIANG))}
_j2010 = {k_: jiang_chem.omega(2010, 0.0, **c_) for k_, c_ in (('ours', jiang_chem.OURS), ('theirs', jiang_chem.JIANG))}
F_J_OWN = _ratio(_j1750['ours'], _j1750['theirs'], _own)
F_J = _ratio(_j1750['ours'], _j1750['theirs'], _com)
F_J10 = _ratio(_j2010['ours'], _j2010['theirs'], _com)
R1750_AR6 = _ratio(jiang_chem.omega(1850, pi_states.X1750 - jiang_chem.X1850_CMIP6, **jiang_chem.JIANG),
                   _j1750['theirs'], _own)
R1850_AR6 = _ratio(jiang_chem.omega(1850, pi_states.X1850 - jiang_chem.X1850_CMIP6, **jiang_chem.JIANG),
                   jiang_chem.omega(1850, 0.0, **jiang_chem.JIANG), _own)
print(f'Jiang et al. recomputed with mocsy (DIC for their fCO2): 1750 own coverage, their constants '
      f'{_own(_j1750["theirs"]):.3f}, ours {_own(_j1750["ours"]):.3f} (published {PI_J_OWN:.3f}); at the AR6 CO2 (their '
      f'constants): 1750 x {R1750_AR6:.4f}, 1850 x {R1850_AR6:.4f}')
PHC_PI_U, PI_J_COM_U = PHC_PI * F_J_OWN, PI_J_COM * F_J
# preindustrial states of STEP_REF after the constraint: at the CO2 of Jiang et al. (constrained 1750 value times the
# modelled ratio of the two CO2 states) and at 278.3 ppm (1750)
PI_JC = PI[STEP_REF]['com'] * S['jiang']['ratio']
PI_1750C = PI[STEP_REF]['com']
PI_E18_J = J10_COM * F_J10 * PI_JC / U10_COM
print(f'conversion of Jiang et al. to our constants ({jiang_chem.JIANG} -> {jiang_chem.OURS}): 1750 x {F_J:.4f}, 2010s x {F_J10:.4f}; 1750 own coverage '
      f'{PHC_PI_U:.3f}, common {PI_J_COM_U:.3f}, 18 ESMs relative to Jiang et al. {PI_E18_J:.3f}')
# Fig. 3c in nine steps: the area and the observational baseline separated. SHARED = the cells where
# both STEP_REF (2024) and Jiang et al. (1750 and 2010s) have values, on the grid of our observation file; from step 3
# on, the weights of this run. Step 4 takes the same observational baseline for both states: the simulated change of
# Jiang et al. (their ratio 1750 / 2010s, our constants) applied to the 2005-2014 state of STEP_REF on the same cells.
# Step 5 then restricts both states to the common area: the slightly larger decline of the cells outside it, the
# whole-ocean adjustment in reverse.
_jf10 = np.isfinite(_J[list(_jy).index(2010)])
_jy_s = np.abs(_jlat[None, :] - lat_s[:, None]).argmin(1)
_jx_s = np.abs(_jlon[None, :] - np.where(lon_s > 180, lon_s - 360, lon_s)[:, None]).argmin(1)
SHARED = np.isfinite(om_s[i24]) & (_jfin & _jf10)[np.ix_(_jy_s, _jx_s)]
_sh = lambda fld: jiang_mean(fld, lat_s, lon_s, SHARED)
F_J_SH = _ratio(_j1750['ours'], _j1750['theirs'], _sh)
F_J10_SH = _ratio(_j2010['ours'], _j2010['theirs'], _sh)
PI_J_SH_U = _sh(_J[list(_jy).index(1750)]) * F_J_SH
J10_SH_U = _sh(_J[list(_jy).index(2010)]) * F_J10_SH
_y10 = (yo_s >= 2005) & (yo_s <= 2014)
U10_SH = float(np.mean(gm_series(yo_s[_y10], om_s[_y10], lat_s, lon_s, SHARED)))
SH24 = mean24(om_s, SHARED)
PI_JB_SH = PI_J_SH_U / J10_SH_U * U10_SH                      # Jiang's change on the baseline of STEP_REF, shared cells
PI_JB_COM = PI_J_COM_U / (J10_COM * F_J10) * U10_COM          # the same on the common area
print(f'shared cells of {STEP_REF} and Jiang et al.: {pct(rc.to_180(lon_s, SHARED)[1][::-1] if lat_s[0] > lat_s[-1] else rc.to_180(lon_s, SHARED)[1]):.1f} % of the ocean; '
      f'2024 {SH24:.4f}; 2005-2014 {STEP_REF} {U10_SH:.4f}, Jiang et al. {J10_SH_U:.4f} '
      f'({100 * (U10_SH / J10_SH_U - 1):+.2f} %); 1750 Jiang et al. {PI_J_SH_U:.4f}, on the baseline of {STEP_REF} '
      f'{PI_JB_SH:.4f}; common area: 2005-2014 {STEP_REF} {U10_COM:.4f}, Jiang et al. {J10_COM * F_J10:.4f} '
      f'({100 * (U10_COM / (J10_COM * F_J10) - 1):+.2f} %), 1750 on the baseline of {STEP_REF} {PI_JB_COM:.4f}')
o = OBS_STEPS
J = ('Jiang 2023', f'{X_J:.0f} (1750)')             # Jiang et al. (2023)
STEPS = [('Planetary\nHealth Check\n(2026)', PHC_OBS, 'full', 'product', '2024', PHC_PI, *J, 'UExP'),
         ('same data,\nour average', o['prod'], 'full', 'product', '2024', PHC_PI, *J, 'UExP'),
         ('mocsy\nchemistry', o['mocsy'], 'full', 'mocsy', '2024', PHC_PI, *J, 'UExP'),
         ('common area\nwith ESMs', o['own'], 'with ESMs', 'mocsy', '2024', PHC_PI, *J, 'UExP'),
         ('trend fit\nfor 2024', o['own_fit'], 'with ESMs', 'mocsy', 'fit', PHC_PI, *J, 'UExP'),
         ('common area\nof all fCO$_2$-products', now_c, 'all products', 'mocsy', 'fit', PHC_PI, *J, 'UExP'),
         ('Jiang 1750 on\nthe common area', now_c, 'all products', 'mocsy', 'fit', PI_J_COM, 'Jiang, common area', J[1],
          'UExP'),
         ('same observed\nbaseline (ESMs)', now_c, 'all products', 'mocsy', 'fit', S['jiang']['y_raw'], 'ESMs', J[1],
          'UExP'),
         ('CO$_2$ of 1750\nfrom AR6', now_c, 'all products', 'mocsy', 'fit', S['1750']['y_raw'], 'ESMs', f'{X_1750:.1f} (1750)',
          'UExP'),
         ('emergent\nconstraint', now_c, 'all products', 'mocsy', 'fit', PI[STEP_REF]['com'], 'ESMs, constr.',
          f'{X_1750:.1f} (1750)', 'UExP'),
         ('this study\n(four\nfCO$_2$-products)', now_m, 'all products', 'mocsy', 'fit', yc_m, 'ESMs, constr.', f'{X_1750:.1f} (1750)',
          'mean of 4')]
STEPS_DET = STEPS                                    # every single change (Extended Data Table 3)
vals_det = [dec(s_[1], s_[5]) for s_ in STEPS_DET]
vals_det[-1] = dec_m
if not DETAILED:                                     # the three consistency steps only (detailed chain: DETAILED = True)
    STEPS = [('Planetary\nHealth Check\n(reproduced)', PHC_OBS_R, 'full', 'product', '2024', PHC_PI, *J, 'UExP'),
             ('same\nequilibrium\nconstants', o['mocsy_f'], 'full', 'mocsy', '2024', PHC_PI_U, *J, 'UExP'),
             ('same cells\nfor 1750\nand 2024', SH24, 'shared', 'mocsy', '2024', PI_J_SH_U,
              'Jiang, shared cells', J[1], 'UExP'),
             ('same\nobservational\nbaseline', SH24, 'shared', 'mocsy', '2024', PI_JB_SH,
              'Jiang change, UExP baseline', J[1], 'UExP'),
             ('common\narea', o['com24'], 'all products', 'mocsy', '2024', PI_JB_COM,
              'Jiang change, UExP baseline', J[1], 'UExP'),
             ('2024 from\nthe trend fit', now_c, 'all products', 'mocsy', 'fit', PI_JB_COM,
              'Jiang change, UExP baseline', J[1], 'UExP'),
             ('18 constrained\nESMs instead\nof 14', now_c, 'all products', 'mocsy', 'fit', PI_JC,
              'ESMs (mocsy)', J[1], 'UExP'),
             (f'CO$_2$ of 1750:\n{X_1750:.1f} ppm\n(AR6)', now_c, 'all products', 'mocsy', 'fit', PI_1750C, 'ESMs (mocsy)',
              f'{X_1750:.1f} (1750)', 'UExP'),
             ('this study,\ncommon area\n(four fCO$_2$-products)', now_m, 'all products', 'mocsy', 'fit', yc_m,
              'ESMs, constr.', f'{X_1750:.1f} (1750)', 'mean of 4')]
vals = [dec(s_[1], s_[5]) for s_ in STEPS]
vals[-1] = dec_m
# consistency of each bar: same equilibrium constants, same ocean area, 1750 consistent with the present-day estimate,
# CO2 of 1750 (1 = consistent)
# bar 2 also contains the saturation state of the annual means (as for the models, about +0.03 points; Extended Data Table 3)
CHECKS = ([(0, 0, 0, 0), (1, 0, 0, 0), (1, 1, 0, 0), (1, 1, 1, 0), (1, 1, 1, 0), (1, 1, 1, 0), (1, 1, 1, 0),
           (1, 1, 1, 1), (1, 1, 1, 1)] if not DETAILED else None)
assert CHECKS is None or len(CHECKS) == len(STEPS)
print('steps: ' + '; '.join(f'{s_[0].replace(chr(10), " ")} {v:.1f} %' for s_, v in zip(STEPS, vals))
      + f' ± {sdec_m:.1f} %')


# ---------------------------------------------------------------- when the decline reaches a given level (used by Fig. 4a
# and the numbers for the text; Figs. 1 and 4 compute the 20 % crossing in the same way and print it for comparison)
def decline_crossing(level, sig=None, _E=dict(SDEC_SHARED=SDEC_SHARED, XCO2=XCO2, refs=list(refs), RCOM=RCOM, LAB=LAB)):
    """year and CO2 at which the forced decline of the global mean surface Omega_arag (common area) reaches level (%)
    relative to the constrained 1750 value, per product and for the mean of the products.
    Forced decline: quadratic fit to the annual means up to LAST_YEAR (the main estimator; also before the first year
    of a product, then marked as extrapolated) and after LAST_YEAR the global projection anchored to the fitted 2024
    value (rc.anchor_projection; CO2 rising at projection.co2_growth, T rising with CO2 as observed, S and ALK constant).
    Mean of the products: the year at which the mean of their decline curves reaches level; range: the curve reaching
    level -+ sig (default SDEC_SHARED, the full 1 sigma of the decline shared by the products, Extended Data Table 2) combined in
    quadrature with the spread of the products' own years (ddof = 1). CO2: NOAA in the record, the projection after.
    Returns dict(t, dt, x, dx, per={q: dict(t, x, extrapolated)}, curve=(years, mean decline in %), curve_x=(CO2, the
    same), sig, year_of (ppm -> year after the record, constant growth))."""
    import numpy as np
    import projection
    import regional_common as rc
    XCO2, refs, RCOM, LAB = _E['XCO2'], _E['refs'], _E['RCOM'], _E['LAB']    # as when this cell ran (other cells reuse names)
    sig = _E['SDEC_SHARED'] if sig is None else sig
    growth = projection.co2_growth(verbose=False)[0]
    y_last = rc.EVAL_YEAR                     # anchor at 2024 (the NOAA file may go further)
    year_of = lambda x_: y_last + (x_ - XCO2[y_last]) / growth
    yrs_rec = np.array(sorted(XCO2))
    x_of = lambda t_: np.interp(t_, yrs_rec, [XCO2[y_] for y_ in yrs_rec], left=np.nan)
    prj = projection.global_projection()
    tt = np.arange(1900.0, rc.LAST_YEAR + 1e-9, 0.01)
    curves, per = [], {}
    for q in refs:
        r = RCOM[q]
        pi_ = r['constrained'][LAB][0]
        yo_, gm_ = np.asarray(r['year'], 'f8'), np.asarray(r['ogm'], 'f8')
        ok_ = np.isfinite(gm_) & (yo_ <= rc.LAST_YEAR)
        d_fit = 100 * (1 - np.polyval(np.polyfit(yo_[ok_] - 2000.0, gm_[ok_], 2), tt - 2000.0) / pi_)
        pa = rc.anchor_projection(prj[q], r, XCO2[rc.EVAL_YEAR])
        xp = np.asarray(pa['x'])[1:]
        d_prj = 100 * (1 - np.asarray(pa['scaled'])[1:] / pi_)
        t_q = np.r_[tt, year_of(xp)]
        d_q = np.r_[d_fit, d_prj]
        x_q = np.r_[x_of(tt), xp]
        i_ = int(np.argmax(d_q >= level))
        if d_q[i_] < level or i_ == 0:
            raise ValueError(f'{q}: decline of {level} % not reached between 1900 and the end of the projection')
        f_ = (level - d_q[i_ - 1]) / (d_q[i_] - d_q[i_ - 1])
        t_c = t_q[i_ - 1] + f_ * (t_q[i_] - t_q[i_ - 1])
        per[q] = dict(t=t_c, x=x_q[i_ - 1] + f_ * (x_q[i_] - x_q[i_ - 1]), extrapolated=bool(t_c < yo_[ok_][0]))
        curves.append((t_q, d_q))
    t_g = np.unique(np.r_[tt, np.arange(rc.LAST_YEAR, min(c_[0][-1] for c_ in curves), 0.01)])
    d_m = np.mean([np.interp(t_g, c_[0], c_[1]) for c_ in curves], axis=0)
    cross_ = lambda lev: float(np.interp(lev, d_m, t_g))                     # d_m increases with time
    t_m = cross_(level)
    sp_t = np.std([v_['t'] for v_ in per.values()], ddof=1)
    dt = float(np.hypot(sp_t, (cross_(level + sig) - cross_(level - sig)) / 2))
    x_m = float(np.mean([v_['x'] for v_ in per.values()]))
    x_t = lambda t_: float(x_of(t_)) if t_ <= rc.LAST_YEAR else XCO2[y_last] + (t_ - y_last) * growth
    dx = float(np.hypot(np.std([v_['x'] for v_ in per.values()], ddof=1),
                        (x_t(cross_(level + sig)) - x_t(cross_(level - sig))) / 2))
    x_g = np.array([x_t(t_) for t_ in t_g])                                  # CO2 of each year of the mean curve
    return dict(t=t_m, dt=dt, x=x_m, dx=dx, per=per, curve=(t_g, d_m), curve_x=(x_g, d_m), sig=sig, year_of=year_of)

# ---------------------------------------------------------------- the global mean: the two adjustments beyond the common area
# (structural_adjustments.py; used by Fig. 4a, Extended Data Tables 1 and 2 and the numbers of the abstract, so that every cell reads
# the same values). (1) Whole ocean instead of the common area: the mean of three estimates of the shift of the decline,
# the products, the 18 ESMs and Jiang et al., each with the unconstrained 1750 state (the multi-model mean of the
# bias-adjusted ESMs) and each filled to the same whole ocean; 1 sigma three times the spread across
# the 18 ESMs (test_polar_shift_models.py, smoothed states of the data product, 1750 from the 1850 state at 278.3 ppm,
# filled like the products) of the shift of the decline and, for the two states, of the whole-ocean mean relative to
# the common-area mean. The states themselves (abs_common, abs_ocean) are the constrained states of the paper. (2) The surface
# alkalinity rise of the calcification feedback (Barrett et al. 2025) that the products and 15 of the 18 ESMs miss:
# chemistry at constant pCO2 cell by cell around the states of each product, distributed in space with the pattern of
# the feedback in four ESMs (GFDL-CM4, GFDL-ESM4, UKESM1-0-LL, GFDL-ESM2M-COBALTv2; feedback_pattern.py), the observed
# rate converted to the common area by its sampling (barrett_sampling.py); 1 sigma = 100 % of the shift; applied after
# the extension, on the whole ocean. Shifts added, uncertainties in quadrature.
import oa_config
_SA = np.load(oa_config.DATA / 'structural' / 'structural_adjustments.npz')
assert list(_SA['products']) == list(refs), f'structural_adjustments.npz is for {list(_SA["products"])}, not {list(refs)}'
POL_SHIFT, POL_SIG = float(_SA['polar_shift']), float(_SA['polar_sigma'])
ALK_SHIFT_C = float(_SA['alk_cell_common'])                           # on the common area (for the text)
ALK_SHIFT = float(_SA['alk_cell_ocean'])                             # on the whole ocean (cell by cell, structural_adjustments.py)
ALK_SIG = abs(ALK_SHIFT)
_ac, _ao = _SA['abs_common'], _SA['abs_ocean']                       # 1750, 2024, boundary; with alkalinity: 3, 4, 5
POL_D50, POL_D24 = float(_ao[0] - _ac[0]), float(_ao[1] - _ac[1])     # whole ocean minus common area, Omega_arag
ALK_D50, ALK_D24 = float(_ao[3] - _ao[0]), float(_ao[4] - _ao[1])     # the alkalinity rise, Omega_arag, whole ocean
POL_S50, POL_S24, POL_SDIST = (float(v_) for v_ in _SA['polar_level_sigma'])   # 1 sigma of the whole-ocean levels
DEC_G = dec_m + POL_SHIFT + ALK_SHIFT
SDEC_G = float(np.sqrt(sdec_m ** 2 + ALK_SIG ** 2 + POL_SIG ** 2))
SDEC_SHARED_G = float(np.sqrt(SDEC_SHARED ** 2 + ALK_SIG ** 2 + POL_SIG ** 2))
OM50_G, OM24_G = yc_m + ALK_D50 + POL_D50, now_m + ALK_D24 + POL_D24
# the crossing of the global mean: D_g(t) = D_c(t) (1 + r) + a, r the whole-ocean shift relative to the decline (the shift
# scales with the decline), a the alkalinity shift on the whole ocean (held at its 2024 value; its growth after 2024,
# about -0.005 points per year, moves the crossing by about 0.1 year): D_g = 20 % where D_c = (20 - a) / (1 + r), with
# the shared 1 sigma including both adjustments (decline_crossing above, as Fig. 4a)
_r_g = POL_SHIFT / dec_m
_lev_g = (20.0 - ALK_SHIFT) / (1 + _r_g)
_dg = decline_crossing(_lev_g, sig=SDEC_SHARED_G / (1 + _r_g))
_fr_c, _fr_o = _SA['frac20_products_common'], _SA['frac20_products_union']
GLOBAL = dict(decline=DEC_G, sigma=SDEC_G, sigma_shared=SDEC_SHARED_G, distance=20 - DEC_G, n_sigma=(20 - DEC_G) / SDEC_G,
              findlay_ratio=5.0 / SDEC_G, omega_1750=OM50_G, omega_2024=OM24_G, boundary=0.8 * OM50_G, r=_r_g, a=ALK_SHIFT,
              cross_year=_dg['t'], cross_dyear=_dg['dt'], cross_ppm=_dg['x'], cross_dppm=_dg['dx'], cross_level_common=_lev_g,
              frac20_common=float(_fr_c[1]), frac20_all=float(_fr_o[1]), frac20_all_area=float(_fr_o[2]),
              frac20_common_noalk=float(_fr_c[0]), frac20_all_noalk=float(_fr_o[0]))
GLOBAL_TEXT = (f'global mean decline 1750-2024: {DEC_G:.2f} ± {SDEC_G:.2f} % ({20 - DEC_G:.2f} points, {(20 - DEC_G) / SDEC_G:.1f} sigma, '
               f'below the boundary; Findlay et al. ±5.0 / {SDEC_G:.2f} = {5.0 / SDEC_G:.1f}); common area {dec_m:.2f} ± {sdec_m:.2f}; '
               f'whole ocean {POL_SHIFT:+.2f} ± {POL_SIG:.2f}; alkalinity on the whole ocean {ALK_SHIFT:+.2f} ± {ALK_SIG:.2f}, on the '
               f'common area {ALK_SHIFT_C:+.2f}\n'
               f'global mean Omega_arag: 1750 {OM50_G:.3f} (whole ocean {POL_D50:+.3f}, alkalinity {ALK_D50:+.3f}), 2024 {OM24_G:.3f} '
               f'({POL_D24:+.3f}, {ALK_D24:+.3f}), boundary {0.8 * OM50_G:.3f}; 1 sigma of the whole-ocean levels {POL_S50:.3f} (1750), '
               f'{POL_S24:.3f} (2024), {POL_SDIST:.3f} (distance)\n'
               f'global crossing of 20 %: {_dg["t"]:.1f} ± {_dg["dt"]:.1f} ({_dg["t"] - _dg["dt"]:.0f}-{_dg["t"] + _dg["dt"]:.0f}), at '
               f'{_dg["x"]:.1f} ± {_dg["dx"]:.1f} ppm (common-area decline reaching {_lev_g:.2f} %)\n'
               f'beyond a 20 % decline in 2024 (mean of the products, with the alkalinity adjustment): common area {_fr_c[1]:.1f} % '
               f'(without {_fr_c[0]:.1f} %); every cell any product covers ({_fr_o[2]:.1f} % of the ocean) {_fr_o[1]:.1f} % (without '
               f'{_fr_o[0]:.1f} %)')
print(GLOBAL_TEXT)


# the variables of this cell, restored by the Fig. 3 cell
_F4_NAMES = ['ABS', 'AREA_C', 'B', 'BCOL', 'BORON', 'B_sig', 'CHECKS', 'CONST_REL', 'DETAILED', 'ENT_CUTOFFS', 'ENT_MAIN',
              'EST', 'F_J', 'F_J10', 'F_J_OWN', 'R1750_AR6', 'R1850_AR6', 'GridSpec', 'GridSpecFromSubplotSpec', 'J', 'PHC_FILES', 'PHC_OBS_R', 'PHC_REPRO', 'W_PHC', 'mean24_phc', 'J10_COM', 'JFILE', 'J_BORON',
              'K_B', 'K_CO2', 'K_CONST', 'K_ESM', 'K_FIT', 'K_K12', 'K_MAP', 'K_METH', 'K_PROD_D', 'LAB', 'Line2D', 'MAP_CUC',
              'MAP_SIG', 'OBSF', 'OBS_STEPS', 'OMEGA', 'PHC_B', 'PHC_OBS', 'PHC_PI', 'PHC_PI_U', 'PI', 'PI_1750C', 'PI_E18_J',
              'PI_JC', 'PI_J_COM', 'PI_J_COM_U', 'PI_J_OWN', 'PK', 'RCOM', 'ROOT', 'ROWN', 'S', 'SDEC_SHARED', 'SIG', 'STEPS',
              'STEPS_DET', 'STEP_REF', 'TS', 'U10_COM', 'V', 'XCO2', 'X_1750', 'X_J', 'YEAR', '_J', '_c', '_cd', '_co2_sig',
              '_com', '_ddec', '_f', '_fc', '_fc_', '_fm', '_here', '_j1750', '_j2010', '_jlat', '_jlon', '_jy', '_m10',
              '_mm', '_mp', '_mt', '_o', '_o10', '_other', '_own', '_pi272', '_rc', '_rel17', '_rel24', '_run_means',
              '_series', '_smoother', '_trend', '_v24', '_v50', '_w_poly', '_wj', '_y_', '_z', 'a_', 'b_', 'c', 'com',
              'com180', 'common', 'cov', 'dec', 'dec_m', 'dec_q', 'decline_crossing', 'e', 'f', 'f_', 'fc', 'fit_now', 'full', 'full180', 'g',
              'gm_series', 'i24', 'importlib', 'jiang_mean', 'k', 'k_', 'keep', 'lab_', 'lat', 'lat_s', 'lon', 'lon180', 'm_',
              'map_terms', 'mean24', 'n_prod', 'name', 'needs', 'netCDF4', 'now_c', 'now_m', 'np', 'o', 'o_', 'ocean', 'ok_',
              'om', 'om_p', 'om_s', 'os', 'out', 'own', 'own_s', 'p_', 'path_', 'pct', 'pickle', 'plt', 'projection', 'q',
              'rc', 'read_obs', 'refs', 'rms', 's_', 'sc_', 'sdec_m', 'so_', 'sys', 't_', 'ta_', 'v', 'v_', 'vals',
              'vals_det', 'w', 'x', 'yc_', 'yc_m', 'yo', 'yo_', 'yo_s',
              '_SA', 'POL_SHIFT', 'POL_SIG', 'ALK_SHIFT', 'ALK_SHIFT_C', 'ALK_SIG', 'POL_D50', 'POL_D24', 'ALK_D50',
              'ALK_D24', 'POL_S50', 'POL_S24', 'POL_SDIST', 'DEC_G', 'SDEC_G', 'SDEC_SHARED_G', 'OM50_G', 'OM24_G',
              'GLOBAL', 'GLOBAL_TEXT']
FIG4 = {k_: globals()[k_] for k_ in _F4_NAMES if k_ in globals()}

## Fig. 1: global mean surface Ωarag from 1750 to 2024

In [ ]:
# Notebook cell: Figure 1, mean surface Omega_arag from 1750 to 2024 on the common area of the four products (the analysis
# before the two adjustments; the best estimate over the whole ocean and when the boundary is crossed are in Fig. 4).
# The global analysis is the function _global, which returns its variables; panel() of it draws the figure.
import oa_config                      # the area weights of the run (OA_WEIGHTS)
import sys
import importlib
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.gridspec import GridSpec, GridSpecFromSubplotSpec

sys.path.insert(0, CODE_DIR)
import regional_common as rc
importlib.reload(rc)


def _global():
    # Global mean surface Omega_arag on the common area, from 1750 to the boundary: the data of every product and of
    # their mean, and panel(), which draws one of them (the mean: Fig. 1; one panel per product: Supplementary Fig. 3).
    #   a (same aspect ratio as b-e, A_HEIGHT times as large, centred): mean of the four products
    #       left column (1750): ESMs (grey dots; bias-adjusted, averaged over the four adjustments), multi-model mean ± 1 sd
    #                           (black), constrained (red, mean of the products ± 1σ)
    #       main axis:          ESMs 1850-2014 (thin grey), multi-model mean (black), mean of the observation-based products
    #                           (red, years covered by all four), projection to 2040 from the fitted 2024 value (CO2 at the
    #                           CO2 growth for constant emissions (projection.co2_growth), T rising with CO2 as observed, S and ALK constant), crossing year
    #       right column:       the same at 80 % (boundary)
    #   b-e: one product each: models adjusted to this product, the product, its
    #        bias-adjustment period (light grey), raw and constrained boundary, 1750 and boundary columns
    # Uncertainties: model spread (raw) and constraint (constrained) only. The CO2 of 1750 (AR6 278.3 ± 2.9 ppm) is
    # printed, not drawn (it moves all values together).
    # Needs data/constraint_results.pkl (with 'states'), the bias-adjusted historical files, data/fig_crossing_projection.pkl
    # (projection.py) and data/co2_annmean_gl.txt.
    import glob
    import os
    import sys
    import pickle
    import importlib
    import numpy as np
    import netCDF4
    from matplotlib.gridspec import GridSpecFromSubplotSpec
    from matplotlib.patches import Patch
    from matplotlib.lines import Line2D
    from matplotlib.legend_handler import HandlerTuple

    sys.path.insert(0, CODE_DIR)
    import regional_common as rc
    importlib.reload(rc)

    ROOT = rc.ROOT
    LAB = rc.LAB
    OMEGA = r'$\Omega_{\mathrm{arag}}$'
    ADJ = {'OceanSODA': f'{ROOT}/omega_arag_biasadj', 'UExP': f'{ROOT}/omega_arag_biasadj_UExP',
           'JMA': f'{ROOT}/omega_arag_biasadj_JMA', 'CMEMS': f'{ROOT}/omega_arag_biasadj_CMEMS'}
    WIN = {q: oa_config.WINDOWS[q] for q in ('OceanSODA', 'UExP', 'JMA', 'CMEMS')}   # the bias windows of the chain
    YEAR_END = 2040                      # end of the projection in a
    MEAN_COL = 'C3'                      # mean of the products (red; no product uses red)

    with open(f'{ROOT}/constraint_results.pkl', 'rb') as f:
        RC = pickle.load(f)['results_common']
    refs = list(RC)

    # ---------------------------------------------------------------- NOAA CO2, years <-> ppm at a constant growth rate
    co2 = np.array([[float(v) for v in l.split()[:2]] for l in open(f'{ROOT}/co2_annmean_gl.txt')
                    if l.strip() and not l.startswith('#')])
    cy, cx = co2[:, 0].astype(int), co2[:, 1]
    import projection as _pj
    growth = _pj.co2_growth()[0]             # constant emissions at the 2024 level: 2.6 ppm/yr
    y_last = rc.EVAL_YEAR                     # anchor at 2024 (the NOAA file may go further)
    x_last = float(cx[cy == y_last][0])
    year_of = lambda x: y_last + (x - x_last) / growth
    x_of = lambda y: x_last + (y - y_last) * growth

    # ---------------------------------------------------------------- per product: bias-adjusted ESMs on the common mask
    P = {}
    for q, rr in RC.items():
        cmask = np.asarray(rr['mask']).astype(bool)
        area = oa_config.area_weights(rr['lat'], rr['lon'])
        A, nm = [], []
        for fh in sorted(glob.glob(f'{ADJ[q]}/historical/omega_arag_biasadj_Oyr_*_historical_*.nc')):
            model = os.path.basename(fh).split('_')[4]
            if model not in rr['names']:
                continue
            with netCDF4.Dataset(fh) as f:
                yrs = np.asarray(f['year'][:])
                h = np.ma.filled(f['omega_arag_adj'][:].astype('f8'), np.nan)
            m = cmask & np.all(np.isfinite(h), axis=0)
            w = area * m
            A.append(np.nansum(np.where(m, h, 0) * w, axis=(1, 2)) / w.sum())
            nm.append(model)
        pi = np.array([np.asarray(rr['y1750'])[list(rr['names']).index(m_)] for m_ in nm])
        yo = np.asarray(rr['year'])
        keep = yo <= rc.LAST_YEAR
        yc, sc = rr['constrained'][LAB][:2]
        P[q] = dict(A=np.array(A), names=nm, yrs=yrs, pi=pi, yc=yc, sc=sc, yo=yo[keep],
                    ob=np.asarray(rr['ogm'])[keep], col=rc.COLORS[q], title=rc.FULL[q], win=WIN[q])

    # ---------------------------------------------------------------- mean of the products (panel a)
    names = [m for m in P[refs[0]]['names'] if all(m in P[q]['names'] for q in refs)]
    A_m = np.mean([[P[q]['A'][P[q]['names'].index(m)] for m in names] for q in refs], axis=0)       # (model, year)
    pi_m = np.mean([[P[q]['pi'][P[q]['names'].index(m)] for m in names] for q in refs], axis=0)
    yo_m = np.array(sorted(set.intersection(*[set(P[q]['yo'].tolist()) for q in refs])))          # years of all four
    ob_m = np.mean([[P[q]['ob'][list(P[q]['yo']).index(y)] for y in yo_m] for q in refs], axis=0)
    yc_m = np.mean([P[q]['yc'] for q in refs])
    sc_m = np.sqrt(np.mean([P[q]['sc'] ** 2 for q in refs]))
    P_mean = dict(A=A_m, yrs=P[refs[0]]['yrs'], pi=pi_m, yc=yc_m, sc=sc_m, yo=yo_m, ob=ob_m, col=MEAN_COL,
                  title='Mean of the four fCO$_2$-products',
                  win=(max(w[0] for w in WIN.values()), min(w[1] for w in WIN.values())))       # 1990-2014, common to all four

    # projection from the fitted 2024 value of each product, mean over the products, crossing year
    import projection
    importlib.reload(projection)
    proj = projection.global_projection()        # cached in the run folder (projection.py)
    X_NOW = dict(zip(cy, cx))[rc.EVAL_YEAR]
    proj = {q: rc.anchor_projection(proj[q], RC[q], X_NOW) for q in refs}
    xg = np.arange(X_NOW, x_of(YEAR_END) + 1, 0.5)
    yg = year_of(xg)
    curve = np.mean([np.interp(xg, proj[q]['x'], proj[q]['scaled']) for q in refs], axis=0)


    def cross(c, level):
        i = np.where(c <= level)[0][0]
        return yg[i - 1] + (c[i - 1] - level) / (c[i - 1] - c[i]) * (yg[i] - yg[i - 1])


    B, B_sig = 0.8 * yc_m, 0.8 * sc_m
    y_prod = {q: cross(np.interp(xg, proj[q]['x'], proj[q]['scaled']), 0.8 * P[q]['yc']) for q in refs}
    y_cross = np.mean(list(y_prod.values()))
    # crossing range: the full 1 sigma of the decline shared by the products (Extended Data Table 2 without the product spread,
    # SDEC_SHARED of the preparation cell), as an Omega_arag offset of the boundary, and the spread of the products
    if 'SDEC_SHARED' not in globals():
        raise NameError('SDEC_SHARED not defined: run the preparation cell first (it comes before this cell)')
    s_om = SDEC_SHARED / 100 * yc_m
    dy = np.hypot(np.std(list(y_prod.values()), ddof=1), (cross(curve, B - s_om) - cross(curve, B + s_om)) / 2)
    S = RC[refs[0]]['states']
    s_co2 = 0.5 * yc_m * abs(S['1750_hi']['ratio'] - S['1750_lo']['ratio'])
    dy_co2 = (cross(curve, 0.8 * (yc_m - s_co2)) - cross(curve, 0.8 * (yc_m + s_co2))) / 2
    for lab_, p in list(P.items()) + [('mean', P_mean)]:
        print(f'{lab_:10s} 1750 models {p["pi"].mean():.3f} ± {p["pi"].std(ddof=1):.3f}, constrained {p["yc"]:.3f} ± '
              f'{p["sc"]:.3f}; boundary {0.8 * p["yc"]:.3f} ± {0.8 * p["sc"]:.3f}')
    print(f'crossing {y_cross:.1f} ± {dy:.1f} (' + ', '.join(f'{q} {v:.1f}' for q, v in y_prod.items())
          + f'); 1 sigma: decline {SDEC_SHARED:.2f} percentage points shared (Extended Data Table 2, of which the CO2 of 1750 '
          f'± {dy_co2:.1f} years) and the spread of the products')

    # common y-limits
    lo = min(min(p['A'].min(), p['ob'].min(), 0.8 * p['pi'].min()) for p in P.values()) - 0.03
    lo = min(lo, B - B_sig - 0.02)
    hi = max(max(p['A'].max(), p['ob'].max(), p['pi'].max()) for p in P.values()) + 0.03


    # ---------------------------------------------------------------- one panel (1750 column | time series | boundary)
    def panel(spec, p, letter, big, fig_=None):
        fg = fig if fig_ is None else fig_                # figure to draw into (Fig. 4 passes its own)
        fs = 10 if big else 8                               # font size of labels inside the panel
        ms = 4.5 if big else 3
        sub = GridSpecFromSubplotSpec(1, 3, subplot_spec=spec, width_ratios=[0.13, 1, 0.17], wspace=0.08)
        ax0 = fg.add_subplot(sub[0])
        ax = fg.add_subplot(sub[1], sharey=ax0)
        ax2 = fg.add_subplot(sub[2], sharey=ax0)
        jit = np.random.default_rng(1).uniform(-0.18, 0.18, p['pi'].size)
        bc, bs = 0.8 * p['yc'], 0.8 * p['sc']

        def column(a_, v, cval, sval, below):
            a_.scatter(-0.25 + jit, v, s=12 if big else 6, color='0.55', lw=0, zorder=2)
            a_.errorbar(0.25, v.mean(), yerr=v.std(ddof=1), fmt='o', color='k', ms=ms, capsize=3, lw=1.1, zorder=3)
            a_.errorbar(0.65, cval, yerr=sval, fmt='o', color=p['col'], ms=ms, capsize=3, lw=1.3, zorder=4)
            if below:
                y_, va = min(v.min(), v.mean() - v.std(ddof=1), cval - sval) - 0.02, 'top'
            else:
                y_, va = max(v.max(), v.mean() + v.std(ddof=1), cval + sval) + 0.02, 'bottom'
            for x_, t_, c_ in ((-0.25, 'ESMs', '0.45'), (0.25, 'multi-model mean', 'k'), (0.65, 'constrained', p['col'])):
                a_.text(x_, y_, t_, rotation=90, ha='center', va=va, fontsize=fs - 0.5, color=c_)
            a_.set_xlim(-0.6, 0.9)
            a_.set_xticks([0.15])
            a_.spines[['top', 'right']].set_visible(False)
            a_.tick_params(labelsize=fs)

        column(ax0, p['pi'], p['yc'], p['sc'], True)
        ax0.set_xticklabels(['1750'])
        ax0.text(-0.9 if big else -1.0, 1.02, letter, fontsize=16 if big else 13, fontweight='bold', va='bottom', transform=ax0.transAxes)
        column(ax2, 0.8 * p['pi'], bc, bs, False)
        ax2.set_xticklabels(['boundary'])
        ax2.tick_params(labelleft=False)

        A, yrs = p['A'], p['yrs']
        if p['win'] is not None:
            ax.axvspan(*p['win'], color='0.92', lw=0, zorder=0)
        for a in A:
            ax.plot(yrs, a, color='0.78', lw=0.6, zorder=1)
        ax.plot(yrs, A.mean(0), color='k', lw=1.5 if big else 1.2, zorder=2)
        ax.plot(p['yo'], p['ob'], color=p['col'], lw=2.0 if big else 1.4, zorder=4)
        bu, bus = 0.8 * p['pi'].mean(), 0.8 * p['pi'].std(ddof=1)
        for y_ in (bu - bus, bu + bus):
            ax.axhline(y_, color='0.45', ls=(0, (3, 3)), lw=0.5, zorder=2)
        ax.axhline(bu, color='0.3', ls=(0, (4, 4)), lw=1.0, zorder=2)
        ax.axhspan(bc - bs, bc + bs, color=p['col'], alpha=0.3, lw=0, zorder=1)
        ax.axhline(bc, color=p['col'], ls=(4, (4, 4)), lw=1.0, zorder=3)
        ax.set_ylim(lo, hi)
        ax.spines[['top', 'right']].set_visible(False)
        ax.tick_params(labelleft=False, labelsize=fs)
        ax.set_title(p['title'], fontsize=13 if big else 10, fontweight='bold')

        if big:
            ax.set_xlim(yrs[0], rc.LAST_YEAR + 1)
            ax0.set_ylabel(f'Mean surface {OMEGA}, common area', fontsize=12)
            hh = [Line2D([], [], color='0.7', lw=0.8), Line2D([], [], color='k', lw=1.5),
                  Line2D([], [], color=p['col'], lw=2.0),
                  Patch(facecolor='0.92', lw=0), Line2D([], [], color='0.3', ls=(0, (4, 4)), lw=1.0),
                  (Patch(facecolor=p['col'], alpha=0.3, lw=0), Line2D([], [], color=p['col'], ls=(4, (4, 4)), lw=1.0))]
            ll = ['ESMs (bias-adjusted)', 'multi-model mean',
                  'fCO$_2$-products (mean of four)', 'bias-adjustment period',
                  f'boundary, unconstrained ({bu:.2f} $\\pm$ {bus:.2f})', f'boundary, constrained ({bc:.2f} $\\pm$ {bs:.2f})']
            ax.legend(hh, ll, fontsize=fs - 1, frameon=False, loc='lower left', bbox_to_anchor=(0.0, 0.14),
                      handler_map={tuple: HandlerTuple(ndivide=1, pad=0)})
        else:
            ax.set_xlim(yrs[0], rc.LAST_YEAR)
            # boundary values as a small legend just above the boundary lines, with the line styles of a
            hh = [Line2D([], [], color='0.3', ls=(0, (4, 4)), lw=1.0),
                  (Patch(facecolor=p['col'], alpha=0.3, lw=0), Line2D([], [], color=p['col'], ls=(4, (4, 4)), lw=1.0))]
            y_txt = (max(bc + bs, bu + bus) + 0.01 - lo) / (hi - lo)              # axes fraction
            leg = ax.legend(hh, [f'Raw: {bu:.2f} $\\pm$ {bus:.2f}', f'Constrained: {bc:.2f} $\\pm$ {bs:.2f}'],
                            title='Boundary', title_fontproperties={'weight': 'bold', 'size': fs}, fontsize=fs,
                            frameon=False, loc='lower left', bbox_to_anchor=(0.0, y_txt), handlelength=2.2,
                            borderaxespad=0.2, handler_map={tuple: HandlerTuple(ndivide=1, pad=0)})
            leg._legend_box.align = 'left'
        return ax0, ax


    return locals()



G = _global()                        # global: P_mean, panel(), projection, crossing year

# ---------------------------------------------------------------- Fig. 1: the common area
plt.close('all')
fig = plt.figure(figsize=(11, 5.6))
gs = GridSpec(1, 1, figure=fig, left=0.08, right=0.98, top=0.95, bottom=0.09)
G['P_mean']['title'] = ''
ax0, ax = G['panel'](gs[0], G['P_mean'], '', True, fig_=fig)
rc.FIGNAMES.setdefault('best', 'Fig1_best_estimate')
rc.savefig(fig, 'best')
plt.show()

### Diagnostics after Fig. 1 (use its variables G and rc; printouts only)

Robustness tests (trend check, leave-one-out and perfect-model tests, model families, sensitivity to the observed ΔSST, uncertainty components, distance to the boundary) and the refined trend check of JMA-MLR.

In [ ]:
# Notebook cell: robustness tests (run after the Fig. 1 cell, same kernel: uses G and rc).
#   1  trend check: do the bias-adjusted ESMs and the products agree on the decline over the product years?
#      (bound on a trend error of the products, which would enter the decline directly)
#   2  leave-one-out and perfect-model tests of the emergent constraint (18 models, 2 predictors)
#   3  model families: one member per family, all combinations
#   4  sensitivity of the constrained 1750 value to the observed dSST and its uncertainty
#   5  constrained and unconstrained 1750 uncertainty with the same components (model spread or constraint, + CO2 of 1750)
#   6  distance to the boundary in percentage points of the decline
# Prints only; nothing is written.
import pickle
import itertools
import numpy as np

assert 'G' in globals(), 'run the Fig. 1 cell first (defines G)'
LAB = rc.LAB
with open(f'{rc.ROOT}/constraint_results.pkl', 'rb') as f:
    RC = pickle.load(f)['results_common']
refs = list(RC)
P = G['P']


def ols(x, y):
    """slope and its standard error; the residuals of annual means are autocorrelated (e.g. ENSO), so the error uses
    the effective number of years n_eff = n (1 - r1) / (1 + r1), r1 the lag-1 autocorrelation of the residuals
    (Santer et al. 2000, J. Geophys. Res. 105, 7337)"""
    x, y = np.asarray(x, 'f8'), np.asarray(y, 'f8')
    b, a = np.polyfit(x, y, 1)
    res = y - (a + b * x)
    r1 = float(np.corrcoef(res[:-1], res[1:])[0, 1])
    neff = x.size * (1 - r1) / (1 + r1) if r1 > 0 else float(x.size)
    se = np.sqrt(res @ res / max(neff - 2, 1.0) / np.sum((x - x.mean()) ** 2))
    return b, se


def constrain(dT, CUC, y, dT_obs, dT_sig, CUC_obs, CUC_sig):
    """constrained value as constraint_refs.fit: prediction sigma at the observed predictors + propagated obs error"""
    X = np.column_stack([np.ones(y.size), dT, CUC])
    x0 = np.array([1.0, dT_obs, CUC_obs])
    beta, *_ = np.linalg.lstsq(X, y, rcond=None)
    res = y - X @ beta
    s2 = res @ res / (X.shape[0] - X.shape[1])
    cov = s2 * np.linalg.inv(X.T @ X)
    sfit = np.sqrt(s2 + x0 @ cov @ x0)
    sobs = np.hypot(beta[1] * dT_sig, beta[2] * CUC_sig)
    return x0 @ beta, np.hypot(sfit, sobs), sfit, sobs, beta


# ------------------------------------------------------------------ 1  trend check: ESMs against the products
# Over the years the product and the historical simulations share (product window, to 2014), the change of the global
# mean is set by the CO2 history and the chemistry, so the bias-adjusted ESMs are an independent estimate of the
# product's trend. A trend error of a product between its reference window (mean year ~1998) and 2024 enters the
# decline directly: d(decline) = d(Omega_2024) / Omega_1750.
print('1  trend of the global mean on the common area, product vs bias-adjusted ESMs (per decade)')
print(f'   {"product":10s} {"years":9s} {"product":>14s} {"ESM mean":>14s} {"ESM sd":>7s} {"diff":>8s} '
      f'{"-> 2024":>8s} {"pp":>6s}')
trend_term = {}
for q in refs:
    p = P[q]
    y0, y1 = p['win'][0], 2014
    yo, ob = np.asarray(p['yo']), np.asarray(p['ob'])
    so = (yo >= y0) & (yo <= y1) & np.isfinite(ob)
    sm = (np.asarray(p['yrs']) >= y0) & (np.asarray(p['yrs']) <= y1)
    bo, seo = ols(yo[so], ob[so])
    bm = np.array([ols(np.asarray(p['yrs'])[sm], a[sm])[0] for a in p['A']])
    diff = bo - bm.mean()                                       # product minus models, per year
    t_mid = 0.5 * (y0 + y1)                                     # mean year of the reference window
    d2024 = diff * (rc.EVAL_YEAR - t_mid)                       # implied error of the 2024 state relative to the window
    pp = 100 * abs(d2024) / p['yc']
    trend_term[q] = (abs(d2024), pp)
    print(f'   {q:10s} {y0}-{y1} {10 * bo:>8.4f}±{10 * seo:.4f} {10 * bm.mean():>8.4f}±{10 * bm.std(ddof=1):.4f} '
          f'{"":7s} {10 * diff:>+8.4f} {d2024:>+8.4f} {pp:>6.2f}')
d_rms = np.sqrt(np.mean([v[0] ** 2 for v in trend_term.values()]))
pp_rms = np.sqrt(np.mean([v[1] ** 2 for v in trend_term.values()]))
pp_max = max(v[1] for v in trend_term.values())
print(f'   -> implied 2024 error (rms over products) {d_rms:.4f} in Omega = {pp_rms:.2f} points of the decline '
      f'(largest product {pp_max:.2f})')
# the products against each other: change from each product's window mean to its fitted 2024 value
print('   change from the window mean to the fitted 2024 value, per product:')
chg = {}
for q in refs:
    p = P[q]
    yo, ob = np.asarray(p['yo']), np.asarray(p['ob'])
    sw = (yo >= p['win'][0]) & (yo <= p['win'][1]) & np.isfinite(ob)
    chg[q] = rc.fit_now(RC[q]) - ob[sw].mean()
    print(f'   {q:10s} {chg[q]:+.4f}  (window {p["win"][0]}-{p["win"][1]}, mean year {np.mean(yo[sw]):.1f})')
print(f'   spread between products {np.std(list(chg.values()), ddof=1):.4f} (part of the "products" term of Extended Data Table 2)')

# ------------------------------------------------------------------ 2  leave-one-out and perfect-model tests
print('\n2  cross-validation of the constraint (regression on 17 models, tested on the 18th)')
for q in refs:
    r = RC[q]
    dT, CUC, y = (np.asarray(r[k], 'f8') for k in ('dT', 'CUC', 'y1750'))
    dT_obs, dT_sig = r['dT_obs']
    CUC_obs, CUC_sig = r['CUC_obs'], rc.CUC_SIG
    n = y.size
    pred, sig, inside = np.empty(n), np.empty(n), 0
    for i in range(n):
        k = np.arange(n) != i
        # perfect model: model i plays the observations (no observational error), the other 17 constrain
        yc_i, _, sfit_i, _, _ = constrain(dT[k], CUC[k], y[k], dT[i], 0.0, CUC[i], 0.0)
        pred[i], sig[i] = yc_i, sfit_i
        inside += abs(y[i] - yc_i) <= sfit_i
    err = y - pred
    rmse = np.sqrt(np.mean(err ** 2))
    yc, stot, sfit, sobs, beta = constrain(dT, CUC, y, dT_obs, dT_sig, CUC_obs, CUC_sig)
    print(f'   {q:10s} leave-one-out rmse {rmse:.4f} vs model sd {y.std(ddof=1):.4f} (skill {1 - rmse / y.std(ddof=1):.0%}); '
          f'perfect-model: {inside}/{n} within ±1σ ({inside / n:.0%}), mean predicted σ {sig.mean():.4f}; '
          f'full: {yc:.3f} ± {stot:.3f} (fit {sfit:.3f}, obs {sobs:.3f})')

# ------------------------------------------------------------------ 3  model families
FAMILIES = [('CESM2', 'CESM2-WACCM'), ('CanESM5', 'CanESM5-CanOE'), ('MPI-ESM1-2-HR', 'MPI-ESM1-2-LR'),
            ('NorESM2-LM', 'NorESM2-MM'), ('GFDL-CM4', 'GFDL-ESM4')]
print('\n3  one member per model family (all combinations; the three PISCES models share only the biogeochemistry and are kept)')
for q in refs:
    r = RC[q]
    names = list(r['names'])
    dT, CUC, y = (np.asarray(r[k], 'f8') for k in ('dT', 'CUC', 'y1750'))
    dT_obs, dT_sig = r['dT_obs']
    CUC_obs, CUC_sig = r['CUC_obs'], rc.CUC_SIG
    fams = [[m for m in fam if m in names] for fam in FAMILIES]
    fams = [f for f in fams if len(f) > 1]
    others = [m for m in names if not any(m in f for f in fams)]
    vals = []
    for pick in itertools.product(*fams):
        keep = np.array([m in others or m in pick for m in names])
        yc_, s_, *_ = constrain(dT[keep], CUC[keep], y[keep], dT_obs, dT_sig, CUC_obs, CUC_sig)
        vals.append((yc_, s_, keep.sum()))
    v = np.array(vals)
    yc, stot, *_ = constrain(dT, CUC, y, dT_obs, dT_sig, CUC_obs, CUC_sig)
    print(f'   {q:10s} {len(vals)} combinations of {int(v[0, 2])} models: constrained 1750 {v[:, 0].min():.3f} to '
          f'{v[:, 0].max():.3f} (mean {v[:, 0].mean():.3f}, all 18: {yc:.3f}), σ {v[:, 1].min():.3f} to {v[:, 1].max():.3f} '
          f'(all 18: {stot:.3f})')

# ------------------------------------------------------------------ 4  observed dSST: sensitivity and uncertainty
print('\n4  observed dSST: effect of +0.1 °C on the constrained value, and σ if the dSST uncertainty were doubled')
for q in refs:
    r = RC[q]
    dT, CUC, y = (np.asarray(r[k], 'f8') for k in ('dT', 'CUC', 'y1750'))
    dT_obs, dT_sig = r['dT_obs']
    CUC_obs, CUC_sig = r['CUC_obs'], rc.CUC_SIG
    yc, stot, sfit, sobs, beta = constrain(dT, CUC, y, dT_obs, dT_sig, CUC_obs, CUC_sig)
    yc2, stot2, *_ = constrain(dT, CUC, y, dT_obs, 2 * dT_sig, CUC_obs, CUC_sig)
    print(f'   {q:10s} dSST_obs {dT_obs:.3f} ± {dT_sig:.3f} °C: b = {beta[1]:+.4f} per °C -> {0.1 * beta[1]:+.4f} per '
          f'+0.1 °C ({100 * 0.1 * abs(beta[1]) / yc:.2f} points of the decline); σ {stot:.4f} -> {stot2:.4f} with ± {2 * dT_sig:.2f} °C')

# ------------------------------------------------------------------ 5  same components for constrained and unconstrained
print('\n5  1750 value ± σ: unconstrained (model spread) and constrained, each without and with the CO2 of 1750')
S = RC[refs[0]]['states']
for q in refs:
    r = RC[q]
    y = np.asarray(r['y1750'], 'f8')
    yu, su = y.mean(), y.std(ddof=1)
    yc, sc = r['constrained'][LAB][:2]
    s_co2 = 0.5 * yc * abs(S['1750_hi']['ratio'] - S['1750_lo']['ratio'])
    print(f'   {q:10s} unconstrained {yu:.3f} ± {su:.3f} | ± {np.hypot(su, s_co2):.3f} with CO2;  '
          f'constrained {yc:.3f} ± {sc:.3f} | ± {np.hypot(sc, s_co2):.3f} with CO2  (CO2 term {s_co2:.3f})')

# ------------------------------------------------------------------ 6  distance to the boundary in points of the decline
print('\n6  decline and distance to the 20 % boundary, per product and mean')
dec = {}
for q in refs:
    now = rc.fit_now(RC[q])
    yc = RC[q]['constrained'][LAB][0]
    dec[q] = 100 * (1 - now / yc)
    print(f'   {q:10s} 2024 {now:.3f}, 1750 {yc:.3f}: decline {dec[q]:.2f} %, {20 - dec[q]:.2f} points short of the boundary')
d = np.array(list(dec.values()))
print(f'   mean {d.mean():.2f} % (products {d.min():.2f} to {d.max():.2f}): {20 - d.mean():.2f} points short of the boundary; '
      f'with σ {sdec_m:.2f} (Extended Data Table 2) that is {(20 - d.mean()) / sdec_m:.1f} σ')

In [ ]:
# Notebook cell: refined trend check (run after the Fig. 1 cell, same kernel: uses G and rc).
# Is the steeper 1990-2014 trend of JMA-MLR a trend error that reaches 2024, or high values in the early 1990s?
# For every product and start year: product trend minus ESM-mean trend (to 2014), its significance, and the implied
# error of the 2024 state relative to the product's reference window, in percentage points of the decline.
import numpy as np

P = G['P']


def ols(x, y):
    """slope and its standard error; the residuals of annual means are autocorrelated (e.g. ENSO), so the error uses
    the effective number of years n_eff = n (1 - r1) / (1 + r1), r1 the lag-1 autocorrelation of the residuals
    (Santer et al. 2000, J. Geophys. Res. 105, 7337)"""
    x, y = np.asarray(x, 'f8'), np.asarray(y, 'f8')
    b, a = np.polyfit(x, y, 1)
    res = y - (a + b * x)
    r1 = float(np.corrcoef(res[:-1], res[1:])[0, 1])
    neff = x.size * (1 - r1) / (1 + r1) if r1 > 0 else float(x.size)
    se = np.sqrt(res @ res / max(neff - 2, 1.0) / np.sum((x - x.mean()) ** 2))
    return b, se


print(f'{"product":10s} {"start":>5s} {"product":>8s} {"ESMs":>8s} {"diff":>8s} {"z":>5s} {"pp":>6s}   (per decade, to 2014)')
for q, p in P.items():
    yo, ob, yr = np.asarray(p['yo']), np.asarray(p['ob']), np.asarray(p['yrs'])
    t_mid = 0.5 * (p['win'][0] + p['win'][1])
    for y0 in sorted({p['win'][0], 1990, 1995, 2000} - set(range(p['win'][0]))):
        so, sm = (yo >= y0) & (yo <= 2014) & np.isfinite(ob), (yr >= y0) & (yr <= 2014)
        bo, seo = ols(yo[so], ob[so])
        bm = np.array([ols(yr[sm], a[sm])[0] for a in p['A']])
        se_m = bm.std(ddof=1) / np.sqrt(bm.size)
        d = bo - bm.mean()
        pp = 100 * abs(d) * (rc.EVAL_YEAR - t_mid) / p['yc']
        print(f'{q:10s} {y0:5d} {10 * bo:8.4f} {10 * bm.mean():8.4f} {10 * d:+8.4f} {d / np.hypot(seo, se_m):5.1f} {pp:6.2f}')
    # residual of the product around the ESM-mean change: where does the difference sit in time?
    A_m = p['A'].mean(0)
    com = [y for y in yo if y in yr and y >= p['win'][0]]
    r = np.array([ob[list(yo).index(y)] - A_m[list(yr).index(y)] for y in com])
    r -= r.mean()
    print(f'{"":10s} product minus ESM mean (window mean removed), by 5 years: '
          + ', '.join(f'{com[k]}-{com[min(k + 4, len(com) - 1)]} {r[k:k + 5].mean():+.3f}' for k in range(0, len(com), 5)))

### Supplementary Fig. 4: trends of the fCO2-products and the ESMs (after the trend check)

In [ ]:
# Notebook cell: Supplementary Fig. 4, the trends of the fCO2-products and of the bias-adjusted
# ESMs over the years they share, from the first year of each fCO2-product to 2014, on the common area (run after the
# Fig. 1 cell and the trend check cell above, same kernel: uses G, rc and ols, the latter with the effective number of
# years of Santer et al. 2000). The printout gives the numbers of the Supplementary Methods ('Mapping uncertainty and
# trend of the fCO2-products').
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.lines import Line2D

rc.FIGNAMES.setdefault('trends', 'FigS29_trends')
P_tr = G['P']
fig, ax = plt.subplots(figsize=(6.5, 3.8))
print(f'{"product":10s} {"years":>9s} {"product":>16s} {"ESMs (mean +- sd)":>20s}   (Omega_arag per decade, common area)')
for k, (q, p) in enumerate(P_tr.items()):
    yo, ob, yr = np.asarray(p['yo']), np.asarray(p['ob']), np.asarray(p['yrs'])
    y0 = int(p['win'][0])
    so, sm = (yo >= y0) & (yo <= 2014) & np.isfinite(ob), (yr >= y0) & (yr <= 2014)
    bo, seo = ols(yo[so], ob[so])
    bm = np.array([ols(yr[sm], a[sm])[0] for a in p['A']])
    xm = k + 0.18 + np.linspace(-0.07, 0.07, bm.size)
    ax.scatter(xm, 10 * bm, s=9, color='0.65', zorder=2)
    ax.errorbar(k + 0.18, 10 * bm.mean(), yerr=10 * bm.std(ddof=1), fmt='s', color='k', ms=5, capsize=3, zorder=3)
    ax.errorbar(k - 0.18, 10 * bo, yerr=10 * seo, fmt='o', color=rc.COLORS[q], ms=7, capsize=3, zorder=3)
    print(f'{q:10s} {y0:4d}-2014 {10 * bo:+8.4f} +- {10 * seo:.4f} {10 * bm.mean():+10.4f} +- {10 * bm.std(ddof=1):.4f}')
ax.set_xticks(range(len(P_tr)))
ax.set_xticklabels([f'{rc.FULL[q].split(" ")[0]}\n{int(p["win"][0])}–2014' for q, p in P_tr.items()], fontsize=8)
ax.set_xlim(-0.6, len(P_tr) - 0.4)
ax.set_ylabel('Trend of $\\Omega_{\\rm arag}$ (per decade)')
ax.legend(handles=[Line2D([], [], marker='o', ls='', color='0.3', label='fCO$_2$-product (±1σ, autocorrelation included)'),
                   Line2D([], [], marker='o', ls='', color='0.65', ms=4, label='ESMs bias-adjusted to it'),
                   Line2D([], [], marker='s', ls='', color='k', label='mean of the ESMs ±1 s.d.')],
          fontsize=7, loc='lower left', frameon=False)
ax.grid(axis='y', lw=0.3, alpha=0.5)
fig.tight_layout()
rc.savefig(fig, 'trends')
plt.show()


## Fig. 2: emergent constraint

In [ ]:
# Notebook cell: Figure 2, the emergent constraint, global and regional (common area).
#   a  global: bias-adjusted 1750 Omega_arag of the 18 ESMs against the regression value (a + b dSST + c CUC),
#      for REF; vertical band = observed predictors ± 1σ; horizontal line = constrained value ± 1σ
#   b  global: constrained vs unconstrained 1750 for all four products (dots = models, bars = 1σ)
#   c  regional: constrained minus unconstrained 1750 Omega_arag (local constraint with cell dSST and CUC), mean of
#      the products
#   d  regional: share of c from the dSST term vs the CUC term, |b dSST'| / (|b dSST'| + |c CUC'|), mean of the
#      products; black dots: |correction| below THR, where the share matters little
# The constrained 1750 field itself is in Supplementary Fig. 8 (per product and mean of the products); the supplementary maps of the
# 1750 and 2024 states (Extended Data Fig. 7) use this cell's variables.
# Needs data/constraint_results.pkl (with 'fields') and regional_common.py.
import sys
import pickle
import importlib
import numpy as np
import netCDF4
import matplotlib.pyplot as plt
from matplotlib.gridspec import GridSpec, GridSpecFromSubplotSpec
from matplotlib.colors import ListedColormap

sys.path.insert(0, CODE_DIR)
import regional_common as rc
importlib.reload(rc)

try:
    import cartopy.crs as ccrs
    HAS_CARTOPY = True
except ImportError:
    HAS_CARTOPY = False

ROOT = rc.ROOT
LAB = rc.LAB
REF = 'OceanSODA'                    # product shown in a (the others in Extended Data)
OMEGA = r'$\Omega_{\mathrm{arag}}$'
VLIM = 0.04
OM_CMAP, OM_LIM = 'cividis', (1.0, 4.2)        # maps of Omega_arag itself (also Extended Data Fig. 7)

with open(f'{ROOT}/constraint_results.pkl', 'rb') as f:
    RC = pickle.load(f)['results_common']
refs = list(RC)
D = rc.load()

# ---------------------------------------------------------------- a: regression for REF
r = RC[REF]
dT, CUC, y = np.asarray(r['dT']), np.asarray(r['CUC']), np.asarray(r['y1750'])
dT_obs, dT_sig = r['dT_obs']
CUC_obs, CUC_sig = r['CUC_obs'], rc.CUC_SIG
X = np.column_stack([np.ones(y.size), dT, CUC])
beta, *_ = np.linalg.lstsq(X, y, rcond=None)
res = y - X @ beta
s_fit = np.sqrt(res @ res / (y.size - X.shape[1]))             # residual standard deviation of the fit
r2 = 1 - res @ res / ((y - y.mean()) @ (y - y.mean()))
xr = X @ beta                                                  # regression value per model
x_obs = beta @ np.array([1.0, dT_obs, CUC_obs])
x_sig = np.hypot(beta[1] * dT_sig, beta[2] * CUC_sig)
yc, sc, _ = r['constrained'][LAB]
print(f'{REF}: y = {beta[0]:.3f} {beta[1]:+.4f} dSST {beta[2]:+.3f} CUC, R2 {r2:.2f}, residual sd {s_fit:.3f}; '
      f'constrained {yc:.3f} ± {sc:.3f}')
# model numbers: alphabetical, ignoring case (the same in Extended Data Fig. 3)
NUM = {m: k + 1 for k, m in enumerate(sorted(r['names'], key=str.lower))}
print('model numbers: ' + ', '.join(f'{NUM[m]} {m}' for m in sorted(r['names'], key=str.lower)))

# ---------------------------------------------------------------- c, d: regional fields (as fig_correction_terms.py)
terms = {}
for q in refs:
    rq = RC[q]
    F = rq['fields']
    mask = np.asarray(rq['mask']).astype(bool)
    Y, DT, CU = (F[k].astype('f8') for k in ('y1750', 'dT', 'CUC'))
    n = Y.shape[0]
    dobs = np.array([F['dsst_obs'][k] for k in F['dsst_obs']], 'f8')
    dTo, dTs = dobs.mean(0), dobs.std(0, ddof=1)
    CUo = F['CUC_obs'].astype('f8')
    with netCDF4.Dataset(rc.OBS[q]) as f:
        yo = np.asarray(f['year'][:])
        om = np.ma.filled(f['omega_arag'][:].astype('f8'), np.nan)[yo <= rc.LAST_YEAR]
    ok = (mask & np.all(np.isfinite(Y), 0) & np.all(np.isfinite(DT), 0) & np.all(np.isfinite(CU), 0)
          & np.all(np.isfinite(om), 0) & np.isfinite(dTo) & np.isfinite(CUo))
    idx = np.where(ok.ravel())[0]
    sel = lambda a: a.reshape(n, -1)[:, idx] if a.ndim == 3 else a.ravel()[idx]
    yq, x1, x2 = sel(Y), sel(DT), sel(CU)
    ycq, st, bq = rc.local_constraint(yq, x1, x2, sel(dTo), sel(dTs), sel(CUo), ok, return_beta=True)
    t_sst = bq[:, 1] * (sel(dTo) - x1.mean(0))
    t_cuc = bq[:, 2] * (sel(CUo) - x2.mean(0))

    def to_map(v):
        m_ = np.full(mask.size, np.nan)
        m_[idx] = v
        return rc.to_180(np.asarray(rq['lon'], 'f8'), m_.reshape(mask.shape))[1]
    terms[q] = dict(corr=D[q]['pi_full'] - D[q]['pi_mean'], sst=to_map(t_sst), cuc=to_map(t_cuc),
                    nogain=(D[q]['sig_full'] >= D[q]['sig_model']).astype(float))
# regional constraint only where it can be done for all four products, i.e. where the observed dSST (three SST products)
# and CUC exist; elsewhere (mostly sea ice) the fields would take the multi-model mean (flag 2): left blank here
both = np.all([D[q]['constrained'] == 1 for q in refs], axis=0)
corr_m = np.where(both, np.nanmean([terms[q]['corr'] for q in refs], axis=0), np.nan)
sst_m = np.where(both, np.nanmean([terms[q]['sst'] for q in refs], axis=0), np.nan)
cuc_m = np.where(both, np.nanmean([terms[q]['cuc'] for q in refs], axis=0), np.nan)
pi_m = np.where(both, np.nanmean([D[q]['pi_full'] for q in refs], axis=0), np.nan)        # constrained 1750
pisig_m = np.where(both, np.sqrt(np.nanmean([D[q]['sig_full'] ** 2 for q in refs], axis=0)), np.nan)
now_m = np.where(both, np.nanmean([D[q]['now'] for q in refs], axis=0), np.nan)         # fitted 2024
print(f'regional constraint possible for all products in {100 * np.sum(D[refs[0]]["area"] * both) / np.sum(D[refs[0]]["area"] * np.isfinite(D[refs[0]]["constrained"])):.1f} % of the common area')
THR = 0.005                          # d: dots where the correction (c) is smaller than this (|constrained - unconstrained|)
tot = np.abs(sst_m) + np.abs(cuc_m)
share = np.where(np.isfinite(tot), 100 * np.abs(sst_m) / np.where(tot > 0, tot, 1), np.nan)
small = np.where(np.isfinite(corr_m) & (np.abs(corr_m) < THR), 1.0, np.nan)
# d: share of the SST term (0 = only CUC, purple: chemistry; 1 = only SST, orange: temperature) on a diverging scale,
# white at 50 % / 50 %; the ends of PuOr are
# cut off so that black dots (correction below THR, share not meaningful) stay visible everywhere
from matplotlib.colors import LinearSegmentedColormap
SHARE_CMAP = LinearSegmentedColormap.from_list('PuOr_r_light', plt.get_cmap('PuOr_r')(np.linspace(0.17, 0.83, 256)))
nogain = np.where(np.nanmean([terms[q]['nogain'] for q in refs], axis=0) >= 0.5, 1.0, np.nan)
lat, lon, land, area = D[refs[0]]['lat'], D[refs[0]]['lon'], D[refs[0]]['land'], D[refs[0]]['area']
v = np.isfinite(corr_m)
print(f'regional: mean |correction| {np.sum(area[v] * np.abs(corr_m[v])) / area[v].sum():.4f}; SST share '
      f'{100 * np.sum(area[v] * np.abs(sst_m[v])) / np.sum(area[v] * (np.abs(sst_m[v]) + np.abs(corr_m[v] - sst_m[v]))):.0f} %')

# ---------------------------------------------------------------- figure
fig = plt.figure(figsize=(12, 8.2))
gs = GridSpec(2, 2, figure=fig, height_ratios=[1, 1.05], hspace=0.3, wspace=0.25,
              left=0.07, right=0.98, top=0.96, bottom=0.1)

# a: regression panel and, to its right, the 1750 range before and after the constraint
sub_a = GridSpecFromSubplotSpec(1, 2, subplot_spec=gs[0, 0], width_ratios=[1, 0.16], wspace=0.04)
ax = fig.add_subplot(sub_a[0])
ax.axvspan(x_obs - x_sig, x_obs + x_sig, color='0.85', lw=0)
ax.axvline(x_obs, color='0.5', lw=0.8)
lim = [min(xr.min(), y.min()) - 0.02, max(xr.max(), y.max()) + 0.02]
ll = np.array(lim)
xx = np.linspace(lim[0], lim[1], 100)
s_pred = s_fit * np.sqrt(1 + 1 / xr.size + (xx - xr.mean()) ** 2 / np.sum((xr - xr.mean()) ** 2))
PRED_COL = '#dfe8ee'                  # prediction band (light blue-grey; grey is the observation band)
ax.fill_between(xx, xx - s_pred, xx + s_pred, color=PRED_COL, lw=0, zorder=0)   # prediction interval of the regression
ax.plot(lim, lim, color='0.6', lw=0.8, ls='--')
ax.scatter(xr, y, s=90, color=rc.COLORS[REF], edgecolor='k', lw=0.4, zorder=3)
CLOSE = 0.004                                        # dots closer than this overlap: number written beside them
for i, (xi, yi) in enumerate(zip(xr, y)):
    d = np.hypot(xr - xi, y - yi)
    d[i] = np.inf
    j = int(np.argmin(d))
    if d[j] < CLOSE:
        left = xi < xr[j] or (xi == xr[j] and i < j)  # left dot of the pair: number on its left, other on its right
        ax.text(xi + (-0.0029 if left else 0.0029), yi, str(NUM[r['names'][i]]), ha='right' if left else 'left', va='center',
                fontsize=6.5, color='k', zorder=4)
    else:
        ax.text(xi, yi, str(NUM[r['names'][i]]), ha='center', va='center', fontsize=6, color='w', zorder=4)
ax.axhspan(yc - sc, yc + sc, color=rc.COLORS[REF], alpha=0.2, lw=0)
ax.axhline(yc, color=rc.COLORS[REF], lw=1.4)
ax.text(lim[0] + 0.002, yc + sc + 0.002, f'constrained: {yc:.3f} ± {sc:.3f}', ha='left', va='bottom', fontsize=8.5,
        color=rc.COLORS[REF])
ax.text(x_obs + x_sig + 0.002, lim[0] + 0.003, 'based on the observed\n$\\Delta$SST and CUC', ha='left', va='bottom',
        fontsize=8.5, color='0.4')
ax.text(lim[0] + 0.002, lim[1] - 0.002, f'R$^2$ = {r2:.2f}', ha='left', va='top', fontsize=8.5)
ax.set_xlim(lim)
ax.set_ylim(lim)
ax.set_xlabel('Regression value $a + b\\,\\Delta$SST$ + c\\,$CUC', fontsize=10)
ax.set_ylabel(f'Bias-adjusted 1750 {OMEGA}', fontsize=10)
ax.spines[['top', 'right']].set_visible(False)
ax.text(-0.12, 1.02, 'a', transform=ax.transAxes, fontsize=14, fontweight='bold', va='bottom')
axr = fig.add_subplot(sub_a[1], sharey=ax)
yu_r, su_r = r['unconstrained'][:2]
# same style as b: models (grey dots) with their mean ± 1 sd (grey square), constrained ± 1 sigma (colour)
axr.scatter(np.random.default_rng(1).uniform(-0.12, 0.12, y.size), y, s=12, color='0.6', edgecolor='k', lw=0.3,
            zorder=2)
axr.errorbar(0, yu_r, yerr=su_r, fmt='s', color='0.3', ms=5, capsize=3, lw=1, zorder=3)
axr.errorbar(1, yc, yerr=sc, fmt='o', color=rc.COLORS[REF], mec='k', mew=0.4, ms=6, capsize=3, lw=1.2, zorder=3)
axr.set_xlim(-0.5, 1.5)
axr.set_xticks([])
axr.text(0, y.max() + 0.008, 'ESMs', rotation=90, ha='center', va='bottom', fontsize=8, color='0.3')   # as in b
axr.text(1, yc + sc + 0.008, 'constrained', rotation=90, ha='center', va='bottom', fontsize=8, color=rc.COLORS[REF])
axr.spines[['top', 'right', 'bottom', 'left']].set_visible(False)
axr.tick_params(left=False, labelleft=False)

# b: constrained vs unconstrained, all products
ax = fig.add_subplot(gs[0, 1])
for k, q in enumerate(refs):
    rq = RC[q]
    yq = np.asarray(rq['y1750'])
    yu, su = rq['unconstrained'][:2]
    ycq, scq, _ = rq['constrained'][LAB]
    ax.scatter(np.full(yq.size, k - 0.18) + np.random.default_rng(1).uniform(-0.05, 0.05, yq.size), yq, s=12,
               color='0.6', edgecolor='k', lw=0.3, zorder=2)
    ax.errorbar(k - 0.18, yu, yerr=su, fmt='s', color='0.3', ms=5, capsize=3, lw=1, zorder=3)
    ax.errorbar(k + 0.18, ycq, yerr=scq, fmt='o', color=rc.COLORS[q], mec='k', mew=0.4, ms=6, capsize=3, lw=1.2, zorder=3)
ax.set_xticks(range(len(refs)))
ax.set_xticklabels([rc.FULL[q].replace(' v', '\nv') for q in refs], fontsize=8)
ax.set_ylabel(f'{OMEGA} in 1750', fontsize=10)
# labels above the first product's columns (as in a): grey = ESMs, colour = constrained
q0 = refs[0]
y0q = np.asarray(RC[q0]['y1750'])
yc0, sc0 = RC[q0]['constrained'][LAB][:2]
ax.text(-0.18, y0q.max() + 0.008, 'ESMs', rotation=90, ha='center', va='bottom', fontsize=8, color='0.3')
ax.text(0.18, yc0 + sc0 + 0.008, 'constrained', rotation=90, ha='center', va='bottom', fontsize=8,
        color=rc.COLORS[q0])
ax.spines[['top', 'right']].set_visible(False)
ax.text(-0.12, 1.02, 'b', transform=ax.transAxes, fontsize=14, fontweight='bold', va='bottom')

# c, d: maps
# Robinson projection, computed directly (no cartopy needed): Snyder's table, linear interpolation in latitude.
# CENTER_LON = 0: Atlantic in the centre; 200: Pacific in the centre, edge through South Africa (20 E)
CENTER_LON = rc.MAP_CENTER         # as all maps (regional_common.py)
_RLAT = np.arange(0, 91, 5.0)
_RX = np.array([1.0000, 0.9986, 0.9954, 0.9900, 0.9822, 0.9730, 0.9600, 0.9427, 0.9216, 0.8962, 0.8679, 0.8350,
                0.7986, 0.7597, 0.7186, 0.6732, 0.6213, 0.5722, 0.5322])
_RY = np.array([0.0000, 0.0620, 0.1240, 0.1860, 0.2480, 0.3100, 0.3720, 0.4340, 0.4958, 0.5571, 0.6176, 0.6769,
                0.7346, 0.7903, 0.8435, 0.8936, 0.9394, 0.9761, 1.0000])


def robinson(lo, la):
    """map coordinates of longitudes relative to the centre (-180..180) and latitudes (degrees)"""
    lo, la = np.asarray(lo, 'f8'), np.asarray(la, 'f8')
    a = np.clip(np.abs(la), 0, 90)
    return (0.8487 * np.interp(a, _RLAT, _RX) * np.deg2rad(lo),
            1.3523 * np.interp(a, _RLAT, _RY) * np.sign(la))


_ls = (np.asarray(lon, 'f8') - CENTER_LON + 180.0) % 360.0 - 180.0     # longitude relative to the centre
om_ = np.argsort(_ls)
lon_m = _ls[om_]
cols = lambda a: np.asarray(a)[:, om_]                  # fields in the column order of the map
_dlo, _dla = np.median(np.diff(lon_m)), np.median(np.diff(lat))
lon_e = np.clip(np.concatenate([lon_m - _dlo / 2, [lon_m[-1] + _dlo / 2]]), -180, 180)
lat_e = np.clip(np.concatenate([np.asarray(lat) - _dla / 2, [lat[-1] + _dla / 2]]), -90, 90)
Xe, Ye = robinson(*np.meshgrid(lon_e, lat_e))           # cell edges on the map
_bl = np.linspace(-90, 90, 181)
outline = np.concatenate([np.column_stack(robinson(np.full(_bl.size, -180.0), _bl)),
                          np.column_stack(robinson(np.full(_bl.size, 180.0), _bl[::-1]))])
outline = np.vstack([outline, outline[:1]])            # closed: left and right edges, pole lines at top and bottom
pcs = []
for col, (field, title, letter, cmap, v0, v1) in enumerate((
        (cols(corr_m), f'Constrained minus unconstrained {OMEGA} in 1750', 'c', 'RdBu_r', -VLIM, VLIM),
        (cols(share / 100), 'What drives the correction: $\\Delta$SST or CUC', 'd', SHARE_CMAP, 0, 1))):
    axm = fig.add_subplot(gs[1, col])
    axm.set_aspect('equal')
    axm.axis('off')
    axm.pcolormesh(Xe, Ye, np.where(cols(land), 1.0, np.nan), cmap=ListedColormap(['0.85']), vmin=0, vmax=1,
                   shading='flat')
    pcs.append(axm.pcolormesh(Xe, Ye, field, cmap=cmap, vmin=v0, vmax=v1, shading='flat'))
    if col == 1:                                         # dots: correction below THR (every second cell)
        LO2, LA2 = np.meshgrid(lon_m, lat)
        sel_ = np.isfinite(cols(small)) & (np.add.outer(np.arange(lat.size), np.arange(lon_m.size)) % 2 == 0) \
            & (np.arange(lat.size)[:, None] % 2 == 0)
        axm.scatter(*robinson(LO2[sel_], LA2[sel_]), s=1.2, color='k', lw=0, zorder=5)
    axm.plot(*outline.T, color='k', lw=0.6, zorder=6, clip_on=False)
    axm.margins(0.01)
    axm.set_title(title, fontsize=10.5)
    axm.text(0.0, 1.02, letter, transform=axm.transAxes, fontsize=14, fontweight='bold', va='bottom')
    axm.apply_aspect()                                   # position after the equal aspect
    pos = axm.get_position()
    cax = fig.add_axes([pos.x0 + 0.15 * pos.width, pos.y0 - 0.045, 0.7 * pos.width, 0.018])
    if col == 0:
        fig.colorbar(pcs[0], cax=cax, orientation='horizontal', extend='both').set_label(f'$\\Delta${OMEGA}', fontsize=9.5)
    else:
        cb = fig.colorbar(pcs[1], cax=cax, orientation='horizontal', extend='neither')
        cb.set_label('Share of the correction from $\\Delta$SST', fontsize=9.5)
        cb.set_ticks([0, 0.5, 1])
        cb.set_ticklabels(['only CUC', '50 % / 50 %', 'only $\\Delta$SST'])
rc.savefig(fig, 'constraint')
plt.show()
# map fields and map projection for Extended Data Fig. 7, kept under one name so that later cells (Fig. 3 uses now_m, lat, ...)
# cannot overwrite them
FIG2 = dict(pi_m=pi_m, pisig_m=pisig_m, now_m=now_m, land=land, area=area, lat=lat, lon_m=lon_m, Xe=Xe, Ye=Ye,
            cols=cols, robinson=robinson, outline=outline, OMEGA=OMEGA, OM_CMAP=OM_CMAP, OM_LIM=OM_LIM)

## Fig. 4: when and where the boundary is crossed

In [ ]:
# Notebook cell: Figure 4, when and where the boundary is crossed: the best estimate (whole ocean, both adjustments)
#   a (top, centred)    decline of global mean surface Omega_arag since 1750 (%), 1950-2040. Bold: the best estimate, the
#                       global mean over the whole ocean with the alkalinity rise (D_g = D_c (1 + r) + a(t), GLOBAL of the
#                       preparation cell; a(t) with the time course of the alkalinity effect of the maps below, held at its
#                       2024 value after 2024) with its 1 sigma (SDEC_G), the years of 10 % and 20 % with their ranges.
#                       Thin: ESMs relative to their own 1750 (mean ± 1 sd), the products' annual means and their forced
#                       mean on the common area (the analysis of Fig. 1), drawn on top of the best estimate
#   b, c (centre)       maps of the best estimate (_best): each product on its own area, the mean of the products that
#                       cover a cell, with the alkalinity adjustment of the data product; decline
#                       from 1750 to 2024 with the -20 % contour, and the year in which each cell crosses the boundary
#   d (bottom, centred) share of the covered area beyond the boundary: the best estimate (forced to 2024, then the
#                       projection; bold) and the same with the annual values of the products instead of the forced
#                       series, from ANN_START (thin): per year and cell the mean of the covering products' annual
#                       Omega_arag with the alkalinity change of that year, relative to the same 1750 state and area;
#                       coloured: each product alone on its own area (forced, then its projection)
# Needs the preparation cell (SDEC_SHARED, sdec_m, decline_crossing, GLOBAL, SDEC_G, XCO2) and the alkalinity files of the
# data product (step 4). The code of the global and the regional analysis is included below as two functions
# (_global, _regional) that run without drawing and return their variables; _best adds the best-estimate maps (own
# areas). The projections are cached (projection.py): a second run recomputes nothing.
ANN_START = 1990                     # thin line of d from here (all four products have data from 1990 on)


def _global():
    # Global mean surface Omega_arag on the common area, from 1750 to the boundary: the data of every product and of
    # their mean, and panel(), which draws one of them (the mean: Fig. 1; one panel per product: Supplementary Fig. 3).
    #   a (same aspect ratio as b-e, A_HEIGHT times as large, centred): mean of the four products
    #       left column (1750): ESMs (grey dots; bias-adjusted, averaged over the four adjustments), multi-model mean ± 1 sd
    #                           (black), constrained (red, mean of the products ± 1σ)
    #       main axis:          ESMs 1850-2014 (thin grey), multi-model mean (black), mean of the observation-based products
    #                           (red, years covered by all four), projection to 2040 from the fitted 2024 value (CO2 at the
    #                           CO2 growth for constant emissions (projection.co2_growth), T rising with CO2 as observed, S and ALK constant), crossing year
    #       right column:       the same at 80 % (boundary)
    #   b-e: one product each: models adjusted to this product, the product, its
    #        bias-adjustment period (light grey), raw and constrained boundary, 1750 and boundary columns
    # Uncertainties: model spread (raw) and constraint (constrained) only. The CO2 of 1750 (AR6 278.3 ± 2.9 ppm) is
    # printed, not drawn (it moves all values together).
    # Needs data/constraint_results.pkl (with 'states'), the bias-adjusted historical files, data/fig_crossing_projection.pkl
    # (projection.py) and data/co2_annmean_gl.txt.
    import glob
    import os
    import sys
    import pickle
    import importlib
    import numpy as np
    import netCDF4
    from matplotlib.gridspec import GridSpecFromSubplotSpec
    from matplotlib.patches import Patch
    from matplotlib.lines import Line2D
    from matplotlib.legend_handler import HandlerTuple

    sys.path.insert(0, CODE_DIR)
    import regional_common as rc
    importlib.reload(rc)

    ROOT = rc.ROOT
    LAB = rc.LAB
    OMEGA = r'$\Omega_{\mathrm{arag}}$'
    ADJ = {'OceanSODA': f'{ROOT}/omega_arag_biasadj', 'UExP': f'{ROOT}/omega_arag_biasadj_UExP',
           'JMA': f'{ROOT}/omega_arag_biasadj_JMA', 'CMEMS': f'{ROOT}/omega_arag_biasadj_CMEMS'}
    WIN = {q: oa_config.WINDOWS[q] for q in ('OceanSODA', 'UExP', 'JMA', 'CMEMS')}   # the bias windows of the chain
    YEAR_END = 2040                      # end of the projection in a
    MEAN_COL = 'C3'                      # mean of the products (red; no product uses red)

    with open(f'{ROOT}/constraint_results.pkl', 'rb') as f:
        RC = pickle.load(f)['results_common']
    refs = list(RC)

    # ---------------------------------------------------------------- NOAA CO2, years <-> ppm at a constant growth rate
    co2 = np.array([[float(v) for v in l.split()[:2]] for l in open(f'{ROOT}/co2_annmean_gl.txt')
                    if l.strip() and not l.startswith('#')])
    cy, cx = co2[:, 0].astype(int), co2[:, 1]
    import projection as _pj
    growth = _pj.co2_growth()[0]             # constant emissions at the 2024 level: 2.6 ppm/yr
    y_last = rc.EVAL_YEAR                     # anchor at 2024 (the NOAA file may go further)
    x_last = float(cx[cy == y_last][0])
    year_of = lambda x: y_last + (x - x_last) / growth
    x_of = lambda y: x_last + (y - y_last) * growth

    # ---------------------------------------------------------------- per product: bias-adjusted ESMs on the common mask
    P = {}
    for q, rr in RC.items():
        cmask = np.asarray(rr['mask']).astype(bool)
        area = oa_config.area_weights(rr['lat'], rr['lon'])
        A, nm = [], []
        for fh in sorted(glob.glob(f'{ADJ[q]}/historical/omega_arag_biasadj_Oyr_*_historical_*.nc')):
            model = os.path.basename(fh).split('_')[4]
            if model not in rr['names']:
                continue
            with netCDF4.Dataset(fh) as f:
                yrs = np.asarray(f['year'][:])
                h = np.ma.filled(f['omega_arag_adj'][:].astype('f8'), np.nan)
            m = cmask & np.all(np.isfinite(h), axis=0)
            w = area * m
            A.append(np.nansum(np.where(m, h, 0) * w, axis=(1, 2)) / w.sum())
            nm.append(model)
        pi = np.array([np.asarray(rr['y1750'])[list(rr['names']).index(m_)] for m_ in nm])
        yo = np.asarray(rr['year'])
        keep = yo <= rc.LAST_YEAR
        yc, sc = rr['constrained'][LAB][:2]
        P[q] = dict(A=np.array(A), names=nm, yrs=yrs, pi=pi, yc=yc, sc=sc, yo=yo[keep],
                    ob=np.asarray(rr['ogm'])[keep], col=rc.COLORS[q], title=rc.FULL[q], win=WIN[q])

    # ---------------------------------------------------------------- mean of the products (panel a)
    names = [m for m in P[refs[0]]['names'] if all(m in P[q]['names'] for q in refs)]
    A_m = np.mean([[P[q]['A'][P[q]['names'].index(m)] for m in names] for q in refs], axis=0)       # (model, year)
    pi_m = np.mean([[P[q]['pi'][P[q]['names'].index(m)] for m in names] for q in refs], axis=0)
    yo_m = np.array(sorted(set.intersection(*[set(P[q]['yo'].tolist()) for q in refs])))          # years of all four
    ob_m = np.mean([[P[q]['ob'][list(P[q]['yo']).index(y)] for y in yo_m] for q in refs], axis=0)
    yc_m = np.mean([P[q]['yc'] for q in refs])
    sc_m = np.sqrt(np.mean([P[q]['sc'] ** 2 for q in refs]))
    P_mean = dict(A=A_m, yrs=P[refs[0]]['yrs'], pi=pi_m, yc=yc_m, sc=sc_m, yo=yo_m, ob=ob_m, col=MEAN_COL,
                  title='Mean of the four fCO$_2$-products',
                  win=(max(w[0] for w in WIN.values()), min(w[1] for w in WIN.values())))       # 1990-2014, common to all four

    # projection from the fitted 2024 value of each product, mean over the products, crossing year
    import projection
    importlib.reload(projection)
    proj = projection.global_projection()        # cached in the run folder (projection.py)
    X_NOW = dict(zip(cy, cx))[rc.EVAL_YEAR]
    proj = {q: rc.anchor_projection(proj[q], RC[q], X_NOW) for q in refs}
    xg = np.arange(X_NOW, x_of(YEAR_END) + 1, 0.5)
    yg = year_of(xg)
    curve = np.mean([np.interp(xg, proj[q]['x'], proj[q]['scaled']) for q in refs], axis=0)


    def cross(c, level):
        i = np.where(c <= level)[0][0]
        return yg[i - 1] + (c[i - 1] - level) / (c[i - 1] - c[i]) * (yg[i] - yg[i - 1])


    B, B_sig = 0.8 * yc_m, 0.8 * sc_m
    y_prod = {q: cross(np.interp(xg, proj[q]['x'], proj[q]['scaled']), 0.8 * P[q]['yc']) for q in refs}
    y_cross = np.mean(list(y_prod.values()))
    # crossing range: the full 1 sigma of the decline shared by the products (Extended Data Table 2 without the product spread,
    # SDEC_SHARED of the preparation cell), as an Omega_arag offset of the boundary, and the spread of the products
    if 'SDEC_SHARED' not in globals():
        raise NameError('SDEC_SHARED not defined: run the preparation cell first (it comes before this cell)')
    s_om = SDEC_SHARED / 100 * yc_m
    dy = np.hypot(np.std(list(y_prod.values()), ddof=1), (cross(curve, B - s_om) - cross(curve, B + s_om)) / 2)
    S = RC[refs[0]]['states']
    s_co2 = 0.5 * yc_m * abs(S['1750_hi']['ratio'] - S['1750_lo']['ratio'])
    dy_co2 = (cross(curve, 0.8 * (yc_m - s_co2)) - cross(curve, 0.8 * (yc_m + s_co2))) / 2
    for lab_, p in list(P.items()) + [('mean', P_mean)]:
        print(f'{lab_:10s} 1750 models {p["pi"].mean():.3f} ± {p["pi"].std(ddof=1):.3f}, constrained {p["yc"]:.3f} ± '
              f'{p["sc"]:.3f}; boundary {0.8 * p["yc"]:.3f} ± {0.8 * p["sc"]:.3f}')
    print(f'crossing {y_cross:.1f} ± {dy:.1f} (' + ', '.join(f'{q} {v:.1f}' for q, v in y_prod.items())
          + f'); 1 sigma: decline {SDEC_SHARED:.2f} percentage points shared (Extended Data Table 2, of which the CO2 of 1750 '
          f'± {dy_co2:.1f} years) and the spread of the products')

    # common y-limits
    lo = min(min(p['A'].min(), p['ob'].min(), 0.8 * p['pi'].min()) for p in P.values()) - 0.03
    lo = min(lo, B - B_sig - 0.07)                     # room for the crossing label below the boundary in a
    hi = max(max(p['A'].max(), p['ob'].max(), p['pi'].max()) for p in P.values()) + 0.03


    # ---------------------------------------------------------------- one panel (1750 column | time series | boundary)
    def panel(spec, p, letter, big, fig_=None):
        fg = fig if fig_ is None else fig_                # figure to draw into (Fig. 4 passes its own)
        fs = 10 if big else 8                               # font size of labels inside the panel
        ms = 4.5 if big else 3
        sub = GridSpecFromSubplotSpec(1, 3, subplot_spec=spec, width_ratios=[0.13, 1, 0.17], wspace=0.08)
        ax0 = fg.add_subplot(sub[0])
        ax = fg.add_subplot(sub[1], sharey=ax0)
        ax2 = fg.add_subplot(sub[2], sharey=ax0)
        jit = np.random.default_rng(1).uniform(-0.18, 0.18, p['pi'].size)
        bc, bs = 0.8 * p['yc'], 0.8 * p['sc']

        def column(a_, v, cval, sval, below):
            a_.scatter(-0.25 + jit, v, s=12 if big else 6, color='0.55', lw=0, zorder=2)
            a_.errorbar(0.25, v.mean(), yerr=v.std(ddof=1), fmt='o', color='k', ms=ms, capsize=3, lw=1.1, zorder=3)
            a_.errorbar(0.65, cval, yerr=sval, fmt='o', color=p['col'], ms=ms, capsize=3, lw=1.3, zorder=4)
            if below:
                y_, va = min(v.min(), v.mean() - v.std(ddof=1), cval - sval) - 0.02, 'top'
            else:
                y_, va = max(v.max(), v.mean() + v.std(ddof=1), cval + sval) + 0.02, 'bottom'
            for x_, t_, c_ in ((-0.25, 'ESMs', '0.45'), (0.25, 'multi-model mean', 'k'), (0.65, 'constrained', p['col'])):
                a_.text(x_, y_, t_, rotation=90, ha='center', va=va, fontsize=fs - 0.5, color=c_)
            a_.set_xlim(-0.6, 0.9)
            a_.set_xticks([0.15])
            a_.spines[['top', 'right']].set_visible(False)
            a_.tick_params(labelsize=fs)

        column(ax0, p['pi'], p['yc'], p['sc'], True)
        ax0.set_xticklabels(['1750'])
        ax0.text(-0.9 if big else -1.0, 1.02, letter, fontsize=16 if big else 13, fontweight='bold', va='bottom', transform=ax0.transAxes)
        column(ax2, 0.8 * p['pi'], bc, bs, False)
        ax2.set_xticklabels(['boundary'])
        ax2.tick_params(labelleft=False)

        A, yrs = p['A'], p['yrs']
        if p['win'] is not None:
            ax.axvspan(*p['win'], color='0.92', lw=0, zorder=0)
        for a in A:
            ax.plot(yrs, a, color='0.78', lw=0.6, zorder=1)
        ax.plot(yrs, A.mean(0), color='k', lw=1.5 if big else 1.2, zorder=2)
        ax.plot(p['yo'], p['ob'], color=p['col'], lw=2.0 if big else 1.4, zorder=4)
        bu, bus = 0.8 * p['pi'].mean(), 0.8 * p['pi'].std(ddof=1)
        for y_ in (bu - bus, bu + bus):
            ax.axhline(y_, color='0.45', ls=(0, (3, 3)), lw=0.5, zorder=2)
        ax.axhline(bu, color='0.3', ls=(0, (4, 4)), lw=1.0, zorder=2)
        ax.axhspan(bc - bs, bc + bs, color=p['col'], alpha=0.3, lw=0, zorder=1)
        ax.axhline(bc, color=p['col'], ls=(4, (4, 4)), lw=1.0, zorder=3)
        ax.set_ylim(lo, hi)
        ax.spines[['top', 'right']].set_visible(False)
        ax.tick_params(labelleft=False, labelsize=fs)
        ax.set_title(p['title'], fontsize=13 if big else 10, fontweight='bold')

        if big:
            ax.plot(yg, curve, color=p['col'], lw=2.0, ls=(0, (3, 2)), zorder=4)
            # crossing: point on the boundary, label in the empty upper right, connected by a thin arrow
            ax.plot(y_cross, B, 'o', ms=6, mfc='white', mec='k', mew=1.2, zorder=6)
            lab_x = p['win'][1] + 1.0                      # right of the bias-adjustment shading
            ax.annotate(f'boundary\ncrossed ≈ {y_cross:.0f}\n({y_cross - dy:.0f}–{y_cross + dy:.0f})', xy=(y_cross, B),
                        xytext=(lab_x, B + 0.25 * (hi - B)), ha='left', va='bottom', fontsize=fs - 2, linespacing=1.3,
                        arrowprops=dict(arrowstyle='-|>,head_length=0.25,head_width=0.12', color='0.4', lw=0.6, shrinkA=2, shrinkB=5,
                                        connectionstyle='arc3,rad=0.0'), zorder=6)
            ax.set_xlim(yrs[0], YEAR_END)
            ax0.set_ylabel(f'Mean surface {OMEGA}, common area', fontsize=12)
            hh = [Line2D([], [], color='0.7', lw=0.8), Line2D([], [], color='k', lw=1.5),
                  Line2D([], [], color=p['col'], lw=2.0), Line2D([], [], color=p['col'], lw=2.0, ls=(0, (3, 2))),
                  Patch(facecolor='0.92', lw=0), Line2D([], [], color='0.3', ls=(0, (4, 4)), lw=1.0),
                  (Patch(facecolor=p['col'], alpha=0.3, lw=0), Line2D([], [], color=p['col'], ls=(4, (4, 4)), lw=1.0))]
            ll = ['ESMs (bias-adjusted)', 'multi-model mean',
                  'fCO$_2$-products (mean of four)',
                  'idealised projection', 'bias-adjustment period',
                  f'boundary, raw models ({bu:.2f} $\\pm$ {bus:.2f})', f'boundary, constrained ({bc:.2f} $\\pm$ {bs:.2f})']
            ax.legend(hh, ll, fontsize=fs - 1, frameon=False, loc='lower left', bbox_to_anchor=(0.0, 0.14),
                      handler_map={tuple: HandlerTuple(ndivide=1, pad=0)})
        else:
            ax.set_xlim(yrs[0], rc.LAST_YEAR)
            # boundary values as a small legend just above the boundary lines, with the line styles of a
            hh = [Line2D([], [], color='0.3', ls=(0, (4, 4)), lw=1.0),
                  (Patch(facecolor=p['col'], alpha=0.3, lw=0), Line2D([], [], color=p['col'], ls=(4, (4, 4)), lw=1.0))]
            y_txt = (max(bc + bs, bu + bus) + 0.01 - lo) / (hi - lo)              # axes fraction
            leg = ax.legend(hh, [f'Raw: {bu:.2f} $\\pm$ {bus:.2f}', f'Constrained: {bc:.2f} $\\pm$ {bs:.2f}'],
                            title='Boundary', title_fontproperties={'weight': 'bold', 'size': fs}, fontsize=fs,
                            frameon=False, loc='lower left', bbox_to_anchor=(0.0, y_txt), handlelength=2.2,
                            borderaxespad=0.2, handler_map={tuple: HandlerTuple(ndivide=1, pad=0)})
            leg._legend_box.align = 'left'
        return ax0, ax


    return locals()


def _regional():
    # The regional crossing on the common area (mean of the four products), computed for Fig. 4: the decline of surface
    # Omega_arag from 1750 to 2024 (1750 = local constraint, pi_full); the year in which each cell reaches -20 % (cells
    # below the boundary in 2024 from the observed record, per-cell quadratic fit; the others from the per-cell chemistry
    # projection from the forced 2024 state: surface pCO2 rising with the atmosphere at the CO2 growth for constant
    # emissions, projection.co2_growth, T rising with CO2 as observed, S and ALK constant); the share of the surface area
    # beyond the boundary, observed and projected, with the year in which the global mean crosses.
    # Needs: data/constraint_results.pkl (with 'states'), regional_common.py, the product files (calc_bias_adjusted),
    # mocsy, pi_states.py, and the NOAA CO2 record (data/co2_annmean_gl.txt, see co2_history.py).
    # The per-cell projection is slow (minutes per product) and cached in data/fig_regional_crossing_projection.pkl.
    import os
    import sys
    import pickle
    import importlib
    import urllib.request
    import numpy as np
    from matplotlib.transforms import offset_copy

    sys.path.insert(0, CODE_DIR)
    import regional_common as rc
    import calc_bias_adjusted as cb
    import calc_omega_arag_obs as co
    import calc_pi_xco2 as cpx
    importlib.reload(rc)

    ROOT = rc.ROOT
    LAB = rc.LAB
    STATE = (2020, 2024)                 # STATE[1]: the first year of the projection
    TREND = (1990, 2024)                 # period for the observed warming per ppm (k_T)
    YEAR_END = 2040                      # last year of panel c
    DX = np.arange(0.0, 202.0, 2.0)      # ppm above the state (to about 620 ppm)
    VARS = cb.VARS
    TXT_LON, TXT_LAT = 80.0, 46.0        # where the numbers are written on the maps (central Asia, on land)
    VMIN, VMAX = -27, -13                # colour range of the decline map
    YMIN, YMAX = 2010, 2050              # colour range of the crossing-year map (arrows: earlier / later)

    # ---------------------------------------------------------------- NOAA CO2 and the years
    co2_file = f'{ROOT}/co2_annmean_gl.txt'
    if not os.path.exists(co2_file):
        urllib.request.urlretrieve('https://gml.noaa.gov/webdata/ccgg/trends/co2/co2_annmean_gl.txt', co2_file)
    co2 = np.array([[float(v) for v in l.split()[:2]] for l in open(co2_file) if l.strip() and not l.startswith('#')])
    cy, cx = co2[:, 0].astype(int), co2[:, 1]
    xco2 = dict(zip(cy, cx))
    import projection as _pj
    growth = _pj.co2_growth()[0]             # constant emissions at the 2024 level: 2.6 ppm/yr
    y_last = rc.EVAL_YEAR                     # anchor at 2024 (the NOAA file may go further)
    x_last = float(cx[cy == y_last][0])
    year_of = lambda x: y_last + (x - x_last) / growth
    x_of = lambda y: x_last + (y - y_last) * growth

    with open(f'{ROOT}/constraint_results.pkl', 'rb') as f:
        RES = pickle.load(f)['results_common']
    refs = list(RES)
    D = rc.load()                        # pi_full, now, om, year, ok, area, land on lon -180..180

    # ---------------------------------------------------------------- per-cell projection (cached)
    import projection
    importlib.reload(projection)
    proj = projection.regional_projection()        # cached in the run folder (projection.py)

    # ---------------------------------------------------------------- crossing year per cell, per product
    x2024 = xco2[2024]
    years_obs = np.arange(1985, 2025)
    per = {}
    for q in refs:
        d, p = D[q], proj[q]
        pi, now, om, yo = d['pi_full'], d['now'], d['om'], np.asarray(d['year'])
        bnd = 0.8 * pi
        # level-adjust the projected curve so that it passes through the fitted 2024 value of each cell
        i2024 = np.interp(x2024, p['x'], np.arange(p['x'].size))
        c2024 = p['curves'][int(np.floor(i2024))] * (1 - i2024 % 1) + p['curves'][int(np.ceil(i2024))] * (i2024 % 1)
        curves = p['curves'] + (now - c2024)[None]
        valid = np.isfinite(pi) & np.isfinite(now) & np.all(np.isfinite(curves), axis=0)
        # already below the boundary in 2024: year from the observed record (per-cell quadratic fit)
        keep = yo <= rc.LAST_YEAR
        flat = om[keep].reshape(keep.sum(), -1)
        idx = np.where(valid.ravel())[0]
        cf = np.polyfit(yo[keep] - 2000.0, flat[:, idx], 2)                                   # (3, ncell)
        fit = np.array([cf[0] * (y - 2000.0) ** 2 + cf[1] * (y - 2000.0) + cf[2] for y in years_obs])   # (nyear, ncell)
        below = fit <= bnd.ravel()[idx][None]
        first_obs = np.where(below.any(0), years_obs[np.argmax(below, axis=0)], np.nan)
        # forced share of the area beyond the boundary in the observed years (per-cell fit, from the first year of the
        # product): ends in 2024 where the projection starts
        w_fit = d['area'].ravel()[idx]
        frac_fit = 100 * (below * w_fit[None]).sum(1) / w_fit.sum()
        own = years_obs >= yo[keep][0]
        # not yet: year from the projection (first x on the grid at or below the boundary)
        cz = curves.reshape(DX.size, -1)[:, idx]
        belowp = cz <= bnd.ravel()[idx][None]
        xcross = np.where(belowp.any(0), p['x'][np.argmax(belowp, axis=0)], np.nan)
        yr = np.where(now.ravel()[idx] <= bnd.ravel()[idx], first_obs, year_of(xcross))
        ycross = np.full(pi.shape, np.nan)
        ycross.ravel()[idx] = yr
        # share of the area beyond the boundary: observed annual values and the projection
        area = d['area']
        frac_obs = rc.area_beyond_series(d, 'pi_full')              # per observed year
        yrs_proj = np.arange(STATE[1], YEAR_END + 1)
        frac_proj = []
        gm_proj = []
        wv = area[valid]
        for y in yrs_proj:
            i = np.interp(x_of(y), p['x'], np.arange(p['x'].size))
            cy_ = curves[int(np.floor(i))] * (1 - i % 1) + curves[int(np.ceil(i))] * (i % 1)
            frac_proj.append(100 * wv[cy_[valid] <= bnd[valid]].sum() / wv.sum())
            gm_proj.append(np.sum(wv * cy_[valid]) / wv.sum())
        gm_proj = np.array(gm_proj)
        yc, sc = RES[q]['constrained'][LAB][:2]
        S = RES[q]['states']
        s_pi = sc                                                    # constraint only (CO2 of 1750 kept separate, as Fig. 1)
        gcross = np.interp(0.8 * yc, gm_proj[::-1], yrs_proj[::-1])  # global crossing year of this product
        s_om = SDEC_SHARED / 100 * yc                                # full shared 1 sigma of the decline (Extended Data Table 2)
        g_early = np.interp(0.8 * yc + s_om, gm_proj[::-1], yrs_proj[::-1])
        g_late = np.interp(0.8 * yc - s_om, gm_proj[::-1], yrs_proj[::-1])
        frac_proj = np.array(frac_proj)          # from the forced 2024 state (per-cell fit), as the global numbers
        per[q] = dict(ycross=ycross, frac_obs=(yo[keep], frac_obs[keep]), frac_proj=(yrs_proj, frac_proj),
                      frac_fit=(years_obs[own], frac_fit[own]),
                      gcross=gcross, g_early=g_early, g_late=g_late, chg=d['chg_full'], valid=valid)
        print(f'{q:10s} global crossing {gcross:.1f} ({g_early:.1f}-{g_late:.1f}); area beyond -20 %: 2024 {frac_proj[0]:.1f} %, '
              f'{gcross:.0f} {np.interp(gcross, yrs_proj, frac_proj):.0f} %, {YEAR_END} {frac_proj[-1]:.0f} %')

    # ---------------------------------------------------------------- mean of the products
    lat, lon = D[refs[0]]['lat'], D[refs[0]]['lon']
    for q in refs[1:]:
        assert np.allclose(D[q]['lat'], lat) and np.allclose(D[q]['lon'], lon), 'products on different grids'
    chg_m = np.nanmean([per[q]['chg'] for q in refs], axis=0)
    ycross_m = np.nanmean([per[q]['ycross'] for q in refs], axis=0)
    area = D[refs[0]]['area']
    land = D[refs[0]]['land']
    yrs_proj = per[refs[0]]['frac_proj'][0]
    frac_proj_m = np.mean([per[q]['frac_proj'][1] for q in refs], axis=0)
    gcross_m = np.mean([per[q]['gcross'] for q in refs])
    g_early_m = np.mean([per[q]['g_early'] for q in refs])
    g_late_m = np.mean([per[q]['g_late'] for q in refs])
    v = np.isfinite(chg_m)
    # share of the area beyond the boundary in 2024: forced state (per-cell quadratic fit, the start of the projection),
    # per product first, then the mean; the annual 2024 values (end of the observed series in d) are printed
    beyond_2024 = np.mean([per[q]['frac_proj'][1][0] for q in refs])
    beyond_2024_annual = np.mean([per[q]['frac_obs'][1][-1] for q in refs])
    print(f'area beyond the boundary in 2024: forced {beyond_2024:.1f} %, annual values {beyond_2024_annual:.1f} %')
    print(f'mean of the products: global crossing {gcross_m:.1f} ({g_early_m:.1f}-{g_late_m:.1f}; products {min(per[q]["gcross"] for q in refs):.1f}-'
          f'{max(per[q]["gcross"] for q in refs):.1f}); area beyond -20 %: 2024 {beyond_2024:.1f} %, '
          f'{gcross_m:.0f} {np.interp(gcross_m, yrs_proj, frac_proj_m):.0f} %, {YEAR_END} {frac_proj_m[-1]:.0f} %; '
          f'crossing-year map: median {np.nanmedian(ycross_m):.0f}, not by {YEAR_END} in '
          f'{100 * area[v & ~(ycross_m <= YEAR_END)].sum() / area[v].sum():.0f} % of the area')

    return locals()


def _best(R):
    # The best estimate per cell (Fig. 4b-d), as the global mean: every product on its own area (constraint_results
    # 'results'), the mean of the products that cover a cell, with the alkalinity adjustment of the data product
    # (carbonate_system_<ref>_alkalinity.nc: the change of Omega_arag of the 1750 state, of every year and of the 2024
    # trend; the pattern of the four ESMs with Omega-dependent calcification, scaled to the observed rise). Per product
    # and cell: the forced series (per-cell quadratic fit of the product years, evaluated from 1980, the first product
    # year, for the products that start later a few years before their record) to 2024, then the per-cell projection from the forced 2024 state of the product's own area
    # (projection.regional_projection('results'), level-adjusted to the fitted 2024 value), the alkalinity change held at
    # its 2024 value after 2024 (as for the global mean). A product enters a cell only with its 1750 state, its whole
    # trajectory and its alkalinity change. Means over the products of the 1750 state and of the trajectory; decline,
    # crossing year (first year at or below 80 % of the 1750 state) and the share of the covered area beyond the boundary
    # all from these means. Cells that no product covers are left blank (the global mean fills them with the mean of
    # their region, structural_adjustments.py).
    import netCDF4
    import importlib
    import numpy as np
    import oa_config
    import projection
    import regional_common as rc
    importlib.reload(projection)
    D = rc.load(which='results')                       # each product on its own area
    proj = projection.regional_projection('results')   # cached (_own)
    refs = list(D)
    lat, lon = D[refs[0]]['lat'], D[refs[0]]['lon']
    assert np.allclose(lat, R['lat']) and np.allclose(lon, R['lon']), 'own-area and common-area maps on different grids'
    xco2, x_of, LY = R['xco2'], R['x_of'], rc.LAST_YEAR
    xs = proj[refs[0]]['x']
    years = np.arange(1980, int(np.floor(R['year_of'](xs[-1]))) + 1)  # to the end of the projection (crossing years)
    yh, yp = years[years <= LY], years[years > LY]
    iE = int(np.where(years == R['YEAR_END'])[0][0])
    area = D[refs[0]]['area']
    PI, TR, PI0, TR0, ANN = [], [], [], [], []
    FRQ = {}                                            # coloured lines of d: each product alone on its own area
    years_ann = np.arange(ANN_START, LY + 1)           # thin line of d: annual values
    PI272 = []                                          # 1750 state at the effective CO2 of Jiang et al. (272.4 ppm)
    for q in refs:
        d, p = D[q], proj[q]
        assert np.allclose(d['lat'], lat) and np.allclose(d['lon'], lon), 'products on different grids'
        assert np.allclose(p['x'], xs), 'projections on different CO2 grids'
        with netCDF4.Dataset(f'{rc.ROOT}/product/carbonate_system_{q}_alkalinity.nc') as f:
            flat, flon = np.asarray(f['lat'][:], 'f8'), np.asarray(f['lon'][:], 'f8')
            fy = np.asarray(f['year'][:]).astype(int)
            names = [str(s) for s in np.asarray(f['state'][:]).ravel()]
            g = lambda v, i=Ellipsis: np.ma.filled(f[v][i].astype('f8'), np.nan)
            a50 = g('omega_arag_state_alk', names.index('1750_AR6'))
            a_all = g('omega_arag_alk')                                       # (year, lat, lon)
            a24 = g(f'omega_arag_{rc.EVAL_YEAR}_trend_alk')
            a272 = g('omega_arag_state_alk', names.index('CO2_of_Jiang_et_al_2023'))
        with netCDF4.Dataset(f'{rc.ROOT}/product/carbonate_system_{q}.nc') as f2:   # the named states of the product
            n2 = [str(s) for s in np.asarray(f2['state'][:]).ravel()]
            s2 = lambda nm_: np.ma.filled(f2['omega_arag_state'][n2.index(nm_)].astype('f8'), np.nan)
            r272 = s2('CO2_of_Jiang_et_al_2023') / s2('1750_AR6')     # per cell: 272.4 ppm state relative to 1750
            assert np.allclose(np.asarray(f2['lat'][:], 'f8'), lat), f'{q}: product file on another latitude grid'
            r272 = rc.to_180(np.asarray(f2['lon'][:], 'f8'), r272)[1]
        assert np.allclose(flat, lat), f'{q}: alkalinity file on another latitude grid'
        l180, a50 = rc.to_180(flon, a50)
        assert np.allclose(l180, lon), f'{q}: alkalinity file on another longitude grid'
        a_all, a24, a272 = rc.to_180(flon, a_all)[1], rc.to_180(flon, a24)[1], rc.to_180(flon, a272)[1]
        a_year = lambda y: a24 if y >= rc.EVAL_YEAR else a_all[int(np.where(fy == y)[0][0])]
        pi, now, om, yo = d['pi_full'], d['now'], d['om'], np.asarray(d['year'])
        keep = yo <= LY
        base = np.isfinite(pi) & np.isfinite(now) & d['ok']
        idx = np.where(base.ravel())[0]
        cf = np.polyfit(yo[keep] - 2000.0, om[keep].reshape(keep.sum(), -1)[:, idx], 2)
        fit = np.full((yh.size,) + pi.shape, np.nan)
        fit.reshape(yh.size, -1)[:, idx] = np.array([np.polyval(cf, y - 2000.0) for y in yh])
        i24 = np.interp(xco2[rc.EVAL_YEAR], p['x'], np.arange(p['x'].size))
        lerp = lambda C, i: C[int(np.floor(i))] * (1 - i % 1) + C[int(np.ceil(i))] * (i % 1)
        curves = p['curves'] + (now - lerp(p['curves'], i24))[None]       # through the fitted 2024 value
        prj = np.array([lerp(curves, np.interp(x_of(y), p['x'], np.arange(p['x'].size))) for y in yp])
        tr0 = np.concatenate([fit, prj])                                   # (year, lat, lon), without alkalinity
        alk = np.array([a_year(min(y, rc.EVAL_YEAR)) for y in years])     # held at 2024 after 2024
        v = (base & np.all(np.isfinite(tr0), 0) & np.isfinite(a50) & np.all(np.isfinite(alk), 0))
        lost = base & np.all(np.isfinite(tr0), 0) & ~v
        print(f'  {q:10s} own area {100 * area[base].sum() / area[D[q]["ok"]].sum():.1f} % with a trajectory; '
              f'{int(lost.sum())} cells without an alkalinity change left out')
        PI.append(np.where(v, pi + a50, np.nan)); PI0.append(np.where(v, pi, np.nan))
        PI272.append(np.where(v, pi * r272 + a272, np.nan))
        TR.append(np.where(v[None], tr0 + alk, np.nan).astype('f4')); TR0.append(np.where(v[None], tr0, np.nan).astype('f4'))
        Aq = area * v                                                      # share of the product's own covered area
        FRQ[q] = 100 * ((TR[-1] <= 0.8 * PI[-1][None]) & v[None]).reshape(years.size, -1) @ Aq.ravel() / Aq.sum()
        print(f'    {q}: beyond the boundary in {LY} {FRQ[q][years == LY][0]:.1f} % of its own covered area')
        del tr0, alk, prj, fit, curves
        # thin line of d: the product's annual values with the alkalinity change of each year, on the same cells (v)
        assert np.isin(years_ann, yo).all(), f'{q}: no annual values in some years from {ANN_START}'
        ka = [int(np.where(yo == y)[0][0]) for y in years_ann]
        alk_a = np.array([a_year(int(y)) for y in years_ann])
        ANN.append(np.where(v[None], om[ka] + alk_a, np.nan).astype('f4'))
        del alk_a
    with np.errstate(invalid='ignore'):
        P, T, P0, T0 = (np.nanmean(np.array(A), axis=0) for A in (PI, TR, PI0, TR0))
    del TR, TR0
    n_prod = np.sum([np.isfinite(A) for A in PI], axis=0)
    cover = np.isfinite(P) & np.all(np.isfinite(T), 0)
    i24 = int(np.where(years == rc.EVAL_YEAR)[0][0])
    chg = np.where(cover, 100 * (T[i24] / P - 1), np.nan)                # change since 1750 (negative)
    chg0 = np.where(cover, 100 * (T0[i24] / P0 - 1), np.nan)
    below = T <= 0.8 * P[None]
    ycross = np.where(cover & below.any(0), years[np.argmax(below, axis=0)], np.nan)
    ycross[cover & ~below.any(0)] = np.inf                                 # not by the end of the projection
    A = area * cover
    frac = 100 * (below & cover[None]).reshape(years.size, -1) @ A.ravel() / A.sum()
    frac0 = 100 * ((T0 <= 0.8 * P0[None]) & cover[None]).reshape(years.size, -1) @ A.ravel() / A.sum()
    frac10 = 100 * ((T <= 0.9 * P[None]) & cover[None]).reshape(years.size, -1) @ A.ravel() / A.sum()   # Findlay et al.
    # the same with the annual values: per year and cell the mean of the products that cover the cell (the same
    # products as P), relative to the same 1750 state (P) on the same area
    with np.errstate(invalid='ignore'):
        M_ann = np.nanmean(np.array(ANN), axis=0)
    del ANN
    assert np.all(np.isfinite(M_ann[:, cover])), 'annual values missing in covered cells'
    frac_ann = 100 * ((M_ann <= 0.8 * P[None]) & cover[None]).reshape(years_ann.size, -1) @ A.ravel() / A.sum()
    del M_ann
    with np.errstate(invalid='ignore'):
        P272 = np.nanmean(np.array(PI272), axis=0)
    c272 = cover & np.isfinite(P272)
    frac272 = 100 * area[c272 & (T[i24] <= 0.8 * P272)].sum() / area[c272].sum()   # share relative to 272.4 ppm
    first_share = lambda f_, lev: float(years[np.argmax(f_ >= lev)]) if np.any(f_ >= lev) else np.nan
    ocean = rc.ocean_on(lat, lon).astype(bool)
    w = oa_config.area_weights(lat, lon)
    cover_pct = 100 * (w * cover).sum() / (w * ocean).sum()
    gm = lambda F: np.sum(F[cover] * area[cover]) / area[cover].sum()
    dec_gm = 100 * (1 - np.array([gm(T[k]) for k in range(years.size)]) / gm(P))
    dec_gm0 = 100 * (1 - np.array([gm(T0[k]) for k in range(years.size)]) / gm(P0))
    a_t = dec_gm0 - dec_gm                    # effect of the alkalinity adjustment on the area-mean decline (points, > 0)
    common = R['D'][R['refs'][0]]['ok']       # common area
    # numbers for the text
    wpct = lambda x, ww, qs: [float(np.interp(q_ / 100, np.cumsum(ww[np.argsort(x)]) / ww.sum(), np.sort(x))) for q_ in qs]
    fin = cover & np.isfinite(ycross)
    yc_fin = np.where(np.isinf(ycross), 1e4, ycross)
    ys_p = wpct(yc_fin[cover], area[cover], (5, 10, 50, 90, 95))
    half = float(np.interp(50.0, frac, years)) if frac.max() >= 50 else np.nan
    la2 = np.broadcast_to(lat[:, None], cover.shape)
    bands = (('south of 60S', la2 < -60), ('60-45S', (la2 >= -60) & (la2 < -45)), ('45-15S', (la2 >= -45) & (la2 < -15)),
             ('15S-15N', np.abs(la2) <= 15), ('15-45N', (la2 > 15) & (la2 <= 45)), ('45-60N', (la2 > 45) & (la2 <= 60)),
             ('north of 60N', la2 > 60))
    print(f'best estimate per cell: {cover_pct:.1f} % of the ocean covered ({oa_config.WEIGHTS_TEXT}); area-mean decline '
          f'1750-{rc.EVAL_YEAR} {dec_gm[i24]:.2f} % (without the alkalinity adjustment {dec_gm0[i24]:.2f} %; the global '
          f'mean of the text, with the uncovered cells filled: {GLOBAL["decline"]:.2f} %)')
    print(f'  beyond the boundary in 2024: {frac[i24]:.1f} % of the covered area (without alkalinity {frac0[i24]:.1f} %); half '
          f'of the area by {half:.1f}; {R["YEAR_END"]}: {frac[iE]:.1f} % ({100 - frac[iE]:.1f} % still within 20 %)')
    f_on = frac[np.isin(years, years_ann)]
    print(f'  beyond the boundary, annual values: {rc.EVAL_YEAR} {frac_ann[-1]:.1f} % (forced {frac[i24]:.1f} %); annual '
          f'minus forced {ANN_START}-{rc.EVAL_YEAR}: mean {np.mean(frac_ann - f_on):+.1f}, s.d. '
          f'{np.std(frac_ann - f_on, ddof=1):.1f}, range {np.min(frac_ann - f_on):+.1f} to {np.max(frac_ann - f_on):+.1f} points')
    print(f'  beyond the boundary in 2024 relative to a 1750 state at the effective CO2 of Jiang et al. (272.4 ppm): '
          f'{frac272:.1f} % of the covered area (with alkalinity; {100 * area[c272].sum() / area[cover].sum():.1f} % of it '
          f'with a 272.4 ppm state)')
    print(f'  beyond a 10 % decline (Findlay et al.): 1980 {frac10[0]:.0f} %, 1985 {frac10[years == 1985][0]:.0f} %, 2000 '
          f'{frac10[years == 2000][0]:.0f} %; half of the area by {first_share(frac10, 50.0):.0f}, 95 % by '
          f'{first_share(frac10, 95.0):.0f}, 99 % by {first_share(frac10, 99.0):.0f} (1980: the first year, before it not computed)')
    print(f'  crossing year of the cells, area percentiles 5/10/50/90/95: ' + ' / '.join(
        ('after ' + str(years[-1])) if y_ > 9000 else f'{y_:.0f}' for y_ in ys_p) + ' (before 1980 counted as 1980)')
    p_ = wpct(-chg[cover], area[cover], (5, 25, 50, 75, 95))
    print('  decline 1750-2024 in the cells, area percentiles 5/25/50/75/95: ' + ' / '.join(f'{x_:.1f}' for x_ in p_) + ' %')
    print('  decline by latitude band (area mean of the cells, %): ' + ', '.join(
        f'{lab} {np.sum(-chg[cover & m] * area[cover & m]) / area[cover & m].sum():.1f}' for lab, m in bands if (cover & m).any()))
    print(f'  alkalinity effect on the area-mean decline (points): 1980 {a_t[0]:.3f}, 2000 {a_t[years == 2000][0]:.3f}, '
          f'{rc.EVAL_YEAR} {a_t[i24]:.3f}')
    ycross_map = np.where(np.isinf(ycross), np.nan, ycross)                # not crossed by the end: blank
    return dict(lat=lat, lon=lon, years=years, chg=chg, chg0=chg0, ycross=ycross_map, frac=frac, frac0=frac0, P=P,
                cover=cover, cover_pct=cover_pct, common=common, n_prod=n_prod, years_ann=years_ann, frac_ann=frac_ann,
                refs=refs, a_t=a_t, frac_prod=FRQ,
                dec_gm=dec_gm, half=half, i24=i24, iE=iE, not_crossed=cover & np.isinf(ycross), frac10=frac10,
                half10=first_share(frac10, 50.0), share95_10=first_share(frac10, 95.0))


import oa_config                      # the area weights of the run (OA_WEIGHTS)
import sys
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.gridspec import GridSpec, GridSpecFromSubplotSpec
from matplotlib.patches import Patch
from matplotlib.lines import Line2D
from matplotlib.legend_handler import HandlerTuple

sys.path.insert(0, CODE_DIR)
import regional_common as rc


G = _global()                        # global: per-product series, projection, crossing year
R = _regional()                      # common area: per-product maps, area series (for the comparison printed below)
B = _best(R)                         # best estimate per cell: own areas, mean of the products, alkalinity adjustment

TOP_W = 0.50                         # width of a and d as a fraction of the figure (maps: full width)
FS_MAP = 8.0                         # labels on the maps
X0_A = 1950                          # first year of a
MEAN_COL = G['MEAN_COL']
BEST_COL = 'k'                       # the best estimate (global mean, whole ocean, both adjustments)
for n_ in ('sdec_m', 'decline_crossing', 'GLOBAL', 'SDEC_G', 'XCO2'):
    if n_ not in globals():
        raise NameError(f'{n_} not defined: run the preparation cell first')

# ---------------------------------------------------------------- a: data, decline relative to 1750 (%)
P, refs_g, LY = G['P'], G['refs'], rc.LAST_YEAR
yrs_m = G['P_mean']['yrs']
esm = 100 * (G['P_mean']['A'] / G['P_mean']['pi'][:, None] - 1)          # each ESM relative to its own 1750 (model, year)
esm_m, esm_s = esm.mean(0), esm.std(0, ddof=1)
dec_obs = {q: 100 * (P[q]['ob'] / P[q]['yc'] - 1) for q in refs_g}        # annual means, constrained 1750 (common area)
# forced decline of the mean of the products on the common area (preparation cell)
DC10, DC20 = decline_crossing(10.0), decline_crossing(20.0)
t0_m = max(P[q]['yo'][0] for q in refs_g)                                   # first year covered by all four
t_c, d_c = DC20['curve']                                                    # from 1900 (fit) and the projection
keep = (t_c >= t0_m) & (t_c <= G['YEAR_END'])
t_all, d_all = t_c[keep], -d_c[keep]                                        # negative: change since 1750 (as the map in b)
# the best estimate: D_g(t) = D_c(t) (1 + r) + a(t), r the whole-ocean shift relative to the decline, a(t) the alkalinity
# shift on the whole ocean: its 2024 value GLOBAL['a'] (structural_adjustments.py), before 2024 with the time course of
# the alkalinity effect on the decline of the best-estimate maps (B['a_t'], the observed rise of the product years and the
# modelled one before), after 2024 held at its 2024 value (as for the crossing, GLOBAL['cross_year'])
a_shape = lambda t_: np.interp(t_, B['years'], B['a_t'] / B['a_t'][B['i24']], left=np.nan)
a_t = lambda t_: GLOBAL['a'] * np.where(np.asarray(t_) >= rc.EVAL_YEAR, 1.0, a_shape(t_))
t_b = t_c[(t_c >= B['years'][0]) & (t_c <= G['YEAR_END'])]
d_b = -d_c[(t_c >= B['years'][0]) & (t_c <= G['YEAR_END'])] * (1 + GLOBAL['r']) - a_t(t_b)   # negative: change
yg, dyg = GLOBAL['cross_year'], GLOBAL['cross_dyear']
# 10 % decline of the best estimate (Findlay et al.): the same curve, range from the shared 1 sigma of the global mean and
# the spread of the products' own 10 % years (as decline_crossing)
crb = lambda lev: float(np.interp(lev, -d_b, t_b))                           # -d_b increases with time
s_g = GLOBAL['sigma_shared']
y10g = crb(10.0)
dy10g = float(np.hypot(np.std([v_['t'] for v_ in DC10['per'].values()], ddof=1), (crb(10.0 + s_g) - crb(10.0 - s_g)) / 2))
yrs_rec = np.array(sorted(XCO2))
x10g = float(np.interp(y10g, yrs_rec, [XCO2[y_] for y_ in yrs_rec]))
GLOBAL10 = dict(year=y10g, dyear=dy10g, ppm=x10g)
print(f'a: best estimate: 20 % in {yg:.1f} ({yg - dyg:.1f}-{yg + dyg:.1f}), {GLOBAL["cross_ppm"]:.0f} ± {GLOBAL["cross_dppm"]:.0f} ppm; '
      f'2024: {-d_b[t_b <= LY][-1]:.2f} % (GLOBAL {GLOBAL["decline"]:.2f} %); 10 % in {y10g:.1f} ({y10g - dy10g:.1f}-{y10g + dy10g:.1f}) '
      f'at {x10g:.0f} ppm; alkalinity shift of the decline 1990 {float(a_t(1990.0)):+.3f}, 2024 {float(a_t(2024.0)):+.3f} points')
print(f'a: common area (for comparison): 20 % in {DC20["t"]:.1f} ± {DC20["dt"]:.1f}, 10 % in {DC10["t"]:.1f} ± {DC10["dt"]:.1f}')

# ---------------------------------------------------------------- figure: a (top, centred), b and c (maps), d (bottom, centred)
fig = plt.figure(figsize=(12, 10.6))
gs = GridSpec(6, 1, figure=fig, height_ratios=[0.62, 0.13, 1.0, 0.04, 0.26, 0.62], hspace=0.0,
              left=0.04, right=0.98, top=0.975, bottom=0.05)
side = [(1 - TOP_W) / 2, TOP_W, (1 - TOP_W) / 2]

ax = fig.add_subplot(GridSpecFromSubplotSpec(1, 3, subplot_spec=gs[0], width_ratios=side, wspace=0)[1])
ax.fill_between(yrs_m, esm_m - esm_s, esm_m + esm_s, color='0.88', lw=0, zorder=1)
ax.plot(yrs_m, esm_m, color='0.55', lw=0.9, zorder=2)
# the best estimate: global mean over the whole ocean with both adjustments, with its 1 sigma (Extended Data Table 2);
# the products and their mean drawn on top of it (thinner), so that both stay visible
ax.fill_between(t_b, d_b - SDEC_G, d_b + SDEC_G, color='0.5', alpha=0.3, lw=0, zorder=4)
ax.plot(t_b[t_b <= LY], d_b[t_b <= LY], color=BEST_COL, lw=2.8, zorder=5)
ax.plot(t_b[t_b >= LY], d_b[t_b >= LY], color=BEST_COL, lw=2.8, ls=(0, (3, 2)), zorder=5)
for q in refs_g:
    ax.plot(P[q]['yo'], dec_obs[q], color=P[q]['col'], lw=1.2, zorder=6)                     # the products, common area
ax.plot(t_all[t_all <= LY], d_all[t_all <= LY], color=MEAN_COL, lw=1.2, zorder=6)
# (no projection of the products' mean: it lies on the projection of the best estimate)
# 10 %: thin dotted line, reached within the observed record
ax.axhline(-10, color='0.35', lw=0.8, ls=(0, (1, 2)), zorder=3)
ax.errorbar(y10g, -10, xerr=dy10g, fmt='o', ms=5, mfc='white', mec='k', mew=1.0, ecolor='k', elinewidth=1.1, capsize=2.5, zorder=7)
ax.annotate(f'10 % decline\n≈ {y10g:.0f} ({y10g - dy10g:.0f}–{y10g + dy10g:.0f})', xy=(y10g, -10), xytext=(y10g - 3, -4.0),
            ha='right', va='center', fontsize=8, linespacing=1.2,
            arrowprops=dict(arrowstyle='-|>,head_length=0.25,head_width=0.12', color='0.4', lw=0.6, shrinkA=2, shrinkB=5))
ax.axhline(-20, color='k', lw=1.0, ls=(0, (4, 3)), zorder=3)
ax.text(X0_A + 1, -19.7, 'boundary: 20 % decline', ha='left', va='bottom', fontsize=8.5)
ax.errorbar(yg, -20, xerr=dyg, fmt='o', ms=6, mfc='white', mec='k', mew=1.2, ecolor='k', elinewidth=1.4, capsize=3, zorder=7)
ax.annotate(f'boundary crossed\n≈ {yg:.0f} ({yg - dyg:.0f}–{yg + dyg:.0f})', xy=(yg, -20),
            xytext=(yg - 4, -22.9), ha='right', va='center', fontsize=8.5, linespacing=1.2,
            arrowprops=dict(arrowstyle='-|>,head_length=0.25,head_width=0.12', color='0.4', lw=0.6, shrinkA=2, shrinkB=5))
ax.set_xlim(X0_A, G['YEAR_END'])
ax.set_ylim(-25, 0.5)
ax.set_ylabel('Change of global mean\n$\\Omega_{\\mathrm{arag}}$ since 1750 (%)', fontsize=10)
ax.set_xlabel('Year', fontsize=10)
ax.spines[['top', 'right']].set_visible(False)
# two legends: the best estimate (whole ocean, both adjustments) and, below it, the fCO2-products on their common area
# without the two adjustments (they nearly cancel: +0.20 for the whole ocean, -0.25 for the alkalinity rise)
# grouped by area: the whole ocean (the best estimate) and the common area (ESMs and products)
COMMON_PCT_A = rc.pct_ocean(np.asarray(B['common'], bool), np.asarray(B['lat'], 'f8'), np.asarray(B['lon'], 'f8'))
hh = [(Patch(facecolor='0.5', alpha=0.3, lw=0), Line2D([], [], color=BEST_COL, lw=2.8)),
      Line2D([], [], color=BEST_COL, lw=2.8, ls=(0, (3, 2)))]
ll = [f'best estimate, forced (± {SDEC_G:.1f} %, 1$\\sigma$)', 'idealised projection']
leg1 = ax.legend(hh, ll, title='Whole ocean, with both adjustments', title_fontsize=7.5, fontsize=7.5, frameon=False,
                 loc='upper left', bbox_to_anchor=(1.02, 1.0), ncol=1, handler_map={tuple: HandlerTuple(ndivide=1, pad=0)})
leg1._legend_box.align = 'left'
ax.add_artist(leg1)
fig.canvas.draw()
y_leg2 = leg1.get_window_extent().transformed(ax.transAxes.inverted()).y0 - 0.05
leg2 = ax.legend([(Patch(facecolor='0.88', lw=0), Line2D([], [], color='0.55', lw=0.9))]
                 + [Line2D([], [], color=P[q]['col'], lw=1.2) for q in refs_g] + [Line2D([], [], color=MEAN_COL, lw=1.2)],
                 ['ESMs, mean ± 1 s.d. (each\nrelative to its own 1750)'] + [f'{rc.FULL[q]}, annual' for q in refs_g]
                 + ['mean of the four fCO$_2$-products, forced'],
                 title=f'Common area ({COMMON_PCT_A:.0f} % of the ocean),\nwithout the adjustments', title_fontsize=7.5,
                 fontsize=7.5, frameon=False, loc='upper left', bbox_to_anchor=(1.02, y_leg2),
                 handler_map={tuple: HandlerTuple(ndivide=1, pad=0)})
leg2._legend_box.align = 'left'
ax.text(-0.14, 1.0, 'a', transform=ax.transAxes, fontsize=16, fontweight='bold', va='bottom')

# ---------------------------------------------------------------- b, c: maps of the best estimate
mid = GridSpecFromSubplotSpec(1, 2, subplot_spec=gs[2], wspace=0.05)
cbr = GridSpecFromSubplotSpec(1, 2, subplot_spec=gs[3], wspace=0.05)
X, Y = np.asarray(B['lon'], 'f8'), np.asarray(B['lat'], 'f8')     # degrees; projected by rc.map_* (Robinson)
land = R['land']
iB24 = B['i24']


def base_map(ax):
    ax.set_xticklabels([])
    ax.set_yticklabels([])
    rc.map_mesh(ax, X, Y, np.where(land, 1.0, np.nan), cmap=plt.matplotlib.colors.ListedColormap(['0.85']), vmin=0, vmax=1,
                shading='auto')


# crossing year: ColorBrewer YlGnBu without its palest end (early: light yellow-green, late: dark blue)
CMAP_YEAR = plt.matplotlib.colors.ListedColormap(plt.get_cmap('YlGnBu')(np.linspace(0.15, 1.0, 256)), name='YlGnBu_from015')
maps = (
    (B['chg'], 'plasma', R['VMIN'], R['VMAX'], -20, '--',
     f"{B['frac'][iB24]:.0f}% of the area\ncrossed in 2024",
     'Change of surface $\\Omega_{\\mathrm{arag}}$ from 1750 to 2024 (%)'),
    (B['ycross'], CMAP_YEAR, R['YMIN'], R['YMAX'], 2024, '-',
     f"global mean crosses\nin {yg:.0f} ({yg - dyg:.0f}–{yg + dyg:.0f})",            # as a
     f"Year in which the boundary is crossed (CO$_2$ growth {R['growth']:.1f} ppm yr$^{{-1}}$)"))
for k, (field, cmap, v0, v1, lev, ls, txt, cblab) in enumerate(maps):
    ax = fig.add_subplot(mid[k])
    base_map(ax)
    pc = rc.map_mesh(ax, X, Y, field, cmap=cmap, vmin=v0, vmax=v1, shading='auto')
    rc.map_contour(ax, X, Y, field, levels=[lev], colors='k', linewidths=0.8, linestyles=ls)
    ax.text(*rc.map_xy(R['TXT_LON'], R['TXT_LAT']), txt, ha='center', va='center', fontsize=FS_MAP, fontweight='bold',
            linespacing=1.1)
    ax.text(0.0, 1.0, 'bc'[k], transform=ax.transAxes, fontsize=16, fontweight='bold', va='bottom')
    cax = fig.add_subplot(GridSpecFromSubplotSpec(1, 3, subplot_spec=cbr[k], width_ratios=[0.15, 1, 0.15])[1])
    fig.colorbar(pc, cax=cax, orientation='horizontal', extend='both').set_label(cblab, fontsize=10)

# ---------------------------------------------------------------- d: area beyond the boundary (best estimate)
ax = fig.add_subplot(GridSpecFromSubplotSpec(1, 3, subplot_spec=gs[5], width_ratios=side, wspace=0)[1])
# the best estimate (bold: forced to 2024, then the projection, dashed) and the same with the annual values of the
# products (thin, from ANN_START)
yb_ = B['years']
sel_o, sel_p = yb_ <= LY, (yb_ >= LY) & (yb_ <= R['YEAR_END'])
# only the best estimate, on the area of b and c (the products alone, on their own areas, are compared in
# Supplementary Fig. 9): forced state (bold; dashed: projection) and annual values (thin with dots)
ax.axhline(50, color='0.8', lw=0.6, ls=(0, (1, 2)), zorder=1)
ax.plot(B['years_ann'], B['frac_ann'], color='0.45', lw=0.8, marker='o', ms=2.2, mfc='0.45', mec='none', zorder=3)
ax.plot(yb_[sel_o], B['frac'][sel_o], color=BEST_COL, lw=2.0, zorder=4)
ax.plot(yb_[sel_p], B['frac'][sel_p], color=BEST_COL, lw=2.0, ls=(0, (4, 2)), zorder=4)
f24_ = float(B['frac'][iB24])
ax.plot(LY, f24_, 'o', ms=5.5, mfc='white', mec='k', mew=1.1, zorder=6)
ax.annotate(f'{f24_:.0f} % in {LY}', xy=(LY, f24_), xytext=(LY - 3.5, f24_ + 14), ha='right', va='center', fontsize=8.5,
            arrowprops=dict(arrowstyle='-|>,head_length=0.25,head_width=0.12', color='0.4', lw=0.6, shrinkA=2, shrinkB=4))
ax.plot(B['half'], 50, 'o', ms=5.5, mfc='white', mec='k', mew=1.1, zorder=6)
ax.annotate(f'half of the area\n≈ {B["half"]:.0f}', xy=(B['half'], 50), xytext=(B['half'] + 3, 34), ha='left', va='center',
            fontsize=8.5, linespacing=1.2,
            arrowprops=dict(arrowstyle='-|>,head_length=0.25,head_width=0.12', color='0.4', lw=0.6, shrinkA=2, shrinkB=4))
ax.axvline(LY, color='k', lw=0.8, ls=(0, (4, 3)))
ax.text(LY - 0.8, 97, 'observation-based', ha='right', va='center', fontsize=7.5, color='0.3')
ax.text(LY + 0.8, 97, 'projection', ha='left', va='center', fontsize=7.5, color='0.3')
for x0_, x1_ in ((LY - 0.8, LY - 9), (LY + 0.8, LY + 4.5)):
    ax.annotate('', xy=(x1_, 92), xytext=(x0_, 92),
                arrowprops=dict(arrowstyle='-|>,head_length=0.4,head_width=0.2', color='0.3', lw=0.8))
ax.set_xlim(1985, R['YEAR_END'])
ax.set_ylim(0, 100)
ax.set_ylabel(f"Area beyond the boundary\n(% of the {B['cover_pct']:.0f} % covered)", fontsize=9)
ax.set_xlabel('Year', fontsize=10)
ax.spines[['top', 'right']].set_visible(False)
ax.legend([Line2D([], [], color=BEST_COL, lw=2.0), Line2D([], [], color=BEST_COL, lw=2.0, ls=(0, (4, 2))),
           Line2D([], [], color='0.45', lw=0.8, marker='o', ms=2.2, mfc='0.45', mec='none')],
          ['forced state (quadratic fit per cell)', 'projection', 'annual values'],
          fontsize=7.5, frameon=False, loc='upper left', bbox_to_anchor=(0.0, 0.88))
ax.text(-0.14, 1.0, 'd', transform=ax.transAxes, fontsize=16, fontweight='bold', va='bottom')

iBY = lambda y_: int(np.where(B['years'] == y_)[0][0])
print(f"best estimate per cell: beyond the boundary 2024 {B['frac'][iB24]:.1f} % (without alkalinity {B['frac0'][iB24]:.1f} %), "
      f"at the global crossing ({yg:.1f}) {np.interp(yg, B['years'], B['frac']):.0f} %, 2030 {B['frac'][iBY(2030)]:.0f} %, "
      f"{R['YEAR_END']} {B['frac'][B['iE']]:.0f} %; half of the area by {B['half']:.1f}; not crossed by {B['years'][-1]}: "
      f"{int(B['not_crossed'].sum())} cells")
print(f"common area, without alkalinity: beyond the boundary 2024 {R['beyond_2024']:.1f} %, "
      f"in 2030 {np.interp(2030.0, R['yrs_proj'], R['frac_proj_m']):.0f} %")
rc.savefig(fig, 'crossing')


### Numbers for the text: 10 % and 20 % of decline (needs the preparation cell)

The years and CO2 at which the global mean decline reaches 10 % and 20 %, from `decline_crossing` (the same numbers as Fig. 4a and Extended Data Fig. 6).

In [ ]:
# Numbers for the text: when were 10 % and 20 % (the boundary) of decline reached, in years and ppm?
# Needs the preparation cell (same kernel): decline_crossing, the same function that draws them in Fig. 4a.
# Per product on the common area: forced decline relative to the constrained 1750 value (quadratic fit to the annual
# means up to 2024, before the first year of a product marked as extrapolated; after 2024 the anchored projection with
# CO2 rising at projection.co2_growth). Mean of the products: the mean curve; range: the full 1 sigma of the decline
# shared by the products (SDEC_SHARED, Extended Data Table 2) combined with the spread of the products' years.
def _numbers_for_text():
    import regional_common as rc
    import projection
    if 'decline_crossing' not in globals():
        raise NameError('decline_crossing not defined: run the preparation cell first (same kernel)')
    out = {}
    for level in (10.0, 20.0):
        d = decline_crossing(level)
        out[level] = d
        for q, v in d['per'].items():
            print(f'{rc.FULL[q]:22s} {level:.0f} %: {v["t"]:.1f}, {v["x"]:.1f} ppm'
                  + (' (before the record: extrapolated)' if v['extrapolated'] else ''))
        print(f'mean of the products: {level:.0f} % in {d["t"]:.1f} ± {d["dt"]:.1f} ({d["t"] - d["dt"]:.0f}-{d["t"] + d["dt"]:.0f}), '
              f'at {d["x"]:.1f} ± {d["dx"]:.1f} ppm (1 sigma of the decline {d["sig"]:.2f} percentage points)\n')
    print(f'CO2 growth after 2024: {projection.co2_growth()[0]:.1f} ppm/yr (constant emissions at the 2024 level)')
    return out


NUMBERS = _numbers_for_text()

### Numbers for the text: area beyond a 10 % decline (after Fig. 4)

Share of the surface ocean beyond a 10 % decline in selected years and the first year with 90, 95, 99, and 100 % of the area (annual values and per-cell fit), for the comparison with Findlay et al. (2025), and the same for 20 % as a check of Fig. 4d.

In [ ]:
# Notebook cell (after Fig. 4): how much of the surface ocean was beyond a 10 % decline, and when nearly all of it was
# (Findlay et al. 2025: "first crossed during the 1980s, with the entire surface ocean having passed this boundary by
# the 2000s"). Per product on the common area, each cell relative to its local constrained 1750 value (pi_full, as the
# maps of Fig. 4), in the observed years only:
#   annual   share of the area beyond -10 % from the annual values (rc.area_beyond_series, as Fig. 4d for -20 %)
#   fit      the same from the per-cell quadratic fit to the annual values (the forced state, as the crossing-year map)
# Printed: the share in selected years and the first year in which it reaches 90, 95, 99 and 100 %; then the mean of the
# products. The -20 % values are printed for comparison with Fig. 4d.
def _area_10():
    import numpy as np
    import regional_common as rc
    if 'R' not in globals():
        raise NameError('run the Fig. 4 cell first (same kernel)')
    D, refs = R['D'], R['refs']
    YEARS = (1985, 1990, 2000, 2010, 2024)
    SHARES = (90.0, 95.0, 99.0, 100.0)
    out = {}
    for thr in (-10.0, -20.0):
        res = {}
        for q in refs:
            d = D[q]
            yo = np.asarray(d['year'])
            keep = yo <= rc.LAST_YEAR
            v = np.isfinite(d['pi_full']) & d['ok']
            a = d['area']
            ann = rc.area_beyond_series(d, 'pi_full', thr=thr)[keep]
            flat = np.asarray(d['om'])[keep].reshape(keep.sum(), -1)[:, v.ravel()]
            ok = np.all(np.isfinite(flat), axis=0)
            cf = np.polyfit(yo[keep] - 2000.0, flat[:, ok], 2)
            fit = np.array([np.polyval(cf, y - 2000.0) for y in yo[keep]])          # (year, cell)
            pi, w = d['pi_full'][v][ok], a[v][ok]
            fsh = np.array([100 * w[100 * (f / pi - 1) < thr].sum() / w.sum() for f in fit])
            first = lambda s, lev: float(yo[keep][np.argmax(s >= lev - 1e-9)]) if np.any(s >= lev - 1e-9) else np.nan
            res[q] = dict(years=yo[keep], annual=ann, fit=fsh,
                          first_annual={lev: first(ann, lev) for lev in SHARES},
                          first_fit={lev: first(fsh, lev) for lev in SHARES})
            sel = lambda s: ', '.join(f'{y} {s[list(yo[keep]).index(y)]:.0f} %' for y in YEARS if y in list(yo[keep]))
            print(f'{rc.FULL[q]:22s} beyond {thr:.0f} %: annual {sel(ann)}; fit {sel(fsh)}')
            print(f'{"":22s}   first year with ' + '; '.join(
                f'{lev:.0f} %: {res[q]["first_annual"][lev]:.0f} (fit {res[q]["first_fit"][lev]:.0f})' for lev in SHARES))
        yc = sorted(set.intersection(*[set(r['years'].tolist()) for r in res.values()]))
        m = {k: np.mean([[r[k][list(r['years']).index(y)] for y in yc] for r in res.values()], axis=0) for k in ('annual', 'fit')}
        yc = np.array(yc)
        first_m = lambda s, lev: float(yc[np.argmax(s >= lev - 1e-9)]) if np.any(s >= lev - 1e-9) else np.nan
        print(f'mean of the products, beyond {thr:.0f} % ({yc[0]}-{yc[-1]}): annual '
              + ', '.join(f'{y} {m["annual"][list(yc).index(y)]:.0f} %' for y in YEARS if y in yc)
              + '; fit ' + ', '.join(f'{y} {m["fit"][list(yc).index(y)]:.0f} %' for y in YEARS if y in yc)
              + '; first year with ' + '; '.join(f'{lev:.0f} %: {first_m(m["annual"], lev):.0f} (fit {first_m(m["fit"], lev):.0f})'
                                                 for lev in SHARES) + '\n')
        out[thr] = dict(per=res, years=yc, mean=m)
    return out


AREA10 = _area_10()

### Diagnostic: the 10 % year from the ESMs (after Fig. 4)

Year of 10 % decline from the bias-adjusted ESMs, unconstrained and with the weights of the emergent constraint, against the fCO2-products (the text reports the fCO2-products only).

In [ ]:
# Notebook cell (after Fig. 4): the year of 10 % decline from the ESMs, unconstrained and constrained, against the products
# Per product (its bias-adjusted ESMs on the common area, G of the Fig. 4 cell), each model relative to its own 1750:
#   models         the year each model reaches 10 % (historical runs, to 2014); mean ± 1 sd over the models
#   constrained    the same years combined with the regression weights of the emergent constraint (prediction at the
#                  observed dSST and CUC, w @ years, test_state_constraint.weights), ± the regression sigma (residual
#                  with leverage and the observed dSST uncertainty)
#   products       the observation-based year of Fig. 4a (decline_crossing(10), per product)
# Check: the same weights applied to the models' 1750 values reproduce the constrained 1750 value of the product.
def _esm_10():
    import numpy as np
    import pickle
    import regional_common as rc
    import test_state_constraint as ts
    if 'G' not in globals() or 'DC10' not in globals():
        raise NameError('run the Fig. 4 cell first (same kernel)')
    with open(f'{rc.ROOT}/constraint_results.pkl', 'rb') as f:
        RC = pickle.load(f)['results_common']
    LEVEL = 10.0
    rows = []
    for q in G['refs']:
        p, r = G['P'][q], RC[q]
        names = list(r['names'])
        idx = [names.index(m) for m in p['names']]                     # the models with historical files
        dT, CUC = np.asarray(r['dT'], 'f8')[idx], np.asarray(r['CUC'], 'f8')[idx]
        dec = 100 * (1 - p['A'] / p['pi'][:, None])                     # (model, year)
        yrs = np.asarray(p['yrs'], 'f8')

        def year_of(d):
            i = int(np.argmax(d >= LEVEL))
            if d[i] < LEVEL or i == 0:
                return np.nan
            return yrs[i - 1] + (LEVEL - d[i - 1]) / (d[i] - d[i - 1])

        t = np.array([year_of(d) for d in dec])
        ok = np.isfinite(t)
        if not ok.all():
            print(f'  {q}: {np.sum(~ok)} models do not reach {LEVEL:.0f} % by {yrs[-1]:.0f} and are left out')
        w, lev, X, x0 = ts.weights([dT[ok], CUC[ok]], [float(r['dT_obs'][0]), float(r['CUC_obs'])])
        t_c = float(w @ t[ok])
        s_c = ts.sigma_of(t[ok], w, lev, X, x0, (float(r['dT_obs'][1]), 0.0))
        pi_check = float(w @ p['pi'][ok])
        rows.append((q, t[ok].mean(), t[ok].std(ddof=1), t_c, s_c, DC10['per'][q]['t']))
        print(f'{rc.FULL[q]:22s} models {t[ok].mean():.1f} ± {t[ok].std(ddof=1):.1f} ({ok.sum()} models); constrained '
              f'{t_c:.1f} ± {s_c:.1f}; products {DC10["per"][q]["t"]:.1f}'
              f'{" (extrapolated)" if DC10["per"][q]["extrapolated"] else ""}; check 1750: weights {pi_check:.3f}, '
              f'constraint {p["yc"]:.3f}')
    a = np.array([x[1:] for x in rows])
    print(f'mean of the products: models {a[:, 0].mean():.1f} ± {np.sqrt(np.mean(a[:, 1] ** 2)):.1f}; constrained '
          f'{a[:, 2].mean():.1f} ± {np.sqrt(np.mean(a[:, 3] ** 2)):.1f}; products {DC10["t"]:.1f} ± {DC10["dt"]:.1f}')
    return rows


ESM10 = _esm_10()

## Fig. 3: why estimates differ

In [ ]:
# Notebook cell: Figure 3, drawn from the variables of the preparation cell
# (description of the panels there). Restores those variables first (FIG4), as other cells reuse some of the names.
if 'FIG4' not in globals():
    raise NameError('FIG4 not defined: run the preparation cell first')
globals().update(FIG4)

# ---------------------------------------------------------------- figure
fig = plt.figure(figsize=(12, 11))
outer = GridSpec(2, 1, figure=fig, height_ratios=[1.0, 0.85], hspace=0.3, left=0.1, right=0.98, top=0.96, bottom=0.25)
top = GridSpecFromSubplotSpec(1, 2, subplot_spec=outer[0], width_ratios=[1.45, 1], wspace=0.2)

# a: annual global means on the own area (dashed) and the common area (solid), with the 1750 value and the boundary of
#    each product on both areas (columns; open = own area, filled = common area)
sub = GridSpecFromSubplotSpec(1, 3, subplot_spec=top[0], width_ratios=[0.2, 1, 0.2], wspace=0.08)
ax0 = fig.add_subplot(sub[0])
ax = fig.add_subplot(sub[1], sharey=ax0)
ax2 = fig.add_subplot(sub[2], sharey=ax0)
xq = {q: -0.45 + 0.3 * k for k, q in enumerate(refs)}


def column(a_, factor):
    for q in refs:
        c = rc.COLORS[q]
        a_.errorbar(xq[q] - 0.06, factor * PI[q]['own'], yerr=factor * PI[q]['sown'], fmt='o', mfc='white', mec=c,
                    mew=1.2, ms=4.5, color=c, capsize=2, lw=0.9)
        a_.errorbar(xq[q] + 0.06, factor * PI[q]['com'], yerr=factor * PI[q]['scom'], fmt='o', color=c, ms=4.5,
                    capsize=2, lw=0.9)
    a_.set_xlim(-0.65, 0.85)
    a_.set_xticks([0])
    a_.spines[['top', 'right']].set_visible(False)


column(ax0, 1.0)
ax0.plot(0.7, PHC_PI, marker='*', color='k', ms=10, ls='none', zorder=4)       # Planetary Health Check
# the preindustrial state consistent with the 2024 state of the Planetary Health Check (open stars): its 2024 value with
# the decline of the same product (UExP-FNN-U) on its own area from our calculation, i.e. both states over the same area,
# with the same constants and at the CO2 of 1750 (the relative decline hardly depends on the present-day state, Methods)
_iu = list(TS['UExP']['year']).index(YEAR)
_d_u = dec(TS['UExP']['own'][_iu], PI['UExP']['own'])
PHC_PI_CONS = PHC_OBS / (1 - _d_u / 100)
print(f'Planetary Health Check: published 1750 {PHC_PI:.2f}, boundary {PHC_B:.2f}, {YEAR} {PHC_OBS:.2f} ({dec(PHC_OBS, PHC_PI):.1f} %); '
      f'consistent with its {YEAR} state (UExP own-area decline {_d_u:.2f} %): 1750 {PHC_PI_CONS:.3f}, boundary {0.8 * PHC_PI_CONS:.3f}')
_arr = dict(arrowstyle='-|>,head_length=0.3,head_width=0.15', color='0.35', lw=0.8, shrinkA=5, shrinkB=5)
ax0.plot(0.7, PHC_PI_CONS, marker='*', mfc='white', mec='k', mew=0.9, ms=10, ls='none', zorder=4)
ax0.annotate('', xy=(0.7, PHC_PI_CONS), xytext=(0.7, PHC_PI), arrowprops=_arr, zorder=3)
ax0.set_xticklabels(['1750'])
ax0.set_ylabel(f'Global mean surface {OMEGA}', fontsize=10)
column(ax2, 0.8)
ax2.plot(0.7, PHC_B, marker='*', color='k', ms=10, ls='none', zorder=4)
ax2.plot(0.7, 0.8 * PHC_PI_CONS, marker='*', mfc='white', mec='k', mew=0.9, ms=10, ls='none', zorder=4)
ax2.annotate('', xy=(0.7, 0.8 * PHC_PI_CONS), xytext=(0.7, PHC_B), arrowprops=_arr, zorder=3)
ax2.set_xticklabels(['boundary'])
ax2.tick_params(labelleft=False)
for q in refs:
    ax.plot(TS[q]['year'], TS[q]['own'], color=rc.COLORS[q], lw=1.0, ls=(0, (3, 2)))
    ax.plot(TS[q]['year'], TS[q]['com'], color=rc.COLORS[q], lw=1.3)
ax.plot(YEAR, PHC_OBS, marker='*', color='k', ms=10, ls='none', zorder=4)
ax.set_xlim(1985, YEAR + 1)
ax.set_xticks(range(1990, YEAR + 1, 10))          # no ticks at the edges, next to the columns
ax.tick_params(labelleft=False)
ax.spines[['top', 'right']].set_visible(False)
hh = [Line2D([], [], color='0.3', lw=1.0, ls=(0, (3, 2))), Line2D([], [], color='0.3', lw=1.3)] + \
     [Line2D([], [], color=rc.COLORS[q], lw=1.3) for q in refs] + \
     [Line2D([], [], marker='*', color='k', ms=9, ls='none'),
      Line2D([], [], marker='*', mfc='white', mec='k', mew=0.9, ms=9, ls='none')]
ax.legend(hh, ['common area with the ESMs (open)', f'common area of all fCO$_2$-products, {AREA_C:.1f}% (filled)'] + [rc.FULL[q] for q in refs]
          + ['Planetary Health Check (2026)', 'its 1750 state and boundary\nconsistent with its 2024 state'],
          fontsize=7.5, frameon=False, loc='upper right')
lo_ = min(min(TS[q]['own'].min(), TS[q]['com'].min(), 0.8 * (PI[q]['own'] - PI[q]['sown'])) for q in refs)
lo_ = min(lo_, PHC_OBS, 0.8 * PHC_PI_CONS) - 0.03
hi_ = max(PI[q][k] + PI[q]['s' + k] for q in refs for k in ('own', 'com')) + 0.03
ax0.set_ylim(lo_, hi_)
ax0.text(-1.1, 1.02, 'a', transform=ax0.transAxes, fontsize=16, fontweight='bold', va='bottom')

# b: decline relative to 1750 over time, each product on its own area (dashed) and on the common area (solid)
ax = fig.add_subplot(top[1])
for q in refs:
    ax.plot(TS[q]['year'], dec(TS[q]['own'], PI[q]['own']), color=rc.COLORS[q], lw=1.0, ls=(0, (3, 2)))
    ax.plot(TS[q]['year'], dec(TS[q]['com'], PI[q]['com']), color=rc.COLORS[q], lw=1.3)
ax.axhline(20, color=BCOL, lw=1.0, ls=(0, (4, 3)))
ax.text(1986, 19.85, 'boundary (20% decline)', ha='left', va='top', fontsize=8.5, color=BCOL)
_d_phc = dec(PHC_OBS, PHC_PI)                       # as published (2.85 and 3.57)
ax.plot(YEAR, _d_phc, marker='*', color='k', ms=10, ls='none', zorder=4)
_d_cons = dec(PHC_OBS, PHC_PI_CONS)                 # with the consistent 1750 state (= the UExP own-area decline)
ax.plot(YEAR, _d_cons, marker='*', mfc='white', mec='k', mew=0.9, ms=10, ls='none', zorder=4)
ax.annotate('', xy=(YEAR, _d_cons), xytext=(YEAR, _d_phc), arrowprops=_arr, zorder=3)
ax.annotate('Planetary Health\nCheck (2026)', xy=(YEAR, _d_phc - 0.12), xytext=(YEAR - 2, _d_phc - 1.1), ha='right',
            va='center', fontsize=8, arrowprops=dict(arrowstyle='-|>,head_length=0.25,head_width=0.12', color='0.4',
                                                     lw=0.6, shrinkA=2, shrinkB=0))      # label below the line, arrow ends just under the star
ax.set_xlim(1985, YEAR + 1)
ax.set_ylabel('Decline from 1750 (%)', fontsize=10)
ax.spines[['top', 'right']].set_visible(False)
ax.text(-0.16, 1.02, 'b', transform=ax.transAxes, fontsize=16, fontweight='bold', va='bottom')

# c: steps from the Planetary Health Check to this study; below: which of the three comparisons is consistent
ax = fig.add_subplot(outer[1])
n = len(STEPS)
C_START, C_END, C_DOWN, C_UP = '0.6', '0.15', '#3d6f8e', '#b9cbd6'
BW = 0.5
for i, v in enumerate(vals):
    if i == 0:
        ax.bar(i, v, width=BW, color=C_START, zorder=2)
    elif i == n - 1:
        # last bar: the mean of the four products (label), drawn at the height of the previous bar (UExP-FNN-U) when both
        # round to the same value, so that the connector meets it; otherwise at its own value
        h_ = vals[i - 1] if round(v, 1) == round(vals[i - 1], 1) else v
        ax.bar(i, h_, width=BW, color=C_END, zorder=2)
    else:
        lo_, hi_ = sorted((vals[i - 1], v))
        ax.bar(i, max(hi_ - lo_, 0.02), bottom=lo_, width=BW, color=C_DOWN if v < vals[i - 1] else C_UP, zorder=2)
    if i > 0:
        ax.plot([i - 1 + BW / 2, i - BW / 2], [vals[i - 1]] * 2, color='0.6', lw=0.6, zorder=1)
    lab = f'{v:.1f}%'
    down = 0 < i < n - 1 and v < vals[i - 1]           # label at the new value: below a decrease, above otherwise
    y_lab = v - 0.12 if down else v + 0.12
    ax.text(i, y_lab, lab, ha='center', va='top' if down else 'bottom', fontsize=9,
            fontweight='bold' if i in (0, n - 1) else 'normal')
ax.axhline(20, color=BCOL, lw=0.9, ls=(0, (4, 3)), zorder=1)
ax.text(n - 0.45, 20.08, 'boundary: 20% decline', ha='right', va='bottom', fontsize=8.5, color=BCOL)
ax.set_xticks(range(n))
ax.set_xticklabels([s_[0] for s_ in STEPS], fontsize=8, linespacing=1.1)
ax.tick_params(axis='x', length=0)
ax.set_xlim(-0.6, n - 0.4)
ax.set_ylim(np.floor(min(vals) - 1.0), np.ceil((max(vals) + 0.4) * 2) / 2)     # top: just above the highest label
ax.set_ylabel(f'Decline of global mean\nsurface {OMEGA} from 1750 (%)', fontsize=10)
ax.spines[['top', 'right']].set_visible(False)
ax.text(-0.075, 1.02, 'c', transform=ax.transAxes, fontsize=16, fontweight='bold', va='bottom')
# consistency matrix: filled = consistent, open = not; the full assumptions are in Extended Data Table 3
if CHECKS is not None:
    tr = ax.get_xaxis_transform()
    names = ('same constants', 'same coverage', 'same fCO$_2$-product', 'CO$_2$ of 1750 (AR6)')
    for r_, name in enumerate(names):
        y_ = -0.3 - 0.1 * r_
        ax.text(-0.6, y_, name, transform=tr, ha='right', va='center', fontsize=9, color='0.25', clip_on=False)
        ax.plot([-0.45, n - 0.55], [y_ - 0.05] * 2, transform=tr, color='0.88', lw=0.6, clip_on=False)
        for i, ck in enumerate(CHECKS):
            ax.plot(i, y_, transform=tr, marker='o', ms=8, mew=1.2, mec=C_DOWN, mfc=C_DOWN if ck[r_] else 'white',
                    clip_on=False, ls='none')
    ax.plot([-0.45, n - 0.55], [-0.25] * 2, transform=tr, color='0.88', lw=0.6, clip_on=False)
    ax.text(n - 0.4, -0.3 - 0.1 * (len(names) - 1) - 0.12, 'filled: the same for 2024 and 1750,  open: different',
            transform=tr, ha='right', va='top', fontsize=8, color='0.4', clip_on=False)
rc.savefig(fig, 'coverage')
plt.show()

## Extended Data Table 1: global summary (needs the preparation cell)

In [ ]:
# Extended Data Table 1, wrapped in a function so that none of its names (pct, om, stats, ...) overwrite
# those of other cells in the same kernel.
# Needs the preparation cell to be run first (same kernel): the uncertainty budget is that of Extended Data Table 2 (_trend, MAP_SIG,
# CONST_REL), applied to every product, area, year and preindustrial state of this table.
def _table_s1():
    #!/usr/bin/env python
    """
    Extended Data Table 1: global mean surface Omega_arag, preindustrial value, boundary (80 % of preindustrial),
    distance to the boundary (observed - boundary) and decline, in the order of the argument:

      1  Planetary Health Check (2026): preindustrial 3.57 (1750, Jiang et al. 2023, CMIP6-based, on Jiang's coverage),
         boundary 2.86, observed 2.85 +- 0.01 in 2024 from one product (UExP-FNN-U) on its own area -> crossed
      2  Findlay et al. (2025): central, lower and upper preindustrial value (range from preindustrial CO2)
      3  This study, each product on its own area, constrained, 1750 (278.3 ppm)
      4  This study, all products on the common area, 1750
      5  This study in 2020 (the year of Findlay et al.), common area, mean of the products: 1750, the effective
         preindustrial CO2 of Jiang et al. (2023, about 272 ppm), 280 ppm, 1850 and the midpoint of 1750 and 1850

    Observed values (this study): quadratic fit to the annual area-weighted means on the area, evaluated in YEAR.
    Single year (this study, last two columns): the area-weighted annual mean of YEAR itself, without the fit, and the
    decline from it to the same constrained preindustrial value; not for the rows adjusted to the global ocean.
    Preindustrial (this study): emergent constraint (dSST + CUC); raw ESM values only in the CSV.
    Uncertainties (1 sigma): the components of Extended Data Table 2, per product: constraint and CO2 of the state, fit (AR(1)),
    fitting method (half range), mapping, equilibrium constants within a formulation (fully correlated), total boron
    and K1/K2 formulation (half the difference to the runs l10 and m10).
    Decline = 1 - observed / preindustrial (%); distance = observed - boundary (constrained where available).
    Preindustrial values for every CO2 state are computed with mocsy (calc_pi_xco2.py) and carried through
    run_constraint.py (results[ref]['states']); nothing is extrapolated.
    All numbers at full precision in data/table_summary.csv; the table (two decimals at most, a year on every
    observed value) as data/table_main.png, .pdf and .tex.
    """
    import csv
    import importlib
    import pickle
    import sys

    import numpy as np
    import netCDF4
    import matplotlib.pyplot as plt

    sys.path.insert(0, globals().get('CODE_DIR', '.'))   # works as a script and a notebook cell
    import regional_common as rc
    importlib.reload(rc)
    import oa_config                  # the area weights of the run (OA_WEIGHTS)

    ROOT = rc.ROOT
    YEAR = 2024
    YEAR_F = 2020                         # year of the observed value of Findlay et al. (2025): last block of the table
    # effective preindustrial CO2 of Jiang et al. (2023): their 1850 model state (CMIP6 forcing, mean 1850-1854, the
    # historical part of their decade 1845-1854) minus their ice-core difference 1852 - 1752 (288.57 - 276.39 ppm)
    from pi_states import TARGETS, XCO2_SIGMA, X1850_CMIP6 as X1850   # preindustrial CO2 states and their 1 sigma (ppm)
    X_JIANG, X1850_AR6 = TARGETS['jiang'], TARGETS['1850']
    CO2_SIG, CO2_SIG_1850 = XCO2_SIGMA['1750'], XCO2_SIGMA['1850']


    LAB = rc.LAB
    OMT = r'$\Omega_{\mathrm{arag}}$'
    import os
    # from the preparation cell (Extended Data Table 2): the 2024 estimators (_trend), the mapping uncertainty of each product's global
    # mean (C2; used for every year and area of the product) and the constants (relative 1 sigma in 1750 and 2024,
    # 1 sigma of the decline in percentage points)
    g_ = globals()
    for n_ in ('_trend', 'MAP_SIG', 'MAP_CUC', 'map_terms', 'CONST_REL'):
        if n_ not in g_:
            raise NameError(f'{n_} not defined: run the preparation cell first (same kernel)')
    trend, MAP, (R17, R24, CDEC) = g_['_trend'], g_['MAP_SIG'], g_['CONST_REL']
    MAP_CUC, map_terms = g_['MAP_CUC'], g_['map_terms']        # mapping error of the observed CUC, combined term
    print('mapping uncertainty of the global means (C2, 1 sigma): ' + ', '.join(f'{q_} {v_:.4f}' for q_, v_ in MAP.items()))

    # previous assessments as published: label, observed, its sigma, year of the observed value, preindustrial,
    # its sigma, boundary, its sigma, decline (%) and its sigma where published, conclusion of the authors
    PHC = [('Planetary Health Check (2026)', 2.85, 0.01, '2024', 3.57, None, 2.86, None, None, None, 'Yes',
            '1750, 276 (effectively 272)')]
    FINDLAY = [('1750 (Jiang et al. 2023)', 2.90, 0.06, '2020', 3.57, None, None, None, None, None, '',
                '1750, 276 (effectively 272)'),
               ('Rockström et al. (2009), taken as 1850', 2.90, 0.06, '2020', 3.44, None, None, None, None, None, '',
                'none, 280 (seawater, box calc.)'),
               ('Midpoint (used by Findlay et al.)', 2.90, 0.06, '2020', 3.51, None, 2.80, None, 17.3, None,
                'Yes, within uncertainty', 'midpoint of the two')]
    NOTES = [
        'Observed: global mean surface value in the year of the block; this study: quadratic fit to the annual means on the '
        'area. Single year: the annual mean of that year itself, without the fit, and the decline from it to the same '
        'preindustrial value (this study, on the areas of the fCO$_2$-products only). Distance = observed − boundary (negative: boundary crossed). This study: all values ± 1σ with the components '
        'of Extended Data Table 2, computed for each row: observed: fit (AR(1)), fitting method, mapping, constants, boron and K$_1$/K$_2$ '
        'formulation; preindustrial: emergent constraint on 18 bias-adjusted ESMs, CO$_2$ of the reference year (AR6; 280 ppm '
        'is a set value), mapping, constants, boron and K$_1$/K$_2$ formulation.',
        'The absolute values are dominated by the aragonite solubility product (σ of Ω$_{arag}$ about 5 %), which is the '
        'same in both states and cancels in the decline and almost entirely in the distance. The mapping error of a fCO$_2$-product '
        '(' + ', '.join(f'{rc.FULL[q_].split()[0]} {v_:.3f}' for q_, v_ in MAP.items()) + ') shifts its observed and, '
        'through the bias adjustment, its preindustrial value alike, and the same error in the observed CUC moves the '
        'constrained preindustrial value further (correlation +0.98, cuc_mapping_uncertainty.py); both are combined per '
        'fCO$_2$-product. Boron and K$_1$/K$_2$: half the difference to the runs with the other formulation.',
        'Mean of the fCO$_2$-products: ± √(spread² + mean σ²) of the components that differ between the fCO$_2$-products; boron and '
        'K$_1$/K$_2$ from the difference of the means; decline and distance computed per fCO$_2$-product first.',
        'Planetary Health Check (2026): 1750 of Jiang et al. (2023), observed UExP-FNN-U on its own area (± 2σ). Findlay et al. '
        '(2025): observed and 1750 of Jiang et al. (2023), 3.44 of Rockström et al. (2009) assigned to 1850, midpoint used in '
        'their analysis. Provenance of these values and of the effective CO$_2$ of Jiang et al. (2023): Methods.',
    ]

    with open(f'{ROOT}/constraint_results.pkl', 'rb') as f:
        P = pickle.load(f)
    GRID = {q: (np.asarray(r['lat'], 'f8'), np.asarray(r['lon'], 'f8')) for q, r in P['results_common'].items()}

    # the ocean area: regional_common.ocean_mask (as in Figure 1 and Supplementary Fig. 1)
    ANYMONTH = {'OceanSODA': 'OceanSODA-ETHZ-v2025', 'UExP': 'UExP-FNN-U-v2026-1', 'JMA': 'JMA-MLR-v2025',
                'CMEMS': 'CMEMS-LSCE-FFNN'}


    def read_obs(path, q):
        with netCDF4.Dataset(path) as f:
            return np.asarray(f['year'][:]), *GRID[q], np.ma.filled(f['omega_arag'][:].astype('f8'), np.nan)


    # one ocean definition for every percentage (regional_common.ocean_mask)


    def fit_gm(q, mask, lat, lon, year, obs=None):
        """quadratic fit to the annual area-weighted means over mask, evaluated in year, the AR(1) standard error of the
        fit, half the range of the alternative fits (the estimators of Extended Data Table 2, _trend of the preparation cell) and the
        annual mean of the year itself (single year, no fit; NaN if the record does not contain the year)"""
        yo, _, _, om = read_obs(obs or rc.OBS[q], q)
        wm = oa_config.area_weights(lat, lon) * mask
        gm = np.array([np.nansum(np.where(np.isfinite(o), o, 0) * wm) / wm[np.isfinite(o)].sum() for o in om])
        ok = np.isfinite(gm) & (yo <= rc.LAST_YEAR)
        e = trend(q, (yo[ok].astype('f8'), gm[ok]), year)
        one = float(gm[yo == year][0]) if np.any(yo == year) else np.nan
        return e['time quadratic'], e['se'], e['range'] / 2, one


    def ours(q, r, obs=None):
        """this study for one product: observed, raw and constrained preindustrial for every CO2 state (mocsy, calc_pi_xco2.py)"""
        mask = np.asarray(r['mask']).astype(bool)
        lat, lon = np.asarray(r['lat'], 'f8'), np.asarray(r['lon'], 'f8')
        now, sfit, smeth, one = fit_gm(q, mask, lat, lon, YEAR, obs)
        now_f, sfit_f, smeth_f, one_f = fit_gm(q, mask, lat, lon, YEAR_F, obs)
        yu, su = r['unconstrained'][:2]
        yc, sc, rng = r['constrained'][LAB]
        d = dict(q=q, area=rc.pct_ocean(mask, lat, np.asarray(r['lon'], 'f8')), smap=MAP[q],
                 now=now, sfit=sfit, smeth=smeth, now_f=now_f, sfit_f=sfit_f, smeth_f=smeth_f, one=one, one_f=one_f)
        if 'states' not in r:
            raise SystemExit('constraint_results.pkl has no CO2 states: rerun calc_pi_xco2.py and run_constraint.py')
        S = r['states']                                                 # tag -> xco2, ratio (to 1750), y_raw
        for key in ('1750', '1850', 'mid', 'jiang', 'x280', 'pi_low', 'pi_high'):
            if key not in S:
                continue
            raw, con = S[key]['y_raw'], yc * S[key]['ratio']
            lo, hi = (key + '_lo', key + '_hi') if key + '_lo' in S else ('1750_lo', '1750_hi')
            s_co2 = 0.5 * yc * abs(S[hi]['ratio'] - S[lo]['ratio'])   # Omega uncertainty from the CO2 of the state (computed)
            if key == 'x280':
                s_co2 = 0.0                                             # 280 ppm is a set value, not an estimate of a year
            d[key] = dict(raw=raw, sraw=np.hypot(su * raw / yu, s_co2), con=con, scon=np.hypot(sc * con / yc, s_co2),
                          xco2=S[key]['xco2'], sxco2=XCO2_SIGMA.get(key, CO2_SIG))
        d['284'], d['275'] = d.get('pi_high'), d.get('pi_low')
        return d


    ORDER = [q for q in ('OceanSODA', 'UExP', 'JMA', 'CMEMS') if q in P['results_common']]
    OWN = {q: ours(q, P['results'][q]) for q in ORDER if q in P['results']}
    COM = {q: ours(q, P['results_common'][q]) for q in ORDER}
    # the same rows in the runs with the other total boron (l10, Lee et al. 2010) and K1/K2 (m10, Millero 2010):
    # their own product files and constraint results; only for the standard run, as in Extended Data Table 2
    OTH = {}
    if rc.RUN == 'u74':
        for run_ in ('l10', 'm10'):
            pk_ = f'{rc.RUNS[run_]}/constraint_results.pkl'
            if not os.path.exists(pk_):
                if oa_config.WEIGHTS == 'ocean':                     # the paper's run: boron and K1/K2 are required
                    raise FileNotFoundError(f'{pk_} missing: the sensitivity run {run_} is needed for boron and K1/K2')
                print(f'{pk_} missing: boron and K1/K2 of {run_} not included (OA_WEIGHTS={oa_config.WEIGHTS}, '
                      'a sensitivity of the area weights; its uncertainties lack these terms)')
                continue
            with open(pk_, 'rb') as f:
                P_ = pickle.load(f)
            ob_ = {q: rc.OBS[q].replace(rc.ROOT, rc.RUNS[run_]) for q in ORDER}
            OTH[run_] = dict(own={q: ours(q, P_['results'][q], ob_[q]) for q in OWN},
                             com={q: ours(q, P_['results_common'][q], ob_[q]) for q in COM})
        print('boron and K1/K2 from the runs ' + ', '.join(f'{k} ({rc.RUNS[k]})' for k in OTH))
    else:
        print(f'run {rc.RUN}: boron and K1/K2 not included (only for the standard run u74)')


    # ---------------------------------------------------------------- table cells
    def om(v, s=None):                    # Omega: two decimals
        if v is None or not np.isfinite(v):
            return ''
        return f'{v:.2f}' + (f' ± {s:.2f}' if s is not None and np.isfinite(s) and round(s, 2) > 0 else '')


    def pct(v, s=None):                   # decline in %: one decimal
        if v is None or not np.isfinite(v):
            return ''
        return f'{v:.1f}' + (f' ± {s:.1f}' if s is not None and round(s, 1) > 0 else '')


    def dist(v, s=None):                  # distance to the boundary: two decimals with sign
        if v is None or not np.isfinite(v):
            return ''
        return (f'{v:+.2f}' + (f' ± {s:.2f}' if s is not None and np.isfinite(s) and round(s, 2) > 0 else '')).replace('-', '−')


    def lit_row(lab, now, snow, yr, pi, spi, b, sb, dec, sdec, concl, pi_ref):
        b = 0.8 * pi if b is None else b
        dec = 100 * (1 - now / pi) if dec is None else dec
        sd = np.hypot(snow or 0.0, sb or 0.0)
        return ('row', lab, [om(now, snow), pi_ref, om(pi, spi), om(b, sb), dist(now - b, sd), pct(dec, sdec), '', ''],
                False)


    KEYS = ('now', 'pi', 'b', 'dec', 'dist')


    def values(d, co2, year):
        now, pi = (d['now'] if year == YEAR else d['now_f']), d[co2]['con']
        one = d['one'] if year == YEAR else d['one_f']                 # single year, no fit (given without uncertainty)
        return dict(now=now, pi=pi, b=0.8 * pi, dec=100 * (1 - now / pi), dist=now - 0.8 * pi,
                    one=one, dec1=100 * (1 - one / pi))


    def variances(d, co2, year, area='com'):
        """1 sigma^2 of one product without boron and K1/K2 (Extended Data Table 2 components)"""
        v = values(d, co2, year)
        now, pi = v['now'], v['pi']
        sf, sm_ = (d['sfit'], d['smeth']) if year == YEAR else (d['sfit_f'], d['smeth_f'])
        s = d[co2]['scon']                                      # constraint and CO2 of the state
        # mapping: the product's level error and the same error in the observed CUC, combined (map_terms, preparation cell)
        mt = map_terms(d['smap'], MAP_CUC[d['q']]['own' if area == 'own' else 'common'], now, pi)
        c24, c17 = R24 * now, R17 * pi                          # constants, fully correlated between the two states
        return dict(now=sf ** 2 + sm_ ** 2 + mt['now'] ** 2 + c24 ** 2,
                    pi=s ** 2 + mt['pi'] ** 2 + c17 ** 2,
                    b=0.64 * (s ** 2 + mt['pi'] ** 2 + c17 ** 2),
                    dec=(100 * now / pi ** 2 * s) ** 2 + (100 * sf / pi) ** 2 + (100 * sm_ / pi) ** 2
                        + mt['dec'] ** 2 + CDEC ** 2,
                    dist=(0.8 * s) ** 2 + sf ** 2 + sm_ ** 2 + mt['dist'] ** 2 + (c24 - 0.8 * c17) ** 2)


    def sens(v, vo):
        """boron and K1/K2: half the difference to the other runs, squared and summed"""
        return {k: sum(((o[k] - v[k]) / 2) ** 2 for o in vo) for k in KEYS}


    def stats(d, co2, year=YEAR, area='com'):
        """one product: observed (in year), preindustrial, boundary, decline, distance and their 1 sigma"""
        v, var = values(d, co2, year), variances(d, co2, year, area)
        sv = sens(v, [values(OTH[run_][area][d['q']], co2, year) for run_ in OTH])
        return {**v, **{'s' + k: np.sqrt(var[k] + sv[k]) for k in KEYS}}


    def mean_stats(D, co2, year=YEAR, area='com'):
        """mean of the products: sqrt(spread^2 + mean(sigma_i^2)) of the components that differ between the products
        (correlated, so not averaged down); boron and K1/K2 from the difference of the means of the runs.
        Decline and distance are computed per product first."""
        S = [values(d, co2, year) for d in D.values()]
        m = {k: np.mean([x[k] for x in S]) for k in KEYS}
        m['one'], m['dec1'] = np.mean([x['one'] for x in S]), np.mean([x['dec1'] for x in S])   # decline per product first
        spread = {k: np.std([x[k] for x in S], ddof=1) for k in KEYS}
        var = {k: np.mean([variances(d, co2, year, area)[k] for d in D.values()]) for k in KEYS}
        mo = [{k: np.mean([values(OTH[run_][area][q], co2, year)[k] for q in D]) for k in KEYS} for run_ in OTH]
        sv = sens(m, mo)
        for k in KEYS:
            m['s' + k] = np.sqrt(spread[k] ** 2 + var[k] + sv[k])
        return m


    def our_row(lab, x, bold=False, pi_ref=f'1750, {TARGETS["1750"]:.1f} ± {CO2_SIG:.1f}'):
        return ('row', lab, [om(x['now'], x['snow']), pi_ref, om(x['pi'], x['spi']), om(x['b'], x['sb']),
                             dist(x['dist'], x['sdist']), pct(x['dec'], x['sdec']), om(x.get('one')), pct(x.get('dec1'))],
                bold)


    T = [('group', f'Planetary Health Check (2026), {YEAR}: preindustrial from Jiang et al. (2023), observed from one '
                   'fCO$_2$-product on its own area')]
    T += [lit_row(*x) for x in PHC]
    T += [('group', f'Findlay et al. (2025), {YEAR_F}')]
    T += [lit_row(*x) for x in FINDLAY]
    area_own = ', '.join(f'{d["area"]:.0f}' for d in OWN.values())
    T += [('group', f'This study, {YEAR}, each fCO$_2$-product on its own area ({area_own}% of the ocean)')]
    T += [our_row(rc.FULL[q], stats(d, '1750', area='own')) for q, d in OWN.items()]
    area_c = np.mean([d['area'] for d in COM.values()])
    T += [('group', f'This study, {YEAR}, all fCO$_2$-products on the common area ({area_c:.1f}% of the ocean)')]
    T += [our_row(rc.FULL[q], stats(d, '1750')) for q, d in COM.items()]
    T += [our_row('Mean of the fCO$_2$-products', mean_stats(COM, '1750'), True)]
    # in the year of Findlay et al. (2025), with the preindustrial states that match theirs row by row
    T += [('group', f'This study, {YEAR_F} (year of Findlay et al. 2025), common area, mean of the fCO$_2$-products')]
    T += [our_row('1750 (AR6)', mean_stats(COM, '1750', YEAR_F))]
    T += [our_row('Effective CO$_2$ of Jiang et al. (2023)', mean_stats(COM, 'jiang', YEAR_F),
                  pi_ref=f'1750, {X_JIANG:.1f} ± {CO2_SIG:.1f}')]
    T += [our_row('280 ppm (as Rockström et al. 2009)', mean_stats(COM, 'x280', YEAR_F), pi_ref='none, 280')]
    T += [our_row('1850 (AR6)', mean_stats(COM, '1850', YEAR_F), pi_ref=f'1850, {X1850_AR6:.1f} ± {CO2_SIG_1850:.1f}')]
    T += [our_row('Midpoint of 1750 and 1850', mean_stats(COM, 'mid', YEAR_F),
                  pi_ref=f'midpoint, {TARGETS["mid"]:.1f} ± {XCO2_SIGMA["mid"]:.1f}')]
    # adjusted to the global ocean (structural_adjustments.py): first the whole ocean instead of the common area, then the
    # alkalinity rise that the products and 15 of the 18 ESMs miss; their uncertainties are the second block of Extended Data Table 2
    SA = np.load(oa_config.DATA / 'structural' / 'structural_adjustments.npz')
    ac_, ao_ = SA['abs_common'], SA['abs_ocean']                  # 1750, 2024, boundary; with alkalinity: 3, 4, 5
    pol_shift, pol_sig = float(SA['polar_shift']), float(SA['polar_sigma'])
    alk_shift = float(SA['alk_cell_ocean'])                      # whole ocean (cell by cell, structural_adjustments.py)
    pd50, pd24 = float(ao_[0] - ac_[0]), float(ao_[1] - ac_[1])
    ad50, ad24 = float(ao_[3] - ao_[0]), float(ao_[4] - ao_[1])
    mc = mean_stats(COM, '1750')
    # 1 sigma of the whole-ocean levels (1750, 2024, distance to the boundary): three times the spread across the ESMs of the
    # whole-ocean mean relative to the common-area mean (as Extended Data Table 2)
    pol_s50, pol_s24, pol_sdist = (float(v_) for v_ in SA['polar_level_sigma'])

    def adjusted(x, d50, d24, ddec, s50, s24, sdec, sdist):
        y_ = dict(x)
        y_['one'] = y_['dec1'] = None                 # the single year only on the areas of the products
        y_['pi'], y_['now'], y_['dec'] = x['pi'] + d50, x['now'] + d24, x['dec'] + ddec
        y_['b'], y_['dist'] = 0.8 * y_['pi'], y_['now'] - 0.8 * y_['pi']
        y_['spi'], y_['snow'], y_['sdec'] = np.hypot(x['spi'], s50), np.hypot(x['snow'], s24), np.hypot(x['sdec'], sdec)
        y_['sb'], y_['sdist'] = np.hypot(x['sb'], 0.8 * s50), np.hypot(x['sdist'], sdist)
        return y_
    wo_ = adjusted(mc, pd50, pd24, pol_shift, pol_s50, pol_s24, pol_sig, pol_sdist)
    wa_ = adjusted(wo_, ad50, ad24, alk_shift, abs(ad50), abs(ad24), abs(alk_shift), abs(ad24 - 0.8 * ad50))
    T += [('group', f'This study, {YEAR}, mean of the fCO$_2$-products, adjusted to the global ocean (Methods; uncertainties: Extended Data Table 2)')]
    T += [our_row('Whole ocean', wo_)]
    T += [our_row('Whole ocean, with the alkalinity rise', wa_, True)]
    sg = lambda v_, f_: format(v_, f_).replace('-', '−')
    NOTES.append(f'Adjusted to the global ocean: the whole ocean instead of the common area ({sg(pol_shift, "+.2f")} ± {pol_sig:.2f} '
                 f'percentage points of decline; Ω$_{{arag}}$ {sg(pd50, "+.2f")} ± {pol_s50:.2f} in 1750 and {sg(pd24, "+.2f")} ± '
                 f'{pol_s24:.2f} in {YEAR}), then the '
                 f'surface alkalinity rise that the fCO$_2$-products and 15 of the 18 ESMs miss ({sg(alk_shift, "+.2f")} ± {abs(alk_shift):.2f} '
                 f'percentage points; Ω$_{{arag}}$ {sg(ad50, "+.3f")} in 1750 and {sg(ad24, "+.3f")} in {YEAR}); Methods and Extended Data Table 2.')

    # ---------------------------------------------------------------- output
    HEAD = [f'Observed\n{OMT}', 'Preindustrial reference\n(year, CO$_2$ in ppm)', f'Preindustrial\n{OMT}',
            'Boundary\n(80%)', 'Distance to\nboundary', 'Decline\n(%)', f'Single year\n{OMT}', 'Decline,\nsingle year (%)']
    HEAD_TXT = ['Observed', 'PI reference (year, ppm)', 'Preindustrial', 'Boundary', 'Distance', 'Decline (%)',
                'Single year', 'Decline 1 yr (%)']

    # CSV, full precision
    out_csv = rc.figpath('table', ext='csv')
    with open(out_csv, 'w', newline='') as f:
        wr = csv.writer(f)
        wr.writerow(['area', 'product', 'area_pct', f'observed_{YEAR}', 'sigma_fit', 'sigma_method', f'observed_{YEAR_F}',
                     f'sigma_fit_{YEAR_F}', f'sigma_method_{YEAR_F}', 'sigma_mapping', f'single_year_{YEAR}',
                     f'single_year_{YEAR_F}'] +
                    [f'{k}_{co2}' for co2 in ('1750', 'x280', '284', '275', '1850', 'mid', 'jiang') for k in ('raw', 'sraw', 'con', 'scon')])
        for name, D in (('own', OWN), ('common', COM)):
            for q, d in D.items():
                wr.writerow([name, q, d['area'], d['now'], d['sfit'], d['smeth'], d['now_f'], d['sfit_f'], d['smeth_f'],
                             d['smap'], d['one'], d['one_f']] + [d[co2][k] for co2 in ('1750', 'x280', '284', '275', '1850', 'mid', 'jiang')
                                                              for k in ('raw', 'sraw', 'con', 'scon')])

    # plain text
    w0 = max(len(r[1]) for r in T if r[0] == 'row') + 2
    wc = [max(len(h), 11) + 2 for h in HEAD_TXT]
    print(f'{"":{w0}s}' + ''.join(f'{h:>{n}s}' for h, n in zip(HEAD_TXT, wc)))
    for r in T:
        print(f'\n{r[1]}' if r[0] == 'group' else f'{r[1]:{w0}s}' + ''.join(f'{c:>{n}s}' for c, n in zip(r[2], wc)))
    for n_ in NOTES:
        print(n_)

    # LaTeX (booktabs)
    esc = lambda t: (t.replace('%', '\\%').replace('±', '$\\pm$').replace('−', '$-$').replace('σ', '$\\sigma$')
                     .replace(OMT, '$\\Omega_\\mathrm{arag}$').replace('\n', ' '))
    with open(rc.figpath('table', ext='tex'), 'w') as f:
        f.write('\\begin{tabular}{l' + 'c' * len(HEAD) + '}\n\\toprule\n')
        f.write(' & ' + ' & '.join(esc(h) for h in HEAD) + ' \\\\\n')
        for r in T:
            if r[0] == 'group':
                f.write('\\midrule\n\\multicolumn{%d}{l}{\\textit{%s}} \\\\\n' % (len(HEAD) + 1, esc(r[1])))
            else:
                cells = [esc(r[1])] + [esc(c) for c in r[2]]
                f.write(' & '.join(f'\\textbf{{{c}}}' if r[3] and c else c for c in cells) + ' \\\\\n')
        f.write('\\bottomrule\n\\end{tabular}\n')

    # rendered table (the notes are printed above for the legend of the manuscript)
    LW, CW, RH, HH, H2 = 3.6, 1.3, 0.28, 0.55, 0.0
    cws = [CW, 2.6, CW, CW, 1.4, CW, 1.25, 1.45]
    W = 0.2 + LW + sum(cws)
    H = 0.25 + H2 + HH + RH * len(T) + 0.25
    fig = plt.figure(figsize=(W, H))
    ax = fig.add_axes([0, 0, 1, 1])
    ax.set_xlim(0, W)
    ax.set_ylim(H, 0)
    ax.axis('off')
    x0 = 0.1
    edges = x0 + LW + np.concatenate([[0], np.cumsum(cws)])
    xc = (edges[:-1] + edges[1:]) / 2
    y = 0.15
    ax.plot([x0, W - 0.1], [y, y], color='k', lw=1.2)
    for xj, h in zip(xc, HEAD):
        ax.text(xj, y + HH / 2, h, ha='center', va='center', fontsize=10, linespacing=1.3, fontweight='bold')
    y += H2 + HH
    ax.plot([x0, W - 0.1], [y, y], color='k', lw=0.8)
    for r in T:
        if r[0] == 'group':
            ax.add_patch(plt.Rectangle((x0, y), W - 0.2, RH, color='0.92', lw=0))
            ax.text(x0 + 0.08, y + RH / 2, r[1], ha='left', va='center', fontsize=10, style='italic')
        else:
            fw = 'bold' if r[3] else 'normal'
            if r[3]:
                ax.plot([x0 + 0.08, W - 0.1], [y, y], color='0.6', lw=0.5)
            ax.text(x0 + 0.25, y + RH / 2, r[1], ha='left', va='center', fontsize=10, fontweight=fw)
            for xj, c in zip(xc, r[2]):
                ax.text(xj, y + RH / 2, c, ha='center', va='center', fontsize=10, fontweight=fw)
        y += RH
    ax.plot([x0, W - 0.1], [y, y], color='k', lw=1.2)
    rc.savefig(fig, 'table')
    plt.show()
    print(f'\nwritten: {out_csv}; {rc.figpath("table", ext="(png, pdf, tex)")}')


_table_s1()

## Extended Data Table 3, block a: steps from the Planetary Health Check to this study (needs the preparation cell)

In [ ]:
# needs the preparation cell to be run first (same kernel): uses its variables. Computes block a of Extended Data Table 3
# (EDT3_A); the table itself is drawn after the cell of Extended Data Fig. 5, which adds the steps from Findlay et al. (2025)
# ---------------------------------------------------------------- Extended Data Table 3: every single change, both sides of the ratio
own180 = rc.to_180(np.asarray(ROWN[STEP_REF]['lon'], 'f8'), np.asarray(ROWN[STEP_REF]['mask']).astype(bool))[1]
A_FULL = pct(cov[STEP_REF]['full'])
P_ = 'UExP-FNN-U'
P_M = 'UExP-FNN-U v2026-1'    # released version, our area-weighted mean (recomputed with mocsy: same to 0.001)
# constants (K1, K2; total boron B): UExP-FNN-U v2026-1 (Zenodo 10.5281/zenodo.22110725): PyCO2SYS, Lueker, Uppstroem;
# this study: mocsy 2.4.0 with optb='u74'; Jiang et al. (2023): CO2System.jl, Lueker, J_BORON
BNAME = {'l10': 'Lee', 'u74': 'Uppström'}
K_PROD = 'Lueker; B: Uppström'
K_OURS = 'Lueker; B: ' + BNAME[BORON]
K_J = 'Lueker; B: ' + BNAME[J_BORON]
# one row per bar of Fig. 3c: step, then 2024 (Omega, product, K1 K2, coverage, year) and 1750 (Omega, source,
# K1 K2, coverage, relative to, CO2), and the decline; entries that change from the previous row are set in bold
T_ROWS = [
    ('Planetary Health Check (2026), reproduced', PHC_OBS_R, 'UExP-FNN-U v2026-pre1', K_PROD, P_, '2024',
     PHC_PI, 'Jiang et al. (2023)', K_J, 'Jiang et al.', 'Jiang et al.', f'{X_J:.0f}'),
    ('same equilibrium constants for 1750 and 2024', o['mocsy_f'], P_M, K_OURS, P_, '2024',
     PHC_PI_U, 'Jiang et al. (2023)', K_OURS, 'Jiang et al.', 'Jiang et al.', f'{X_J:.0f}'),
    ('the same cells for 1750 and 2024', SH24, P_M, K_OURS, 'shared', '2024',
     PI_J_SH_U, 'Jiang et al. (2023)', K_OURS, 'shared', 'Jiang et al.', f'{X_J:.0f}'),
    ('the same observational baseline for 1750 and 2024', SH24, P_M, K_OURS, 'shared', '2024',
     PI_JB_SH, '14 ESMs of Jiang et al.', K_OURS, 'shared', P_, f'{X_J:.0f}'),
    ('the common area of this study', o['com24'], P_M, K_OURS, 'common', '2024',
     PI_JB_COM, '14 ESMs of Jiang et al.', K_OURS, 'common', P_, f'{X_J:.0f}'),
    ('2024 from the trend fit instead of a single year', now_c, P_M, K_OURS, 'common', '2024 (fit)',
     PI_JB_COM, '14 ESMs of Jiang et al.', K_OURS, 'common', P_, f'{X_J:.0f}'),
    ('18 constrained ESMs instead of the 14 ESMs of Jiang et al.', now_c, P_M, K_OURS, 'common', '2024 (fit)',
     PI_JC, '18 ESMs, constrained', K_OURS, 'common', P_, f'{X_J:.0f}'),
    (f'CO$_2$ in 1750 of {X_1750:.1f} ppm (AR6) instead of {X_J:.0f} ppm', now_c, P_M, K_OURS, 'common', '2024 (fit)',
     PI_1750C, '18 ESMs, constrained', K_OURS, 'common', P_, f'{X_1750:.1f}'),
    ('this study (mean of the four fCO$_2$-products)', now_m, 'four fCO$_2$-products', K_OURS, 'common', '2024 (fit)',
     yc_m, '18 ESMs, constrained', K_OURS, 'common', 'each fCO$_2$-product', f'{X_1750:.1f}')]
T_DEC = [dec(r_[1], r_[6]) for r_ in T_ROWS]
T_DEC[-1] = dec_m
assert np.allclose(T_DEC, vals), 'block a of Extended Data Table 3 and Fig. 3c differ'
GROUPS = [('', 2), ('2024', 5), ('Preindustrial (1750)', 6), ('', 1)]
HEAD = ['', 'Step (bars of Fig. 3c)', 'Ω$_{arag}$', 'fCO$_2$-product', 'K$_1$, K$_2$', 'coverage', 'year',
        'Ω$_{arag}$', 'source', 'K$_1$, K$_2$', 'coverage', 'relative to', 'CO$_2$ (ppm)', 'Decline (%)']
CELLS = [[str(k + 1), r_[0], f'{r_[1]:.3f}', r_[2], r_[3], r_[4], r_[5], f'{r_[6]:.3f}', r_[7], r_[8], r_[9], r_[10],
          r_[11], f'{d_:.1f}']
         for k, (r_, d_) in enumerate(zip(T_ROWS, T_DEC))]
NOTES = [
    f'Coverage: {P_} = all cells of {P_} with data in all months ({A_FULL:.1f}% of the area covered by any of the four fCO$_2$-products); Jiang et al. = the coverage '
    'of their 1° product (without the Mediterranean, Red Sea, Hudson Bay and part of the coasts); common = cells shared '
    f'by the four fCO$_2$-products and all 18 ESMs ({AREA_C:.1f}%); shared = cells where both {P_} and Jiang et al. have values.',
    'Constants: K$_1$, K$_2$ and total boron (B). All use K$_1$, K$_2$ of Lueker et al. (2000). UExP-FNN-U computes '
    'Ω$_{arag}$ with PyCO2SYS and total boron of Uppström (1974) (doi:10.5281/zenodo.22110725). This study recomputes '
    'Ω$_{arag}$ with mocsy 2.4.0 from T, S, DIC and TA (GLODAPv2 Si and PO$_4$), with total boron of '
    + {'l10': 'Lee et al. (2010)', 'u74': 'Uppström (1974)'}[BORON] +
    ', K$_{SO4}$ Dickson (1990), K$_F$ Perez and Fraga (1987) and the aragonite solubility of Mucci (1983). '
    + (f'Jiang et al. (2023) use total boron of {BNAME[J_BORON]}; from step 2 on, their Ω$_{{arag}}$ is converted to '
       f'{BNAME[BORON]} by recomputing their states from their own fCO$_2$, TA, T and S with both formulations (mocsy, DIC solved for their fCO$_2$), '
       f'which raises their Ω$_{{arag}}$ by {100 * (F_J_OWN - 1):.1f}%.'
       if J_BORON != BORON else 'Jiang et al. (2023) use the same constants.'),
    'Relative to: the observations the preindustrial models are bias-adjusted to, so that 1750 = the observations of 2024 '
    'minus the modelled change (Jiang et al.: their gridded SOCAT, GLODAPv2 and CODAP-NA climatology). Step 4 keeps '
    f'the change simulated by the 14 ESMs of Jiang et al. (their ratio of 1750 to 2005-2014) and applies it to the '
    f'2005-2014 mean of {P_} on the same cells, so that both states rest on {P_}; step 5 restricts both states to the '
    f'common area (the slightly larger decline of the cells outside it, the whole-ocean adjustment in reverse); step 7 '
    f'replaces the 14 ESMs by our 18 constrained ESMs. Year: 2024 = '
    'annual mean of 2024; 2024 (fit) = quadratic fit to the annual means, evaluated in 2024. CO$_2$: 272 = effective CO$_2$ of the 1750 value of Jiang et al. (their 1850 state minus the '
    'ice-core change 1752-1852); 278.3 = AR6 for 1750.',
    f'2024: step 1 reproduces the published {PHC_OBS:.2f} ({dec(PHC_OBS, PHC_PI):.1f}%) from the pre-release '
    f'UExP-FNN-U v2026-pre1 with the weights of the file, cell area on the ellipsoid times the ocean proportion of each '
    f'cell from ESA Land Cover CCI ({PHC_OBS_R:.4f}; the fCO$_2$-product\'s own Ω$_{{arag}}$ averaged over the 12 months of '
    f'2024). From step 2 on, the released v2026-1 is used (with the same weights {PHC_REPRO[("v2026-1", "file")]:.4f}, '
    f'{dec(PHC_REPRO[("v2026-1", "file")], PHC_PI) - dec(PHC_OBS_R, PHC_PI):+.2f} points). Step 2 recomputes '
    f'Ω$_{{arag}}$ with mocsy and the equilibrium constants used for 1750, as for the models once from the annual means '
    f'of T, S, DIC and TA, since their fields are annual means ({o["mocsy_f"]:.3f}). The constants alone give '
    f'{o["mocsy_mm_f"]:.3f} (mean of the monthly saturation states, '
    f'{dec(o["mocsy_mm_f"], PHC_PI_U) - dec(PHC_REPRO[("v2026-1", "file")], PHC_PI):+.2f} points), the annual means '
    f'{dec(o["mocsy_f"], PHC_PI_U) - dec(o["mocsy_mm_f"], PHC_PI_U):+.2f} points. Steps 1 and 2 keep the weights of the '
    f'file. From step 3 on, the weights of this study ({oa_config.WEIGHTS_TEXT}). Separately, whole cells instead of their ocean fraction change the decline from the fixed 3.57 by '
    f'{dec(PHC_REPRO[("v2026-1", "cos")], PHC_PI) - dec(PHC_REPRO[("v2026-1", "file")], PHC_PI):+.2f} points '
    f'(v2026-1, full area).',
    'Decline = 1 − Ω$_{arag}$(2024) / Ω$_{arag}$(1750). Bold: what changes from the previous row. Last row: mean of the four fCO$_2$-products; '
    'its uncertainty is given in Extended Data Table 2.']

# block a of Extended Data Table 3 (drawn with blocks b and c after the cell of Extended Data Fig. 5)
EDT3_A = dict(cells=CELLS, notes=NOTES, K_J=K_J, K_OURS=K_OURS, X_J=X_J, X_1750=X_1750)
print('Extended Data Table 3, block a: ' + '; '.join(f'{r_[1]} {r_[-1]} %' for r_ in CELLS))

## Extended Data Table 2: uncertainty budget (needs the preparation cell)

In [ ]:
# needs the preparation cell to be run first (same kernel): uses its variables
import csv
X0 = 0.15                            # left margin of the rendered tables (inches)
# ---------------------------------------------------------------- Extended Data Table 2: uncertainty of the decline (1 sigma)
rc.FIGNAMES.setdefault('uncertainty_table', 'TableS3_uncertainty')
HOW = {K_ESM: 'regression across the 18 bias-adjusted ESMs and the observed predictors (ΔSST, CUC)',
       K_CO2: 'preindustrial states at 278.3 ± 1.8 ppm (AR6 very likely range ±2.9 ppm)',
       K_PROD_D: 'standard deviation across the four fCO$_2$-products; offsets largely cancel in the decline',
       K_FIT: 'standard error of the 2024 value of the quadratic fit, residuals as AR(1) process',
       K_METH: 'half the range of quadratic fits and Enting splines (15, 20, 25 yr), in time and against CO$_2$',
       K_B: 'half the difference between runs with both formulations (difference = 2σ, as Orr et al., 2018)',
       K_K12: 'half the difference between runs with both formulations (same assumption as for boron)',
       K_CONST: 'mocsy error propagation on the 2024 and 1750 states, fully correlated; K$_{sp}$ cancels in the decline',
       K_MAP: 'random fields (Ford et al. 2024), same in all months, also in the observed CUC'}
# order of the rows: 1750, 2024, products, chemistry
_ORDER = [K_ESM, K_CO2, K_PROD_D, K_MAP, K_FIT, K_METH, K_B, K_K12, K_CONST]
_ORDER = [k_ for k_ in _ORDER if k_ in SIG] + [k_ for k_ in SIG if k_ not in _ORDER]
f2 = lambda v: f'{v:.2f}' if np.isfinite(v) else '–'
f3 = lambda v: f'{v:.3f}' if np.isfinite(v) else '–'
T3 = [(k_, f3(ABS[k_][0]), f3(ABS[k_][1]),
       f2(SIG[k_]) if np.isfinite(SIG[k_]) else 'after the sensitivity run',
       HOW.get(k_, '')) for k_ in _ORDER]
_s50 = float(np.sqrt(np.nansum([a_[0] ** 2 for a_ in ABS.values()])))
_s24 = float(np.sqrt(np.nansum([a_[1] ** 2 for a_ in ABS.values()])))
T3_TOT = [('total (in quadrature)', f3(_s50), f3(_s24), f2(sdec_m), '')]
HEAD3 = ['Component', '1σ of Ω$_{arag}$ 1750', '1σ of Ω$_{arag}$ 2024', '1σ of decline (%)', 'Estimated from']
with open(rc.figpath('uncertainty_table', ext='csv'), 'w', newline='') as f_:
    wr = csv.writer(f_)
    wr.writerow(['Component', '1 sigma Omega_arag 1750', '1 sigma Omega_arag 2024', '1 sigma decline (percentage points)',
                 'Estimated from'])
    wr.writerows([[c_.replace('$', '') for c_ in r_] for r_ in T3 + T3_TOT])
T3 = T3 + T3_TOT
# ---------------------------------------------------------------- the second block: the additional uncertainties of the global mean
# The two adjustments (whole ocean, alkalinity rise) and their 1 sigma come from the preparation cell (POL_*, ALK_*,
# DEC_G, SDEC_G, OM50_G, OM24_G, from structural_adjustments.npz). The table shows only the additional uncertainties;
# the adjustments themselves are in Extended Data Table 1 and the text.
for n_ in ('POL_SHIFT', 'POL_SIG', 'ALK_SHIFT', 'ALK_SIG', 'ALK_D50', 'ALK_D24', 'POL_S50', 'POL_S24', 'DEC_G', 'SDEC_G', '_SA'):
    if n_ not in globals():
        raise NameError(f'{n_} not defined: run the preparation cell first')
_b = _SA['barrett']
T3_G = [('Additional uncertainty of the global mean', '1σ of Ω$_{arag}$ 1750', '1σ of Ω$_{arag}$ 2024', '1σ of decline (%)', 'Estimated from'),
        ('whole ocean instead of the common area', f3(POL_S50), f3(POL_S24), f2(POL_SIG),
         f'three times the spread across {int(_SA["polar_models_n"])} ESMs, each extended to the whole ocean like the fCO$_2$-products'),
        ('alkalinity rise missing in the fCO$_2$-products and 15 of the 18 ESMs', f3(abs(ALK_D50)), f3(abs(ALK_D24)), f2(ALK_SIG),
         f'1σ = the adjustment (TA rise {_b[0]:.3f} ± {_b[1]:.3f} µmol kg$^{{-1}}$ yr$^{{-1}}$, mean of the two regressions of Barrett et al., 2025)'),
        ('total of the global mean (in quadrature)', f3(np.sqrt(_s50 ** 2 + POL_S50 ** 2 + ALK_D50 ** 2)),
         f3(np.sqrt(_s24 ** 2 + POL_S24 ** 2 + ALK_D24 ** 2)), f2(SDEC_G), '')]
with open(rc.figpath('uncertainty_table', ext='csv'), 'a', newline='') as f_:
    csv.writer(f_).writerows([[c_.replace('$', '') for c_ in r_] for r_ in T3_G])
_n1 = len(T3)
T3 = T3 + T3_G
BOLD3 = {_n1 - len(T3_TOT), _n1, len(T3) - 1}                        # total, header of the second block, global total
SEP3 = {_n1 - len(T3_TOT), _n1, _n1 + 1, len(T3) - 1}
print(f'global mean: decline {DEC_G:.2f} ± {SDEC_G:.2f} % (common area {dec_m:.2f} ± {sdec_m:.2f}; whole ocean '
      f'{POL_SHIFT:+.2f} ± {POL_SIG:.2f}; alkalinity on the whole ocean {ALK_SHIFT:+.2f} ± {ALK_SIG:.2f}, on the common area '
      f'{ALK_SHIFT_C:+.2f}); Omega_arag 1750 {OM50_G:.3f} (whole ocean {POL_D50:+.3f}, alkalinity {ALK_D50:+.3f}), '
      f'2024 {OM24_G:.3f} ({POL_D24:+.3f}, {ALK_D24:+.3f}), boundary {0.8 * OM50_G:.3f}; 1 sigma of Omega_arag 1750 '
      f'{np.sqrt(_s50 ** 2 + POL_S50 ** 2 + ALK_D50 ** 2):.3f}, 2024 {np.sqrt(_s24 ** 2 + POL_S24 ** 2 + ALK_D24 ** 2):.3f}')
NOTE3 = ('All values 1σ; components combined in quadrature; per-product values as root mean square over the four '
         'fCO$_2$-products. Errors shared by the two states cancel largely in the decline, so the decline column is not a '
         f'combination of the Ω$_{{arag}}$ columns (0.01 in Ω$_{{1750}}$ alone changes the decline by about '
         f'{now_m / yc_m ** 2:.2f} points, in Ω$_{{2024}}$ alone by about {1 / yc_m:.2f}). The absolute values are '
         'dominated by the aragonite solubility product (Orr et al., 2018), which is the same in 1750 and 2024 and cancels '
         'in the decline. The mapping error of a fCO$_2$-product shifts its 1750 value (through the bias adjustment) and its 2024 '
         'value alike; the decline then changes only by σ (Ω$_{1750}$ − '
         'Ω$_{2024}$) / Ω$_{1750}^2$. The same mapping error also changes the observed CUC (correlation +0.98 with the '
         'Ω$_{arag}$ error), which moves the constrained 1750 value further and partly offsets that remainder; both are '
         'combined per fCO$_2$-product. This is the uncertainty of each fCO$_2$-product; the spread across the fCO$_2$-products is the difference '
         'between the mapping methods. Both are included. Whole ocean: the ESMs are not bias-adjusted, so their spread of the '
         'level outside the common area is an upper bound; their 1750 state is the 1850 state at 278.3 ppm.')
print('Notes for the legend of Extended Data Table 2:\n' + NOTE3.replace('$', ''))
CW3, RH3 = [5.2, 1.6, 1.6, 2.7, 7.0], 0.3
W3, H3 = X0 * 2 + sum(CW3), 0.35 + RH3 * (len(T3) + 1) + 0.35
fig = plt.figure(figsize=(W3, H3))
ax = fig.add_axes([0, 0, 1, 1])
ax.set_xlim(0, W3)
ax.set_ylim(H3, 0)
ax.axis('off')
xe3 = X0 + np.concatenate([[0], np.cumsum(CW3)])
y = 0.2
ax.plot([X0, W3 - X0], [y, y], color='k', lw=1.2)
for j, h_ in enumerate(HEAD3):
    ax.text(xe3[j] + 0.06, y + RH3 / 2, h_, ha='left', va='center', fontsize=9.5, fontweight='bold')
y += RH3
ax.plot([X0, W3 - X0], [y, y], color='k', lw=0.7)
for k, row in enumerate(T3):
    tot = k in BOLD3
    if k in SEP3:
        ax.plot([X0, W3 - X0], [y, y], color='0.5' if k != _n1 else 'k', lw=0.5 if k != _n1 else 0.9)
    for j, c_ in enumerate(row):
        ax.text(xe3[j] + 0.06, y + RH3 / 2, c_, ha='left', va='center', fontsize=9 if j < 4 else 8.5,
                fontweight='bold' if tot else 'normal', color='k' if j < 4 else '0.3')
    y += RH3
ax.plot([X0, W3 - X0], [y, y], color='k', lw=1.2)
rc.savefig(fig, 'uncertainty_table')
plt.show()

### Numbers for the abstract: the global mean (computed in the preparation cell)

In [ ]:
# The global mean: computed in the preparation cell (GLOBAL,
# GLOBAL_TEXT), printed here again next to the tables. GLOBAL holds decline, sigma, omega_1750, omega_2024, boundary,
# cross_year, cross_dyear, cross_ppm, cross_dppm, frac20_common, frac20_all (with the alkalinity adjustment) and more.
print(GLOBAL_TEXT)


## Extended Data Fig. 4: the two adjustments to the global mean (needs the structural files and the alkalinity files of the data product)

In [ ]:
# Notebook cell: Extended Data Fig. 4, the two adjustments from the common area to the global mean
# (structural_adjustments.py, feedback_pattern.py, barrett_sampling.py, test_polar_shift_models.py, make_product_alkalinity.py).
#   a  the spatial pattern of the calcification feedback in surface alkalinity (four ESMs with Omega-dependent
#      calcification; mean 1 over the common area), with the cells of the GLODAPv2.2023 surface measurements of the
#      cruises of Barrett et al. (2025) over which the observed rate is defined (dots; p = pattern mean over them)
#   b  the missing surface alkalinity on the common area, m(t), as OA-Track adds it: to the series
#      of the constrained ESMs (model formula, mean over the four products' rows; historical to 2014, black, then the four
#      scenarios, colour) and to the fCO2-products (product formula, the full observed rise; mean of the four and their
#      range over the years all four cover, red, dashed)
#   c  the shift of the decline 1750-2024 from the common area to the whole ocean in each of the 18 ESMs (bars; smoothed
#      states of the data product, 1750 from the 1850 state at 278.3 ppm, filled to the ocean of the paper, on the decline
#      of this study), the products (red line; unconstrained 1750 state), Jiang et al. (blue, dash-dot), and the 1 sigma
#      used, three times the spread of the models (band), around the adjustment, the mean of the three (black)
#   d  the level: whole-ocean mean relative to the common-area mean of each ESM in 2024 against 1750 (filled levels), and
#      of the products (2024 against 1750); the two gaps of a model move together (r printed), so they largely cancel in
#      the decline
# Reads the structural files of data/<run>/structural and the alkalinity files of the data product; nothing recomputed.
import netCDF4
import numpy as np
from matplotlib.gridspec import GridSpec
from matplotlib.lines import Line2D
from matplotlib.patches import Patch
from matplotlib.legend_handler import HandlerTuple
import oa_config

rc.FIGNAMES.setdefault('adjustments', 'FigS28_adjustments')
SD = oa_config.DATA / 'structural'
PD = oa_config.DATA / 'product'
SA = np.load(SD / 'structural_adjustments.npz')
FP = np.load(SD / 'feedback_pattern.npz')
BS = np.load(SD / 'barrett_sampling.npz')
A_REFS = [str(q) for q in SA['products']]
A_LAT, A_LON = np.asarray(SA['lat'], 'f8'), np.asarray(SA['lon'], 'f8')
assert np.allclose(FP['lat'], A_LAT) and np.allclose(FP['lon'], A_LON), 'feedback_pattern.npz: grid differs'
PATTERN = np.asarray(SA['alk_pattern'], 'f8')                      # the pattern the adjustment used (FP['pattern'])
A_OCEAN = rc.ocean_on(A_LAT, A_LON).astype(bool)
A_W = oa_config.area_weights(A_LAT, A_LON)
NPTS = np.asarray(BS['n_points'], 'f8')
ok_p = (NPTS > 0) & np.isfinite(PATTERN)                            # as barrett_rate in structural_adjustments.py
p_s = float(np.sum(NPTS[ok_p] * PATTERN[ok_p]) / NPTS[ok_p].sum())
assert abs(p_s - float(SA['alk_p_sampling'])) < 1e-6, f'p from the files {p_s:.4f} differs from the run {float(SA["alk_p_sampling"]):.4f}'

# b: the missing alkalinity, from the alkalinity files of the data product (common-area mean, umol/kg)
for q in A_REFS + ['common']:
    assert (PD / f'carbonate_system_{q}_alkalinity.nc').exists(), f'{PD}/carbonate_system_{q}_alkalinity.nc missing: run make_product_alkalinity.py'
M_PATH, M_SSP = {}, {}
for q in A_REFS:
    with netCDF4.Dataset(PD / f'carbonate_system_{q}_alkalinity.nc') as f_:
        M_PATH[q] = (np.asarray(f_['year'][:]).astype(int), np.ma.filled(f_['talk_missing'][:].astype('f8'), np.nan))
with netCDF4.Dataset(PD / 'carbonate_system_common_alkalinity.nc') as f_:
    yr_ = np.asarray(f_['year'][:]).astype(int)
    m_mod = np.ma.filled(f_['talk_missing_switch1990_models'][:].astype('f8'), np.nan)
    m_pro = np.ma.filled(f_['talk_missing_switch1990_products'][:].astype('f8'), np.nan)
    M_PATH['common'] = (yr_, np.where(yr_ < 1990, m_mod, m_pro))      # switch1990: the models to 1989, then the products
    ys_ = np.asarray(f_['year_ssp'][:]).astype(int)
    for s_ in ('ssp126', 'ssp245', 'ssp370', 'ssp585'):
        if f'talk_missing_{s_}' in f_.variables:
            M_SSP[s_] = (ys_, np.ma.filled(f_[f'talk_missing_{s_}'][:].astype('f8'), np.nan))
SSP_LAB = {'ssp126': 'SSP1-2.6', 'ssp245': 'SSP2-4.5', 'ssp370': 'SSP3-7.0', 'ssp585': 'SSP5-8.5'}
SSP_COL = {'ssp126': '#1f4e79', 'ssp245': '#d8c200', 'ssp370': '#e41a1c', 'ssp585': '#7a0019'}
PROD_COL = '#c0392b'                                               # the fCO2-products (b, c)

# c, d: the 18 ESMs (numbered as in Fig. 2)
FIG2_ORDER = ['ACCESS-ESM1-5', 'CanESM5', 'CanESM5-CanOE', 'CESM2', 'CESM2-WACCM', 'CMCC-ESM2', 'CNRM-ESM2-1', 'EC-Earth3-CC',
              'GFDL-CM4', 'GFDL-ESM4', 'IPSL-CM6A-LR', 'MIROC-ES2L', 'MPI-ESM1-2-HR', 'MPI-ESM1-2-LR', 'MRI-ESM2-0', 'NorESM2-LM',
              'NorESM2-MM', 'UKESM1-0-LL']
A_MODELS = [str(m) for m in SA['polar_models']]
assert set(A_MODELS) == set(FIG2_ORDER), f'models of structural_adjustments.npz: {A_MODELS}'
A_NUM = [FIG2_ORDER.index(m) + 1 for m in A_MODELS]
A_REL = np.asarray(SA['polar_models_rel'], 'f8')                   # shift relative to the model's own decline
A_GAP = 100 * np.asarray(SA['polar_models_gap'], 'f8')             # (model, [1750, 2024]) ocean / common - 1, %
P_GAP = 100 * np.asarray(SA['polar_products_gap'], 'f8')           # products: 1750, 2024
A_DM = float(SA['polar_products_common'])                          # the common-area decline of the products (%)
A_SHIFT_M = A_REL * A_DM                                           # points on the decline of this study
A_SIG = float(SA['polar_sigma'])                                   # 3 x the spread (polar_sigma_factor)
A_F = float(SA['polar_sigma_factor'])
A_SHIFT_P = float(SA['polar_products_shift'])                      # the products
A_SHIFT_J = float(SA['polar_jiang_shift'])                         # Jiang et al., relative, on the decline of this study
A_CEN = float(SA['polar_shift'])                                   # central: mean of the products, the ESMs and Jiang et al.
r_gap = float(np.corrcoef(A_GAP[:, 0], A_GAP[:, 1])[0, 1])
print(f'pattern p over the measurements {p_s:.3f}; common-area rate {float(SA["alk_rate_common"]):.4f} umol/kg per year; '
      f'missing alkalinity 1750 {", ".join(f"{q} {M_PATH[q][1][0]:+.2f}" for q in A_REFS)} (common file {M_PATH["common"][1][0]:+.2f}), '
      f'2024 common file {M_PATH["common"][1][-1]:+.2f}; 2100: ' + ', '.join(f'{SSP_LAB[s_]} {v_[1][-1]:+.1f}' for s_, v_ in M_SSP.items()))
print(f'models: shift {A_SHIFT_M.mean():+.3f} (sd {A_SHIFT_M.std(ddof=1):.3f}; {A_F:g} sd {A_SIG:.3f}) points, products {A_SHIFT_P:+.3f}, Jiang {A_SHIFT_J:+.3f}, central {A_CEN:+.3f}; '
      f'level gap 1750 {A_GAP[:, 0].mean():+.2f} %, 2024 {A_GAP[:, 1].mean():+.2f} %, r {r_gap:+.2f}; products {P_GAP[0]:+.2f} %, {P_GAP[1]:+.2f} %')

fig = plt.figure(figsize=(12, 9.4))
gs = GridSpec(2, 2, figure=fig, height_ratios=[1.0, 0.95], width_ratios=[1.15, 1.0], hspace=0.38, wspace=0.22,
              left=0.06, right=0.98, top=0.96, bottom=0.12)
# ---------------------------------------------------------------- a: the pattern
ax = fig.add_subplot(gs[0, 0])
rc.map_mesh(ax, A_LON, A_LAT, np.where(~A_OCEAN, 1.0, np.nan), cmap=plt.matplotlib.colors.ListedColormap(['0.85']), vmin=0, vmax=1)
# diverging, centred on zero (blue: the rise the other way round), lower end from the data (0.5th percentile, in steps of 0.5)
# the full range of the pattern, rounded outward to 0.25, so that no colour saturates
PAT_LO = min(-0.25, float(np.floor(4 * np.nanmin(PATTERN[A_OCEAN])) / 4))
PAT_HI = float(np.ceil(4 * np.nanmax(PATTERN[A_OCEAN])) / 4)
print(f'a: pattern on the ocean: min {np.nanmin(PATTERN[A_OCEAN]):.2f}, 0.5 % {np.nanpercentile(PATTERN[A_OCEAN], 0.5):.2f}, '
      f'99.5 % {np.nanpercentile(PATTERN[A_OCEAN], 99.5):.2f}, max {np.nanmax(PATTERN[A_OCEAN]):.2f}; colour range {PAT_LO:g} to {PAT_HI:g}; '
      f'negative on {100 * A_W[A_OCEAN & (PATTERN < 0)].sum() / A_W[A_OCEAN & np.isfinite(PATTERN)].sum():.1f} % of the ocean')
# arcsinh scale as Supplementary Fig. 23: full range without saturation, contrast between 0 and 2
ASINH_C = 0.25                    # width of the linear part of the arcsinh scale
pc = rc.map_mesh(ax, A_LON, A_LAT, np.where(A_OCEAN, np.arcsinh(PATTERN / ASINH_C), np.nan), cmap='RdBu_r',
                 norm=plt.matplotlib.colors.TwoSlopeNorm(vmin=np.arcsinh(PAT_LO / ASINH_C), vcenter=0.0, vmax=np.arcsinh(PAT_HI / ASINH_C)))
PAT_TICKS = [t for t in (-15, -5, -2, -1, -0.5, 0, 0.5, 1, 2, 5) if PAT_LO <= t <= PAT_HI]
jy_, ix_ = np.where(ok_p)
ax.scatter(*rc.map_xy(A_LON[ix_], A_LAT[jy_]), s=0.9, color='k', lw=0, alpha=0.7, zorder=5)
cb = fig.colorbar(pc, ax=ax, orientation='horizontal', fraction=0.05, pad=0.03)
cb.set_ticks(np.arcsinh(np.asarray(PAT_TICKS) / ASINH_C))
cb.set_ticklabels([f'{t:g}'.replace('-', '−') for t in PAT_TICKS])
cb.set_label('Pattern of the alkalinity rise (common-area mean = 1; blue: negative; arcsinh scale)', fontsize=9)
ax.set_title(f'Pattern of the feedback in four ESMs\n(dots: surface measurements of Barrett et al., p = {p_s:.2f})', fontsize=9.5)
ax.text(0.0, 1.0, 'a', transform=ax.transAxes, fontsize=16, fontweight='bold', va='bottom')
# ---------------------------------------------------------------- b: m(t)
ax = fig.add_subplot(gs[0, 1])
# the missing alkalinity as OA-Track adds it (structural_adjustments.npz: alk_m_model, alk_m_product, alk_m_<scenario>,
# one row per product): to the series of the constrained ESMs with the model formula (the part of the rise that the
# multi-model mean does not already contain; historical to 2014, then the scenarios; mean over the four products' rows)
# and to the fCO2-products with the product formula (the full observed rise, as they contain none of it; mean of the
# four and their range over the years all four cover).
ya_ = np.asarray(SA['alk_years']).astype(int)
row_ = lambda key: dict(zip(A_REFS, np.asarray(SA[key], 'f8')))
MM, MP = row_('alk_m_model'), row_('alk_m_product')
FIRST = dict(zip(A_REFS, np.asarray(SA['alk_first_product_year']).astype(int)))
mm_ = np.mean([MM[q] for q in A_REFS], axis=0)
kh_ = ya_ <= 2014
ax.plot(ya_[kh_], mm_[kh_], color='k', lw=1.8, zorder=3)
for s_ in SSP_LAB:
    if f'alk_m_{s_}' in SA.files:
        ms_ = np.mean(list(row_(f'alk_m_{s_}').values()), axis=0)
        ks_ = ya_ >= 2014
        ax.plot(ya_[ks_], ms_[ks_], color=SSP_COL[s_], lw=1.4, zorder=3)
Y0_P = max(FIRST.values())                                         # from here on all four products have values
kp_ = (ya_ >= Y0_P) & (ya_ <= rc.LAST_YEAR)
mp_ = np.array([MP[q][kp_] for q in A_REFS])
ax.fill_between(ya_[kp_], mp_.min(axis=0), mp_.max(axis=0), color=PROD_COL, alpha=0.2, lw=0, zorder=2)
ax.plot(ya_[kp_], mp_.mean(axis=0), color=PROD_COL, lw=1.8, ls=(0, (4, 2)), zorder=4)
ax.axhline(0, color='0.6', lw=0.6)
ax.axvline(rc.LAST_YEAR, color='0.6', lw=0.6, ls=':')
ax.set_xlim(1750, 2100)
ax.set_xlabel('Year', fontsize=10)
ax.set_ylabel('Missing surface alkalinity, common area (µmol kg$^{-1}$)', fontsize=9.5)
hb_ = [Line2D([], [], color='k', lw=1.8)] + [Line2D([], [], color=SSP_COL[s_], lw=1.4) for s_ in SSP_LAB] + \
      [(Patch(facecolor=PROD_COL, alpha=0.2, lw=0), Line2D([], [], color=PROD_COL, lw=1.8, ls=(0, (4, 2))))]
lb_ = ['constrained ESMs, 1750–2014'] + [SSP_LAB[s_] for s_ in SSP_LAB] + \
      [f'fCO$_2$-products, {Y0_P}–{rc.LAST_YEAR} (mean and range of the four)']
ax.legend(hb_, lb_, frameon=False, fontsize=8, loc='upper left', handler_map={tuple: HandlerTuple(ndivide=1, pad=0)})
for q in A_REFS:
    print(f'b: {q}: ESMs (model formula) 1750 {MM[q][0]:+.2f}, 2014 {MM[q][ya_ == 2014][0]:+.2f}; product formula '
          f'{FIRST[q]} {MP[q][ya_ == FIRST[q]][0]:+.2f}, {rc.LAST_YEAR} {MP[q][ya_ == rc.LAST_YEAR][0]:+.2f} umol/kg')
print(f'b: products, mean of the four: {Y0_P} {mp_.mean(axis=0)[0]:+.2f}, {rc.LAST_YEAR} {mp_.mean(axis=0)[-1]:+.2f} umol/kg '
      f'(range in {rc.LAST_YEAR} {mp_[:, -1].min():+.2f} to {mp_[:, -1].max():+.2f})')
ax.set_title('Missing surface alkalinity added by the adjustment', fontsize=9.5)
ax.text(-0.12, 1.0, 'b', transform=ax.transAxes, fontsize=16, fontweight='bold', va='bottom')
ax.spines[['top', 'right']].set_visible(False)
# ---------------------------------------------------------------- c: the shift of the decline per model
ax = fig.add_subplot(gs[1, 0])
o_ = np.argsort(A_SHIFT_M)
xs_ = np.arange(len(A_MODELS))
ax.axhspan(A_CEN - A_SIG, A_CEN + A_SIG, color='0.9', lw=0, zorder=0)
ax.axhline(A_CEN, color='k', lw=1.4, zorder=2)
ax.axhline(A_SHIFT_P, color=PROD_COL, lw=1.0, zorder=2)
ax.axhline(A_SHIFT_J, color='#2471a3', lw=1.0, ls='-.', zorder=2)
ax.axhline(A_SHIFT_M.mean(), color='0.4', lw=0.9, ls='--', zorder=2)
ax.bar(xs_, A_SHIFT_M[o_], color='#3b6ea5', width=0.7, zorder=3)
ax.axhline(0, color='0.5', lw=0.6)
ax.set_xticks(xs_)
ax.set_xticklabels([A_MODELS[i_] for i_ in o_], rotation=50, ha='right', fontsize=7.5)
ax.set_ylabel('Whole ocean minus common area\n(percentage points of the decline 1750–2024)', fontsize=9.5)
hc_ = [Patch(facecolor='#3b6ea5'), Line2D([], [], color='0.4', lw=0.9, ls='--'), Line2D([], [], color=PROD_COL, lw=1.0),
       Line2D([], [], color='#2471a3', lw=1.0, ls='-.'), (Patch(facecolor='0.9'), Line2D([], [], color='k', lw=1.4))]
lc_ = ['single ESMs', f'mean of the ESMs ({A_SHIFT_M.mean():+.2f} ± {A_SHIFT_M.std(ddof=1):.2f})',
       f'fCO$_2$-products ({A_SHIFT_P:+.2f})', f'Jiang et al. ({A_SHIFT_J:+.2f})',
       f'adjustment: mean of the three ({A_CEN:+.2f})\n± {A_F:g} s.d. of the ESMs (± {A_SIG:.2f}, 1σ)']
ax.legend(hc_, lc_, frameon=False, fontsize=8, loc='upper left', ncol=2, columnspacing=1.2,
          handler_map={tuple: HandlerTuple(ndivide=1, pad=0)})
ax.set_ylim(0, max(A_SHIFT_M.max(), A_CEN + A_SIG) + 0.17)
ax.set_title('Shift of the decline from the common area to the whole ocean', fontsize=9.5)
ax.text(-0.12, 1.0, 'c', transform=ax.transAxes, fontsize=16, fontweight='bold', va='bottom')
ax.spines[['top', 'right']].set_visible(False)
# ---------------------------------------------------------------- d: the level gaps
ax = fig.add_subplot(gs[1, 1])
lo_, hi_ = min(A_GAP.min(), P_GAP.min()) - 0.4, max(A_GAP.max(), P_GAP.max()) + 0.4
# the decline shift follows from the two gaps: shift = rho (G1750 - G2024) / (1 + G1750 / 100), rho = 2024 / 1750 on the
# common area; on the 1:1 line the decline does not shift, the solid line is the shift used (A_CEN)
rho_ = 1 - A_DM / 100
xl_ = np.array([lo_, hi_])
ax.plot(xl_, xl_, color='0.7', lw=0.8, ls='--', zorder=1, label='same gap in both years: no shift of the decline')
ax.plot(xl_, xl_ - A_CEN * (1 + xl_ / 100) / rho_, color='k', lw=1.0, zorder=1, label=f'shift used ({A_CEN:+.2f} points)')
ax.scatter(A_GAP[:, 0], A_GAP[:, 1], s=28, color='#3b6ea5', zorder=3, label='single ESMs (numbered as in Fig. 2)')
for i_, n_ in enumerate(A_NUM):
    ax.annotate(str(n_), (A_GAP[i_, 0], A_GAP[i_, 1]), xytext=(3, 3), textcoords='offset points', fontsize=7, color='0.3')
ax.scatter(P_GAP[0], P_GAP[1], s=90, marker='*', color='k', zorder=4, label='fCO$_2$-products')
ax.errorbar(A_GAP[:, 0].mean(), A_GAP[:, 1].mean(), xerr=A_F * A_GAP[:, 0].std(ddof=1), yerr=A_F * A_GAP[:, 1].std(ddof=1),
            fmt='s', ms=5, color='0.3', capsize=3, lw=0.9, zorder=2, label=f'mean of the ESMs ± {A_F:g} s.d. (the 1σ used)')
ax.set_xlim(lo_, hi_)
ax.set_ylim(lo_, hi_)
ax.set_aspect('equal')
ax.set_xlabel('Whole-ocean mean $\\Omega_{\\mathrm{arag}}$ relative to\nthe common-area mean in 1750 (%)', fontsize=9.5)
ax.set_ylabel('The same in 2024 (%)', fontsize=9.5)
ax.legend(frameon=False, fontsize=8, loc='upper left')
ax.text(0.97, 0.04, f'ESMs: r = {r_gap:.2f}', transform=ax.transAxes, ha='right', va='bottom', fontsize=8)
ax.set_title('Whole-ocean level relative to the common area', fontsize=9.5)
ax.text(-0.18, 1.0, 'd', transform=ax.transAxes, fontsize=16, fontweight='bold', va='bottom')
ax.spines[['top', 'right']].set_visible(False)
rc.savefig(fig, 'adjustments')
plt.show()


## Supplementary Fig. 1: the fCO2-products in 2024

In [ ]:
# Notebook cell: Supplementary Fig. 1, the four observation-based products in 2024.
# a-d: surface Omega_arag in 2024 for the four products on an equal-area style projection
#      (Robinson with cartopy; Mollweide from matplotlib otherwise). Coloured = cells used (fixed mask,
#      complete in every month); white = ocean without data; grey = land, i.e. cells where none of the
#      products has data in any year (on the products' own 1 deg grid).
# e:   annual global mean over all cells with data in that year, any number of months (no consistent mask;
#      needs calc_omega_arag_anymonth.py, falls back to the 12-month annual means otherwise).
# f:   annual global mean of each product on its own fixed mask.
# g:   the same on the common mask of the analysis (4 products and all ESMs valid; from run_constraint.py).
import oa_config                      # the area weights of the run (OA_WEIGHTS)
import numpy as np
import netCDF4
import sys
sys.path.insert(0, CODE_DIR)
import regional_common as rc
import matplotlib.pyplot as plt
from matplotlib.colors import ListedColormap
from matplotlib.gridspec import GridSpec, GridSpecFromSubplotSpec
from matplotlib.patches import Patch
from matplotlib.transforms import offset_copy

try:
    import cartopy.crs as ccrs
    HAS_CARTOPY = False   # maps are drawn with rc.map_* (Robinson, no cartopy)
except ImportError:
    HAS_CARTOPY = False

ROOT = f'{rc.ROOT}/obs'
PRODUCTS = {'OceanSODA': f'{ROOT}/omega_arag_Oyr_OceanSODA-ETHZ-v2025.nc',
            'UExP': f'{ROOT}/omega_arag_Oyr_UExP-FNN-U-v2026-1.nc',
            'JMA': f'{ROOT}/omega_arag_Oyr_JMA-MLR-v2025.nc',
            'CMEMS': f'{ROOT}/omega_arag_Oyr_CMEMS-LSCE-FFNN.nc'}
# annual means from all available months (>= 1 month), from calc_omega_arag_anymonth.py, for panel e
ANYMONTH = {'OceanSODA': f'{ROOT}/omega_arag_anymonth_Oyr_OceanSODA-ETHZ-v2025.nc',
            'UExP': f'{ROOT}/omega_arag_anymonth_Oyr_UExP-FNN-U-v2026-1.nc',
            'JMA': f'{ROOT}/omega_arag_anymonth_Oyr_JMA-MLR-v2025.nc',
            'CMEMS': f'{ROOT}/omega_arag_anymonth_Oyr_CMEMS-LSCE-FFNN.nc'}
MEAN_PERIOD = (1990, 2024)            # period of the means written into panels e-g
COLORS = {'OceanSODA': 'C0', 'UExP': 'C1', 'JMA': 'C2', 'CMEMS': 'C4'}
FULL = {'OceanSODA': 'OceanSODA-ETHZ v2025', 'UExP': 'UExP-FNN-U v2026-1',
        'JMA': 'JMA-MLR v2025', 'CMEMS': 'CMEMS-LSCE-FFNN'}
YEAR = 2024
ERSST = str(oa_config.OBS2_DIR / 'ERSST/ersst_v6_data/ersst.v6.185001_202512_r.nc')
VMIN, VMAX, CMAP = 1.0, 4.2, 'viridis'
TXT_LON, TXT_LAT = 82.0, 49.0          # where mean and coverage are written on each map
OMEGA = r'$\Omega_{\mathrm{arag}}$'


def to_180(lon, a):
    """Longitudes to -180..180, sorted, with the field reordered to match."""
    l = np.where(lon > 180, lon - 360, lon)
    o = np.argsort(l)
    return l[o], a[..., o]


# ocean for the coverage percentage: ERSSTv6 cells with SST in the last year (incl. ice-covered ocean);
# ERSST centres sit at whole degrees, the products at x.5, so shift by 0.5 deg before the nearest match
with netCDF4.Dataset(ERSST) as f:
    _s = np.ma.filled(f['sst'][-12:].astype('f8'), np.nan)
    _s = _s[:, 0] if _s.ndim == 4 else _s
    OCEAN = np.all(np.isfinite(_s) & (np.abs(_s) < 100), axis=0)
    E_LAT, E_LON = np.asarray(f['lat'][:], 'f8'), np.asarray(f['lon'][:], 'f8')


def ocean_on(lat, lon):
    jy = np.abs(lat[:, None] - E_LAT[None, :]).argmin(1)
    ix = np.abs(((lon[:, None] - 0.5 - E_LON[None, :]) + 180) % 360 - 180).argmin(1)
    return OCEAN[np.ix_(jy, ix)]


# read all products onto the common -179.5..179.5 grid
D = {}
for name, path in PRODUCTS.items():
    with netCDF4.Dataset(path) as f:
        lat = np.asarray(f['lat'][:], 'f8')
        lon0 = np.asarray(f['lon'][:], 'f8')
        yr = np.asarray(f['year'][:])
        om = np.ma.filled(f['omega_arag'][:].astype('f8'), np.nan)
        mask = np.asarray(f['mask'][:]).astype(bool)
    _, mask = to_180(lon0, mask)
    lon, om = to_180(lon0, om)
    D[name] = dict(lat=lat, lon=lon, year=yr, om=om, mask=mask)
    try:
        with netCDF4.Dataset(ANYMONTH[name]) as f:
            _, D[name]['om_any'] = to_180(np.asarray(f['lon'][:], 'f8'),
                                          np.ma.filled(f['omega_arag'][:].astype('f8'), np.nan))
            D[name]['year_any'] = np.asarray(f['year'][:])
    except (FileNotFoundError, OSError):
        print(f'{name}: no any-month file, panel e uses annual means with all 12 months')
lat, lon = D['OceanSODA']['lat'], D['OceanSODA']['lon']
area = oa_config.area_weights(lat, lon)
ANY_DATA = np.any([np.isfinite(d['om']).any(axis=0) for d in D.values()], axis=0)
ersst_ocean = ocean_on(lat, lon)
# ocean for the percentage and the maps: cells where at least one product has data in at least one year, on the
# products' own 1 deg grid. ERSST (2 deg) is not used as denominator: mapped to 1 deg it turns
# coastal land cells into "ocean" and would lower every product's coverage by a few percent.
# inland seas (Black Sea incl. Azov and Marmara, Caspian) count as land: grey on the maps and not in the
# coverage. Boxes in -180..180 degrees east.
# One definition for every percentage of the paper: regional_common.ocean_mask (any month of any year, any product,
# without the inland seas).
ocean = rc.ocean_on(lat, lon)
_miss = ersst_ocean & ~ANY_DATA
_hi = _miss & (np.abs(lat)[:, None] >= 66)
print(f'ERSST ocean not covered by any product: {100 * (area * _miss).sum() / (area * ersst_ocean).sum():.1f} % '
      f'of the ERSST ocean area ({100 * (area * _hi).sum() / (area * ersst_ocean).sum():.1f} % poleward of 66 deg)')

# g: common mask of the analysis (product masks and all 18 ESMs valid), saved by run_constraint.py;
#    without it the intersection of the product masks
import os, pickle
common_prod = np.all([d['mask'] for d in D.values()], axis=0)
CONSTRAINT = f'{rc.ROOT}/constraint_results.pkl'
common_ana = None
if os.path.exists(CONSTRAINT):
    with open(CONSTRAINT, 'rb') as f:
        _c = pickle.load(f)
    _, common_ana = to_180(np.asarray(_c['lon_c'], 'f8'), np.asarray(_c['cm']).astype(bool))
else:
    print('no constraint_results.pkl: panel h is left out (run run_constraint.py)')
pct = lambda m: 100 * (area * (m & ocean)).sum() / (area * ocean).sum()
print(f'common mask of the four products: {pct(common_prod):.1f} % of the ocean')
if common_ana is not None:
    print(f'common mask of the analysis:      {pct(common_ana):.1f} % of the ocean '
          f'({100 * (area * (common_ana & common_prod)).sum() / (area * common_prod).sum():.1f} % of the product common mask; '
          f'cells outside it: {(common_ana & ~common_prod).sum()})')


def gmean_series(d, m):
    w = area * m
    return np.array([np.nansum(np.where(m, o, 0) * w) / (w * (m & np.isfinite(o))).sum() for o in d['om']])


# figure layout: 2 x 2 maps, colorbar, spacer, three time series rows spanning both columns
N_TS = 3
fig = plt.figure(figsize=(12, 16.5 + 2.4 * (N_TS - 3)))
gs = GridSpec(5, 2, figure=fig, height_ratios=[1, 1, 0.06, 0.05, 0.97 * N_TS], hspace=0.32, wspace=0.03,
              left=0.07, right=0.98, top=0.97, bottom=0.05)
proj = {'projection': ccrs.Robinson()} if HAS_CARTOPY else {'projection': 'mollweide'}
kw = {'transform': ccrs.PlateCarree()} if HAS_CARTOPY else {}
xy = (lambda lo, la: (lo, la)) if HAS_CARTOPY else (lambda lo, la: (np.deg2rad(lo), np.deg2rad(la)))
xy = lambda lo, la: (np.asarray(lo, 'f8'), np.asarray(la, 'f8'))   # degrees; projected by rc.map_* (Robinson)
proj = proj_ = {}
kw = {}
X, Y = xy(lon, lat)
letters = iter('abcdefg')

for k, (name, d) in enumerate(D.items()):
    ax = fig.add_subplot(gs[k // 2, k % 2], **proj)
    ax.set_facecolor('white')
    if HAS_CARTOPY:
        ax.set_global()
    else:
        ax.set_xticklabels([])
        ax.set_yticklabels([])
    iy = int(np.where(d['year'] == YEAR)[0][0])
    rc.map_mesh(ax, X, Y, np.where(~ocean, 1.0, np.nan), cmap=ListedColormap(['0.85']),
                  vmin=0, vmax=1, shading='auto', **kw)
    pc = rc.map_mesh(ax, X, Y, np.where(d['mask'] & ocean, d['om'][iy], np.nan), cmap=CMAP, vmin=VMIN, vmax=VMAX,
                       shading='auto', **kw)
    gm = gmean_series(d, d['mask'])[iy]
    cover = 100 * (area * (d['mask'] & ocean)).sum() / (area * ocean).sum()
    ax.set_title(FULL[name], fontsize=11.5, fontweight='bold')
    # values written on central Asia (land), in map coordinates: mean in bold, coverage below
    base = kw.get('transform', ax.transData)
    base = base._as_mpl_transform(ax) if hasattr(base, '_as_mpl_transform') else base
    ax.text(*rc.map_xy(TXT_LON, TXT_LAT), f'Mean: {gm:.2f}', ha='center', va='bottom', fontsize=9.5, fontweight='bold',
            transform=offset_copy(base, fig=fig, y=1, units='points'))
    ax.text(*rc.map_xy(TXT_LON, TXT_LAT), f'{cover:.0f}% coverage', ha='center', va='top', fontsize=9.5,
            transform=offset_copy(base, fig=fig, y=-1, units='points'))
    ax.text(0.0, 1.02, next(letters), transform=ax.transAxes, fontsize=14, fontweight='bold', va='bottom')

cax = fig.add_subplot(gs[2, :])
pos = cax.get_position()
cax.set_position([pos.x0 + 0.25 * pos.width, pos.y0, 0.5 * pos.width, pos.height])
cb = fig.colorbar(pc, cax=cax, orientation='horizontal')
cb.set_label(f'Surface {OMEGA} in {YEAR}', fontsize=12)
fig.legend(handles=[Patch(facecolor='white', edgecolor='0.6', label='ocean without data'),
                    Patch(facecolor='0.85', label='land')],
           loc='center right', bbox_to_anchor=(0.98, pos.y0 + pos.height / 2), fontsize=10, frameon=False)

Y0 = min(int(d['year'][0]) for d in D.values())
Y1 = max(int(d['year'][-1]) for d in D.values())
ALL = np.ones(area.shape, bool)


def series_all(d):
    if 'om_any' in d:
        return d['year_any'], gmean_series(dict(om=d['om_any']), ALL)
    return d['year'], gmean_series(d, ALL)


PANELS = ((series_all, 'Global mean over all cells with data'),
          (lambda d: (d['year'], gmean_series(d, d['mask'])), 'Global mean over cells with data in every month (each fCO$_2$-product separately)'))
if common_ana is not None:
    PANELS += ((lambda d: (d['year'], gmean_series(d, common_ana)),
                'Global mean over cells with data in every month in all four fCO$_2$-products'),)
else:
    PANELS += ((lambda d: (d['year'], gmean_series(d, common_prod)), 'Global mean over cells with data in every month in all four fCO$_2$-products'),)
# time series: centred, two thirds of the figure width, with their own vertical spacing
TS_HSPACE = 0.42
gts = GridSpecFromSubplotSpec(N_TS, 3, subplot_spec=gs[4, :], width_ratios=[1, 4, 1], hspace=TS_HSPACE, wspace=0)
for row, (series, title) in enumerate(PANELS):
    ax = fig.add_subplot(gts[row, 1])
    ax.text(0.99, 0.96, f'Mean {MEAN_PERIOD[0]}\u2013{MEAN_PERIOD[1]}', transform=ax.transAxes, ha='right',
            va='top', fontsize=9, fontweight='bold')
    for k, (name, d) in enumerate(D.items()):
        yy, gg = series(d)
        ax.plot(yy, gg, color=COLORS[name], lw=1.3, label=FULL[name])
        sel = (yy >= MEAN_PERIOD[0]) & (yy <= MEAN_PERIOD[1])
        ax.text(0.99, 0.96, f'{gg[sel].mean():.2f}', color=COLORS[name], fontsize=9, fontweight='bold', ha='right', va='top',
                transform=offset_copy(ax.transAxes, fig=fig, y=-11.5 * (k + 1), units='points'))
    ax.set_title(title, fontsize=11, loc='left', x=0.04)
    ax.set_xlim(Y0, Y1)
    ax.set_ylabel(OMEGA, fontsize=12)
    ax.grid(alpha=0.3)
    ax.text(-0.065, 1.06, next(letters), transform=ax.transAxes, fontsize=14, fontweight='bold', va='bottom')
    if row == 0:
        ax.legend(ncol=2, fontsize=9, frameon=False, loc='lower left')
    if row == len(PANELS) - 1:
        ax.set_xlabel('Year', fontsize=12)

print(f'projection: Robinson centred at {rc.MAP_CENTER:.0f} E (rc.map_*)')
rc.savefig(fig, 'maps_products')
plt.show()

## Supplementary Fig. 3: global mean per fCO2-product

In [ ]:
# Supplementary Fig. 3, one panel per product. Wrapped in a function so that its names (pi_m, yc_m, A_m, ...) do not overwrite
# those of other cells (Fig. 2, Fig. 3, Extended Data Table 3 and Extended Data Fig. 7 use the same names for other things).
import oa_config                      # the area weights of the run (OA_WEIGHTS)
def _fig_products():
    # Global mean surface Omega_arag on the common area, from 1750 to the boundary: the data of every product and of
    # their mean, and panel(), which draws one of them (the mean: Fig. 1; one panel per product: Supplementary Fig. 3).
    #   a (same aspect ratio as b-e, A_HEIGHT times as large, centred): mean of the four products
    #       left column (1750): ESMs (grey dots; bias-adjusted, averaged over the four adjustments), multi-model mean ± 1 sd
    #                           (black), constrained (red, mean of the products ± 1σ)
    #       main axis:          ESMs 1850-2014 (thin grey), multi-model mean (black), mean of the observation-based products
    #                           (red, years covered by all four), projection to 2040 from the fitted 2024 value (CO2 at the
    #                           CO2 growth for constant emissions (projection.co2_growth), T rising with CO2 as observed, S and ALK constant), crossing year
    #       right column:       the same at 80 % (boundary)
    #   b-e: one product each: models adjusted to this product, the product, its
    #        bias-adjustment period (light grey), raw and constrained boundary, 1750 and boundary columns
    # Uncertainties: model spread (raw) and constraint (constrained) only. The CO2 of 1750 (AR6 278.3 ± 2.9 ppm) is
    # printed, not drawn (it moves all values together).
    # Needs data/constraint_results.pkl (with 'states'), the bias-adjusted historical files, data/fig_crossing_projection.pkl
    # (projection.py) and data/co2_annmean_gl.txt.
    import glob
    import os
    import sys
    import pickle
    import importlib
    import numpy as np
    import netCDF4
    import matplotlib.pyplot as plt
    from matplotlib.gridspec import GridSpec, GridSpecFromSubplotSpec
    from matplotlib.patches import Patch
    from matplotlib.lines import Line2D
    from matplotlib.legend_handler import HandlerTuple

    sys.path.insert(0, CODE_DIR)
    import regional_common as rc
    importlib.reload(rc)

    ROOT = rc.ROOT
    LAB = rc.LAB
    OMEGA = r'$\Omega_{\mathrm{arag}}$'
    ADJ = {'OceanSODA': f'{ROOT}/omega_arag_biasadj', 'UExP': f'{ROOT}/omega_arag_biasadj_UExP',
           'JMA': f'{ROOT}/omega_arag_biasadj_JMA', 'CMEMS': f'{ROOT}/omega_arag_biasadj_CMEMS'}
    WIN = {q: oa_config.WINDOWS[q] for q in ('OceanSODA', 'UExP', 'JMA', 'CMEMS')}   # the bias windows of the chain
    YEAR_END = 2040                      # end of the projection in a
    MEAN_COL = 'C3'                      # mean of the products (red; no product uses red)

    with open(f'{ROOT}/constraint_results.pkl', 'rb') as f:
        RC = pickle.load(f)['results_common']
    refs = list(RC)

    # ---------------------------------------------------------------- NOAA CO2, years <-> ppm at a constant growth rate
    co2 = np.array([[float(v) for v in l.split()[:2]] for l in open(f'{ROOT}/co2_annmean_gl.txt')
                    if l.strip() and not l.startswith('#')])
    cy, cx = co2[:, 0].astype(int), co2[:, 1]
    import projection as _pj
    growth = _pj.co2_growth()[0]             # constant emissions at the 2024 level: 2.6 ppm/yr
    y_last = rc.EVAL_YEAR                     # anchor at 2024 (the NOAA file may go further)
    x_last = float(cx[cy == y_last][0])
    year_of = lambda x: y_last + (x - x_last) / growth
    x_of = lambda y: x_last + (y - y_last) * growth

    # ---------------------------------------------------------------- per product: bias-adjusted ESMs on the common mask
    P = {}
    for q, rr in RC.items():
        cmask = np.asarray(rr['mask']).astype(bool)
        area = oa_config.area_weights(rr['lat'], rr['lon'])
        A, nm = [], []
        for fh in sorted(glob.glob(f'{ADJ[q]}/historical/omega_arag_biasadj_Oyr_*_historical_*.nc')):
            model = os.path.basename(fh).split('_')[4]
            if model not in rr['names']:
                continue
            with netCDF4.Dataset(fh) as f:
                yrs = np.asarray(f['year'][:])
                h = np.ma.filled(f['omega_arag_adj'][:].astype('f8'), np.nan)
            m = cmask & np.all(np.isfinite(h), axis=0)
            w = area * m
            A.append(np.nansum(np.where(m, h, 0) * w, axis=(1, 2)) / w.sum())
            nm.append(model)
        pi = np.array([np.asarray(rr['y1750'])[list(rr['names']).index(m_)] for m_ in nm])
        yo = np.asarray(rr['year'])
        keep = yo <= rc.LAST_YEAR
        yc, sc = rr['constrained'][LAB][:2]
        P[q] = dict(A=np.array(A), names=nm, yrs=yrs, pi=pi, yc=yc, sc=sc, yo=yo[keep],
                    ob=np.asarray(rr['ogm'])[keep], col=rc.COLORS[q], title=rc.FULL[q], win=WIN[q])

    # ---------------------------------------------------------------- mean of the products (panel a)
    names = [m for m in P[refs[0]]['names'] if all(m in P[q]['names'] for q in refs)]
    A_m = np.mean([[P[q]['A'][P[q]['names'].index(m)] for m in names] for q in refs], axis=0)       # (model, year)
    pi_m = np.mean([[P[q]['pi'][P[q]['names'].index(m)] for m in names] for q in refs], axis=0)
    yo_m = np.array(sorted(set.intersection(*[set(P[q]['yo'].tolist()) for q in refs])))          # years of all four
    ob_m = np.mean([[P[q]['ob'][list(P[q]['yo']).index(y)] for y in yo_m] for q in refs], axis=0)
    yc_m = np.mean([P[q]['yc'] for q in refs])
    sc_m = np.sqrt(np.mean([P[q]['sc'] ** 2 for q in refs]))
    P_mean = dict(A=A_m, yrs=P[refs[0]]['yrs'], pi=pi_m, yc=yc_m, sc=sc_m, yo=yo_m, ob=ob_m, col=MEAN_COL,
                  title='Mean of the four fCO$_2$-products',
                  win=(max(w[0] for w in WIN.values()), min(w[1] for w in WIN.values())))       # 1990-2014, common to all four

    # projection from the fitted 2024 value of each product, mean over the products, crossing year
    import projection
    importlib.reload(projection)
    proj = projection.global_projection()        # cached in the run folder (projection.py)
    X_NOW = dict(zip(cy, cx))[rc.EVAL_YEAR]
    proj = {q: rc.anchor_projection(proj[q], RC[q], X_NOW) for q in refs}
    xg = np.arange(X_NOW, x_of(YEAR_END) + 1, 0.5)
    yg = year_of(xg)
    curve = np.mean([np.interp(xg, proj[q]['x'], proj[q]['scaled']) for q in refs], axis=0)


    def cross(c, level):
        i = np.where(c <= level)[0][0]
        return yg[i - 1] + (c[i - 1] - level) / (c[i - 1] - c[i]) * (yg[i] - yg[i - 1])


    B, B_sig = 0.8 * yc_m, 0.8 * sc_m
    y_prod = {q: cross(np.interp(xg, proj[q]['x'], proj[q]['scaled']), 0.8 * P[q]['yc']) for q in refs}
    y_cross = np.mean(list(y_prod.values()))
    # crossing range: the full 1 sigma of the decline shared by the products (Extended Data Table 2 without the product spread,
    # SDEC_SHARED of the preparation cell), as an Omega_arag offset of the boundary, and the spread of the products
    if 'SDEC_SHARED' not in globals():
        raise NameError('SDEC_SHARED not defined: run the preparation cell first (it comes before this cell)')
    s_om = SDEC_SHARED / 100 * yc_m
    dy = np.hypot(np.std(list(y_prod.values()), ddof=1), (cross(curve, B - s_om) - cross(curve, B + s_om)) / 2)
    S = RC[refs[0]]['states']
    s_co2 = 0.5 * yc_m * abs(S['1750_hi']['ratio'] - S['1750_lo']['ratio'])
    dy_co2 = (cross(curve, 0.8 * (yc_m - s_co2)) - cross(curve, 0.8 * (yc_m + s_co2))) / 2
    for lab_, p in list(P.items()) + [('mean', P_mean)]:
        print(f'{lab_:10s} 1750 models {p["pi"].mean():.3f} ± {p["pi"].std(ddof=1):.3f}, constrained {p["yc"]:.3f} ± '
              f'{p["sc"]:.3f}; boundary {0.8 * p["yc"]:.3f} ± {0.8 * p["sc"]:.3f}')
    print(f'crossing {y_cross:.1f} ± {dy:.1f} (' + ', '.join(f'{q} {v:.1f}' for q, v in y_prod.items())
          + f'); 1 sigma: decline {SDEC_SHARED:.2f} percentage points shared (Extended Data Table 2, of which the CO2 of 1750 '
          f'± {dy_co2:.1f} years) and the spread of the products')

    # common y-limits
    lo = min(min(p['A'].min(), p['ob'].min(), 0.8 * p['pi'].min()) for p in P.values()) - 0.03
    lo = min(lo, B - B_sig - 0.07)                     # room for the crossing label below the boundary in a
    hi = max(max(p['A'].max(), p['ob'].max(), p['pi'].max()) for p in P.values()) + 0.03


    # ---------------------------------------------------------------- one panel (1750 column | time series | boundary)
    def panel(spec, p, letter, big, fig_=None):
        fg = fig if fig_ is None else fig_                # figure to draw into (Fig. 4 passes its own)
        fs = 10 if big else 8                               # font size of labels inside the panel
        ms = 4.5 if big else 3
        sub = GridSpecFromSubplotSpec(1, 3, subplot_spec=spec, width_ratios=[0.13, 1, 0.17], wspace=0.08)
        ax0 = fg.add_subplot(sub[0])
        ax = fg.add_subplot(sub[1], sharey=ax0)
        ax2 = fg.add_subplot(sub[2], sharey=ax0)
        jit = np.random.default_rng(1).uniform(-0.18, 0.18, p['pi'].size)
        bc, bs = 0.8 * p['yc'], 0.8 * p['sc']

        def column(a_, v, cval, sval, below):
            a_.scatter(-0.25 + jit, v, s=12 if big else 6, color='0.55', lw=0, zorder=2)
            a_.errorbar(0.25, v.mean(), yerr=v.std(ddof=1), fmt='o', color='k', ms=ms, capsize=3, lw=1.1, zorder=3)
            a_.errorbar(0.65, cval, yerr=sval, fmt='o', color=p['col'], ms=ms, capsize=3, lw=1.3, zorder=4)
            if below:
                y_, va = min(v.min(), v.mean() - v.std(ddof=1), cval - sval) - 0.02, 'top'
            else:
                y_, va = max(v.max(), v.mean() + v.std(ddof=1), cval + sval) + 0.02, 'bottom'
            for x_, t_, c_ in ((-0.25, 'ESMs', '0.45'), (0.25, 'multi-model mean', 'k'), (0.65, 'constrained', p['col'])):
                a_.text(x_, y_, t_, rotation=90, ha='center', va=va, fontsize=fs - 0.5, color=c_)
            a_.set_xlim(-0.6, 0.9)
            a_.set_xticks([0.15])
            a_.spines[['top', 'right']].set_visible(False)
            a_.tick_params(labelsize=fs)

        column(ax0, p['pi'], p['yc'], p['sc'], True)
        ax0.set_xticklabels(['1750'])
        ax0.text(-0.9 if big else -1.0, 1.02, letter, fontsize=16 if big else 13, fontweight='bold', va='bottom', transform=ax0.transAxes)
        column(ax2, 0.8 * p['pi'], bc, bs, False)
        ax2.set_xticklabels(['boundary'])
        ax2.tick_params(labelleft=False)

        A, yrs = p['A'], p['yrs']
        if p['win'] is not None:
            ax.axvspan(*p['win'], color='0.92', lw=0, zorder=0)
        for a in A:
            ax.plot(yrs, a, color='0.78', lw=0.6, zorder=1)
        ax.plot(yrs, A.mean(0), color='k', lw=1.5 if big else 1.2, zorder=2)
        ax.plot(p['yo'], p['ob'], color=p['col'], lw=2.0 if big else 1.4, zorder=4)
        bu, bus = 0.8 * p['pi'].mean(), 0.8 * p['pi'].std(ddof=1)
        for y_ in (bu - bus, bu + bus):
            ax.axhline(y_, color='0.45', ls=(0, (3, 3)), lw=0.5, zorder=2)
        ax.axhline(bu, color='0.3', ls=(0, (4, 4)), lw=1.0, zorder=2)
        ax.axhspan(bc - bs, bc + bs, color=p['col'], alpha=0.3, lw=0, zorder=1)
        ax.axhline(bc, color=p['col'], ls=(4, (4, 4)), lw=1.0, zorder=3)
        ax.set_ylim(lo, hi)
        ax.spines[['top', 'right']].set_visible(False)
        ax.tick_params(labelleft=False, labelsize=fs)
        ax.set_title(p['title'], fontsize=13 if big else 10, fontweight='bold')

        if big:
            ax.plot(yg, curve, color=p['col'], lw=2.0, ls=(0, (3, 2)), zorder=4)
            # crossing: point on the boundary, label in the empty upper right, connected by a thin arrow
            ax.plot(y_cross, B, 'o', ms=6, mfc='white', mec='k', mew=1.2, zorder=6)
            lab_x = p['win'][1] + 1.0                      # right of the bias-adjustment shading
            ax.annotate(f'boundary\ncrossed ≈ {y_cross:.0f}\n({y_cross - dy:.0f}–{y_cross + dy:.0f})', xy=(y_cross, B),
                        xytext=(lab_x, B + 0.25 * (hi - B)), ha='left', va='bottom', fontsize=fs - 2, linespacing=1.3,
                        arrowprops=dict(arrowstyle='-|>,head_length=0.25,head_width=0.12', color='0.4', lw=0.6, shrinkA=2, shrinkB=5,
                                        connectionstyle='arc3,rad=0.0'), zorder=6)
            ax.set_xlim(yrs[0], YEAR_END)
            ax0.set_ylabel(f'Global mean surface {OMEGA}', fontsize=12)
            hh = [Line2D([], [], color='0.7', lw=0.8), Line2D([], [], color='k', lw=1.5),
                  Line2D([], [], color=p['col'], lw=2.0), Line2D([], [], color=p['col'], lw=2.0, ls=(0, (3, 2))),
                  Patch(facecolor='0.92', lw=0), Line2D([], [], color='0.3', ls=(0, (4, 4)), lw=1.0),
                  (Patch(facecolor=p['col'], alpha=0.3, lw=0), Line2D([], [], color=p['col'], ls=(4, (4, 4)), lw=1.0))]
            ll = ['ESMs (bias-adjusted)', 'multi-model mean',
                  'fCO$_2$-products (mean of four)',
                  'idealised projection', 'bias-adjustment period',
                  f'boundary, raw models ({bu:.2f} $\\pm$ {bus:.2f})', f'boundary, constrained ({bc:.2f} $\\pm$ {bs:.2f})']
            ax.legend(hh, ll, fontsize=fs - 1, frameon=False, loc='lower left', bbox_to_anchor=(0.0, 0.14),
                      handler_map={tuple: HandlerTuple(ndivide=1, pad=0)})
        else:
            ax.set_xlim(yrs[0], rc.LAST_YEAR)
            # boundary values as a small legend just above the boundary lines, with the line styles of a
            hh = [Line2D([], [], color='0.3', ls=(0, (4, 4)), lw=1.0),
                  (Patch(facecolor=p['col'], alpha=0.3, lw=0), Line2D([], [], color=p['col'], ls=(4, (4, 4)), lw=1.0))]
            y_txt = (max(bc + bs, bu + bus) + 0.01 - lo) / (hi - lo)              # axes fraction
            leg = ax.legend(hh, [f'Raw: {bu:.2f} $\\pm$ {bus:.2f}', f'Constrained: {bc:.2f} $\\pm$ {bs:.2f}'],
                            title='Boundary', title_fontproperties={'weight': 'bold', 'size': fs}, fontsize=fs,
                            frameon=False, loc='lower left', bbox_to_anchor=(0.0, y_txt), handlelength=2.2,
                            borderaxespad=0.2, handler_map={tuple: HandlerTuple(ndivide=1, pad=0)})
            leg._legend_box.align = 'left'
        return ax0, ax


    # ---------------------------------------------------------------- Supplementary Fig. 3: one panel per product
    fig = plt.figure(figsize=(13.5, 8.4))
    gs = GridSpec(2, 2, figure=fig, hspace=0.34, wspace=0.16, left=0.07, right=0.99, top=0.95, bottom=0.08)
    for k, q in enumerate(refs):
        ax0, ax = panel(gs[k // 2, k % 2], P[q], 'abcd'[k], False)
        if k % 2 == 0:
            ax0.set_ylabel(f'Surface {OMEGA}', fontsize=10)
        if k // 2 == 1:
            ax.set_xlabel('Year', fontsize=10)
    rc.savefig(fig, 'global_products')
    plt.show()


_fig_products()

## Supplementary Fig. 5: emergent constraint per fCO2-product

In [ ]:
import oa_config                      # the bias windows of the chain (oa_config.WINDOWS)
# Notebook cell: Supplementary Fig. 5, the emergent constraint (dT + CUC) on preindustrial (1750) Omega_arag.
# One figure, one row per product (REFS_PLOT, product name left of the row), the four columns titled and lettered a-d
# in the first row only; the key of the model numbers once on the right. Per product:
# a, b: partial regressions for one product (REF). Each model's bias-adjusted 1750 Omega is moved along the
#       other predictor to its observed value, so that the panel shows the effect of one predictor alone:
#         a: y - b_dT  * (dT  - dT_obs)  against CUC
#         b: y - b_CUC * (CUC - CUC_obs) against dT
#       Line: the regression at the observed value of the other predictor, band: 1 sigma prediction
#       interval of the fit. Vertical line and band: observed value and its uncertainty.
# c:    both predictors combined: model 1750 Omega against b0 + b1 dSST + b2 CUC, 1:1 line, observation.
# d:    1750 Omega without (grey) and with (colour) constraint for REF; dots: the 18 models, binned in
#       0.005 bins and stacked (a dot histogram; the binning belongs in the caption).
# Needs data/constraint_results.pkl from run_constraint.py. All on the common mask.
import pickle
import numpy as np
import sys
sys.path.insert(0, CODE_DIR)
import regional_common as rc
import matplotlib.pyplot as plt
from matplotlib.gridspec import GridSpec, GridSpecFromSubplotSpec
from matplotlib.lines import Line2D
from matplotlib.colors import to_rgb

ROOT = rc.ROOT
REFS_PLOT = ['OceanSODA', 'UExP', 'JMA', 'CMEMS']    # one row per product, in this order
LAB = 'dT + CUC'
COLORS = {'OceanSODA': 'C0', 'UExP': 'C1', 'JMA': 'C2', 'CMEMS': 'C4'}
FULL = {'OceanSODA': 'OceanSODA-ETHZ v2025', 'UExP': 'UExP-FNN-U v2026-1',
        'JMA': 'JMA-MLR v2025', 'CMEMS': 'CMEMS-LSCE-FFNN'}
LABEL_MODELS = True                  # number each model dot (key printed; same order in every figure)
EST_DARKEN = 0.35                   # constrained estimate: product colour mixed with this much black
PARTIAL = False                     # True: models moved along the other predictor (partial regression);
                                    # False: raw model values with a single-predictor regression
WIN = {q: oa_config.WINDOWS[q] for q in ('OceanSODA', 'UExP', 'JMA', 'CMEMS')}   # the bias windows of the chain
OMEGA = r'$\Omega_{\mathrm{arag}}$'

with open(f'{ROOT}/constraint_results.pkl', 'rb') as f:
    RC = pickle.load(f)['results_common']


def plot_constraint(REF, fig, gs, row, last):
    """the four panels of one product in row `row` of the GridSpec `gs` (columns 0-3), letters from 4 * row"""
    r = RC[REF]
    NUM = {m_: k_ + 1 for k_, m_ in enumerate(sorted(r['names'], key=str.lower))}   # alphabetical (as Fig. 2)
    dT, CUC, y = np.asarray(r['dT']), np.asarray(r['CUC']), np.asarray(r['y1750'])
    dT_obs, dT_sig = r['dT_obs']
    CUC_obs = r['CUC_obs']
    CUC_SIG = rc.cuc_sigma(REF, 'results_common')[0]   # observed CUC: the product's mapping uncertainty (vertical band)
    w0, w1 = WIN[REF]                    # window of the product (CUC state and dSST)

    # the regression of the constraint (same as fit() in constraint_refs.py)
    X = np.column_stack([np.ones(y.size), dT, CUC])
    beta, *_ = np.linalg.lstsq(X, y, rcond=None)
    res = y - X @ beta
    s2 = res @ res / (y.size - X.shape[1])
    cov = s2 * np.linalg.inv(X.T @ X)
    r2 = 1 - res @ res / ((y - y.mean()) @ (y - y.mean()))
    yc, stot, rng = r['constrained'][LAB]
    yu, su = r['unconstrained'][:2]
    if LABEL_MODELS:
        print('model numbers: ' + ', '.join(f'{NUM[m]} {m}' for m in sorted(r['names'], key=str.lower)))
    print(f'{REF}: y = {beta[0]:.3f} {beta[1]:+.4f} dT {beta[2]:+.3f} CUC, R2 {r2:.2f}; '
          f'constrained {yc:.3f} +- {stot:.3f}, unconstrained {yu:.3f} +- {su:.3f}')


    def pred_band(xx, which):
        """regression and 1 sigma prediction interval along one predictor, the other at its observed value"""
        X0 = np.column_stack([np.ones(xx.size), xx if which == 'dT' else np.full(xx.size, dT_obs),
                              xx if which == 'CUC' else np.full(xx.size, CUC_obs)])
        m = X0 @ beta
        s = np.sqrt(s2 + np.einsum('ij,jk,ik->i', X0, cov, X0))
        return m, s


    def model_points(ax, xv, yv):
        """model dots; with LABEL_MODELS a number in each dot (1..18, alphabetical, see the table printed)"""
        if not LABEL_MODELS:
            ax.scatter(xv, yv, s=28, color='0.35', edgecolor='white', lw=0.5, zorder=3)
            return
        ax.scatter(xv, yv, s=90, color='0.35', edgecolor='white', lw=0.5, zorder=3)
        for i_, (xi, yi) in enumerate(zip(xv, yv)):
            ax.text(xi, yi, str(NUM[r['names'][i_]]), ha='center', va='center_baseline', fontsize=6, color='white',
                    fontweight='bold', zorder=4)

    def note(ax, txt, slope):
        """grey text in the lower corner that the regression line leaves empty: lower right for a
        rising line, lower left for a falling one"""
        right = slope > 0
        ax.text(0.98 if right else 0.02, 0.03, txt, transform=ax.transAxes, ha='right' if right else 'left',
                va='bottom', fontsize=9.5, color='0.3', linespacing=1.4)

    ax_a = fig.add_subplot(gs[row, 0])
    ax_b = fig.add_subplot(gs[row, 1], sharey=ax_a)
    ax_e = fig.add_subplot(gs[row, 2], sharey=ax_a)
    c = COLORS[REF]
    est = tuple((1 - EST_DARKEN) * np.array(to_rgb(c)))     # darker shade of the product colour

    for ax, x, x_obs, x_sig, which, xlabel in (
            (ax_a, CUC, CUC_obs, CUC_SIG, 'CUC',
         f'CUC, {w0}$-${w1} ' r'($\mu$mol kg$^{-1}$ $\mu$atm$^{-1}$)'),
            (ax_b, dT, dT_obs, dT_sig, 'dT', f'$\\Delta$SST, {w0}$-${w1} minus 1850$-$1900 ($^\\circ$C)')):
        pad = 0.1 * (x.max() - x.min())
        xx = np.linspace(min(x.min(), x_obs - x_sig) - pad, max(x.max(), x_obs + x_sig) + pad, 200)
        if PARTIAL:
            # models moved along the other predictor to its observed value; line from the 2-predictor fit
            other = beta[1] * (dT - dT_obs) if which == 'CUC' else beta[2] * (CUC - CUC_obs)
            ya = y - other
            m, s = pred_band(xx, which)
        else:
            # raw model values against one predictor; line: simple regression on this predictor alone
            ya = y
            X1_ = np.column_stack([np.ones(y.size), x])
            b1, *_ = np.linalg.lstsq(X1_, y, rcond=None)
            r1 = y - X1_ @ b1
            s21 = r1 @ r1 / (y.size - 2)
            cov1 = s21 * np.linalg.inv(X1_.T @ X1_)
            X0 = np.column_stack([np.ones(xx.size), xx])
            m = X0 @ b1
            s = np.sqrt(s21 + np.einsum('ij,jk,ik->i', X0, cov1, X0))
            r2_1 = 1 - r1 @ r1 / ((y - y.mean()) @ (y - y.mean()))
            pname = 'CUC' if which == 'CUC' else r'$\Delta$SST'
            note_txt = f'$R^2$ = {r2_1:.2f}\n({pname} alone)'
        ax.fill_between(xx, m - s, m + s, color='0.85', lw=0, zorder=0)
        ax.plot(xx, m, color='k', lw=1.2, zorder=1)
        model_points(ax, x, ya)
        ax.axvspan(x_obs - x_sig, x_obs + x_sig, color=c, alpha=0.18, lw=0, zorder=0)
        ax.axvline(x_obs, color=c, lw=1.5, zorder=2)
        ax.set_xlim(xx[0], xx[-1])
        ax.set_xlabel(xlabel, fontsize=11)
        ax.spines[['top', 'right']].set_visible(False)
        if not PARTIAL:
            note(ax, note_txt, b1[1])

    # c: both predictors combined, x = regression value b0 + b1 dSST + b2 CUC of each model; 1:1 line
    xhat = X @ beta
    x0 = beta @ np.array([1.0, dT_obs, CUC_obs])
    sx0 = np.hypot(beta[1] * dT_sig, beta[2] * CUC_SIG)
    pad = 0.1 * (xhat.max() - xhat.min())
    xx = np.linspace(min(xhat.min(), x0 - sx0) - pad, max(xhat.max(), x0 + sx0) + pad, 50)
    ax_e.fill_between(xx, xx - np.sqrt(s2), xx + np.sqrt(s2), color='0.85', lw=0, zorder=0)
    ax_e.plot(xx, xx, color='k', lw=1.2, zorder=1)
    model_points(ax_e, xhat, y)
    ax_e.axvspan(x0 - sx0, x0 + sx0, color=c, alpha=0.18, lw=0, zorder=0)
    ax_e.axvline(x0, color=c, lw=1.5, zorder=2)
    # constrained estimate: horizontal line and +-1 sigma (fit and observational uncertainty combined)
    ax_e.axhspan(yc - stot, yc + stot, color=est, alpha=0.18, lw=0, zorder=0)
    ax_e.axhline(yc, color=est, lw=1.3, ls='--', zorder=2)
    ax_e.set_xlim(xx[0], xx[-1])
    sg = lambda v: '+' if v >= 0 else '$-$'
    ax_e.set_xlabel(f'{beta[2]:.2f} CUC {sg(beta[1])} {abs(beta[1]):.3f} $\\Delta$SST {sg(beta[0])} {abs(beta[0]):.2f}',
                    fontsize=11)
    ax_e.spines[['top', 'right']].set_visible(False)
    ax_e.tick_params(labelleft=False)
    note(ax_e, f'$R^2$ = {r2:.2f}\n(both predictors)', 1.0)
    ax_e.text(0.02, (yc + stot - ax_e.get_ylim()[0]) / np.ptp(ax_e.get_ylim()) + 0.01,
              f'constrained\n{yc:.3f} $\\pm$ {stot:.3f}', transform=ax_e.transAxes, ha='left', va='bottom',
              fontsize=9.5, color=est, linespacing=1.3)

    ax_a.set_ylabel(f'Preindustrial (1750) {OMEGA}', fontsize=12)
    ax_b.tick_params(labelleft=False)
    # the product of the row, left of the row and turned by 90 degrees; the column titles in the first row only
    ax_a.text(-0.24, 0.5, FULL[REF], transform=ax_a.transAxes, rotation=90, ha='right', va='center', fontsize=13,
              fontweight='bold')
    if PARTIAL:
        ax_a.text(0.98, 0.03, f'models moved to the observed $\\Delta$SST', transform=ax_a.transAxes, ha='right',
                  fontsize=9, color='0.3')
        ax_b.text(0.98, 0.03, 'models moved to the observed CUC', transform=ax_b.transAxes, ha='right',
                  fontsize=9, color='0.3')

    # d: before and after
    ax_c = fig.add_subplot(gs[row, 3])
    yy = np.linspace(min(yu - 3.5 * su, y.min() - 0.005), max(yu + 3.5 * su, y.max() + 0.005), 400)
    gauss = lambda mu, sd: np.exp(-0.5 * ((yy - mu) / sd) ** 2) / (sd * np.sqrt(2 * np.pi))
    gu, gc = gauss(yu, su), gauss(yc, stot)
    ax_c.fill_between(yy, 0, gu, color='0.6', alpha=0.35, lw=0, zorder=1, label='models without constraint')
    ax_c.fill_between(yy, 0, gc, color=est, alpha=0.3, lw=0, label='constrained')
    ax_c.plot(yy, gu, color='0.4', lw=1)
    ax_c.plot(yy, gc, color=est, lw=1.2)
    top = gc.max()
    ax_c.text(yc, top * 1.04, f'{yc:.3f} $\\pm$ {stot:.3f}', ha='center', va='bottom', fontsize=9.5, color=est)
    ax_c.text(yu - 1.6 * su, gauss(yu, su)[np.argmin(abs(yy - (yu - 1.6 * su)))] + 0.03 * top,
              f'{yu:.3f} $\\pm$ {su:.3f}', ha='right', va='bottom', fontsize=9.5, color='0.35')
    # the 18 models as a dot histogram: binned (BIN wide), one dot per model stacked at the bin centre
    BIN = 0.005
    DOT_H = (0.1 if LABEL_MODELS else 0.075) * top     # vertical distance of stacked dots
    ctr = (np.floor(y / BIN) + 0.5) * BIN
    xs, ys, ids = [], [], []
    for c_ in np.unique(ctr):
        members = [i_ for i_ in np.argsort(y) if np.isclose(ctr[i_], c_)]
        for k_, i_ in enumerate(members):
            xs.append(c_)
            ys.append(DOT_H * (0.6 + k_))
            ids.append(i_)
    ax_c.scatter(xs, ys, s=90 if LABEL_MODELS else 32, color='0.3', edgecolor='white', lw=0.6, zorder=5,
                 label=f'models ({y.size})')
    if LABEL_MODELS:
        for xi, yi, i_ in zip(xs, ys, ids):
            ax_c.text(xi, yi, str(NUM[r['names'][i_]]), ha='center', va='center_baseline', fontsize=6, color='white',
                      fontweight='bold', zorder=6)
    ax_c.set_ylim(0, top * 1.2)
    ax_c.set_xlim(yy[0], yy[-1])
    ax_c.set_yticks([])
    ax_c.spines[['top', 'right', 'left']].set_visible(False)
    ax_c.set_xlabel(f'Preindustrial (1750) {OMEGA}', fontsize=12)
    ax_c.legend(fontsize=9, frameon=False, loc='upper left')

    if row == 0:
        for ax, t_ in zip((ax_a, ax_b, ax_e, ax_c), COL_TITLES):
            ax.set_title(t_, fontsize=12.5, pad=14)
    if row == 0:                                   # letters a-d for the columns; the rows are named by the product
        for ax, l in ((ax_a, 'a'), (ax_b, 'b'), (ax_e, 'c'), (ax_c, 'd')):
            ax.text(-0.03 if ax is ax_c else -0.08, 1.03, l, transform=ax.transAxes, fontsize=14, fontweight='bold',
                    va='bottom', ha='right' if ax is ax_c else 'left')
    return sorted(r['names'], key=str.lower)


COL_TITLES = ('Partial regression on CUC', 'Partial regression on $\\Delta$SST', 'Both predictors',
              'Without and with the constraint')
refs_ = [q for q in REFS_PLOT if q in RC]
fig = plt.figure(figsize=(22.6, 5.2 * len(refs_) + 0.3))
gs = GridSpec(len(refs_), 5, figure=fig, width_ratios=[1, 1, 1, 1.15, 0.42], hspace=0.42, wspace=0.14,
              left=0.07, right=0.995, top=1 - 0.75 / (5.2 * len(refs_) + 0.3), bottom=0.04)
names_ = None
for i_, REF in enumerate(refs_):
    names_ = plot_constraint(REF, fig, gs, i_, i_ == len(refs_) - 1)
if LABEL_MODELS:
    # the key of the model numbers (alphabetical, the same in every row), once, on the right
    ax_k = fig.add_subplot(gs[:, 4])
    ax_k.axis('off')
    ax_k.text(0.02, 0.62, 'models', transform=ax_k.transAxes, fontsize=10, color='0.35', va='bottom', fontweight='bold')
    for n_, m_ in enumerate(names_):
        ax_k.text(0.02, 0.61 - n_ * 0.0125, f'{n_ + 1:2d}  {m_}', transform=ax_k.transAxes, fontsize=9, color='0.35',
                  va='top', ha='left')
rc.savefig(fig, 'constraint_product')
plt.show()

## Supplementary Fig. 6: ΔSST bias of the ESMs

In [ ]:
# Notebook cell: SI map, model minus observed SST change per cell, for each product window.
# dSST = window mean minus 1850-1900 (models drift-corrected, as in the constraint);
# observations: mean of ERSSTv6, COBE-SST3 and DCENT-I. Positive: models warm more than observed.
# Dots: |model mean - obs mean| smaller than the spread of the three SST products (1 sigma).
# Needs regional_common.py and data/constraint_results.pkl (run_constraint.py, STORE_FIELDS = True).
import oa_config                      # the area weights of the run (OA_WEIGHTS)
import sys
import pickle
import importlib
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.gridspec import GridSpec, GridSpecFromSubplotSpec
from matplotlib.colors import ListedColormap

sys.path.insert(0, CODE_DIR)
import regional_common as rc
importlib.reload(rc)

try:
    import cartopy.crs as ccrs
    HAS_CARTOPY = False   # maps are drawn with rc.map_* (Robinson, no cartopy)
except ImportError:
    HAS_CARTOPY = False

VLIM = 1.5                                   # colour range (deg C)
WIN = {q: '{}-{}'.format(*oa_config.WINDOWS[q]) for q in ('OceanSODA', 'UExP', 'JMA', 'CMEMS')}   # the bias windows of the chain

with open(f'{rc.ROOT}/constraint_results.pkl', 'rb') as f:
    RC = pickle.load(f)['results_common']
land = rc.load()[list(RC)[0]]['land']

fig = plt.figure(figsize=(12, 7.4))
gs = GridSpec(3, 2, figure=fig, height_ratios=[1, 1, 0.05], hspace=0.18, wspace=0.05,
              left=0.03, right=0.97, top=0.95, bottom=0.08)
proj = {'projection': ccrs.Robinson()} if HAS_CARTOPY else {'projection': 'mollweide'}
kw = {'transform': ccrs.PlateCarree()} if HAS_CARTOPY else {}
xy = (lambda lo, la: (lo, la)) if HAS_CARTOPY else (lambda lo, la: (np.deg2rad(lo), np.deg2rad(la)))
xy = lambda lo, la: (np.asarray(lo, 'f8'), np.asarray(la, 'f8'))   # degrees; projected by rc.map_* (Robinson)
proj = proj_ = {}
kw = {}

for k, (ref, r) in enumerate(RC.items()):
    F = r['fields']
    lat, lon0 = np.asarray(r['lat'], 'f8'), np.asarray(r['lon'], 'f8')
    mask = np.asarray(r['mask']).astype(bool)
    dT = F['dT'].astype('f8')
    dobs = np.array([F['dsst_obs'][q] for q in F['dsst_obs']], 'f8')
    mod = np.where(mask, dT.mean(0), np.nan)
    obs = np.where(mask, dobs.mean(0), np.nan)
    sd_obs = np.where(mask, dobs.std(0, ddof=1), np.nan)
    diff = mod - obs
    lon, diff = rc.to_180(lon0, diff)
    _, small = rc.to_180(lon0, np.where(np.abs(mod - obs) < sd_obs, 1.0, np.nan))
    X, Y = xy(lon, lat)
    ax = fig.add_subplot(gs[k // 2, k % 2], **proj)
    if HAS_CARTOPY:
        ax.set_global()
    else:
        ax.set_xticklabels([])
        ax.set_yticklabels([])
    rc.map_mesh(ax, X, Y, np.where(land, 1.0, np.nan), cmap=ListedColormap(['0.85']), vmin=0, vmax=1,
                  shading='auto', **kw)
    pc = rc.map_mesh(ax, X, Y, diff, cmap='RdBu_r', vmin=-VLIM, vmax=VLIM, shading='auto', **kw)
    rc.map_contourf(ax, X, Y, small, levels=[0.5, 1.5], colors='none', hatches=['....'], **kw)
    ax.set_title(f'{rc.FULL[ref]} window ({WIN.get(ref, "")})', fontsize=11)
    ax.text(0.0, 1.02, 'abcd'[k], transform=ax.transAxes, fontsize=14, fontweight='bold', va='bottom')
    w = oa_config.area_weights(lat, lon0)                 # mod, obs, sd_obs are on the original longitude lon0
    v = np.isfinite(mod - obs)
    am = lambda a: np.sum(w[v] * a[v]) / w[v].sum()
    print(f'{ref:10s} area mean: models {am(mod):.2f}, obs {am(obs):.2f}, models - obs {am(mod - obs):+.2f} degC; '
          f'|diff| < obs spread in {100 * w[v & (np.abs(mod - obs) < sd_obs)].sum() / w[v].sum():.0f} % of the area')
cax = fig.add_subplot(GridSpecFromSubplotSpec(1, 3, subplot_spec=gs[2, :], width_ratios=[1, 1.5, 1])[1])
cb = fig.colorbar(pc, cax=cax, orientation='horizontal', extend='both')
cb.set_label(r'$\Delta$SST, model mean minus observations ($^\circ$C)', fontsize=11)
rc.savefig(fig, 'dsst_bias')
plt.show()

## Supplementary Fig. 7: terms of the cell-by-cell constraint

In [ ]:
# Notebook cell: SI figure, what drives the local constraint (constrained minus unconstrained 1750 Omega_arag).
# The regression passes through the model means, so per cell (where the local constraint is used)
#   constrained - unconstrained = b_dT * (dSST_obs - mean dSST_models) + b_CUC * (CUC_obs - mean CUC_models)
# One row per product:  SST term | CUC term | share of the SST term, |SST| / (|SST| + |CUC|)
# Local constraint everywhere (pi_full). Dots: its uncertainty exceeds the model spread.
# Share map: dark grey where the correction is negligible (|SST term| + |CUC term| < THR).
# Printout also: across-model correlation of dSST and CUC per cell (collinearity of the two predictors).
# Needs regional_common.py and data/constraint_results.pkl (run_constraint.py, STORE_FIELDS = True).
import sys
import pickle
import importlib
import numpy as np
import netCDF4
import matplotlib.pyplot as plt
from matplotlib.gridspec import GridSpec, GridSpecFromSubplotSpec
from matplotlib.colors import ListedColormap

sys.path.insert(0, CODE_DIR)
import regional_common as rc
importlib.reload(rc)

try:
    import cartopy.crs as ccrs
    HAS_CARTOPY = False   # maps are drawn with rc.map_* (Robinson, no cartopy)
except ImportError:
    HAS_CARTOPY = False

OMEGA = r'$\Omega_{\mathrm{arag}}$'
VLIM = 0.04                                  # as panel b of fig_regional_local.py
THR = 0.005                                  # share only where the correction exceeds this
NA_BOX = dict(lat0=45, lon0=-60, lon1=10)    # subpolar North Atlantic for the printout

with open(f'{rc.ROOT}/constraint_results.pkl', 'rb') as f:
    RC = pickle.load(f)['results_common']
D = rc.load()                                # for land and panel b (check)


def terms(r, ref):
    """SST term, CUC term and use mask per cell, on the product grid (same cells as rc.load)"""
    F = r['fields']
    mask = np.asarray(r['mask']).astype(bool)
    Y, DT, CU = (F[k].astype('f8') for k in ('y1750', 'dT', 'CUC'))
    n = Y.shape[0]
    dobs = np.array([F['dsst_obs'][k] for k in F['dsst_obs']], 'f8')
    dTo, dTs = dobs.mean(0), dobs.std(0, ddof=1)
    CUo = F['CUC_obs'].astype('f8')
    with netCDF4.Dataset(rc.OBS[ref]) as f:
        yo = np.asarray(f['year'][:])
        om = np.ma.filled(f['omega_arag'][:].astype('f8'), np.nan)[yo <= rc.LAST_YEAR]
    ok = (mask & np.all(np.isfinite(Y), 0) & np.all(np.isfinite(DT), 0) & np.all(np.isfinite(CU), 0)
          & np.all(np.isfinite(om), 0) & np.isfinite(dTo) & np.isfinite(CUo))
    idx = np.where(ok.ravel())[0]
    sel = lambda a: a.reshape(n, -1)[:, idx] if a.ndim == 3 else a.ravel()[idx]
    y, x1, x2 = sel(Y), sel(DT), sel(CU)
    yc, st, beta = rc.local_constraint(y, x1, x2, sel(dTo), sel(dTs), sel(CUo), ok, return_beta=True,
                                       CUs=sel(rc.cuc_sigma(ref, 'results_common')[1]))
    use = st < y.std(0, ddof=1)                         # 0: sigma above the model spread (dots)
    t_sst = beta[:, 1] * (sel(dTo) - x1.mean(0))
    t_cuc = beta[:, 2] * (sel(CUo) - x2.mean(0))
    a1, a2 = x1 - x1.mean(0), x2 - x2.mean(0)
    rr = (a1 * a2).sum(0) / np.sqrt((a1 ** 2).sum(0) * (a2 ** 2).sum(0))

    def to_map(v):
        m_ = np.full(mask.size, np.nan)
        m_[idx] = v
        return rc.to_180(np.asarray(r['lon'], 'f8'), m_.reshape(mask.shape))[1]
    return to_map(t_sst), to_map(t_cuc), to_map(use.astype(float)), to_map(rr)


refs = list(RC)
nr = len(refs)
fig = plt.figure(figsize=(14, 3.0 * nr + 0.9))
gs = GridSpec(nr + 1, 3, figure=fig, height_ratios=[1] * nr + [0.06], hspace=0.12, wspace=0.04,
              left=0.05, right=0.99, top=0.96, bottom=0.05)
proj = {'projection': ccrs.Robinson()} if HAS_CARTOPY else {'projection': 'mollweide'}
kw = {'transform': ccrs.PlateCarree()} if HAS_CARTOPY else {}
xy = (lambda lo, la: (lo, la)) if HAS_CARTOPY else (lambda lo, la: (np.deg2rad(lo), np.deg2rad(la)))
xy = lambda lo, la: (np.asarray(lo, 'f8'), np.asarray(la, 'f8'))   # degrees; projected by rc.map_* (Robinson)
proj = proj_ = {}
kw = {}
COLS = [(f'SST term, 1750 {OMEGA}', 'RdBu_r', -VLIM, VLIM, 'both'),
        (f'CUC term, 1750 {OMEGA}', 'RdBu_r', -VLIM, VLIM, 'both'),
        ('Share of the SST term (%)', 'cividis', 0, 100, 'neither')]
letters = iter('abcdefghijklmnopqrstuvwxyz')
pcs = [None] * 3
for row, ref in enumerate(refs):
    d = D[ref]
    s_, c_, use, rr = terms(RC[ref], ref)
    tot = np.abs(s_) + np.abs(c_)
    share = np.where(tot > 0, 100 * np.abs(s_) / np.where(tot > 0, tot, 1), np.nan)
    share = np.where(np.isfinite(s_) & (tot >= THR), share, np.nan)
    small = np.where(np.isfinite(s_) & (tot < THR), 1.0, np.nan)
    X, Y = xy(d['lon'], d['lat'])
    fallback = np.where(use == 0, 1.0, np.nan)
    for col, (a, (title, cmap, v0, v1, ext)) in enumerate(zip([s_, c_, share], COLS)):
        ax = fig.add_subplot(gs[row, col], **proj)
        if HAS_CARTOPY:
            ax.set_global()
        else:
            ax.set_xticklabels([])
            ax.set_yticklabels([])
        rc.map_mesh(ax, X, Y, np.where(d['land'], 1.0, np.nan), cmap=ListedColormap(['0.85']), vmin=0, vmax=1,
                      shading='auto', **kw)
        if col == 2:
            rc.map_mesh(ax, X, Y, small, cmap=ListedColormap(['0.55']), vmin=0, vmax=1, shading='auto', **kw)
        pcs[col] = rc.map_mesh(ax, X, Y, a, cmap=cmap, vmin=v0, vmax=v1, shading='auto', **kw)
        rc.map_contourf(ax, X, Y, fallback, levels=[0.5, 1.5], colors='none', hatches=['....'], **kw)
        if row == 0:
            ax.set_title(title, fontsize=11)
        if col == 0:
            ax.text(-0.04, 0.5, rc.FULL[ref], transform=ax.transAxes, rotation=90, ha='right', va='center',
                    fontsize=10.5, fontweight='bold')
        ax.text(0.0, 1.0, next(letters), transform=ax.transAxes, fontsize=13, fontweight='bold', va='bottom')

    # numbers: area-weighted share of the correction from SST, check against panel b
    w = d['area']
    v = np.isfinite(s_)
    corr = d['pi_full'] - d['pi_mean']
    sh = 100 * np.sum(w[v] * np.abs(s_[v])) / np.sum(w[v] * tot[v])
    dom = 100 * w[v & (np.abs(s_) > np.abs(c_))].sum() / w[v].sum()
    LA, LO = np.meshgrid(d['lat'], d['lon'], indexing='ij')
    nat = v & (LA > NA_BOX['lat0']) & (LO > NA_BOX['lon0']) & (LO < NA_BOX['lon1'])
    big = v & (tot >= THR)
    vr = np.isfinite(rr)
    amr = lambda m: np.sum(w[m] * np.abs(rr[m])) / w[m].sum()
    sh_na = 100 * np.sum(w[nat] * np.abs(s_[nat])) / np.sum(w[nat] * tot[nat])
    print(f'{ref:10s} share of |correction| from SST: {sh:.0f} % (area mean); SST term larger in {dom:.0f} % of the '
          f'area; North Atlantic > 45N: {sh_na:.0f} %; check vs panel b: max |diff| '
          f'{np.nanmax(np.abs(s_ + c_ - corr)):.1e}')
    print(f'{"":10s} correction >= {THR}: {100 * w[big].sum() / w[v].sum():.0f} % of the area, SST share there '
          f'{100 * np.sum(w[big] * np.abs(s_[big])) / np.sum(w[big] * tot[big]):.0f} %')
    print(f'{"":10s} corr(dSST, CUC) across models: area mean |r| {amr(vr):.2f}, |r| > 0.7 in '
          f'{100 * w[vr & (np.abs(rr) > 0.7)].sum() / w[vr].sum():.0f} % of the area; North Atlantic box: '
          f'mean r {np.sum(w[nat] * rr[nat]) / w[nat].sum():+.2f}, mean |r| {amr(nat):.2f}')
for col, (title, cmap, v0, v1, ext) in enumerate(COLS):
    cax = fig.add_subplot(GridSpecFromSubplotSpec(1, 3, subplot_spec=gs[nr, col], width_ratios=[0.12, 1, 0.12])[1])
    cb = fig.colorbar(pcs[col], cax=cax, orientation='horizontal', extend=ext)
    if col == 2:
        cb.set_label(f'dark grey: correction < {THR}', fontsize=9)
rc.savefig(fig, 'correction_terms')
plt.show()

## Supplementary Fig. 8: effect of the cell-by-cell constraint

In [ ]:
# Notebook cell: supplementary figure, what the cell-by-cell constraint does (Fig. 4 uses the same constraint).
# One row per product and a last row for the mean of the four products (cells where all four have a value),
# three columns:
#      constrained preindustrial (1750) surface Omega_arag per cell
#      constrained minus unconstrained (multi-model mean of the bias-adjusted fields)
#      uncertainty reduction 1 - sigma_constrained / sigma_models (incl. observational uncertainty)
# The local constraint is used everywhere (pi_full). Dots: the uncertainty of the constrained value exceeds the
# model spread (the observed dSST/CUC are too uncertain to narrow the range; negative reduction).
# Last panel: area beyond -20 % in 2024, on the basis of Fig. 4 (own areas, alkalinity adjustment), for each
# fCO2-product and their mean: no constraint, global scaling, cell-by-cell constraint
# Needs regional_common.py and data/constraint_results.pkl (run_constraint.py, STORE_FIELDS = True).
import sys
import importlib
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.gridspec import GridSpec, GridSpecFromSubplotSpec
from matplotlib.colors import ListedColormap

sys.path.insert(0, CODE_DIR)
import regional_common as rc
importlib.reload(rc)

try:
    import cartopy.crs as ccrs
    HAS_CARTOPY = False   # maps are drawn with rc.map_* (Robinson, no cartopy)
except ImportError:
    HAS_CARTOPY = False

OMEGA = r'$\Omega_{\mathrm{arag}}$'
D = rc.load()
refs = list(D)
# mean of the products as an additional row (same grid for all products)
_both = np.all([np.isfinite(D[q]['pi_full']) for q in refs], axis=0)
_mean = lambda k: np.where(_both, np.nanmean([D[q][k] for q in refs], axis=0), np.nan)
D_ROWS = dict(D)
D_ROWS['mean'] = dict(lon=D[refs[0]]['lon'], lat=D[refs[0]]['lat'], land=D[refs[0]]['land'], area=D[refs[0]]['area'],
                      pi_full=_mean('pi_full'), pi_mean=_mean('pi_mean'),
                      sig_full=np.where(_both, np.sqrt(np.nanmean([D[q]['sig_full'] ** 2 for q in refs], axis=0)), np.nan),
                      sig_model=np.where(_both, np.sqrt(np.nanmean([D[q]['sig_model'] ** 2 for q in refs], axis=0)), np.nan))
rows = refs + ['mean']
ROW_LABEL = dict(rc.FULL, mean='Mean of the four fCO$_2$-products')
nr = len(rows)

fig = plt.figure(figsize=(14, 3.0 * nr + 3.2))
gs = GridSpec(nr + 2, 3, figure=fig, height_ratios=[1] * nr + [0.06, 1.05], hspace=0.12, wspace=0.04,
              left=0.05, right=0.99, top=0.97, bottom=0.04)
proj = {'projection': ccrs.Robinson()} if HAS_CARTOPY else {'projection': 'mollweide'}
kw = {'transform': ccrs.PlateCarree()} if HAS_CARTOPY else {}
xy = (lambda lo, la: (lo, la)) if HAS_CARTOPY else (lambda lo, la: (np.deg2rad(lo), np.deg2rad(la)))
xy = lambda lo, la: (np.asarray(lo, 'f8'), np.asarray(la, 'f8'))   # degrees; projected by rc.map_* (Robinson)
proj = proj_ = {}
kw = {}
COLS = [(f'Constrained 1750 {OMEGA}', 'viridis', 1.0, 4.2, 'both', False),
        (f'Constrained minus unconstrained 1750 {OMEGA}', 'RdBu_r', -0.04, 0.04, 'both', True),
        ('Uncertainty reduction (%)', 'Blues', 0, 60, 'both', True)]
letters = iter('abcdefghijklmnopqrstuvwxyz')
pcs = [None] * 3
for row, q in enumerate(rows):
    d = D_ROWS[q]
    X, Y = xy(d['lon'], d['lat'])
    fallback = np.where(d['sig_full'] >= d['sig_model'], 1.0, np.nan)
    reduction = 1 - d['sig_full'] / d['sig_model']
    d['reduction'] = reduction
    fields = [d['pi_full'], d['pi_full'] - d['pi_mean'], 100 * reduction]
    for col, (a, (title, cmap, v0, v1, ext, dots)) in enumerate(zip(fields, COLS)):
        ax = fig.add_subplot(gs[row, col], **proj)
        if HAS_CARTOPY:
            ax.set_global()
        else:
            ax.set_xticklabels([])
            ax.set_yticklabels([])
        rc.map_mesh(ax, X, Y, np.where(d['land'], 1.0, np.nan), cmap=ListedColormap(['0.85']), vmin=0, vmax=1,
                      shading='auto', **kw)
        pcs[col] = rc.map_mesh(ax, X, Y, a, cmap=cmap, vmin=v0, vmax=v1, shading='auto', **kw)
        if dots:
            rc.map_contourf(ax, X, Y, fallback, levels=[0.5, 1.5], colors='none', hatches=['....'], **kw)
        if row == 0:
            ax.set_title(title, fontsize=11)
        if col == 0:
            ax.text(-0.04, 0.5, ROW_LABEL[q], transform=ax.transAxes, rotation=90, ha='right', va='center',
                    fontsize=10.5, fontweight='bold')
        ax.text(0.0, 1.0, next(letters), transform=ax.transAxes, fontsize=13, fontweight='bold', va='bottom')
for col, (title, cmap, v0, v1, ext, dots) in enumerate(COLS):
    cax = fig.add_subplot(GridSpecFromSubplotSpec(1, 3, subplot_spec=gs[nr, col], width_ratios=[0.12, 1, 0.12])[1])
    fig.colorbar(pcs[col], cax=cax, orientation='horizontal', extend=ext)

# last: area beyond -20 % in 2024, three ways to set the preindustrial value, on the basis of Fig. 4: each
# fCO2-product on its own area, and the mean of
# the fCO2-products that cover a cell (98 % of the ocean), all with the alkalinity adjustment of the best estimate
# (rc.alkalinity_change: the change of the 1750 state and of the 2024 trend value, applied to each variant)
ax = fig.add_subplot(GridSpecFromSubplotSpec(2, 3, subplot_spec=gs[nr + 1, :], width_ratios=[1, 1.4, 1],
                                             height_ratios=[0.25, 1])[1, 1])
DO = rc.load(which='results')                                      # each fCO2-product on its own area, as Fig. 4
ALK8 = {q: rc.alkalinity_change(q, DO[q]['lat'], DO[q]['lon']) for q in refs}
AREA8 = DO[refs[0]]['area']


def beyond8(pkey, qs):
    """share (%) of the covered area beyond -20 % in 2024: from the mean over the fCO2-products qs that cover a cell
    of the 1750 state pkey and of the 2024 trend value, each with the alkalinity adjustment (one product: its own area)"""
    pis = np.array([np.where(DO[q]['ok'], DO[q][pkey] + ALK8[q][0], np.nan) for q in qs])
    nows = np.array([np.where(DO[q]['ok'], DO[q]['now'] + ALK8[q][2], np.nan) for q in qs])
    v = np.isfinite(pis) & np.isfinite(nows)
    n = v.sum(axis=0)
    pi_m = np.where(n > 0, np.where(v, pis, 0.0).sum(axis=0) / np.maximum(n, 1), np.nan)
    now_m = np.where(n > 0, np.where(v, nows, 0.0).sum(axis=0) / np.maximum(n, 1), np.nan)
    return rc.area_beyond(dict(area=AREA8, c=100 * (now_m / pi_m - 1)), 'c')


groups = refs + ['mean']
VARIANTS = (('pi_mean', 0.35), ('pi_scaled', 0.65), ('pi_full', 1.0))
VALS8 = {pkey: [beyond8(pkey, refs if g == 'mean' else [g]) for g in groups] for pkey, _ in VARIANTS}
for pkey, _ in VARIANTS:
    print(f'Supp. Fig. 8, last panel, area beyond -20 % in {rc.EVAL_YEAR} ({pkey}; own areas, alkalinity adjustment): '
          + ', '.join(f'{g} {v_:.1f} %' for g, v_ in zip(groups, VALS8[pkey])))
xpos = np.arange(len(groups))
for j, (pkey, alpha) in enumerate(VARIANTS):
    xb = xpos + (j - 1) * 0.27
    ax.bar(xb, VALS8[pkey], width=0.25, color=[rc.COLORS[q] for q in refs] + ['C3'], alpha=alpha, edgecolor='k', lw=0.5)
    for x_, v_ in zip(xb, VALS8[pkey]):
        ax.text(x_, v_ + 0.4, f'{v_:.1f}', ha='center', va='bottom', fontsize=7.5, rotation=90)
ax.set_xticks(xpos)
ax.set_xticklabels(refs + ['mean'], fontsize=10)
ax.set_ylabel(f'Area beyond $-$20%\nin {rc.EVAL_YEAR} (%)', fontsize=10)
ax.set_ylim(0, 1.45 * max(max(v_) for v_ in VALS8.values()))
ax.spines[['top', 'right']].set_visible(False)
from matplotlib.patches import Patch
ax.legend([Patch(facecolor='0.3', alpha=al, edgecolor='k', lw=0.5) for al in (0.35, 0.65, 1.0)],
          ['no constraint', 'global scaling', 'cell-by-cell constraint'], fontsize=8.5, frameon=False,
          loc='upper center', ncol=3, handlelength=1.2, columnspacing=1)
ax.text(-0.15, 1.02, next(letters), transform=ax.transAxes, fontsize=13, fontweight='bold', va='bottom')

for q in rows:
    d = D_ROWS[q]
    w = d['area']
    v = np.isfinite(d['pi_full'])
    dd_ = d['pi_full'] - d['pi_mean']
    reduction = d['reduction']
    print(f'{q}: sigma above the model spread in {100 * w[v & (d["sig_full"] >= d["sig_model"])].sum() / w[v].sum():.0f} % of the area; '
          f'constrained minus unconstrained: area mean {np.sum(w[v] * dd_[v]) / w[v].sum():+.4f}, '
          f'5-95 % {np.percentile(dd_[v], 5):+.3f} to {np.percentile(dd_[v], 95):+.3f}; '
          f'uncertainty reduction area mean {100 * np.sum(w[v] * reduction[v]) / w[v].sum():.0f} %')
rc.savefig(fig, 'regional_local')
plt.show()

## Diagnostic: cell-by-cell against global constraint

Area mean of the cell-by-cell constraint against the global constraint, per fCO2-product, with the resulting decline (printed table, no figure).

In [ ]:
# Notebook cell: global mean of the cell-by-cell constraint against the global constraint (common area).
# For every product: area-weighted means over the analysis area (cells with all model 1750 values and
# observations) of the unconstrained multi-model mean, the global constraint distributed proportionally
# (pi_scaled), the cell-by-cell constraint everywhere (pi_full, the paper's regional field), with the threshold
# (pi_local) and the Bayesian variant (pi_bayes); and the decline to EVAL_YEAR that follows from each.
# Needs data/constraint_results.pkl and regional_common.py (same kernel settings as the regional figures).
import pickle
import numpy as np
import regional_common as rc

D = rc.load()                                            # common mask, lon -180..180
with open(f'{rc.ROOT}/constraint_results.pkl', 'rb') as f_:
    RC = pickle.load(f_)['results_common']
KEYS = ('pi_mean', 'pi_scaled', 'pi_full', 'pi_local', 'pi_bayes')
print(f'1750 global mean Omega_arag, common area, run {rc.RUN}; decline = 1 - now/1750 of the area means (%)')
print(f'{"product":10s} {"n cells":>7s} | {"unconstr.":>9s} {"global":>8s} {"cell":>8s} {"thresh.":>8s} {"Bayes":>8s} '
      f'| {"cell-glob":>9s} | {"decline global":>14s} {"cell":>7s} {"thresh.":>7s} {"Bayes":>7s} | {"global (pkl)":>12s}')
rows = {}
for q, d in D.items():
    v = np.isfinite(d['now'])
    for k in KEYS:
        v &= np.isfinite(d[k])
    w = d['area'] * v
    am = lambda a: float(np.sum(np.where(v, a, 0) * w) / w.sum())
    g = {k: am(d[k]) for k in KEYS}
    now = am(d['now'])
    dec = {k: 100 * (1 - now / g[k]) for k in KEYS}
    r = RC[q]
    yc, sc, _ = r['constrained'][rc.LAB]
    dec_pkl = 100 * (1 - r['now'] / yc)
    rows[q] = (g, dec)
    print(f'{q:10s} {int(v.sum()):7d} | {g["pi_mean"]:9.3f} {g["pi_scaled"]:8.3f} {g["pi_full"]:8.3f} {g["pi_local"]:8.3f} '
          f'{g["pi_bayes"]:8.3f} | {g["pi_full"] - g["pi_scaled"]:+9.4f} | {dec["pi_scaled"]:14.2f} {dec["pi_full"]:7.2f} '
          f'{dec["pi_local"]:7.2f} {dec["pi_bayes"]:7.2f} | {dec_pkl:12.2f}')
    # consistency: the area mean of pi_scaled is the global constrained value, that of pi_mean the unconstrained one
    if abs(g['pi_scaled'] - yc) > 0.002 or abs(g['pi_mean'] - r['unconstrained'][0]) > 0.002:
        print(f'{"":10s} note: area means differ from the global values of the pkl (constrained {yc:.3f}, '
              f'unconstrained {r["unconstrained"][0]:.3f}): different cells (now-field or product coverage)')
gm_ = {k: np.mean([rows[q][0][k] for q in rows]) for k in KEYS}
dm_ = {k: np.mean([rows[q][1][k] for q in rows]) for k in KEYS}
print(f'{"mean":10s} {"":7s} | {gm_["pi_mean"]:9.3f} {gm_["pi_scaled"]:8.3f} {gm_["pi_full"]:8.3f} {gm_["pi_local"]:8.3f} '
      f'{gm_["pi_bayes"]:8.3f} | {gm_["pi_full"] - gm_["pi_scaled"]:+9.4f} | {dm_["pi_scaled"]:14.2f} {dm_["pi_full"]:7.2f} '
      f'{dm_["pi_local"]:7.2f} {dm_["pi_bayes"]:7.2f} |')

## Extended Data Fig. 6: crossing of the boundary against CO2

In [ ]:
# Notebook cell: Extended Data Fig. 6, when is the global boundary crossed? Decline of the global mean surface Omega_arag since 1750
# against atmospheric CO2 (the CO2 counterpart of Fig. 4a).
#   observed:   annual global means of each product on the common area, relative to its constrained 1750 value, against
#               the NOAA global mean xCO2 of the year (thin, product colours)
#   forced:     mean of the products (decline_crossing of the preparation cell, as Fig. 4a): quadratic fit to the
#               annual means up to 2024, then the global projection started at the fitted 2024 value (surface pCO2
#               rising with the atmosphere, T rising with CO2 as observed, S and ALK constant), with the full 1 sigma of
#               the decline (Extended Data Table 2, sdec_m, held fixed); no band on the boundary
#   boundary:   20 % decline from 1750 (278.3 ppm); thin lines: 80 % of the state at the CO2 of 1850 (285.5 ppm) or at
#               the effective preindustrial CO2 of Jiang et al. (2023, 272.4 ppm), drawn as decline from 1750; the legend
#               names each line by its preindustrial CO2
#   crossing:   CO2 and year at which the forced decline reaches each boundary, one marker per definition (no lines),
#               with the range of the text (decline_crossing; the same 1 sigma of the decline for all three)
#   top axis:   year: NOAA record up to its last year, then constant growth (projection.co2_growth, 2.6 ppm/yr)
# Needs the preparation cell (sdec_m, decline_crossing) and data/constraint_results.pkl with 'states'.
def _fig_s8():
    import pickle
    import numpy as np
    import matplotlib.pyplot as plt
    from matplotlib.patches import Patch
    from matplotlib.lines import Line2D
    from matplotlib.legend_handler import HandlerTuple
    import regional_common as rc
    import projection
    from pi_states import TARGETS          # CO2 of the states: 1750 (AR6), 1850 (AR6), Jiang et al. (effective)
    for n_ in ('sdec_m', 'decline_crossing'):
        if n_ not in globals():
            raise NameError(f'{n_} not defined: run the preparation cell first (same kernel)')
    with open(f'{rc.ROOT}/constraint_results.pkl', 'rb') as f:
        RES = pickle.load(f)['results_common']
    refs = [q for q in ('OceanSODA', 'UExP', 'JMA', 'CMEMS') if q in RES]
    co2 = projection.noaa_co2()
    growth = projection.co2_growth()[0]
    y_last = rc.EVAL_YEAR                     # anchor at 2024 (the NOAA file may go further)
    yr_rec = np.array(sorted(y for y in co2 if y <= y_last), 'f8')
    x_rec = np.array([co2[int(y)] for y in yr_rec])
    # year <-> ppm: the NOAA record, then constant growth (both monotonic)
    x_ext = np.r_[x_rec, x_rec[-1] + growth * 60]
    y_ext = np.r_[yr_rec, yr_rec[-1] + 60]
    year_of = lambda x: np.interp(x, x_ext, y_ext)
    x_of = lambda y: np.interp(y, y_ext, x_ext)
    MEAN_COL = 'C3'
    YEAR_END = 2040

    # alternative definitions of the preindustrial state, as decline levels relative to 1750: 80 % of the state with
    # the CO2 of 1850 (or of Jiang et al.) is 100 (1 - 0.8 ratio) % below the 1750 value
    ratio = {k: np.mean([RES[q]['states'][k]['ratio'] for q in refs]) for k in ('1850', 'jiang')}
    LEVELS = {'1750 (this study)': 20.0,
              '1850': 100 * (1 - 0.8 * ratio['1850']),
              'the CO$_2$ of Jiang et al. (2023)': 100 * (1 - 0.8 * ratio['jiang'])}
    DC = {k: decline_crossing(v) for k, v in LEVELS.items()}
    for k, d in DC.items():
        print(f'boundary from {k:55s}: decline from 1750 {LEVELS[k]:5.2f} %, reached at {d["x"]:.0f} ± {d["dx"]:.0f} ppm '
              f'({d["x"] - d["dx"]:.0f}-{d["x"] + d["dx"]:.0f}), {d["t"]:.0f} ({d["t"] - d["dt"]:.0f}-{d["t"] + d["dt"]:.0f})')
    D20 = DC['1750 (this study)']
    x_c, d_c = D20['curve_x']
    t_c = D20['curve'][0]
    t0 = max(float(np.asarray(RES[q]['year'])[0]) for q in refs)           # first year covered by all four
    keep = np.isfinite(x_c) & (t_c >= t0) & (t_c <= YEAR_END)
    xa, da = x_c[keep], -d_c[keep]                                           # negative: change since 1750 (as Fig. 4a)
    x_now = co2[rc.EVAL_YEAR]

    # diagnostic: rate of decline per ppm, observed and projected (does the curve bend at 2024?)
    #   annual  linear regression of each product's annual decline on CO2 over the last 10 and 20 years of the record
    #   fit     slope of the forced mean curve (quadratic fit in time) over the last 5 years before 2024
    #   proj    slope of the projection over the first 10 ppm after 2024
    sl = lambda x_, d_: float(np.polyfit(x_, d_, 1)[0])
    for n_yr in (10, 20):
        v_ = []
        for q in refs:
            r = RES[q]
            yo, gm = np.asarray(r['year']), np.asarray(r['ogm'], 'f8')
            k_ = (yo <= rc.LAST_YEAR) & (yo > rc.LAST_YEAR - n_yr) & np.isin(yo, list(co2))
            v_.append(sl([co2[int(y)] for y in yo[k_]], 100 * (1 - gm[k_] / r['constrained'][rc.LAB][0])))
        print(f'decline per ppm, annual values, last {n_yr} years: ' + ', '.join(f'{q} {v:.3f}' for q, v in zip(refs, v_))
              + f'; mean {np.mean(v_):.3f} % per ppm')
    ok_ = np.isfinite(x_c)
    kf = ok_ & (t_c > rc.LAST_YEAR - 5) & (t_c <= rc.LAST_YEAR)
    kp = ok_ & (x_c > x_now) & (x_c <= x_now + 10)
    print(f'decline per ppm, forced fit (last 5 years) {sl(x_c[kf], d_c[kf]):.3f}, projection (first 10 ppm) '
          f'{sl(x_c[kp], d_c[kp]):.3f} % per ppm; in time: fit {sl(t_c[kf], d_c[kf]):.3f}, projection '
          f'{sl(t_c[kp], d_c[kp]):.3f} % per year')

    fig, ax = plt.subplots(figsize=(7.4, 4.6))
    for q in refs:
        r = RES[q]
        yo, gm = np.asarray(r['year']), np.asarray(r['ogm'], 'f8')
        k_ = (yo <= rc.LAST_YEAR) & np.isin(yo, list(co2))
        ax.plot([co2[int(y)] for y in yo[k_]], 100 * (gm[k_] / r['constrained'][rc.LAB][0] - 1), color=rc.COLORS[q],
                lw=1.1, zorder=5, label=rc.FULL[q])
    ax.fill_between(xa, da - sdec_m, da + sdec_m, color=MEAN_COL, alpha=0.25, lw=0, zorder=2)
    ax.plot(xa[xa <= x_now], da[xa <= x_now], color=MEAN_COL, lw=1.6, alpha=0.8, zorder=4)
    ax.plot(xa[xa >= x_now], da[xa >= x_now], color=MEAN_COL, lw=1.6, ls=(0, (3, 2)), zorder=4)
    # where the boundary is crossed for the three definitions of the preindustrial state: one marker each, on the forced
    # curve (no horizontal lines), with the range in CO2 (the same 1 sigma of the decline for all three; the
    # uncertainty of the CO2 of each state is not included)
    ORDER = ['1750 (this study)', '1850', 'the CO$_2$ of Jiang et al. (2023)']
    NAMES = {'1750 (this study)': f'This study, 1750 ({TARGETS["1750"]:.0f} ppm, AR6)',
             '1850': f'This study, 1850 ({TARGETS["1850"]:.0f} ppm, AR6)',
             'the CO$_2$ of Jiang et al. (2023)': f'Jiang et al. (2023) (effective {TARGETS["jiang"]:.0f} ppm)'}
    MARK = {'1750 (this study)': dict(marker='o', ms=7, mfc='white', mec='k', mew=1.4, color='k'),
            'the CO$_2$ of Jiang et al. (2023)': dict(marker='s', ms=5.5, mfc='0.65', mec='k', mew=0.9, color='k'),
            '1850': dict(marker='D', ms=5.0, mfc='k', mec='k', mew=0.9, color='k')}
    ax.axhline(-20, color='0.3', lw=0.8, ls=(0, (1, 2)), zorder=3)              # the boundary as defined in this study
    for k in ORDER:
        d = DC[k]
        m = MARK[k]
        ax.errorbar(d['x'], -LEVELS[k], xerr=d['dx'], fmt=m['marker'], ms=m['ms'], mfc=m['mfc'], mec=m['mec'], mew=m['mew'],
                    ecolor='k', elinewidth=1.3 if k == ORDER[0] else 0.9, capsize=3 if k == ORDER[0] else 2.5, zorder=6)
    X0 = min(co2[int(np.asarray(RES[q]['year'])[0])] for q in refs) - 2
    X1 = float(x_of(YEAR_END))
    ax.set_xlim(X0, X1)
    ax.text(X0 + 1.5, -19.85, '20 % decline from 1750 (AR6)', ha='left', va='bottom', fontsize=7.5, color='0.3')
    ax.set_ylim(-27, -8)
    ax.set_yticks(np.arange(-25, -9, 5))
    ax.set_xlabel('Atmospheric CO$_2$ (ppm)', fontsize=10)
    ax.set_ylabel('Change of global mean\n$\\Omega_{\\mathrm{arag}}$ since 1750 (%)', fontsize=10)
    ax.spines[['top', 'right']].set_visible(False)
    sec = ax.secondary_xaxis('top', functions=(year_of, x_of))
    sec.set_xlabel(f'Year (NOAA to {y_last}, then {growth:.1f} ppm yr$^{{-1}}$)', fontsize=9)
    hh = [Line2D([], [], color=rc.COLORS[q], lw=1.1) for q in refs] + \
         [(Patch(facecolor=MEAN_COL, alpha=0.25, lw=0), Line2D([], [], color=MEAN_COL, lw=1.6, alpha=0.8)),
          Line2D([], [], color=MEAN_COL, lw=1.6, ls=(0, (3, 2)))]
    ll = [rc.FULL[q] for q in refs] + [f'mean of the fCO$_2$-products, forced (± {sdec_m:.1f} %, 1$\\sigma$)', 'idealised projection']
    leg = ax.legend(hh, ll, fontsize=7.5, frameon=False, loc='upper right', handler_map={tuple: HandlerTuple(ndivide=1, pad=0)})
    ax.add_artist(leg)
    hb, lb = [], []
    for k in ORDER:
        d = DC[k]
        m = MARK[k]
        hb.append(Line2D([], [], ls='none', marker=m['marker'], ms=m['ms'], mfc=m['mfc'], mec=m['mec'], mew=m['mew']))
        lb.append(f'{NAMES[k]}: {d["x"]:.0f} ppm ({d["x"] - d["dx"]:.0f}–{d["x"] + d["dx"]:.0f}), '
                  f'≈ {d["t"]:.0f} ({d["t"] - d["dt"]:.0f}–{d["t"] + d["dt"]:.0f})')
    ax.legend(hb, lb, fontsize=7.5, frameon=False, loc='lower left', alignment='left', title_fontsize=8,
              title='Boundary (80 % of the preindustrial state) crossed at')
    fig.tight_layout()
    rc.savefig(fig, 'crossing_co2')
    plt.show()
    return DC


S8 = _fig_s8()

## Supplementary Fig. 9: regional decline per fCO2-product

In [ ]:
# Notebook cell: Supplementary Fig. 9, regional decline of surface Omega_arag.
# a-d: relative change in 2024 (per-cell quadratic fit) against 1750 for each product, on its own area (dotted:
#      the common area, the same values there);
#      1750 per cell (PI): 'full' = cell-by-cell constraint with cell dSST and CUC everywhere (observations
#      taken as they are); 'local' = same, model mean where it does not reduce the uncertainty;
#      'bayes' = observations weighted against the models; 'scaled' = multi-model mean x global factor.
#      Black contour: -20 % (the boundary); numbers: area beyond it on the product's own area and on the common area.
# e:   share of the area beyond -20 % in every year (annual values) for each product, on the common area (solid) and on
#      the product's own area (dashed).
# f:   the same for the forced state (the per-cell quadratic fit of the maps, evaluated in every product year).
# All panels with the alkalinity adjustment of the best estimate, as Fig. 4: per
# product the change of Omega_arag (rc.alkalinity_change, carbonate_system_<ref>_alkalinity.nc) added to the 1750 state,
# to the 2024 trend value, to the annual values and to the forced state of every year (the fit of the product's own
# values plus the change of that year); the shares without it are printed for comparison.
# Needs regional_common.py, data/constraint_results.pkl (run_constraint.py, STORE_FIELDS = True) and
# product/carbonate_system_<ref>_alkalinity.nc (make_product_alkalinity.py).
import sys
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.gridspec import GridSpec, GridSpecFromSubplotSpec
from matplotlib.colors import ListedColormap
from matplotlib.transforms import offset_copy

sys.path.insert(0, CODE_DIR)
import importlib
import regional_common as rc
importlib.reload(rc)                  # pick up changes to regional_common.py without restarting the kernel

try:
    import cartopy.crs as ccrs
    HAS_CARTOPY = False   # maps are drawn with rc.map_* (Robinson, no cartopy)
except ImportError:
    HAS_CARTOPY = False

# continuous, perceptually uniform colour map without white (white = ocean without data, as in Figure 1):
# plasma, stronger decline = darker; -20 % marked by the black contour
VMIN, VMAX = -27, -13      # the range the maps actually cover; -20 % in the middle of the colour map
CMAP = plt.get_cmap('plasma')
TXT_LON, TXT_LAT = 82.0, 49.0          # where the area beyond -20 % is written (central Asia)
PI = 'full'                            # 1750 per cell: 'full' (cell-by-cell constraint everywhere), 'local',
                                       # 'bayes' or 'scaled' (see above)
CHG, PIK = f'chg_{PI}', f'pi_{PI}'
FIRST_YEAR = 1985                      # time series from here

D = rc.load()
D_OWN = rc.load(which='results')       # each product on its own area (maps a-d, dashed lines in e, f)
for q_ in D:
    assert np.allclose(D[q_]['lat'], D_OWN[q_]['lat']) and np.allclose(D[q_]['lon'], D_OWN[q_]['lon']), f'{q_}: grids differ'
    in_ = np.isfinite(D[q_][CHG]) & np.isfinite(D_OWN[q_][CHG])
    print(f'{q_:10s} own area against common area, inside the common area: max |difference| of the decline '
          f'{np.max(np.abs(D_OWN[q_][CHG] - D[q_][CHG])[in_]):.2g} points; cells in the common area without an own-area value '
          f'{int((np.isfinite(D[q_][CHG]) & ~np.isfinite(D_OWN[q_][CHG])).sum())}')


def with_alkalinity(d, a50, a_year, a24):
    """a copy of the fields of a product with the alkalinity adjustment; the fields without it as *_noalk and om0"""
    e = dict(d)
    A = np.array([a_year(int(y)) for y in d['year']])
    ok = np.isfinite(a50) & np.isfinite(a24) & np.all(np.isfinite(A), axis=0)
    e['pi_noalk'], e['chg_noalk'], e['om0'], e['alk_year'] = d[PIK], d[CHG], d['om'], a_year
    e[PIK] = np.where(ok, d[PIK] + a50, np.nan)
    e['now'] = d['now'] + a24
    e[CHG] = 100 * (e['now'] / e[PIK] - 1)
    e['om'] = d['om'] + A
    e['alk_lost'] = int((np.isfinite(d[PIK]) & d['ok'] & ~ok).sum())
    return e


for q_ in D:
    alk_ = rc.alkalinity_change(q_, D[q_]['lat'], D[q_]['lon'])
    D[q_], D_OWN[q_] = with_alkalinity(D[q_], *alk_), with_alkalinity(D_OWN[q_], *alk_)
    print(f'{q_:10s} alkalinity adjustment: {D_OWN[q_]["alk_lost"]} cells of the own area without a change left out')

fig = plt.figure(figsize=(12, 11.2))
# rows: maps, maps, colour bar, empty spacer, time series
gs = GridSpec(5, 2, figure=fig, height_ratios=[1, 1, 0.045, 0.16, 0.8], hspace=0.12, wspace=0.05,
              left=0.04, right=0.98, top=0.97, bottom=0.05)
proj = {'projection': ccrs.Robinson()} if HAS_CARTOPY else {'projection': 'mollweide'}
kw = {'transform': ccrs.PlateCarree()} if HAS_CARTOPY else {}
xy = (lambda lo, la: (lo, la)) if HAS_CARTOPY else (lambda lo, la: (np.deg2rad(lo), np.deg2rad(la)))
xy = lambda lo, la: (np.asarray(lo, 'f8'), np.asarray(la, 'f8'))   # degrees; projected by rc.map_* (Robinson)
proj = proj_ = {}
kw = {}
letters = iter('abcdef')


def draw_map(ax, d, d_own, title):
    if HAS_CARTOPY:
        ax.set_global()
    else:
        ax.set_xticklabels([])
        ax.set_yticklabels([])
    X, Y = xy(d['lon'], d['lat'])
    rc.map_mesh(ax, X, Y, np.where(d['land'], 1.0, np.nan), cmap=ListedColormap(['0.85']), vmin=0, vmax=1,
                  shading='auto', **kw)                                    # land grey, ocean outside the mask white
    pc = rc.map_mesh(ax, X, Y, d_own[CHG], cmap=CMAP, vmin=VMIN, vmax=VMAX, shading='auto', **kw)
    rc.map_contour(ax, X, Y, d_own[CHG], levels=[-20], colors='k', linewidths=0.8, linestyles='solid', **kw)
    rc.map_contour(ax, X, Y, np.isfinite(d[CHG]).astype(float), levels=[0.5], colors='0.25', linewidths=0.5,
                   linestyles=':', **kw)                                   # the common area
    ax.set_title(title, fontsize=11.5, fontweight='bold')
    base = kw.get('transform', ax.transData)
    base = base._as_mpl_transform(ax) if hasattr(base, '_as_mpl_transform') else base
    ax.text(*rc.map_xy(TXT_LON, TXT_LAT), 'beyond $-$20%', ha='center', va='bottom', fontsize=9.5,
            transform=offset_copy(base, fig=fig, y=1, units='points'))
    ax.text(*rc.map_xy(TXT_LON, TXT_LAT), f'{rc.area_beyond(d_own, CHG):.0f}% of its own area\n'
            f'{rc.area_beyond(d, CHG):.0f}% of the common area', ha='center', va='top', fontsize=8.5, linespacing=1.25,
            transform=offset_copy(base, fig=fig, y=-1, units='points'))
    ax.text(0.0, 1.02, next(letters), transform=ax.transAxes, fontsize=16, fontweight='bold', va='bottom')
    return pc


for k, (ref, d) in enumerate(D.items()):
    pc = draw_map(fig.add_subplot(gs[k // 2, k % 2], **proj), d, D_OWN[ref], rc.FULL[ref])
cax = fig.add_subplot(GridSpecFromSubplotSpec(1, 3, subplot_spec=gs[2, :], width_ratios=[1, 1.5, 1])[1])
cb = fig.colorbar(pc, cax=cax, orientation='horizontal', extend='both')
cb.set_label(r'Change in surface $\Omega_{\mathrm{arag}}$ from 1750 to ' + f'{rc.EVAL_YEAR} (%)', fontsize=11)

# e, f: area beyond -20 % over time, annual values (e) and forced state (f)
# the time series with more room on the left than the maps, so that the y label and the letter e are not cut
# (the same rows, its own left edge)
gs_ts = GridSpec(5, 2, figure=fig, height_ratios=[1, 1, 0.045, 0.16, 0.8], hspace=0.12, wspace=0.22,
                 left=0.085, right=0.98, top=0.97, bottom=0.05)
gef = [gs_ts[4, 0], gs_ts[4, 1]]
yrs = np.arange(FIRST_YEAR, rc.LAST_YEAR + 1)


def on_years(d, factor=1.0):
    s = rc.area_beyond_series(d, PIK, factor=factor)
    out = np.full(yrs.size, np.nan)
    for i, y in enumerate(d['year']):
        if FIRST_YEAR <= y <= rc.LAST_YEAR:
            out[y - FIRST_YEAR] = s[i]
    return out


def forced_on_years(d):
    # share beyond -20 % of the per-cell quadratic fit (as d['now'] and the maps) in every product year
    pi, yo = d[PIK], np.asarray(d['year'])
    v = np.isfinite(pi) & d['ok']
    idx = np.where(v.ravel())[0]
    cf = np.polyfit(yo - 2000.0, d['om0'].reshape(yo.size, -1)[:, idx], 2)    # the product's own values
    a, p_ = d['area'].ravel()[idx], pi.ravel()[idx]
    out = np.full(yrs.size, np.nan)
    for y in yo:
        if FIRST_YEAR <= y <= rc.LAST_YEAR:
            f_ = np.polyval(cf, y - 2000.0) + d['alk_year'](int(y)).ravel()[idx]
            out[y - FIRST_YEAR] = 100 * a[100 * (f_ / p_ - 1) < -20.0].sum() / a.sum()
    return out


S = {q: on_years(d) for q, d in D.items()}
F = {q: forced_on_years(d) for q, d in D.items()}
S_OWN = {q: on_years(d) for q, d in D_OWN.items()}
F_OWN = {q: forced_on_years(d) for q, d in D_OWN.items()}
top = 5 * np.ceil(max(np.nanmax(s_[q]) for s_ in (S, F, S_OWN, F_OWN) for q in D) / 5)
from matplotlib.lines import Line2D
for k, (ser, own, sub) in enumerate(((S, S_OWN, 'annual values'), (F, F_OWN, 'forced state (quadratic fit per cell)'))):
    ax = fig.add_subplot(gef[k])
    for q in D:
        ax.plot(yrs, ser[q], color=rc.COLORS[q], lw=1.5, label=rc.FULL[q])
        ax.plot(yrs, own[q], color=rc.COLORS[q], lw=1.1, ls=(0, (4, 2)))
    ax.set_xlim(FIRST_YEAR, rc.LAST_YEAR)
    ax.set_ylim(0, top)
    ax.set_title(sub, fontsize=10)
    ax.set_ylabel('Area beyond $-$20% (%)', fontsize=11)
    ax.set_xlabel('Year', fontsize=11)
    ax.spines[['top', 'right']].set_visible(False)
    if k == 0:
        ax.legend(fontsize=8.5, frameon=False, loc='upper left')
    else:
        ax.legend([Line2D([], [], color='0.3', lw=1.5), Line2D([], [], color='0.3', lw=1.1, ls=(0, (4, 2)))],
                  ['common area (as the maps)', "each fCO$_2$-product's own area"], fontsize=8.5, frameon=False, loc='upper left')
    ax.text(-0.14, 1.02, next(letters), transform=ax.transAxes, fontsize=16, fontweight='bold', va='bottom')

for ref, d in D.items():
    print(f'{ref:10s} 1750: {PI}; area beyond -20 % in {rc.EVAL_YEAR} (fit): {rc.area_beyond(d, CHG):.1f} % (panel f: '
          f'{F[ref][rc.EVAL_YEAR - FIRST_YEAR]:.1f} %; annual value: {S[ref][rc.EVAL_YEAR - FIRST_YEAR]:.1f} %); own area: '
          f'map {rc.area_beyond(D_OWN[ref], CHG):.1f} %, forced {F_OWN[ref][rc.EVAL_YEAR - FIRST_YEAR]:.1f} %, annual '
          f'{S_OWN[ref][rc.EVAL_YEAR - FIRST_YEAR]:.1f} %; without the alkalinity adjustment: map {rc.area_beyond(d, "chg_noalk"):.1f} %, '
          f'own area {rc.area_beyond(D_OWN[ref], "chg_noalk"):.1f} %')
rc.savefig(fig, 'regional_decline')
plt.show()

## Supplementary Fig. 2: coverage of the fCO2-products

In [ ]:
# Notebook cell: Supplementary Fig. 2, coverage of the observation-based products:
# number of products with complete data in each cell (an annual value in every year up to 2024, i.e. all 12 months),
# with the common area outlined.
import oa_config                      # the area weights of the run (OA_WEIGHTS)
import sys
import pickle
import importlib
import numpy as np
import netCDF4
import matplotlib.pyplot as plt
from matplotlib.gridspec import GridSpec, GridSpecFromSubplotSpec
from matplotlib.colors import ListedColormap, BoundaryNorm
from matplotlib.lines import Line2D

sys.path.insert(0, CODE_DIR)
import regional_common as rc
importlib.reload(rc)

try:
    import cartopy.crs as ccrs
    HAS_CARTOPY = False   # maps are drawn with rc.map_* (Robinson, no cartopy)
except ImportError:
    HAS_CARTOPY = False

ROOT = rc.ROOT
LAB = rc.LAB
YEAR = 2024
OMEGA = r'$\Omega_{\mathrm{arag}}$'
# Planetary Health Check (2026), ocean acidification section: observed 2.85 ± 0.01 (2σ) in 2024 from UExP-FNN-U on its
# full area; preindustrial 3.57 = Jiang et al. (2023, Table S5) for 1750; boundary 2.86 (80 % of 3.57, as published)
PHC_OBS, PHC_PI, PHC_B = 2.85, 3.57, 2.86
STEP_REF = 'UExP'                     # product of the Planetary Health Check
BCOL = 'C3'                           # boundary

with open(f'{ROOT}/constraint_results.pkl', 'rb') as f:
    PK = pickle.load(f)
RCOM, ROWN = PK['results_common'], PK['results']
refs = [q for q in ('OceanSODA', 'UExP', 'JMA', 'CMEMS') if q in RCOM]


def read_obs(q):
    r = RCOM[q]
    with netCDF4.Dataset(rc.OBS[q]) as f:
        return (np.asarray(f['year'][:]), np.asarray(r['lat'], 'f8'), np.asarray(r['lon'], 'f8'),
                np.ma.filled(f['omega_arag'][:].astype('f8'), np.nan))


def gm_series(yo, om, lat, lon, mask):
    """annual area-weighted means over mask (the weights of the run)"""
    wm = oa_config.area_weights(lat, lon) * mask
    return np.array([np.nansum(np.where(np.isfinite(o), o, 0) * wm) / wm[np.isfinite(o)].sum() for o in om])


def fit_now(yo, gm):
    """quadratic fit to the annual means, evaluated in YEAR (as Extended Data Table 1)"""
    ok = np.isfinite(gm) & (yo <= rc.LAST_YEAR)
    return np.polyval(np.polyfit(yo[ok] - 2000.0, gm[ok], 2), YEAR - 2000.0)


# ---------------------------------------------------------------- coverage and observed values per product
cov, V, TS = {}, {}, {}
for q in refs:
    yo, lat, lon, om = read_obs(q)
    full = np.isfinite(om[yo <= rc.LAST_YEAR]).all(0)          # complete: every year (and so every month) up to 2024
    own = np.asarray(ROWN[q]['mask']).astype(bool)
    com = np.asarray(RCOM[q]['mask']).astype(bool)
    g = {k: gm_series(yo, om, lat, lon, m_) for k, m_ in (('full', full), ('own', own), ('com', com))}
    keep = yo <= rc.LAST_YEAR
    TS[q] = dict(year=yo[keep], own=g['own'][keep], com=g['com'][keep])
    V[q] = {k: fit_now(yo, v) for k, v in g.items()}
    # one map grid for all products: longitudes -180..180 (products differ in convention), latitudes ascending
    lon180, full180 = rc.to_180(lon, full)
    com180 = rc.to_180(lon, com)[1]
    if lat[0] > lat[-1]:
        lat, full180, com180 = lat[::-1], full180[::-1], com180[::-1]
    cov[q] = dict(lat=lat, lon=lon180, full=full180, com=com180)
lat, lon = cov[refs[0]]['lat'], cov[refs[0]]['lon']
for q in refs[1:]:
    assert np.allclose(cov[q]['lat'], lat) and np.allclose(cov[q]['lon'], lon), 'products on different grids'
n_prod = sum(cov[q]['full'].astype(int) for q in refs)
common = cov[refs[0]]['com']
w = oa_config.area_weights(lat, lon)
ocean = rc.ocean_on(lat, lon)          # one ocean definition for every percentage (regional_common.ocean_mask)
pct = lambda m: 100 * w[m & ocean].sum() / w[ocean].sum()
AREA_C = pct(common & ocean)

# preindustrial (constrained, 1750) and decline, own area and common area
dec = lambda o, p: 100 * (1 - o / p)
PI = {}
for q in refs:
    yo_, so_ = ROWN[q]['constrained'][LAB][:2]
    yc_, sc_ = RCOM[q]['constrained'][LAB][:2]
    PI[q] = dict(own=yo_, sown=so_, com=yc_, scom=sc_,
                 dec_own=dec(V[q]['own'], yo_), sdec_own=100 * V[q]['own'] / yo_ ** 2 * so_,
                 dec_com=dec(V[q]['com'], yc_), sdec_com=100 * V[q]['com'] / yc_ ** 2 * sc_)
    print(f"{q:10s} 2024: full area {V[q]['full']:.3f}, own {V[q]['own']:.3f}, common ({AREA_C:.1f} %) {V[q]['com']:.3f}; "
          f"1750: own {yo_:.3f}, common {yc_:.3f}; decline: own {PI[q]['dec_own']:.1f} %, common {PI[q]['dec_com']:.1f} %")
yc_m = np.mean([PI[q]['com'] for q in refs])
B = 0.8 * yc_m
B_sig = 0.8 * np.sqrt(np.mean([PI[q]['scom'] ** 2 for q in refs]))

# ---------------------------------------------------------------- supplementary figure: coverage map
fig = plt.figure(figsize=(8, 5))
proj_ = {'projection': ccrs.Robinson()} if HAS_CARTOPY else {'projection': 'mollweide'}
kw = {'transform': ccrs.PlateCarree()} if HAS_CARTOPY else {}
X, Y = lon, lat                      # degrees; projected by rc.map_* (Robinson)
proj = proj_ = {}
kw = {}
ax = fig.add_subplot(1, 1, 1, **proj_)
if HAS_CARTOPY:
    ax.set_global()
else:
    ax.set_xticklabels([])
    ax.set_yticklabels([])
cmap = ListedColormap(['0.85', '#e66101', '#fdb863', '#b2abd2', '#8073ac'])    # land, 1, 2, 3, 4 products
pc = rc.map_mesh(ax, X, Y, n_prod, cmap=cmap, norm=BoundaryNorm(np.arange(-0.5, 5), cmap.N), shading='auto', **kw)
rc.map_contour(ax, X, Y, common.astype(float), levels=[0.5], colors='k', linewidths=0.9, **kw)
cb = fig.colorbar(pc, ax=ax, orientation='horizontal', fraction=0.05, pad=0.04, shrink=0.7, ticks=range(1, 5))
cb.ax.set_xlim(0.5, 4.5)
cb.set_label(f'Number of fCO$_2$-products with data in all months and years\n(black line: common area, {AREA_C:.1f}% of the ocean)',
             fontsize=9.5)
rc.savefig(fig, 'coverage_map')
plt.show()

## Extended Data Fig. 1: estimators of the 2024 state (needs the preparation cell)

In [ ]:
# needs the preparation cell to be run first (same kernel): uses its variables
# ---------------------------------------------------------------- Extended Data Fig. 1: estimators of the 2024 state, all products in one figure
# one row per product (named left of the row, turned by 90 degrees); left column: in time, right column: against
# atmospheric CO2 (titles in the first row); each with the quadratic fit (bold, the main estimate), the Enting spline
# with 20 yr cutoff (dashed) and with 15 and 25 yr (thin; in CO2 the equivalent cutoffs); all enter the method range;
# dots: annual means on the common coverage; insets: the last 15 years
rc.FIGNAMES.setdefault('trend_fits', 'FigS11_trend_fits')
co = ENT_MAIN
CUT_COL = {15.0: '#5aa0c8', 25.0: '#c8783c'}   # thin lines: shorter and longer cutoff
fig, AX = plt.subplots(len(refs), 2, figsize=(11, 4.1 * len(refs)), sharey='row', squeeze=False)
letters = 'abcdefghijklmnop'
for r_, q in enumerate(refs):
    e, c = EST[q], rc.COLORS[q]
    axs = AX[r_]
    for k_, (ax_, xk, tag, ttl) in enumerate(((axs[0], 'yrs', 'time', 'in time'),
                                              (axs[1], 'xco2', 'CO2', 'against atmospheric CO$_2$'))):
        ent_lab = lambda c_: f'Enting spline ({c_:.0f} yr)' if tag == 'time' else f'Enting spline ({c_ * e["gr"]:.0f} ppm, {c_:.0f} yr)'
        lines = ((f'fit {tag} quadratic', 'quadratic fit', 'k', '-', 1.6),
                 (f'fit {tag} Enting {co:.0f} yr', ent_lab(co), '0.45', (0, (4, 2)), 1.3))
        lines += tuple((f'fit {tag} Enting {c_:.0f} yr', ent_lab(c_), CUT_COL.get(c_, '0.6'), '-', 0.7)
                       for c_ in ENT_CUTOFFS if c_ != co)
        xs = e[xk]
        ax_.plot(xs, e['y'], 'o', ms=3.5, color=c, alpha=0.7, zorder=1, label='annual mean')
        for kk, lab_, col, ls_, lw_ in lines:
            ax_.plot(xs, e[kk], color=col, lw=lw_, ls=ls_, label=lab_, zorder=3 if lw_ > 1 else 2)
        if r_ == 0:
            ax_.set_title(f'Fits {ttl}', fontsize=11.5, pad=12)
        ax_.text(-0.12 if k_ == 0 else -0.06, 1.03, letters[2 * r_ + k_], transform=ax_.transAxes, fontsize=14,
                 fontweight='bold')
        ax_.spines[['top', 'right']].set_visible(False)
        ax_.legend(fontsize=8, frameon=False, loc='upper right')
        ins = ax_.inset_axes([0.08, 0.08, 0.4, 0.4])
        sel = e['yrs'] >= YEAR - 14
        ins.plot(xs[sel], e['y'][sel], 'o', ms=3, color=c, alpha=0.7)
        for kk, lab_, col, ls_, lw_ in lines:
            ins.plot(xs[sel], e[kk][sel], color=col, lw=lw_, ls=ls_, zorder=3 if lw_ > 1 else 2)
        ins.tick_params(labelsize=7)
        ins.set_title('last 15 years', fontsize=7)
    axs[0].set_ylabel(f'Global mean surface {OMEGA}', fontsize=10)
    axs[0].text(-0.17, 0.5, rc.FULL[q], transform=axs[0].transAxes, rotation=90, ha='right', va='center',
                fontsize=11.5, fontweight='bold')
    if r_ == len(refs) - 1:
        axs[0].set_xlabel('Year')
        axs[1].set_xlabel('Atmospheric CO$_2$ (ppm, NOAA global annual mean)')
    print(f'{rc.FULL[q]}, 2024: ' + ', '.join(f'{k} {v:.3f}' for k, v in e.items() if isinstance(v, float)
                                              and k[:4] in ('time', 'CO2 ')) + f', annual mean {e["y"][-1]:.3f}')
fig.tight_layout(rect=(0.03, 0, 1, 1))
rc.savefig(fig, 'trend_fits')
plt.show()

## Extended Data Fig. 3: why the emergent constraint works (needs test_state_constraint.py)

In [ ]:
# Notebook cell: Extended Data Fig. 3, why the emergent constraint works: the mechanism behind the
# 1750 state, as in the state constraint of the data product. Global means over the cells valid in all models,
# 18 ESMs bias-adjusted to one product (REF, panels a-e); f: all four products.
#   first row, the carbon: dDIC - dALK (ALK - DIC approximates the carbonate ion, so this is close to its loss)
#   a  dDIC - dALK against CUC, each model moved to the observed dSST along the joint regression
#   b  dDIC - dALK against dSST, each model moved to the observed CUC
#   c  dDIC - dALK against its regression on dSST and CUC (1:1 line); observation and constrained value
#   second row, the state:
#   d  the models' 1750 Omega_arag against dDIC - dALK: the carbon change sets the preindustrial state
#   e  SST of the window relative to piControl (dT) against the predictor dSST (relative to 1850-1900):
#      regression, 1 sigma prediction band, observed dSST (+-1 sigma) and the constrained warming (the temperature of
#      the 1750 state; matters little for Omega_arag, much for pCO2, pH and [H+])
#   f  1750 Omega_arag for the four products: models (mean +- 1 s.d.), direct regression of Omega_arag on dSST and CUC
#      (Fig. 2) and the state constraint (dT from dSST, dDIC - dALK from dSST and CUC, dS and dALK multi-model mean,
#      Omega_arag with mocsy), each +- 1 sigma
# Needs data/<run>/state_constraint/state_<ref>_common.npz from test_state_constraint.py --area common
# (run_checks.sh, 'state'): the common area of the global constraint (Fig. 2).
# Wrapped in a function so that its names do not overwrite those of other cells.
def _state_constraint_figure():
    import os
    import importlib
    import numpy as np
    import matplotlib.pyplot as plt
    import regional_common as rc
    importlib.reload(rc)
    rc.FIGNAMES.setdefault('state_constraint', 'FigS12_state_constraint')
    REF = 'OceanSODA'
    REFS = ('OceanSODA', 'UExP', 'JMA', 'CMEMS')
    OMEGA = r'$\Omega_{\mathrm{arag}}$'
    DC = r'$\Delta$DIC $-$ $\Delta$ALK ($\mu$mol kg$^{-1}$)'
    D = {}
    for q in REFS:
        p = f'{rc.ROOT}/state_constraint/state_{q}_common.npz'
        if not os.path.exists(p):
            raise FileNotFoundError(f'{p} missing: run test_state_constraint.py --data {rc.ROOT} --area common (run_checks.sh {rc.RUN}) first')
        D[q] = dict(np.load(p))
    d = D[REF]
    names = [str(x) for x in d['names']]
    NUM = {m: i + 1 for i, m in enumerate(sorted(names, key=str.lower))}
    col, est = rc.COLORS[REF], '#1f4e79'
    x1, x2 = d['dSST'], d['CUC']
    x1o, s1o, x2o, s2o = float(d['dSST_obs']), float(d['dSST_obs_sigma']), float(d['CUC_obs']), float(d['CUC_obs_sigma'])
    dT, dC, y = d['dT'], d['dDIC'] - d['dALK'], d['y1750']
    n = y.size

    def fit(P, t):
        """least squares t on the predictors P (list of (n,)); coefficients, fitted values, residual variance, (X'X)^-1"""
        X = np.column_stack([np.ones(n)] + list(P))
        b = np.linalg.lstsq(X, t, rcond=None)[0]
        r = t - X @ b
        return b, X @ b, r @ r / (n - X.shape[1]), np.linalg.inv(X.T @ X)

    def band(ax, xx, X0, b, s2, inv, color='0.85'):
        """regression line and 1 sigma prediction band at the design rows X0"""
        yy = X0 @ b
        sp = np.sqrt(s2 * (1 + np.einsum('ij,jk,ik->i', X0, inv, X0)))
        ax.fill_between(xx, yy - sp, yy + sp, color=color, lw=0, zorder=0)
        ax.plot(xx, yy, color='k', lw=1.2, zorder=1)
        ax.set_xlim(xx[0], xx[-1])                     # the band reaches both edges

    def points(ax, xs, ys):
        ax.scatter(xs, ys, s=95, color='0.3', edgecolor='white', lw=0.6, zorder=5)
        for xi, yi, m in zip(xs, ys, names):
            ax.text(xi, yi, str(NUM[m]), ha='center', va='center_baseline', fontsize=6, color='white', fontweight='bold',
                    zorder=6)

    def obs(ax, x0, sx):
        ax.axvspan(x0 - sx, x0 + sx, color=col, alpha=0.18, lw=0, zorder=0)
        ax.axvline(x0, color=col, lw=1.5, zorder=2)

    def constrained(ax, v, sv, lab):
        ax.axhspan(v - sv, v + sv, color=est, alpha=0.15, lw=0, zorder=0)
        ax.axhline(v, color=est, lw=1.3, ls='--', zorder=2)
        ax.text(0.02, 0.97, f'{lab}\n{v:.3g} $\\pm$ {sv:.2g}', transform=ax.transAxes, ha='left', va='top', fontsize=9,
                color=est, linespacing=1.3)

    def note(ax, txt, slope):
        """R2 in the corner away from the line and its bands: lower right for a rising, upper right for a falling line"""
        up = slope < 0
        ax.text(0.98, 0.97 if up else 0.03, txt, transform=ax.transAxes, ha='right', va='top' if up else 'bottom',
                fontsize=9, color='0.3', multialignment='right')

    r2 = lambda t, f: 1 - np.var(t - f) / np.var(t)
    fig, AX = plt.subplots(2, 3, figsize=(15.5, 9.4))
    (b_, c_, d_), (e_, a_, f_) = AX                      # variables by content, panels in reading order

    # a: dT on dSST
    bT, fT, s2T, invT = fit([x1], dT)
    xx = np.linspace(min(x1.min(), x1o - s1o) - 0.03, max(x1.max(), x1o + s1o) + 0.03, 100)
    band(a_, xx, np.column_stack([np.ones_like(xx), xx]), bT, s2T, invT)
    points(a_, x1, dT)
    obs(a_, x1o, s1o)
    constrained(a_, float(d['dT_constrained']), float(d['dT_constrained_sigma']), 'constrained')
    a_.set_xlabel(r'$\Delta$SST relative to 1850-1900 ($^\circ$C)')
    a_.set_ylabel(r'$\Delta$SST relative to piControl ($^\circ$C)')
    note(a_, f'$R^2$ = {r2(dT, fT):.2f}', bT[1])
    a_.set_title('Warming since 1750', fontsize=11.5)

    # b, c, d: dDIC - dALK on dSST and CUC
    bC, fC, s2C, invC = fit([x1, x2], dC)
    yb = dC - bC[1] * (x1 - x1o)                          # moved to the observed dSST
    xx2 = np.linspace(min(x2.min(), x2o - s2o), max(x2.max(), x2o + s2o), 100)
    pad2 = 0.05 * np.ptp(xx2)
    xx2 = np.linspace(xx2[0] - pad2, xx2[-1] + pad2, 100)
    band(b_, xx2, np.column_stack([np.ones_like(xx2), np.full_like(xx2, x1o), xx2]), bC, s2C, invC)
    points(b_, x2, yb)
    obs(b_, x2o, s2o)
    b_.set_xlabel(r'CUC of the window state, $\langle$DIC / R$\rangle$ / $p$CO$_2$')
    b_.set_ylabel(DC)
    note(b_, f'$R^2$ = {r2(yb, bC[0] + bC[1] * x1o + bC[2] * x2):.2f}\n(CUC alone)', bC[2])
    b_.set_title(r'$\Delta$DIC $-$ $\Delta$ALK and uptake capacity', fontsize=11.5)

    yc = dC - bC[2] * (x2 - x2o)                          # moved to the observed CUC
    band(c_, xx, np.column_stack([np.ones_like(xx), xx, np.full_like(xx, x2o)]), bC, s2C, invC)
    points(c_, x1, yc)
    obs(c_, x1o, s1o)
    c_.set_xlabel(r'$\Delta$SST relative to 1850-1900 ($^\circ$C)')
    c_.set_ylabel(DC)
    # b and c on the same y range, wide enough for the models moved either way and the bands
    lo_, hi_ = min(yb.min(), yc.min()), max(yb.max(), yc.max())
    pd_ = 0.08 * (hi_ - lo_)
    for ax in (b_, c_):
        ax.set_ylim(lo_ - pd_, hi_ + pd_)
    note(c_, f'$R^2$ = {r2(yc, bC[0] + bC[1] * x1 + bC[2] * x2o):.2f}\n($\\Delta$SST alone)', bC[1])
    c_.set_title(r'$\Delta$DIC $-$ $\Delta$ALK and warming', fontsize=11.5)

    xhat = fC
    x0h = float(bC @ [1.0, x1o, x2o])
    lo, hi = min(xhat.min(), dC.min()), max(xhat.max(), dC.max())
    pad = 0.06 * (hi - lo)
    xx3 = np.linspace(lo - pad, hi + pad, 100)
    d_.fill_between(xx3, xx3 - np.sqrt(s2C), xx3 + np.sqrt(s2C), color='0.85', lw=0, zorder=0)
    d_.plot(xx3, xx3, color='k', lw=1.2, zorder=1)
    d_.set_xlim(xx3[0], xx3[-1])
    points(d_, xhat, dC)
    obs(d_, x0h, float(np.hypot(bC[1] * s1o, bC[2] * s2o)))
    constrained(d_, float(d['dC_constrained']), float(d['dC_constrained_sigma']), 'constrained')
    d_.set_xlabel('Regression value $a + b\\,\\Delta$SST$ + c\\,$CUC ($\\mu$mol kg$^{-1}$)')
    d_.set_ylabel(DC)
    note(d_, f'$R^2$ = {r2(dC, fC):.2f}\n(both predictors)', 1.0)
    d_.set_title(r'$\Delta$DIC $-$ $\Delta$ALK, both predictors', fontsize=11.5)

    # e: 1750 Omega against dDIC - dALK
    be, fe, s2e, inve = fit([dC], y)
    xx4 = np.linspace(dC.min() - pad, dC.max() + pad, 100)
    band(e_, xx4, np.column_stack([np.ones_like(xx4), xx4]), be, s2e, inve)
    points(e_, dC, y)
    e_.axvspan(float(d['dC_constrained']) - float(d['dC_constrained_sigma']),
               float(d['dC_constrained']) + float(d['dC_constrained_sigma']), color=est, alpha=0.15, lw=0, zorder=0)
    e_.axvline(float(d['dC_constrained']), color=est, lw=1.3, ls='--', zorder=2)
    e_.set_xlabel(DC)
    e_.set_ylabel(f'Preindustrial (1750) {OMEGA}')
    note(e_, f'$R^2$ = {r2(y, fe):.2f}', be[1])
    e_.set_title(f'From $\\Delta$DIC $-$ $\\Delta$ALK to 1750 {OMEGA}', fontsize=11.5)

    # f: the four products, direct regression and state constraint
    for k, q in enumerate(REFS):
        g = D[q]
        c = rc.COLORS[q]
        f_.errorbar(k - 0.22, float(g['unconstrained']), yerr=float(g['unconstrained_sd']), fmt='o', color='0.55', ms=6,
                    capsize=3, label='models (mean $\\pm$ 1 s.d.)' if k == 0 else None)
        f_.errorbar(k, float(g['direct']), yerr=float(g['direct_sigma']), fmt='o', mfc='white', mec=c, color=c, ms=7,
                    capsize=3, mew=1.6, label=f'direct regression of {OMEGA} (Fig. 2)' if k == 0 else None)
        f_.errorbar(k + 0.22, float(g['state']), yerr=float(g['state_sigma']), fmt='s', color=c, ms=6.5, capsize=3,
                    label='state constraint (warming, $\\Delta$DIC $-$ $\\Delta$ALK)' if k == 0 else None)
        print(f'  {rc.FULL[q]:22s} models {float(g["unconstrained"]):.4f} +- {float(g["unconstrained_sd"]):.4f}; direct '
              f'{float(g["direct"]):.4f} +- {float(g["direct_sigma"]):.4f}; state {float(g["state"]):.4f} +- '
              f'{float(g["state_sigma"]):.4f} ({float(g["state"]) - float(g["direct"]):+.4f})')
    f_.set_xticks(range(len(REFS)), [rc.FULL[q].split()[0].split('-')[0] for q in REFS], fontsize=9.5)
    f_.set_xlim(-0.6, len(REFS) - 0.4)
    f_.set_ylabel(f'Preindustrial (1750) {OMEGA}')
    f_.legend(fontsize=8.5, frameon=False, loc='upper right')
    f_.set_title('Direct regression and state constraint', fontsize=11.5)

    for ax, l in zip(AX.ravel(), 'abcdef'):
        ax.spines[['top', 'right']].set_visible(False)
        ax.text(-0.14, 1.04, l, transform=ax.transAxes, fontsize=14, fontweight='bold', va='bottom')
    fig.text(0.5, 0.005, f'a-e: the 18 ESMs bias-adjusted to {rc.FULL[REF]}, means over the common area; numbers as in Fig. 2', ha='center',
             fontsize=9, color='0.35')
    fig.tight_layout(rect=(0, 0.02, 1, 1), h_pad=2.5, w_pad=2.0)
    print(f'{rc.FULL[REF]}: dT = {bT[0]:.3f} + {bT[1]:.3f} dSST (degC) (R2 {r2(dT, fT):.2f}); dDIC - dALK = {bC[0]:.1f} {bC[1]:+.2f} dSST '
          f'{bC[2]:+.1f} CUC (R2 {r2(dC, fC):.2f}); Omega_1750 = {be[0]:.3f} {be[1]:+.5f} (dDIC - dALK) (R2 {r2(y, fe):.2f})')
    rc.savefig(fig, 'state_constraint')
    plt.show()


_state_constraint_figure()

## Diagnostic: local mapping uncertainty of the observed CUC (needs cuc_mapping_uncertainty.py)

In [ ]:
# Notebook cell (diagnostic, not a figure of the paper): the local mapping uncertainty of the observed CUC in the
# cell-by-cell constraint, common area, one row per product (named left of the row):
#   left    local 1 sigma of the observed CUC from the product's mapping errors (window mean, C2)
#   middle  its effect on the local constrained 1750 Omega_arag, |beta_CUC| * sigma_CUC (beta of the local regression)
#   right   relative change of the local constraint sigma when the set value 0.001 is replaced by this local sigma
# Land grey, ocean without data white (as the other maps); the colour maps start above white, so that a
# value of zero is not taken for a missing value.
# Needs data/<run>/product_uncertainty/cuc_mapping_<ref>.npz (cuc_mapping_uncertainty.py).
# Wrapped in a function so that its names do not overwrite those of other cells.
import oa_config                      # the area weights of the run (OA_WEIGHTS)
def _cuc_local_maps():
    import os
    import importlib
    import numpy as np
    import matplotlib.pyplot as plt
    import regional_common as rc
    importlib.reload(rc)
    rc.FIGNAMES.setdefault('cuc_local', 'Diag_cuc_local_uncertainty')
    REFS = [q for q in ('OceanSODA', 'UExP', 'JMA', 'CMEMS')
            if os.path.exists(f'{rc.ROOT}/product_uncertainty/cuc_mapping_{q}.npz')]
    if not REFS:
        raise FileNotFoundError(f'no cuc_mapping_<ref>.npz in {rc.ROOT}/product_uncertainty: run cuc_mapping_uncertainty.py')
    D = {q: np.load(f'{rc.ROOT}/product_uncertainty/cuc_mapping_{q}.npz') for q in REFS}
    g = lambda q, k: np.asarray(D[q][f'common_field_{k}'], 'f8')
    allv = lambda k: np.concatenate([g(q, k)[np.isfinite(g(q, k))] for q in REFS])
    v1 = np.nanpercentile(allv('sigma_cuc'), 98)
    v2 = np.nanpercentile(allv('shift'), 98)
    rel = {q: 100 * (g(q, 'sigma_constraint_mapping') / g(q, 'sigma_constraint_set') - 1) for q in REFS}
    v3 = np.nanpercentile(np.concatenate([r[np.isfinite(r)] for r in rel.values()]), 98)
    fig, AX = plt.subplots(len(REFS), 3, figsize=(15, 2.9 * len(REFS) + 1.4), squeeze=False)
    from matplotlib.colors import ListedColormap
    trunc = lambda name, a: ListedColormap(plt.get_cmap(name)(np.linspace(a, 1.0, 256)))
    cols = (('sigma_cuc', 'Local 1$\\sigma$ of the observed CUC', trunc('magma_r', 0.06), 0, v1, 'CUC'),
            ('shift', 'Effect on 1750 $\\Omega_{\\mathrm{arag}}$, $|\\beta_{\\mathrm{CUC}}|\\,\\sigma_{\\mathrm{CUC}}$',
             trunc('magma_r', 0.06), 0, v2, '$\\Omega_{\\mathrm{arag}}$'),
            ('rel', 'Change of the local constraint $\\sigma$ against a set 0.001', trunc('Reds', 0.12), 0, v3, '%'))
    ims = {}
    for r_, q in enumerate(REFS):
        lat, lon = g(q, 'lat'), g(q, 'lon')
        land = ~rc.ocean_on(lat, lon).astype(bool)
        for k_, (key, ttl, cmap, lo, hi, unit) in enumerate(cols):
            ax = AX[r_, k_]
            a = rel[q] if key == 'rel' else g(q, key)
            rc.map_mesh(ax, lon, lat, np.where(land, 1.0, np.nan), cmap=ListedColormap(['0.85']), vmin=0, vmax=1, rasterized=True)
            ims[k_] = rc.map_mesh(ax, lon, lat, np.where(land, np.nan, a), cmap=cmap, vmin=lo, vmax=hi, rasterized=True)
            if r_ == 0:
                ax.set_title(ttl, fontsize=11)
            ax.text(0.0, 1.02, 'abcdefghijkl'[3 * r_ + k_], transform=ax.transAxes, fontsize=16, fontweight='bold')
        AX[r_, 0].text(-0.06, 0.5, rc.FULL[q], transform=AX[r_, 0].transAxes, rotation=90, ha='right', va='center',
                       fontsize=11, fontweight='bold')
        w = oa_config.area_weights(lat, lon) * np.isfinite(rel[q])
        print(f'{rc.FULL[q]:22s} local sigma CUC median {np.nanmedian(g(q, "sigma_cuc")):.4f}; effect on 1750 median '
              f'{np.nanmedian(g(q, "shift")):.4f}, 95 % {np.nanpercentile(g(q, "shift"), 95):.4f}; constraint sigma: '
              f'median change {np.nanmedian(rel[q]):+.1f} %, area > 10 %: {100 * np.sum(w * (rel[q] > 10)) / w.sum():.1f} %')
    for k_, (key, ttl, cmap, lo, hi, unit) in enumerate(cols):
        cb = fig.colorbar(ims[k_], ax=AX[:, k_], orientation='horizontal', fraction=0.025, pad=0.03, extend='max')
        cb.set_label(unit, fontsize=9)
    rc.savefig(fig, 'cuc_local')
    plt.show()


_cuc_local_maps()

## Extended Data Fig. 5: from Findlay et al. (2025) to this study

(Runs on its own; Jiang et al. 2023 aragonite fields, historical and SSP2-4.5, in data/jiang2023)

In [ ]:
# Notebook cell: Extended Data Fig. 5, from Findlay et al. (2025) to this study, in 2020 (their year).
# Needs the preparation cell (FIG4).
#   a  one change per bar, as decline of global mean surface Omega_arag from the preindustrial state (%); 1750 and 2020
#      stay consistent in every bar (same data, area and constants, as in Findlay et al.). The chain stays within the data
#      of Jiang et al. (2023) until the last step, which changes to the four products:
#      Findlay et al. (2025) with the 1750 value of Jiang et al., both on the area of Jiang et al.
#      -> the area of this study (Jiang et al. fields for 1750 and 2020, the latter from their SSP2-4.5 file)
#      -> observed instead of projected CO2: their 2020 state moved from the SSP2-4.5 CO2 (2015-2024 mean) to the NOAA
#         CO2 of 2015-2024 with their own chemistry (their 2005-2014 fields, surface fCO2 raised by the CO2 difference,
#         jiang_chem.omega with their constants: ratio G of the two states)
#      -> total boron of this study (Uppstroem) for both states (1750: F_J of the preparation cell; 2020: the same
#         ratio on the state of the previous step, F20)
#      -> our 18 constrained ESMs instead of their 14: the constrained 1750 state of their observations through the chain
#         of this study (constraint_results_jiang.pkl, Extended Data Fig. 2), at their CO2 (272.4 ppm)
#      -> CO2 of 1750 from AR6 (278.3 ppm)
#      -> the four products instead of Jiang et al., observed (2020 fit) and preindustrial state together = this study
#   b  the same chain for 1850, from the 3.44 of Findlay et al. and the 1850 of Jiang et al. (what Findlay et al. would
#      have obtained without the box value of Rockstroem et al.) to this study's 1850 (AR6 CO2, 285.5 ppm)
#   c  range of the decline from the definition of the preindustrial state: Findlay et al. as published (1750 of Jiang
#      et al., the box value 3.44 given as their 1850 (dashed part of the range), their midpoint, for
#      comparison the 1850 of Jiang et al.) and this study (1750, midpoint, 1850). ±1σ: constraint and the CO2 of the year.
# Data: the preparation cell (FIG4), so that this figure uses the same
# boron conversion, constants and uncertainty budget as Figs. 1 to 4 and Extended Data Tables 1 to 3 (run that cell first).
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.gridspec import GridSpec
from matplotlib.lines import Line2D

assert 'FIG4' in globals(), 'run the preparation cell first'
C = dict(FIG4)
rc = C['rc']
rc.FIGNAMES.setdefault('findlay', 'FigS_findlay')
refs, PI, RCOM, dec = C['refs'], C['PI'], C['RCOM'], C['dec']
YEAR_F = 2020
OMEGA = r'$\Omega_{\mathrm{arag}}$'
# Findlay et al. (2025): observed 2020 = Jiang et al. (2023); 1750 = Jiang et al.; 1850 = 3.44 in their Table S1, given as
# the 1850 value of Jiang et al. (whose 1850 field gives 3.48 on their coverage; 3.44 is also the value of the original
# boundary, Rockstroem et al. 2009); their central value: the midpoint
F_OBS, F_1750, F_1850 = 2.90, 3.57, 3.44
F_MID = 0.5 * (F_1750 + F_1850)

now_f = {q: rc.fit_now(RCOM[q], YEAR_F) for q in refs}               # quadratic fit, common area, in 2020
now_fm = np.mean(list(now_f.values()))
PI_J = C['PI_J_COM_U']                       # 1750 of Jiang et al. on the common area, with the constants of this run


def ours(key):
    """mean of the products in 2020 for a preindustrial state: decline and 1 sigma (the budget of Extended Data Table 2: spread of
    the products, constraint and CO2 of the preindustrial year per product, and the other terms)"""
    d, s = [], []
    for q in refs:
        S = RCOM[q]['states']
        pi = PI[q]['com'] * S[key]['ratio']
        lo, hi = (key + '_lo', key + '_hi') if key + '_lo' in S else ('1750_lo', '1750_hi')
        s_co2 = 0.5 * PI[q]['com'] * abs(S[hi]['ratio'] - S[lo]['ratio'])
        if key == 'x280':
            s_co2 = 0.0                                  # set value, not an estimate of a year
        spi = np.hypot(PI[q]['scom'] * S[key]['ratio'], s_co2)
        d.append(dec(now_f[q], pi))
        s.append(100 * now_f[q] / pi ** 2 * spi)
    # the other terms of the budget (Extended Data Table 2 without the spread of the products, the ESM constraint and the CO2 of
    # 1750, which are replaced by their values for this preindustrial year)
    other2 = C['SDEC_SHARED'] ** 2 - C['SIG'][C['K_ESM']] ** 2 - C['SIG'][C['K_CO2']] ** 2
    return np.mean(d), np.sqrt(np.std(d, ddof=1) ** 2 + np.mean(np.square(s)) + other2)


# Every bar keeps 1750 and 2020 consistent (same data, same area, same constants), as Findlay et al. did: their 2020
# value and their 1750 value both come from Jiang et al. (2023). Only one thing changes per bar.
_jy, _J, _wj = C['_jy'], C['_J'], C['_wj']
_f10 = _J[list(_jy).index(2010)]
J10_OWN = np.nansum(np.where(np.isfinite(_f10), _f10, 0) * _wj) / _wj[np.isfinite(_f10)].sum()   # Jiang 2010s, own coverage
# their 2020 field (SSP2-4.5; the scenarios differ by < 0.01 in 2020), read and sorted as their historical file,
# so that their 2020 value can be put on our common area directly
import netCDF4
JSSP = f'{rc.ROOT}/jiang2023/Aragonite_median_ssp245.nc'
with netCDF4.Dataset(JSSP) as f:
    _sy = np.asarray(f['time'][:]).astype(int)
    _slon = (np.asarray(f['longitude'][:], 'f8')[0, :] + 180.0) % 360.0 - 180.0
    _S = np.ma.filled(f['aragonite'][:].astype('f8'), np.nan)
_S[~np.isfinite(_S) | (_S <= 0) | (_S > 10)] = np.nan
_S = _S[:, :, np.argsort(_slon)]
_f20 = _S[list(_sy).index(YEAR_F)]
assert _f20.shape == _f10.shape, 'SSP file on a different grid than the historical file'
J20_OWN = np.nansum(np.where(np.isfinite(_f20), _f20, 0) * _wj) / _wj[np.isfinite(_f20)].sum()   # Jiang 2020, own coverage
_rq = RCOM[refs[0]]
J20_COM = C['jiang_mean'](_f20, _rq['lat'], _rq['lon'], _rq['mask'])                        # Jiang 2020, common area
J10_COM, PI_J_COM, F_J, F_J10 = C['J10_COM'], C['PI_J_COM'], C['F_J'], C['F_J10']
mdec = lambda f: np.mean([dec(now_f[q], f(q)) for q in refs])          # mean over the products of the decline
_com_j = lambda fld: C['jiang_mean'](fld, _rq['lat'], _rq['lon'], _rq['mask'])


def _ratio_j(a_, b_):
    """common-area mean of a_ over that of b_, on the cells where both are valid"""
    m_ = np.isfinite(a_) & np.isfinite(b_)
    return _com_j(np.where(m_, a_, np.nan)) / _com_j(np.where(m_, b_, np.nan))


# observed instead of projected CO2, within the data of Jiang et al.: their 2020 state (SSP2-4.5, decade 2015-2024)
# moved to the observed CO2 of the same years with their chemistry, as the ratio of their 2005-2014 state with surface
# fCO2 raised to the observed and to the SSP2-4.5 CO2 of 2015-2024 (both relative to the CMIP6 historical CO2 of
# 2005-2014, which their 2005-2014 state follows)
import pickle
import co2_history
import jiang_chem
_hist, _ssp, _noaa = co2_history.historical(), co2_history.scenario('ssp245'), co2_history.noaa()
assert _hist and _ssp, 'CMIP6 CO2 files (historical, SSP2-4.5) missing in inputs/co2'
assert all(y_ in _noaa for y_ in range(2015, 2025)), 'NOAA global annual means of 2015-2024 missing'
X_J10 = float(np.mean([_hist[y_] for y_ in range(2005, 2015)]))
X_SSP20 = float(np.mean([_ssp[y_] for y_ in range(2015, 2025)]))
X_OBS20 = float(np.mean([_noaa[y_] for y_ in range(2015, 2025)]))
G_OBS = _ratio_j(jiang_chem.omega(2010, X_OBS20 - X_J10), jiang_chem.omega(2010, X_SSP20 - X_J10))
J20_O = J20_COM * G_OBS                                         # Jiang et al. 2020 at the observed CO2, their constants
# total boron of this study: the ratio of the states recomputed with our constants and with theirs (2020: the 2005-2014
# fields at the observed CO2 of 2015-2024; 1750: F_J of the preparation cell; 1850: their 1850 fields)
F20 = _ratio_j(jiang_chem.omega(2010, X_OBS20 - X_J10, **jiang_chem.OURS), jiang_chem.omega(2010, X_OBS20 - X_J10))
F50 = _ratio_j(jiang_chem.omega(1850, 0.0, **jiang_chem.OURS), jiang_chem.omega(1850, 0.0))
J20_OU = J20_O * F20
print(f'CO2 2015-2024: SSP2-4.5 {X_SSP20:.2f}, NOAA {X_OBS20:.2f} ppm (CMIP6 2005-2014 {X_J10:.2f}); Jiang et al. 2020 '
      f'(common area) {J20_COM:.4f} -> {J20_O:.4f} at the observed CO2 (x {G_OBS:.5f}); boron factors 1750 {F_J:.5f}, '
      f'1850 {F50:.5f}, 2005-2014 {F_J10:.5f}, 2020 {F20:.5f}')
# their observations through the chain of this study: constrained 1750 state on the common area (Extended Data Fig. 2)
with open(f'{rc.ROOT}/constraint_results_jiang.pkl', 'rb') as f_:
    _rj = pickle.load(f_)['common']['Jiang']
YJ = float(_rj['constrained'][rc.LAB][0])                       # 1750 (278.3 ppm), constrained, our constants
X50_CMIP6 = 284.59


def _r_cmip6_1850(S):
    """1850 state at the CMIP6 CO2 of 1850 (284.59 ppm) relative to 1750: linear between the states at 284.0 and 285.5 ppm"""
    w = (X50_CMIP6 - S['pi_high']['xco2']) / (S['1850']['xco2'] - S['pi_high']['xco2'])
    return S['pi_high']['ratio'] + w * (S['1850']['ratio'] - S['pi_high']['ratio'])


def ratio_cmip6_1850(q):
    return _r_cmip6_1850(RCOM[q]['states'])


_SJ = _rj.get('states')
if _SJ is not None and all(k_ in _SJ for k_ in ('jiang', '1850', 'pi_high')):
    RJ = {'jiang': _SJ['jiang']['ratio'], '1850': _SJ['1850']['ratio'], 'cmip6_1850': _r_cmip6_1850(_SJ)}
    _rsrc = 'Jiang reference'
else:          # the modelled ratios between CO2 states (the same 18 ESMs): mean over the products
    RJ = {'jiang': np.mean([RCOM[q]['states']['jiang']['ratio'] for q in refs]),
          '1850': np.mean([RCOM[q]['states']['1850']['ratio'] for q in refs]),
          'cmip6_1850': np.mean([ratio_cmip6_1850(q) for q in refs])}
    _rsrc = 'mean of the products (no states in constraint_results_jiang.pkl)'
print(f'Jiang et al. through the chain of this study: 1750 {YJ:.4f} (278.3 ppm); ratios to it ({_rsrc}): 272.4 ppm '
      f'{RJ["jiang"]:.5f}, 1850 CMIP6 {RJ["cmip6_1850"]:.5f}, 1850 AR6 {RJ["1850"]:.5f}; their 2005-2014 state in our '
      f'chemistry {float(_rj["obs10"]):.4f} (chain) and {J10_COM * F_J10:.4f} (boron factor)')
# short labels (9 bars in b; the full wording is in Extended Data Table 3)
L_AREA = 'ocean area\nof this study'
L_OBS = 'observed\ninstead of\nprojected CO$_2$'
L_BOR = 'total boron\nof this study\n(Uppström)'
L_ESM = '18 constrained\nESMs instead\nof 14'
L_PROD = 'four fCO$_2$-\nproducts instead\nof Jiang et al.'
STEPS = [('Findlay et al.\n(2025, 1750 of\nJiang et al.)', dec(F_OBS, F_1750)),
         (L_AREA, dec(J20_COM, PI_J_COM)),
         (L_OBS, dec(J20_O, PI_J_COM)),
         (L_BOR, dec(J20_OU, PI_J_COM * F_J)),
         (L_ESM, dec(J20_OU, YJ * RJ['jiang'])),
         (f'CO$_2$ of 1750:\n{C["X_1750"]:.1f} ppm (AR6)\ninstead of {C["X_J"]:.0f}', dec(J20_OU, YJ)),
         (L_PROD, mdec(lambda q: PI[q]['com']))]
STEPS += [(f'this study\n({YEAR_F}, 1750)', STEPS[-1][1])]       # final bar: the same value, drawn from zero
vals = [v for _, v in STEPS]
# the same chain for 1850: Findlay et al. would have obtained the second bar with the 1850 of Jiang et al. (their 1850
# model state, forced with the CMIP6 CO2 of 1850-1854, 284.59 ppm) instead of the box value of Rockstroem et al.
_f50 = _J[list(_jy).index(1850)]
J50_OWN = np.nansum(np.where(np.isfinite(_f50), _f50, 0) * _wj) / _wj[np.isfinite(_f50)].sum()   # Jiang 1850, own coverage
J50_COM = C['jiang_mean'](_f50, _rq['lat'], _rq['lon'], _rq['mask'])                        # Jiang 1850, common area
X50_AR6 = RCOM[refs[0]]['states']['1850']['xco2']
STEPS50 = [('Findlay et al.\n(2025, 1850:\n3.44)', dec(F_OBS, F_1850)),
           (f'1850 of\nJiang et al.\n({J50_OWN:.2f})\ninstead of 3.44', dec(F_OBS, J50_OWN)),
           (L_AREA, dec(J20_COM, J50_COM)),
           (L_OBS, dec(J20_O, J50_COM)),
           (L_BOR, dec(J20_OU, J50_COM * F50)),
           (L_ESM, dec(J20_OU, YJ * RJ['cmip6_1850'])),
           (f'CO$_2$ of 1850:\n{X50_AR6:.1f} ppm (AR6)\ninstead of {X50_CMIP6:.1f}', dec(J20_OU, YJ * RJ['1850'])),
           (L_PROD, mdec(lambda q: PI[q]['com'] * RCOM[q]['states']['1850']['ratio']))]
STEPS50 += [(f'this study\n({YEAR_F}, 1850)', STEPS50[-1][1])]
print('steps 1850: ' + '; '.join(f'{l.replace(chr(10), " ")} {v:.1f} %' for l, v in STEPS50))
# b: decline against the preindustrial CO2 that each value corresponds to (ppm)
XCO2 = {k: RCOM[refs[0]]['states'][k]['xco2'] for k in ('jiang', '1750', 'x280', 'mid', '1850')}
OURS = {k: ours(k) for k in XCO2}
DEF = [('1750', dec(F_OBS, F_1750), OURS['1750']), ('midpoint', dec(F_OBS, F_MID), OURS['mid']),
       ('1850', dec(F_OBS, F_1850), OURS['1850'])]
print(f'Jiang et al. {YEAR_F} (SSP2-4.5): own coverage {J20_OWN:.3f} (Findlay et al.: {F_OBS}), common area {J20_COM:.3f}; '
      f'1750 own coverage {C["PI_J_OWN"]:.3f} (Findlay et al.: {F_1750}); decline own coverage '
      f'{dec(J20_OWN, C["PI_J_OWN"]):.1f} %, common area {dec(J20_COM, PI_J_COM):.1f} %')
print(f'run {rc.RUN}; {YEAR_F}: products {now_fm:.3f} (Findlay {F_OBS}); Jiang et al. 2010s: own coverage {J10_OWN:.3f}, '
      f'common area {J10_COM:.3f}; 1750 common area {PI_J_COM:.3f}')
print('steps: ' + '; '.join(f'{l.replace(chr(10), " ")} {v:.1f} %' for l, v in STEPS))
print('definitions (Findlay | this study): ' + '; '.join(f'{k} {f:.1f} | {o[0]:.1f} ± {o[1]:.1f} %' for k, f, o in DEF))
print(f'range from the definition: Findlay {max(d[1] for d in DEF) - min(d[1] for d in DEF):.1f}, this study '
      f'{max(d[2][0] for d in DEF) - min(d[2][0] for d in DEF):.1f} points')

plt.close('all')
fig = plt.figure(figsize=(14, 9.5))
gs = GridSpec(2, 2, figure=fig, width_ratios=[2.4, 1.25], hspace=0.62, wspace=0.18, left=0.07, right=0.98, top=0.95,
              bottom=0.12)
C_START, C_END, C_DOWN, C_UP = '0.6', '0.15', '#3d6f8e', '#b9cbd6'
BW = 0.5


def waterfall(ax, steps, letter, title):
    vals_ = [v for _, v in steps]
    n = len(steps)
    for i, v in enumerate(vals_):
        if i == 0:
            ax.bar(i, v, width=BW, color=C_START, zorder=2)
        elif i == n - 1:
            ax.bar(i, v, width=BW, color=C_END, zorder=2)
        else:
            l_, h_ = sorted((vals_[i - 1], v))
            ax.bar(i, max(h_ - l_, 0.02), bottom=l_, width=BW, color=C_DOWN if v < vals_[i - 1] else C_UP, zorder=2)
        if i > 0:
            ax.plot([i - 1 + BW / 2, i - BW / 2], [vals_[i - 1]] * 2, color='0.6', lw=0.6, zorder=1)
        down = 0 < i < n - 1 and v < vals_[i - 1]
        ax.text(i, v - 0.12 if down else v + 0.12, f'{v:.1f}%', ha='center', va='top' if down else 'bottom', fontsize=9,
                fontweight='bold' if i in (0, n - 1) else 'normal')
    ax.axhline(20, color='k', lw=0.9, ls=(0, (4, 3)), zorder=1)
    ax.text(n - 0.45, 20.08, 'boundary: 20% decline', ha='right', va='bottom', fontsize=8.5)
    ax.set_xticks(range(n))
    ax.set_xticklabels([l for l, _ in steps], fontsize=8 if n <= 6 else 7.2, linespacing=1.1)
    ax.tick_params(axis='x', length=0)
    ax.set_xlim(-0.6, n - 0.4)
    ax.set_ylim(np.floor(2 * (min(vals_) - 0.7)) / 2, 20.5)
    ax.set_ylabel(f'Decline of global mean\nsurface {OMEGA} in {YEAR_F} (%)', fontsize=10)
    ax.set_title(title, fontsize=10, loc='left')
    ax.spines[['top', 'right']].set_visible(False)
    ax.text(-0.1, 1.04, letter, transform=ax.transAxes, fontsize=16, fontweight='bold', va='bottom')


waterfall(fig.add_subplot(gs[0, 0]), STEPS, 'a', 'Preindustrial state in 1750')
waterfall(fig.add_subplot(gs[1, 0]), STEPS50, 'b', 'Preindustrial state in 1850')

# ---------------------------------------------------------------- c: range from the definition of the preindustrial state
# three columns, each with 1750, 1850 and their midpoint (larger dot), decline with the 2020 value of the column:
#   Findlay et al. as published (area and 2020 value of Jiang et al.; 1850 = 3.44 of their Table S1, which the 1850
#     field of Jiang et al. does not support: the part of the range below it is dashed, ending at the open square)
#   Jiang et al. at the AR6 CO2 of 1750 and 1850 (their data and method, CO2 corrected: their 1750 and 1850 times the
#     ratios R1750_AR6 and R1850_AR6 of the preparation cell, recomputed with mocsy and their constants, jiang_chem.py)
#   this study (AR6 CO2, common area, four products)
ax = fig.add_subplot(gs[:, 1])
R1750_AR6, R1850_AR6 = C['R1750_AR6'], C['R1850_AR6']
FD = dict(d1750=dec(F_OBS, F_1750), d1850J=dec(F_OBS, J50_OWN), dbox=dec(F_OBS, F_1850), dmid=dec(F_OBS, F_MID))
JA = {'1750': F_1750 * R1750_AR6, '1850': J50_OWN * R1850_AR6}
JD_ = {k: dec(F_OBS, v) for k, v in JA.items()}
JD_['mid'] = dec(F_OBS, 0.5 * (JA['1750'] + JA['1850']))
OM = {k: ours(k)[0] for k in ('1750', 'mid', '1850')}
FG, JG, FS = '0.55', '0.35', 6.8                    # grey of Findlay et al., of Jiang et al. (AR6 CO2), label size
X = [0.0, 1.0, 2.0]


def column(x, top, bottom, mid, col, labels):
    ax.plot([x, x], [bottom, top], color=col, lw=1.5, solid_capstyle='butt', zorder=2)
    ax.plot(x, top, 'o', color=col, ms=4.5, zorder=3)
    ax.plot(x, bottom, 'o', color=col, ms=4.5, zorder=3)
    ax.plot(x, mid, 'o', color=col, ms=8, zorder=3)
    for y_, t_ in zip((top, mid, bottom), labels):
        ax.annotate(t_, (x, y_), xytext=(8, 0), textcoords='offset points', va='center', fontsize=FS, color=col)


# Findlay et al.: solid down to the 1850 of Jiang et al., dashed to the 3.44 they used (open square)
column(X[0], FD['d1750'], FD['d1850J'], FD['dmid'], FG, ('1750 (272 ppm)', 'midpoint\nused', '1850 of\nJiang et al.'))
ax.plot([X[0], X[0]], [FD['dbox'], FD['d1850J']], color=FG, lw=1.0, ls=(0, (2, 2)), zorder=2)
ax.plot(X[0], FD['dbox'], 's', mfc='white', mec=FG, mew=1.0, ms=5, zorder=3)
ax.annotate('3.44 used as 1850\n(= original boundary)', (X[0], FD['dbox']), xytext=(8, 0), textcoords='offset points',
            va='center', fontsize=FS, color=FG)
column(X[1], JD_['1750'], JD_['1850'], JD_['mid'], JG, ('1750 (278.3 ppm)', 'midpoint', '1850 (285.5 ppm)'))
column(X[2], OM['1750'], OM['1850'], OM['mid'], C_END, ('1750 (278.3 ppm)', 'midpoint', '1850 (285.5 ppm)'))
# what changes from column to column (arrows above the columns)
YA = 19.45
for xa, xb, t_ in ((X[0], X[1], 'CO$_2$ of 1750 and\n1850 from AR6;\n1850 of Jiang et al.'),
                   (X[1], X[2], 'area, observed CO$_2$,\nboron and 18 constrained\nESMs of this study,\nfour fCO$_2$-products')):
    ax.annotate('', xy=(xb - 0.05, YA), xytext=(xa + 0.05, YA),
                arrowprops=dict(arrowstyle='-|>,head_length=0.3,head_width=0.15', color='0.5', lw=0.8))
    ax.text(0.5 * (xa + xb), YA + 0.05, t_, ha='center', va='bottom', fontsize=FS, color='0.3', linespacing=1.15)
ax.axhline(20, color='k', lw=0.9, ls=(0, (4, 3)), zorder=1)
ax.set_xticks(X)
ax.set_xticklabels(['Findlay et al.\n(2025)', 'Jiang et al.\nat AR6 CO$_2$', f'this study\n({YEAR_F})'], fontsize=8.5)
ax.set_xlim(-0.3, 2.95)
ax.set_ylabel(f'Decline in {YEAR_F} (%)', fontsize=10)
ax.spines[['top', 'right']].set_visible(False)
ax.tick_params(axis='x', length=0)
ax.set_title('Range from the preindustrial state', fontsize=10)
ax.text(-0.2, 1.02, 'c', transform=ax.transAxes, fontsize=16, fontweight='bold', va='bottom')
print('decline in 2020 (1750 | midpoint | 1850): Findlay et al. as published '
      f'{FD["d1750"]:.1f} | {FD["dmid"]:.1f} | {FD["dbox"]:.1f} % (1850 of Jiang et al.: {FD["d1850J"]:.1f} %); '
      f'Jiang et al. at AR6 CO2 {JD_["1750"]:.1f} | {JD_["mid"]:.1f} | {JD_["1850"]:.1f} %; '
      f'this study {OM["1750"]:.1f} | {OM["mid"]:.1f} | {OM["1850"]:.1f} %')
rc.savefig(fig, 'findlay')
plt.show()

## Extended Data Table 3: steps from the Planetary Health Check 2026 and Findlay et al. (2025) to this study (needs the cells of block a and of Extended Data Fig. 5)

In [ ]:
# Notebook cell: Extended Data Table 3, the steps from the Planetary Health Check 2026 (block a, rows of the cell
# block a, bars of Fig. 3c) and from Findlay et al. (2025) (blocks b and c, the bars of Extended Data Fig. 5,
# preindustrial state in 1750 and in 1850) to this study. Needs the cells of block a of Extended Data Table 3 (EDT3_A) and of Extended Data Fig. 5 (STEPS,
# STEPS50 and their inputs) of the same kernel: run in the order of the notebook. Blocks b and c: values of the
# fCO2-products are means over the four, declines the mean of their declines (as the bars of Extended Data Fig. 5).
import csv
import numpy as np
import matplotlib.pyplot as plt

assert 'EDT3_A' in globals(), 'run the cell of block a of Extended Data Table 3 first'
assert 'STEPS50' in globals(), 'run the cell of Extended Data Fig. 5 first'
A_ = EDT3_A
KJ, KO = A_['K_J'], A_['K_OURS']
x50 = RCOM[refs[0]]['states']['1850']['xco2']
J_SSP = 'Jiang et al. (2023), SSP2-4.5'
J_OBS = 'Jiang et al., observed CO$_2$'
Y_J = f'{YEAR_F} (decade)'
Y_F = f'{YEAR_F} (fit)'
mpi = lambda f: float(np.mean([f(q) for q in refs]))                   # mean over the products of a preindustrial value
f3 = lambda x: f'{x:.3f}'
E18 = '18 ESMs, constrained'
xJ, x17 = f'{A_["X_J"]:.0f}', f'{A_["X_1750"]:.1f}'
x5C, x5A = f'{X50_CMIP6:.1f}', f'{X50_AR6:.1f}'
# (step, observed: Omega, product, K, coverage, year; preindustrial: Omega, source, K, coverage, relative to, CO2);
# within the data of Jiang et al. until the last row, which changes to the four products
ROWS_B = [
    ('Findlay et al. (2025), as published', '2.90', J_SSP, KJ, 'Jiang et al.', Y_J,
     '3.57', 'Jiang et al. (2023)', KJ, 'Jiang et al.', 'Jiang et al.', xJ),
    ('ocean area of this study instead of the area of Jiang et al.', f3(J20_COM), J_SSP, KJ, 'common', Y_J,
     f3(PI_J_COM), 'Jiang et al. (2023)', KJ, 'common', 'Jiang et al.', xJ),
    ('observed instead of projected CO$_2$ (SSP2-4.5) for Jiang et al.', f3(J20_O), J_OBS, KJ, 'common', Y_J,
     f3(PI_J_COM), 'Jiang et al. (2023)', KJ, 'common', 'Jiang et al.', xJ),
    ('total boron of this study (Uppström) for both states', f3(J20_OU), J_OBS, KO, 'common', Y_J,
     f3(PI_J_COM * F_J), 'Jiang et al. (2023)', KO, 'common', 'Jiang et al.', xJ),
    ('18 constrained ESMs instead of the 14 ESMs of Jiang et al.', f3(J20_OU), J_OBS, KO, 'common', Y_J,
     f3(YJ * RJ['jiang']), E18, KO, 'common', 'Jiang et al.', xJ),
    (f'CO$_2$ in 1750 of {A_["X_1750"]:.1f} ppm (AR6) instead of {A_["X_J"]:.0f} ppm', f3(J20_OU), J_OBS, KO, 'common', Y_J,
     f3(YJ), E18, KO, 'common', 'Jiang et al.', x17),
    ('the four fCO$_2$-products instead of Jiang et al.: this study', f3(now_fm), 'four fCO$_2$-products', KO, 'common', Y_F,
     f3(mpi(lambda q: PI[q]['com'])), E18, KO, 'common', 'each fCO$_2$-product', x17)]
ROWS_C = [
    ('Findlay et al. (2025), 1850 = 3.44', '2.90', J_SSP, KJ, 'Jiang et al.', Y_J,
     '3.44', 'original boundary', '–', '–', '–', '280'),
    ('1850 state of Jiang et al. instead of 3.44', '2.90', J_SSP, KJ, 'Jiang et al.', Y_J,
     f3(J50_OWN), 'Jiang et al. (2023), 1850', KJ, 'Jiang et al.', 'Jiang et al.', x5C),
    ('ocean area of this study instead of the area of Jiang et al.', f3(J20_COM), J_SSP, KJ, 'common', Y_J,
     f3(J50_COM), 'Jiang et al. (2023), 1850', KJ, 'common', 'Jiang et al.', x5C),
    ('observed instead of projected CO$_2$ (SSP2-4.5) for Jiang et al.', f3(J20_O), J_OBS, KJ, 'common', Y_J,
     f3(J50_COM), 'Jiang et al. (2023), 1850', KJ, 'common', 'Jiang et al.', x5C),
    ('total boron of this study (Uppström) for both states', f3(J20_OU), J_OBS, KO, 'common', Y_J,
     f3(J50_COM * F50), 'Jiang et al. (2023), 1850', KO, 'common', 'Jiang et al.', x5C),
    ('18 constrained ESMs instead of the 14 ESMs of Jiang et al.', f3(J20_OU), J_OBS, KO, 'common', Y_J,
     f3(YJ * RJ['cmip6_1850']), E18, KO, 'common', 'Jiang et al.', x5C),
    (f'CO$_2$ in 1850 of {X50_AR6:.1f} ppm (AR6) instead of {X50_CMIP6:.1f} ppm (CMIP6)', f3(J20_OU), J_OBS, KO, 'common', Y_J,
     f3(YJ * RJ['1850']), E18, KO, 'common', 'Jiang et al.', x5A),
    ('the four fCO$_2$-products instead of Jiang et al.: this study', f3(now_fm), 'four fCO$_2$-products', KO, 'common', Y_F,
     f3(mpi(lambda q: PI[q]['com'] * RCOM[q]['states']['1850']['ratio'])), E18, KO, 'common', 'each fCO$_2$-product', x5A)]
DEC_B, DEC_C = [v for _, v in STEPS[:len(ROWS_B)]], [v for _, v in STEPS50[:len(ROWS_C)]]
assert np.isclose(DEC_B[-1], STEPS[-1][1]) and np.isclose(DEC_C[-1], STEPS50[-1][1]), 'last rows differ from this study'
# the declines of the single-source rows follow from the values in the table (blocks b and c: means of the products)
for r_, d_ in list(zip(ROWS_B[:-1], DEC_B[:-1])) + list(zip(ROWS_C[:-1], DEC_C[:-1])):
    assert abs(dec(float(r_[1]), float(r_[6])) - d_) < 0.05, f'{r_[0]}: table and Extended Data Fig. 5 differ'
cells_ = lambda rows, decs: [[str(k + 1), r_[0]] + list(r_[1:]) + [f'{d_:.1f}'] for k, (r_, d_) in enumerate(zip(rows, decs))]
BLOCKS = [('a  From the Planetary Health Check 2026 to this study, 2024 (bars of Fig. 3c)', A_['cells']),
          (f'b  From Findlay et al. (2025) to this study, {YEAR_F}, preindustrial state in 1750 (Extended Data Fig. 5a)',
           cells_(ROWS_B, DEC_B)),
          (f'c  From Findlay et al. (2025) to this study, {YEAR_F}, preindustrial state in 1850 (Extended Data Fig. 5b)',
           cells_(ROWS_C, DEC_C))]
NOTES_ALL = A_['notes'] + [
    f'Blocks b and c: {YEAR_F} (decade) = the ensemble median of Jiang et al. for their decade {YEAR_F} (2015–2024) under '
    'SSP2-4.5, as used by Findlay et al.; 2.90, 3.57 and 3.44 as published. 3.44 = the value of the original boundary '
    '(Rockström et al., 2009) for representative surface water at 280 µatm, which Findlay et al. give as 1850. The chain '
    'stays within the data of Jiang et al. until the last step. "Observed instead of projected CO$_2$" moves their state '
    f'of 2015–2024 from the CO$_2$ of SSP2-4.5 ({X_SSP20:.1f} ppm) to the observed ({X_OBS20:.1f} ppm, NOAA) with their own '
    'chemistry (their 2005–2014 fields with surface fCO$_2$ raised by the difference, recomputed with mocsy). "Total boron" '
    'recomputes both states with the boron of this study. "18 constrained ESMs" takes the constrained preindustrial state of '
    'their observations through the method of this study (Extended Data Fig. 2), first at their CO$_2$ (272 ppm for 1750, '
    '284.6 ppm, CMIP6, for 1850). The last step changes the observed and the preindustrial state together to the four '
    'fCO$_2$-products. Values of the fCO$_2$-products: means over the four; decline: the mean of their declines.']

with open(rc.figpath('steps_table', ext='csv'), 'w', newline='') as f_:
    wr = csv.writer(f_)
    wr.writerow(['#', 'Step', 'Observed Omega', 'Observed product', 'Observed K1 K2', 'Observed coverage', 'Observed year',
                 'Preindustrial Omega', 'Preindustrial source', 'Preindustrial K1 K2', 'Preindustrial coverage',
                 'Preindustrial relative to', 'Preindustrial CO2 (ppm)', 'Decline (%)'])
    for t_, rows in BLOCKS:
        wr.writerow([t_])
        wr.writerows([[c_.replace('$', '') for c_ in r_] for r_ in rows])
    for n_ in NOTES_ALL:
        wr.writerow([n_.replace('$', '')])

# rendered table (booktabs style: rules, no grid; per block the entries that change from the previous row highlighted)
GROUPS = [('', 2), ('Observed state', 5), ('Preindustrial state', 6), ('', 1)]
HEAD = ['', 'Step', 'Ω$_{arag}$', 'fCO$_2$-product', 'K$_1$, K$_2$', 'coverage', 'year',
        'Ω$_{arag}$', 'source', 'K$_1$, K$_2$', 'coverage', 'relative to', 'CO$_2$ (ppm)', 'Decline (%)']
CW = [0.3, 4.6, 0.75, 2.05, 1.75, 1.0, 1.15, 0.75, 2.35, 1.75, 1.05, 1.45, 1.0, 1.15]      # column widths (inches)
RH, X0 = 0.27, 0.15
WIDTH = X0 * 2 + sum(CW)
n_rows = sum(len(r_) + 1 for _, r_ in BLOCKS)
HEIGHT = 0.35 + 2 * RH + RH * n_rows + 0.1 * (len(BLOCKS) - 1) + 0.3
plt.close('all')
fig = plt.figure(figsize=(WIDTH, HEIGHT))
ax = fig.add_axes([0, 0, 1, 1])
ax.set_xlim(0, WIDTH)
ax.set_ylim(HEIGHT, 0)
ax.axis('off')
xe = X0 + np.concatenate([[0], np.cumsum(CW)])
ALIGN = ['center', 'left'] + ['center'] * 12
xt = lambda j: xe[j] + 0.06 if ALIGN[j] == 'left' else (xe[j] + xe[j + 1]) / 2
y = 0.25
ax.plot([X0, WIDTH - X0], [y, y], color='k', lw=1.2)
j0 = 0
for g_, nspan in GROUPS:                             # group headers with a short rule underneath
    if g_:
        ax.text((xe[j0] + xe[j0 + nspan]) / 2, y + RH / 2, g_, ha='center', va='center', fontsize=10, fontweight='bold')
        ax.plot([xe[j0] + 0.08, xe[j0 + nspan] - 0.08], [y + RH - 0.03] * 2, color='0.3', lw=0.7)
    j0 += nspan
y += RH
for j, h_ in enumerate(HEAD):
    ax.text(xt(j), y + RH / 2, h_, ha=ALIGN[j], va='center', fontsize=9.5, fontweight='bold')
y += RH
ax.plot([X0, WIDTH - X0], [y, y], color='k', lw=0.7)
for b_, (title_, rows) in enumerate(BLOCKS):
    if b_:
        y += 0.1
        ax.plot([X0, WIDTH - X0], [y, y], color='0.3', lw=0.6)
    ax.text(X0 + 0.06, y + RH / 2, title_, ha='left', va='center', fontsize=9.5, fontweight='bold', style='italic')
    y += RH
    for k, row in enumerate(rows):
        last = k == len(rows) - 1
        if last:
            ax.plot([X0, WIDTH - X0], [y, y], color='0.5', lw=0.5)
        for j, c_ in enumerate(row):
            changed = k > 0 and 2 <= j <= 12 and c_ != rows[k - 1][j]
            if changed:
                ax.add_patch(plt.Rectangle((xe[j] + 0.02, y + 0.03), CW[j] - 0.04, RH - 0.06, color='#e3edf3', lw=0))
            ax.text(xt(j), y + RH / 2, c_, ha=ALIGN[j], va='center', fontsize=9,
                    fontweight='bold' if (changed or last or j == 13) else 'normal',
                    color='k' if (changed or j < 2 or j == 13) else '0.4')
        y += RH
ax.plot([X0, WIDTH - X0], [y, y], color='k', lw=1.2)
rc.savefig(fig, 'steps_table')
for t_, rows in BLOCKS:
    print(t_ + ': ' + '; '.join(f'{r_[1]} {r_[-1]} %' for r_ in rows))
print('Notes for the legend of Extended Data Table 3:\n' + '\n'.join(n_.replace('$', '') for n_ in NOTES_ALL))
plt.show()

## Extended Data Fig. 7: 1750 and 2024 states (needs the Fig. 2 cell)

In [ ]:
# needs the Fig. 2 cell to be run first (same kernel): uses its fields and its map projection
# ---------------------------------------------------------------- Extended Data Fig. 7: preindustrial and present state, mean of the products
# runs on the map fields of Fig. 2 (FIG2), in a function so that no other cell can overwrite them
def _fig_s14(pi_m, pisig_m, now_m, land, area, lat, lon_m, Xe, Ye, cols, robinson, outline, OMEGA, OM_CMAP, OM_LIM):
    #   a  constrained 1750 Omega_arag (cell-by-cell constraint), b its 1 sigma, c fitted 2024 Omega_arag,
    #   d 2024 minus the boundary (80 % of 1750): negative where the boundary is crossed
    # every cell that at least one product covers: per cell the mean of the products that cover it, each on its own area
    # with its own cell-by-cell constrained 1750 state and fitted 2024 state, as the best-estimate maps of Fig. 4; without the
    # alkalinity adjustment, as Fig. 4: per product the change of the
    # 1750 state and of the 2024 trend value (rc.alkalinity_change) added. 1 sigma: root mean square of the covering
    # products' sigma (the constraint)
    rc.FIGNAMES.setdefault('states_maps', 'FigS14_states_1750_2024')
    import warnings
    pi_c = pi_m                                                  # common area (Fig. 2), for the comparison printed below
    Dw = rc.load(which='results')                                # each product on its own area
    qs = list(Dw)
    assert all(np.allclose(Dw[q]['lat'], lat) and Dw[q]['pi_full'].shape == land.shape for q in qs), 'own areas on another grid'
    alk = {q: rc.alkalinity_change(q, Dw[q]['lat'], Dw[q]['lon']) for q in qs}
    use = {q: np.isfinite(Dw[q]['pi_full']) & np.isfinite(Dw[q]['now']) & np.asarray(Dw[q]['ok'], bool)
              & np.isfinite(alk[q][0]) & np.isfinite(alk[q][2]) for q in qs}
    stack = lambda k: np.array([np.where(use[q], Dw[q][k], np.nan) for q in qs])
    with warnings.catch_warnings():
        warnings.simplefilter('ignore', RuntimeWarning)              # cells without any product
        pi_m0, now_m0 = np.nanmean(stack('pi_full'), axis=0), np.nanmean(stack('now'), axis=0)    # without
        pi_m = np.nanmean(np.array([np.where(use[q], Dw[q]['pi_full'] + alk[q][0], np.nan) for q in qs]), axis=0)
        pisig_m = np.sqrt(np.nanmean(stack('sig_full') ** 2, axis=0))
        now_m = np.nanmean(np.array([np.where(use[q], Dw[q]['now'] + alk[q][2], np.nan) for q in qs]), axis=0)
    n_prod = np.sum([use[q] for q in qs], axis=0)
    ocean = ~np.asarray(land, bool)
    print(f'cells covered by at least one product: {100 * area[(n_prod > 0) & ocean].sum() / area[ocean].sum():.1f} % of the ocean '
          f'(all four: {100 * area[(n_prod == 4) & ocean].sum() / area[ocean].sum():.1f} %; common area of Fig. 2: '
          f'{100 * area[np.isfinite(pi_c)].sum() / area[ocean].sum():.1f} %)')


    def map_panel(fig_, spec, field, cmap, v0, v1, title, letter, cblab, extend='both', contour=None):
        axm = fig_.add_subplot(spec)
        axm.set_aspect('equal')
        axm.axis('off')
        axm.pcolormesh(Xe, Ye, np.where(cols(land), 1.0, np.nan), cmap=ListedColormap(['0.85']), vmin=0, vmax=1,
                       shading='flat')
        pc = axm.pcolormesh(Xe, Ye, cols(field), cmap=cmap, vmin=v0, vmax=v1, shading='flat')
        if contour is not None:                              # e.g. the zero line (boundary crossed)
            LO2, LA2 = np.meshgrid(lon_m, lat)
            xc, yc_ = robinson(LO2, LA2)
            axm.contour(xc, yc_, cols(field), levels=[contour], colors='k', linewidths=0.7)
        axm.plot(*outline.T, color='k', lw=0.6, zorder=6, clip_on=False)
        axm.margins(0.01)
        axm.set_title(title, fontsize=10.5)
        axm.text(0.0, 1.02, letter, transform=axm.transAxes, fontsize=16, fontweight='bold', va='bottom')
        axm.apply_aspect()
        pos = axm.get_position()
        cax = fig_.add_axes([pos.x0 + 0.15 * pos.width, pos.y0 - 0.04, 0.7 * pos.width, 0.016])
        fig_.colorbar(pc, cax=cax, orientation='horizontal', extend=extend).set_label(cblab, fontsize=9.5)
        return axm


    fig_s = plt.figure(figsize=(12, 8.8))           # room below the lower colour bars for their labels
    gs_s = GridSpec(2, 2, figure=fig_s, hspace=0.45, wspace=0.08, left=0.02, right=0.98, top=0.95, bottom=0.14)
    dist = now_m - 0.8 * pi_m
    map_panel(fig_s, gs_s[0, 0], pi_m, OM_CMAP, *OM_LIM, f'Constrained {OMEGA} in 1750', 'a', OMEGA)
    map_panel(fig_s, gs_s[0, 1], pisig_m, 'viridis', 0, 0.06, f'Uncertainty of {OMEGA} in 1750 (1$\\sigma$)', 'b',
              f'$\\sigma$ of {OMEGA}', extend='max')
    map_panel(fig_s, gs_s[1, 0], now_m, OM_CMAP, *OM_LIM, f'{OMEGA} in 2024 (trend fit)', 'c', OMEGA)
    map_panel(fig_s, gs_s[1, 1], dist, 'RdBu', -0.6, 0.6, f'{OMEGA} in 2024 minus the boundary', 'd',
              f'$\\Delta${OMEGA} (negative: boundary crossed)', contour=0.0)
    v = np.isfinite(pi_m)
    print(f'1750: area mean {np.sum(area[v] * pi_m[v]) / area[v].sum():.3f}, sigma {np.sum(area[v] * pisig_m[v]) / area[v].sum():.3f}; '
          f'2024 (forced): {np.sum(area[v] * now_m[v]) / area[v].sum():.3f}')
    # area beyond the boundary in 2024, forced state (per-cell quadratic fit) in both: here from the mean of the products'
    # fields on the area of this figure (cell-by-cell constraint possible for all four); in Fig. 4b per product on the
    # common area, then the mean (the number of the text)
    here = 100 * np.sum(area[v] * (dist[v] < 0)) / area[v].sum()
    here0 = 100 * np.sum(area[v] * ((now_m0 - 0.8 * pi_m0)[v] < 0)) / area[v].sum()
    print(f'with the alkalinity adjustment: 1750 {np.sum(area[v] * (pi_m - pi_m0)[v]) / area[v].sum():+.4f}, 2024 '
          f'{np.sum(area[v] * (now_m - now_m0)[v]) / area[v].sum():+.4f} (area means of the change); beyond the boundary in '
          f'2024 {here:.1f} %, without the adjustment {here0:.1f} %')
    fig3 = globals()['R']['beyond_2024'] if 'R' in globals() else np.nan
    print(f'area beyond the boundary in 2024 (forced): {here:.1f} % in panel d (mean of the products, this area); '
          f'{fig3:.1f} % in Fig. 3b (per product on the common area, then the mean; the number of the text)')
    rc.savefig(fig_s, 'states_maps')
    plt.show()


_fig_s14(**FIG2)

## Extended Data Fig. 2: the observations of Jiang et al. (2023) through the chain of this study (needs the preparation cell and run_constraint_jiang.py)

In [ ]:
# Notebook cell: Extended Data Fig. 2, the observations of Jiang et al. (2023) through the chain of this study.
# Their observation-based state of the decade 2010 (2005-2014; their T, S, DIC, ALK, Omega_arag recomputed with mocsy and
# the constants of this run) is used as a fifth reference: same 18 ESMs, bias adjustment, piControl drift, 1750 CO2
# (AR6, 278.3 ppm) and emergent constraint (run_constraint_jiang.py). Common area of the paper (restricted to the
# coverage of Jiang et al. where needed), decade 2005-2014 (the only one of Jiang et al.):
#   a  global decline from 1750 to 2005-2014 (%): Jiang et al. as published (their 1750 and 2010 fields), the same with
#      the total boron of this study (Uppstroem 1974), Jiang et al.'s observations with the constrained 1750 state of this
#      study, and the four products of this study (± 1 sigma of the constraint, the term the four products and Jiang
#      et al. share; Extended Data Table 2 has the full budget); mean of the products: that sigma and their spread in quadrature
#   b  constrained 1750 Omega_arag from Jiang et al.'s observations (cell-by-cell constraint) minus their 1750 field in
#      the chemistry of this study (their fCO2 + TA, Uppstroem boron): the difference from the definition of 1750 alone
#      (CO2 of 1750, ESMs, constraint)
# Needs the preparation cell (FIG4: Jiang et al.'s fields and the boron conversion), data/constraint_results.pkl,
# data/constraint_results_jiang.pkl (run_constraint_jiang.py), data/obs/omega_arag_Oyr_Jiang2023-2010.nc and, if it
# exists, product_uncertainty/cuc_mapping_Jiang.npz (local mapping uncertainty of the observed CUC; else none).
import oa_config                      # the area weights of the run (OA_WEIGHTS)
def _fig_s_jiang():
    import os
    import pickle
    import numpy as np
    import netCDF4
    import matplotlib.pyplot as plt
    from matplotlib.gridspec import GridSpec, GridSpecFromSubplotSpec
    from matplotlib.colors import ListedColormap
    import regional_common as rc
    if 'FIG4' not in globals():
        raise NameError('FIG4 not defined: run the preparation cell first (same kernel)')
    F4 = globals()['FIG4']
    rc.FIGNAMES.setdefault('jiang_chain', 'FigS15_jiang_chain')
    ROOT, LAB = rc.ROOT, rc.LAB
    W10 = (2005, 2014)
    dec = lambda now, pi: 100 * (1 - now / pi)
    with open(f'{ROOT}/constraint_results.pkl', 'rb') as f:
        RCOM = pickle.load(f)['results_common']
    with open(f'{ROOT}/constraint_results_jiang.pkl', 'rb') as f:
        JK = pickle.load(f)
    refs = [q for q in ('OceanSODA', 'UExP', 'JMA', 'CMEMS') if q in RCOM]

    # ---------------------------------------------------------------- a: global values (common area)
    rows = []                                      # (label, decline, sigma, colour, marker, filled)
    pj, j10 = F4['PI_J_COM'], F4['J10_COM']
    rows.append(('Jiang et al. (2023), published', dec(j10, pj), np.nan, '0.45', 's', False))
    rows.append(('Jiang et al. (2023), Uppström boron', dec(j10 * F4['F_J10'], pj * F4['F_J']), np.nan, '0.45', 's', True))
    rj = JK['common']['Jiang']
    yc, sc = rj['constrained'][LAB][:2]
    rows.append(('Jiang et al. observations,\n1750 of this study', dec(rj['obs10'], yc), 100 * rj['obs10'] / yc ** 2 * sc,
                 'k', 's', True))
    dq, sq, prod_rows = [], [], []
    for q in refs:
        r = RCOM[q]
        yc, sc = r['constrained'][LAB][:2]
        dq.append(dec(r['obs10'], yc))
        sq.append(100 * r['obs10'] / yc ** 2 * sc)
        prod_rows.append((rc.FULL[q], dq[-1], sq[-1], rc.COLORS[q], 'o', True))
    # mean of the products: the constraint term is shared (not reduced by averaging), the spread of the products added
    # in quadrature, as for the mean in Extended Data Table 1
    # directly below Jiang et al. (the comparison of the panel), then the four products
    rows.append(('mean of the four fCO$_2$-products', np.mean(dq), float(np.hypot(np.sqrt(np.mean(np.square(sq))), np.std(dq, ddof=1))),
                 'C3', 'o', True))
    rows += prod_rows
    for lab_, v, s, *_ in rows:
        print(f'{lab_.replace(chr(10), " "):45s} decline 1750 to {W10[0]}-{W10[1]}: {v:5.2f} %' + (f' ± {s:.2f}' if np.isfinite(s) else ''))
    print(f'common area with Jiang et al.: {JK["coverage_test"]["common_fraction_of_own"]:.0f} % of their own area; '
          f'decline on their own area {JK["coverage_test"]["own"]:.2f} %, on the common area {JK["coverage_test"]["common"]:.2f} %')

    # ---------------------------------------------------------------- b, c: fields on the grid of the products (lon -180..180)
    D = rc.load(refs=refs)                          # products: cell-by-cell constrained 1750 (pi_full), annual fields
    lat, lon = D[refs[0]]['lat'], D[refs[0]]['lon']
    land = D[refs[0]]['land']
    nn = lambda src, dst, per=False: (np.abs(((dst[:, None] - src[None, :]) + 180) % 360 - 180).argmin(1) if per
                                      else np.abs(dst[:, None] - src[None, :]).argmin(1))
    regrid = lambda fld, la, lo: np.asarray(fld)[np.ix_(nn(np.asarray(la, 'f8'), lat), nn(np.asarray(lo, 'f8') % 360, lon % 360, True))]
    # products: decline from 1750 to 2005-2014 per cell, mean of the four
    d10 = []
    for q in refs:
        d = D[q]
        y_ = np.asarray(d['year'])
        with np.errstate(invalid='ignore'):
            import warnings
            with warnings.catch_warnings():
                warnings.simplefilter('ignore', RuntimeWarning)          # land: no values
                o10 = np.nanmean(np.asarray(d['om'])[(y_ >= W10[0]) & (y_ <= W10[1])], axis=0)
        d10.append(dec(o10, d['pi_full']))
    d10_m = np.mean(d10, axis=0)                     # nan where any product has none (common area)
    # Jiang et al. through the chain: cell-by-cell constraint on their own grid (as rc.load)
    r = JK['own']['Jiang']
    Fj = r['fields']
    Y, DT, CU = (Fj[k].astype('f8') for k in ('y1750', 'dT', 'CUC'))
    n = Y.shape[0]
    dobs = np.array([Fj['dsst_obs'][k] for k in Fj['dsst_obs']], 'f8')
    dTo, dTs = dobs.mean(0), dobs.std(0, ddof=1)
    CUo = Fj['CUC_obs'].astype('f8')
    p_ = f'{ROOT}/product_uncertainty/cuc_mapping_Jiang.npz'
    CUs_f = rc.cuc_sigma('Jiang', 'results')[1] if os.path.exists(p_) else np.zeros(CUo.shape)
    print('observed CUC of Jiang et al.: ' + ('local mapping uncertainty from ' + os.path.basename(p_) if os.path.exists(p_)
                                             else 'no mapping uncertainty (cuc_mapping_Jiang.npz missing)'))
    with netCDF4.Dataset(f'{ROOT}/obs/omega_arag_Oyr_Jiang2023-2010.nc') as f:
        jo = np.ma.filled(f['omega_arag'][:].astype('f8'), np.nan)[0]
    mask = np.asarray(r['mask']).astype(bool)
    ok = (mask & np.all(np.isfinite(Y), 0) & np.all(np.isfinite(DT), 0) & np.all(np.isfinite(CU), 0) & np.isfinite(dTo)
          & np.isfinite(CUo) & np.isfinite(jo))
    idx = np.where(ok.ravel())[0]
    sel = lambda a: a.reshape(n, -1)[:, idx] if a.ndim == 3 else a.ravel()[idx]
    yj, sj = rc.local_constraint(sel(Y), sel(DT), sel(CU), sel(dTo), sel(dTs), sel(CUo), ok, CUs=sel(CUs_f))
    pij = np.full(mask.size, np.nan)
    pij[idx] = yj
    pij = pij.reshape(mask.shape)
    pij_p = regrid(pij, r['lat'], r['lon'])          # onto the grid of the products
    jo_p = regrid(jo, r['lat'], r['lon'])
    dj_p = dec(jo_p, pij_p)
    # their 1750 field in the chemistry of this study (Uppstroem boron), else as published
    j1750, j1750_lab = F4['_j1750']['ours'], 'their 1750 in the chemistry of this study'   # recomputed, constants of this run
    j1750_p = regrid(j1750, F4['_jlat'], F4['_jlon'])
    v = np.isfinite(d10_m)                            # common area of the four products
    diff_b = np.where(v, pij_p - j1750_p, np.nan)
    area = oa_config.area_weights(lat, lon)
    am = lambda fld: float(np.nansum(np.where(np.isfinite(fld), fld, 0) * area) / area[np.isfinite(fld)].sum())
    print(f'b: 1750 of this study from Jiang et al. minus {j1750_lab}: area mean {am(diff_b):+.3f}, '
          f'5-95 % {np.nanpercentile(diff_b, 5):+.3f} to {np.nanpercentile(diff_b, 95):+.3f}')
    print(f'area-mean declines to {W10[0]}-{W10[1]} on the common area (cell-by-cell constraint): Jiang et al. observations '
          f'{am(np.where(v, dj_p, np.nan)):.2f} %, mean of the products {am(d10_m):.2f} %')

    # ---------------------------------------------------------------- figure: a left, b map right
    fig = plt.figure(figsize=(13, 4.6))
    gs = GridSpec(2, 2, figure=fig, width_ratios=[1.0, 1.25], height_ratios=[1.0, 0.06], hspace=0.12, wspace=0.12,
                  left=0.17, right=0.99, top=0.9, bottom=0.12)
    ax = fig.add_subplot(gs[:, 0])
    yy = np.arange(len(rows))[::-1]
    for (lab_, v_, s_, c_, m_, fill_), y_ in zip(rows, yy):
        ax.errorbar(v_, y_, xerr=s_ if np.isfinite(s_) else None, fmt=m_, ms=7, color=c_, mfc=c_ if fill_ else 'white',
                    mec=c_, mew=1.3, capsize=3, lw=1.2, zorder=3)
    ax.axhspan(yy[-1] - 0.5, yy[-5] + 0.5, color='0.94', lw=0, zorder=0)        # the products of this study
    ax.axvline(np.mean(dq), color='C3', lw=0.8, ls=(0, (4, 3)), zorder=1)
    ax.set_yticks(yy)
    ax.set_yticklabels([r_[0] for r_ in rows], fontsize=8.5)
    ax.set_ylim(yy[-1] - 0.7, yy[0] + 0.7)
    ax.set_xlabel(f'Decline of surface $\\Omega_{{\\mathrm{{arag}}}}$ from 1750 to {W10[0]}–{W10[1]} (%)', fontsize=10)
    ax.spines[['top', 'right']].set_visible(False)
    ax.text(-0.5, 1.02, 'a', transform=ax.transAxes, fontsize=16, fontweight='bold', va='bottom')
    lim_b = float(np.nanpercentile(np.abs(diff_b), 98))
    panels = ((diff_b, lim_b, '1750 $\\Omega_{\\mathrm{arag}}$: this study (Jiang et al. observations)\nminus Jiang et al. (2023)',
               'b', '$\\Delta\\Omega_{\\mathrm{arag}}$'),)
    for k, (fld, lim, ttl, let, cbl) in enumerate(panels):
        axm = fig.add_subplot(gs[0, 1])
        axm.set_xticklabels([])
        axm.set_yticklabels([])
        rc.map_mesh(axm, lon, lat, np.where(land, 1.0, np.nan), cmap=ListedColormap(['0.85']), vmin=0, vmax=1)
        pc = rc.map_mesh(axm, lon, lat, fld, cmap='RdBu_r', vmin=-lim, vmax=lim)
        axm.set_title(ttl, fontsize=9.5)
        axm.text(0.0, 1.0, let, transform=axm.transAxes, fontsize=16, fontweight='bold', va='bottom')
        cax = fig.add_subplot(GridSpecFromSubplotSpec(1, 3, subplot_spec=gs[1, 1], width_ratios=[0.2, 1, 0.2])[1])
        fig.colorbar(pc, cax=cax, orientation='horizontal', extend='both').set_label(cbl, fontsize=9.5)
    rc.savefig(fig, 'jiang_chain')
    plt.show()
    return dict(rows=rows, diff_b=diff_b)


FIGS_JIANG = _fig_s_jiang()

## Supplementary Figs. 10 to 21: every indicator of the data product

In [ ]:
# Notebook cell: Supplementary Figs. 10 to 21, every indicator of the data product (one figure per indicator), from 1750 to 2100.
#   a  1750 (the constrained state at the AR6 CO2 of 278.3 ppm), b  2024 (the trend value, mean of the fCO2-products),
#   c  2024 minus 1750;  d-g  2100 under SSP1-2.6, SSP2-4.5, SSP3-7.0 and SSP5-8.5 (constrained ESMs of each scenario)
#   h  global mean on the common area: the historical series 1750-2014 (global constraint) with 1 sigma, the mean of the
#      fCO2-products (years with all four), and the four scenarios 1750-2100 (each with its own ESMs and its own 1750)
#      with 1 sigma (the references' sigma and their spread)
# OA-Track (make_oatrack.py) with the alkalinity adjustment: maps OA-Track_all.nc (mean of the fCO2-products covering a
# cell), series OA-Track_common.nc.
# Wrapped in a function so that its names do not overwrite those of other cells.
def _indicator_figures():
    import os
    import numpy as np
    import netCDF4
    import matplotlib.pyplot as plt
    from matplotlib.gridspec import GridSpec
    import regional_common as rc

    PROD = f'{rc.ROOT}/oatrack'
    REFS = ('OceanSODA', 'UExP', 'JMA', 'CMEMS')
    SSPS = ('ssp126', 'ssp245', 'ssp370', 'ssp585')
    SSP_LAB = {'ssp126': 'SSP1-2.6', 'ssp245': 'SSP2-4.5', 'ssp370': 'SSP3-7.0', 'ssp585': 'SSP5-8.5'}
    SSP_COL = {'ssp126': '#1d3354', 'ssp245': '#e9dc3d', 'ssp370': '#f11111', 'ssp585': '#830b22'}   # IPCC AR6 scenario colours
    IND = {  # name: (label, colormap of the level)
        'omega_arag': (r'$\Omega_{\mathrm{arag}}$', 'viridis'),
        'omega_calc': (r'$\Omega_{\mathrm{calc}}$', 'viridis'),
        'ph': (r'pH$_\mathrm{T}$', 'viridis'),
        'hplus': (r'[H$^+$]$_\mathrm{T}$', 'magma_r'),
        'hplus_free': (r'[H$^+$]$_\mathrm{F}$', 'magma_r'),
        'pco2': (r'$p$CO$_2$', 'magma_r'),
        'fco2': (r'$f$CO$_2$', 'magma_r'),
        'co2': (r'CO$_2^*$', 'magma_r'),
        'hco3': (r'HCO$_3^-$', 'magma_r'),
        'co3': (r'CO$_3^{2-}$', 'viridis'),
        'revelle': ('Revelle factor', 'magma_r'),
        'dic': ('DIC', 'magma_r'),
    }
    UNIT_LAB = {'1': '', 'total scale': '', 'uatm': r' ($\mu$atm)', 'umol kg-1': r' ($\mu$mol kg$^{-1}$)',
                'nmol kg-1': r' (nmol kg$^{-1}$)'}
    first = 16
    for k, q in enumerate(IND):
        rc.FIGNAMES.setdefault(f'ind_{q}', f'FigS{first + k}_indicator_{q}')

    pa, pc = f'{PROD}/OA-Track_all.nc', f'{PROD}/OA-Track_common.nc'
    if not (os.path.exists(pa) and os.path.exists(pc)):
        print(f'no combined OA-Track files in {PROD} (make_oatrack.py): nothing drawn')
        return
    A, C = netCDF4.Dataset(pa), netCDF4.Dataset(pc)
    rd = lambda f, n, i=None: np.ma.filled((f[n][:] if i is None else f[n][i]).astype('f8'), np.nan)
    lat, lon = rd(A, 'lat'), rd(A, 'lon')
    years = np.asarray(C['year'][:]).astype(int)
    ys = np.asarray(A['year_ssp'][:]).astype(int) if 'year_ssp' in A.dimensions else None
    yo = np.asarray(C['year_obs'][:]).astype(int)
    all4 = np.asarray(C['n_products_gm'][:]).astype(int) == 4          # the years with all four fCO2-products
    ssps = [s for s in SSPS if ys is not None and f'omega_arag_{s}' in A.variables]
    print(f'indicator figures: maps {pa}, series {pc}, scenarios {ssps}')

    for q, (lab, cmap) in IND.items():
        if f'{q}_state' not in A.variables:
            print(f'  {q}: not in the OA-Track files, skipped')
            continue
        unit = UNIT_LAB.get(A[f'{q}_state'].units, f' ({A[f"{q}_state"].units})')
        m1750, m2024 = rd(A, f'{q}_state', 0), rd(A, f'{q}_{rc.EVAL_YEAR}_trend')
        m2100 = {s: rd(A, f'{q}_{s}', int(np.where(ys == 2100)[0][0])) for s in ssps}
        allv = np.concatenate([a[np.isfinite(a)] for a in [m1750, m2024] + list(m2100.values())])
        vmin, vmax = np.percentile(allv, [2, 98])
        d = m2024 - m1750
        dmax = np.nanpercentile(np.abs(d), 98)

        # layout: bold letters as in the other figures; every colour bar directly under the maps it
        # belongs to (a, b: the level; c: the change; d-g: the level on the same scale as a, b)
        fig = plt.figure(figsize=(12, 11.4))
        gs = GridSpec(7, 24, figure=fig, height_ratios=[1.0, 0.045, 0.24, 0.72, 0.045, 0.34, 1.0], hspace=0.16,
                      wspace=0.2, left=0.06, right=0.98, top=0.93, bottom=0.05)

        def mpanel(spec, field, title, letter, cm, lo, hi):
            ax_ = fig.add_subplot(spec)
            im_ = None
            if field is None:
                rc.map_frame(ax_)
                ax_.text(0, 0, 'not available', ha='center', va='center', fontsize=9, color='0.4')
            else:
                im_ = rc.map_mesh(ax_, lon, lat, field, cmap=cm, vmin=lo, vmax=hi, rasterized=True)
            ax_.set_title(title, fontsize=10.5)
            ax_.text(0.0, 1.02, letter, transform=ax_.transAxes, fontsize=16, fontweight='bold', va='bottom')
            return im_

        def cbar(im_, spec, label_):
            if im_ is not None:
                cb_ = fig.colorbar(im_, cax=fig.add_subplot(spec), orientation='horizontal', extend='both')
                cb_.set_label(label_, fontsize=9.5)
                cb_.ax.tick_params(labelsize=8.5)

        im_a = mpanel(gs[0, 0:8], m1750, '1750', 'a', cmap, vmin, vmax)
        im_b = mpanel(gs[0, 8:16], m2024, f'{rc.EVAL_YEAR}', 'b', cmap, vmin, vmax)
        im_c = mpanel(gs[0, 16:24], d, f'{rc.EVAL_YEAR} minus 1750', 'c', 'RdBu_r', -dmax, dmax)
        im_2 = [mpanel(gs[3, 6 * j:6 * j + 6], m2100.get(s), f'2100, {SSP_LAB[s]}', 'defg'[j], cmap, vmin, vmax)
                for j, s in enumerate(SSPS)]
        cbar(im_a if im_a is not None else im_b, gs[1, 3:13], lab + unit)
        cbar(im_c, gs[1, 18:22], r'$\Delta$' + lab + unit)
        cbar(next((i_ for i_ in im_2 if i_ is not None), None), gs[4, 7:17], lab + unit + ' (same scale as a, b)')

        ax = fig.add_subplot(gs[6, 1:23])
        gm_, sg_ = rd(C, f'{q}_gm_global'), rd(C, f'{q}_gm_global_sigma')
        for s in ssps:
            v, sv = rd(C, f'{q}_gm_{s}_global'), rd(C, f'{q}_gm_{s}_global_sigma')
            ax.fill_between(ys, v - sv, v + sv, color=SSP_COL[s], alpha=0.15, lw=0)
            ax.plot(ys, v, color=SSP_COL[s], lw=1.3, label=SSP_LAB[s])
        ax.fill_between(years, gm_ - sg_, gm_ + sg_, color='k', alpha=0.15, lw=0)
        ax.plot(years, gm_, color='k', lw=1.8, label='historical series (constrained ESMs)')
        go = rd(C, f'{q}_gm_obs')
        ax.plot(yo[all4], go[all4], color='r', lw=1.6, ls=(0, (4, 2)), zorder=5,     # dashed: distinct from SSP3-7.0
                label='mean of the fCO$_2$-products')                                   # also without colour
        ax.set_xlim(1750, 2100)
        ax.set_ylabel(lab + unit)
        ax.set_title('Global mean on the common area (bands: 1$\\sigma$)', fontsize=10.5)
        ax.text(-0.06, 1.02, 'h', transform=ax.transAxes, fontsize=16, fontweight='bold', va='bottom')
        ax.legend(frameon=False, fontsize=8.5, ncol=2, loc='best')
        ax.grid(alpha=0.3)
        ax.spines[['top', 'right']].set_visible(False)
        fig.suptitle(f'{lab}: 1750, 2024 and 2100 in OA-Track (mean of the fCO$_2$-products and constrained ESMs)',
                     fontsize=12)
        rc.savefig(fig, f'ind_{q}')
        plt.show()
        plt.close(fig)
        print(f'  {q}: 1750 (series) {gm_[0]:.4g}, {rc.EVAL_YEAR} trend {float(C[f"{q}_gm_{rc.EVAL_YEAR}_trend"][...]):.4g}'
              + ''.join(f', 2100 {SSP_LAB[s]} {rd(C, f"{q}_gm_{s}_global")[-1]:.4g}' for s in ssps))
    for f in (A, C):
        f.close()


_indicator_figures()

## Supplementary Table 1: the carbonate system of the common area in 1750 and 2024 (data product)

In [ ]:
# Notebook cell: Supplementary Table 1, the carbonate system of the common area in 1750 and
# 2024 from the data product: every indicator, mean of the four products with the range across them (OceanSODA-ETHZ,
# UExP-FNN-U, JMA-MLR, CMEMS-LSCE-FFNN), 1750 at the AR6 CO2 of 278.3 ppm (<q>_gm_common_global_state, state 1750),
# 2024 from the quadratic trend fit of the product years (<q>_gm_common_2024_trend), change in percent (pH,
# temperature and salinity: the difference, as in OA-Track)
# (<q>_gm_common_global_decline_from_AR6_1750). Reads OA-Track_<ref>.nc (make_oatrack.py, with the alkalinity
# adjustment); writes the table as png, pdf and csv.
# Wrapped in a function so that its names do not overwrite those of other cells.
def _product_table():
    import csv
    import os
    import numpy as np
    import netCDF4
    import matplotlib.pyplot as plt
    import regional_common as rc
    import oa_config

    rc.FIGNAMES.setdefault('product_table', 'TableS4_product_indicators')
    PROD = f'{rc.ROOT}/oatrack'
    REFS = ('OceanSODA', 'UExP', 'JMA', 'CMEMS')
    #        name           label                              unit (text)        unit (csv)   decimals
    ROWS = [('omega_arag',  r'$\Omega_{\mathrm{arag}}$',       '',                '1',         3),
            ('omega_calc',  r'$\Omega_{\mathrm{calc}}$',       '',                '1',         3),
            ('ph',          r'pH$_\mathrm{T}$',                '',                'total scale', 3),
            ('hplus',       r'[H$^+$]$_\mathrm{T}$',           r'nmol kg$^{-1}$', 'nmol kg-1', 2),
            ('hplus_free',  r'[H$^+$]$_\mathrm{F}$',           r'nmol kg$^{-1}$', 'nmol kg-1', 2),
            ('pco2',        r'$p$CO$_2$',                      r'$\mu$atm',       'uatm',      1),
            ('fco2',        r'$f$CO$_2$',                      r'$\mu$atm',       'uatm',      1),
            ('co2',         r'CO$_2^*$',                       r'$\mu$mol kg$^{-1}$', 'umol kg-1', 2),
            ('hco3',        r'HCO$_3^-$',                      r'$\mu$mol kg$^{-1}$', 'umol kg-1', 0),
            ('co3',         r'CO$_3^{2-}$',                    r'$\mu$mol kg$^{-1}$', 'umol kg-1', 1),
            ('revelle',     'Revelle factor',                  '',                '1',         2),
            ('dic',         'DIC',                             r'$\mu$mol kg$^{-1}$', 'umol kg-1', 0),
            ('talk',        'Alkalinity',                      r'$\mu$mol kg$^{-1}$', 'umol kg-1', 0),
            ('thetao',      'Temperature',                     '°C',              'degC',      2),
            ('so',          'Salinity',                        '',                'psu',       3)]
    files = {r: f'{PROD}/OA-Track_{r}.nc' for r in REFS}
    missing = [r for r, p in files.items() if not os.path.exists(p)]
    if missing:
        print(f'OA-Track files missing in {PROD}: {missing} (make_oatrack.py): no table')
        return
    rd = lambda f, n: np.ma.filled(f[n][:].astype('f8'), np.nan)
    V = {}                                                       # name -> (4, 3): 1750, 2024, change per product
    for r in REFS:
        with netCDF4.Dataset(files[r]) as f:
            states = [str(s) for s in f['state'][:]]
            k = states.index('1750')
            for q, *_ in ROWS:
                a = V.setdefault(q, np.full((len(REFS), 3), np.nan))
                i = REFS.index(r)
                a[i, 0] = rd(f, f'{q}_gm_common_global_state')[k]
                a[i, 1] = float(f[f'{q}_gm_common_2024_trend'][...])
                a[i, 2] = rd(f, f'{q}_gm_common_global_decline_from_AR6_1750')[0] if f[f'{q}_gm_common_global_decline_from_AR6_1750'].ndim else float(f[f'{q}_gm_common_global_decline_from_AR6_1750'][...])

    def cell(vals, d):
        """mean (min to max) over the products"""
        m, lo, hi = np.nanmean(vals), np.nanmin(vals), np.nanmax(vals)
        return f'{m:.{d}f} ({lo:.{d}f} to {hi:.{d}f})'

    ABS_CHANGE = ('ph', 'thetao', 'so')          # change in the units of the quantity (as OA-Track); all others in %
    T = []
    for q, lab, unit, unit_csv, d in ROWS:
        a = V[q]
        if q in ABS_CHANGE:
            a[:, 2] = a[:, 1] - a[:, 0]
        T.append((lab, unit, cell(a[:, 0], d), cell(a[:, 1], d), cell(a[:, 2], d if q in ABS_CHANGE else 1), unit_csv, a))
    # csv: per product and mean
    with open(rc.figpath('product_table', ext='csv'), 'w', newline='') as f_:
        wr = csv.writer(f_)
        wr.writerow(['indicator', 'unit'] + [f'{r} 1750' for r in REFS] + ['mean 1750'] + [f'{r} 2024' for r in REFS] + ['mean 2024']
                    + [f'{r} change (% or unit)' for r in REFS] + ['mean change (% or unit)'])
        for (q, lab, unit, unit_csv, d), (_, _, _, _, _, _, a) in zip(ROWS, T):
            wr.writerow([q, unit_csv] + [f'{v:.6g}' for v in a[:, 0]] + [f'{np.nanmean(a[:, 0]):.6g}']
                        + [f'{v:.6g}' for v in a[:, 1]] + [f'{np.nanmean(a[:, 1]):.6g}']
                        + [f'{v:.4f}' for v in a[:, 2]] + [f'{np.nanmean(a[:, 2]):.4f}'])
    # the common area in % of the ocean, from the common mask of the constraint (as Extended Data Table 1)
    import pickle
    with open(f'{rc.ROOT}/constraint_results.pkl', 'rb') as f_:
        RC_ = pickle.load(f_)['results_common']
    pct_ = [rc.pct_ocean(np.asarray(RC_[r]['mask']).astype(bool), np.asarray(RC_[r]['lat'], 'f8'), np.asarray(RC_[r]['lon'], 'f8'))
            for r in REFS]
    assert max(pct_) - min(pct_) < 0.05, f'common masks differ between the products: {pct_}'
    COMMON_PCT = float(np.mean(pct_))
    # rendered table
    HEAD = ['Indicator', 'Unit', '1750 (278.3 ppm)', '2024', 'Change 1750 to 2024']
    CW = [1.35, 0.95, 1.75, 1.75, 2.1]
    X0, RH = 0.15, 0.26
    W = sum(CW) + 2 * X0
    NOTE = [f'Mean of the four fCO$_2$-products on the common area ({COMMON_PCT:.1f}% of the ocean), range across the fCO$_2$-products in parentheses;',
            '1750: the constrained preindustrial state at the AR6 CO2 of 278.3 ppm; 2024: the quadratic trend fit of the fCO$_2$-product',
            'years evaluated in 2024; change relative to 1750 in %, for pH, temperature, and salinity in their units. From OA-Track (OA-Track_<product>.nc) with the alkalinity',
            'adjustment, on the common area and thus without the extension to the whole ocean.']
    print('Notes for the legend of Supplementary Table 1: ' + ' '.join(NOTE))
    H = 0.2 + RH * (len(T) + 1) + 0.35
    fig = plt.figure(figsize=(W, H))
    ax = fig.add_axes([0, 0, 1, 1])
    ax.set_xlim(0, W)
    ax.set_ylim(H, 0)
    ax.axis('off')
    xe = X0 + np.concatenate([[0], np.cumsum(CW)])
    y = 0.2
    ax.plot([X0, W - X0], [y, y], color='k', lw=1.2)
    for j, h_ in enumerate(HEAD):
        ax.text(xe[j] + 0.06, y + RH / 2, h_, ha='left', va='center', fontsize=9.5, fontweight='bold')
    y += RH
    ax.plot([X0, W - X0], [y, y], color='k', lw=0.7)
    for k, row in enumerate(T):
        if k in (3, 5, 8, 11, 13):                                # groups: saturation and pH, [H+], CO2 species, DIC/ALK, physics
            ax.plot([X0, W - X0], [y, y], color='0.5', lw=0.5)
        for j, c_ in enumerate(row[:5]):
            ax.text(xe[j] + 0.06, y + RH / 2, c_, ha='left', va='center', fontsize=9)
        y += RH
    ax.plot([X0, W - X0], [y, y], color='k', lw=1.2)
    rc.savefig(fig, 'product_table')
    plt.show()
    print(f'common area {COMMON_PCT:.1f} % of the ocean ({oa_config.WEIGHTS_TEXT})')
    print('Supplementary Table 1: ' + '; '.join(f'{q} {np.nanmean(V[q][:, 0]):.4g} -> {np.nanmean(V[q][:, 1]):.4g} ({np.nanmean(V[q][:, 2]):+.4g}{"" if q in ABS_CHANGE else " %"})' for q, *_ in ROWS))

_product_table()


### Numbers for the text: CO2 at a 20 % decline with and without further warming (needs the preparation cell)

In [ ]:
# Notebook cell (diagnostic, after the preparation cell): CO2 at which the decline reaches 20 % with and without the
# warming that accompanies the CO2 rise (projection 'scaled': T rising with CO2 as observed 1990-2024; 'fixed': T
# constant), each anchored to the fitted 2024 state of the product (as decline_crossing). For the
# Methods ('Projection and crossing').
import numpy as np
import projection
C = dict(FIG4)
rc, refs, RCOM, LAB, XCO2 = C['rc'], C['refs'], C['RCOM'], C['LAB'], C['XCO2']
prj = projection.global_projection()
out = {}
for q in refs:
    r = RCOM[q]
    pi_ = r['constrained'][LAB][0]
    pa = rc.anchor_projection(prj[q], r, XCO2[rc.EVAL_YEAR])
    x = np.asarray(pa['x'])
    res = {}
    for k in ('scaled', 'fixed'):
        d = 100 * (1 - np.asarray(pa[k]) / pi_)
        res[k] = float(np.interp(20.0, d, x)) if d[-1] >= 20 else np.nan
    out[q] = res
    print(f'{q:10s} k_T {100 * prj[q]["k_T"]:.2f} degC per 100 ppm: 20 % at {res["scaled"]:.1f} ppm with warming, '
          f'{res["fixed"]:.1f} ppm without ({res["fixed"] - res["scaled"]:+.1f} ppm)')
dm = np.mean([v['fixed'] - v['scaled'] for v in out.values()])
print(f'mean: without warming {dm:+.1f} ppm ({dm / projection.co2_growth(verbose=False)[0]:+.1f} years at 2.6 ppm/yr)')

## Supplementary Fig. 22: leave-one-out test of the emergent constraint

In [ ]:
# Notebook cell: Supplementary Fig. 22, the leave-one-out test of the emergent
# constraint with the real models (functions of test_constraint_loo.py): each ESM in turn is the truth, the other 17
# give the constrained prediction at its own dSST and CUC and the unconstrained prediction (their mean). Global means on
# the area of each product (constraint_results.pkl 'results', as the constraint of the paper and the numbers of the
# Methods and the Supplementary Methods), and per cell with the local predictors (cell-by-cell constraint). The printout
# gives the numbers of the text ('in the units of the indicator' block of test_constraint_loo.py).
# Figure: only the global predictions, one panel per product, constrained only; in each panel the RMS error
# with the constraint and as the mean of the other 17 models and how many models lie within 1 sigma. The cell-by-cell
# numbers (Supplementary Methods) are computed and printed.
import pickle
import importlib
import numpy as np
import matplotlib.pyplot as plt
import oa_config
import regional_common as rc
import test_constraint_loo as tl
importlib.reload(tl)

rc.FIGNAMES.setdefault('loo', 'FigS30_leave_one_out')
with open(f'{rc.ROOT}/constraint_results.pkl', 'rb') as f:
    PK_loo = pickle.load(f)
REFS_loo = [q for q in ('OceanSODA', 'UExP', 'JMA', 'CMEMS') if q in PK_loo['results']]
fig = plt.figure(figsize=(11.5, 3.4))
gs = fig.add_gridspec(1, len(REFS_loo), wspace=0.38, left=0.07, right=0.99, top=0.86, bottom=0.17)
ST_loo = {}
for k, q in enumerate(REFS_loo):
    R = PK_loo['results'][q]
    y = np.asarray(R['y1750'], 'f8')
    dT, CUC = np.asarray(R['dT'], 'f8'), np.asarray(R['CUC'], 'f8')
    n = y.size
    ec, sc, eu, su = tl.loo_global(y, [dT, CUC])
    ax = fig.add_subplot(gs[0, k])
    lo, hi = min(y.min(), (y + ec - sc).min()) - 0.01, max(y.max(), (y + ec + sc).max()) + 0.01
    ax.plot([lo, hi], [lo, hi], 'k--', lw=0.7, zorder=1)
    ax.errorbar(y, y + ec, yerr=sc, fmt='o', color=rc.COLORS[q], ms=4, capsize=2, lw=0.8, zorder=3,
                label='constrained prediction (±1σ)')
    ax.text(0.97, 0.04, f'RMS error {np.sqrt(np.mean(ec ** 2)):.3f}\n(mean of the other 17: {np.sqrt(np.mean(eu ** 2)):.3f})\n'
            f'{int(np.sum(np.abs(ec) < sc))} of {n} within 1σ', transform=ax.transAxes, ha='right', va='bottom', fontsize=7)
    ax.set_xlim(lo, hi)
    ax.set_ylim(lo, hi)
    ax.set_aspect('equal')
    ax.set_title(rc.FULL[q], fontsize=9.5)
    ax.text(-0.3 if k == 0 else -0.18, 1.04, 'abcd'[k], transform=ax.transAxes, fontsize=16, fontweight='bold', va='bottom')
    ax.set_xlabel('true 1750 $\\Omega_{\\rm arag}$', fontsize=8)
    if k == 0:
        ax.set_ylabel('predicted 1750 $\\Omega_{\\rm arag}$', fontsize=8)
        ax.legend(fontsize=7, loc='upper left', frameon=False, handletextpad=0.3)
    ax.tick_params(labelsize=7)
    # cell by cell, local dSST and CUC (as the cell-by-cell constraint)
    Fd = R['fields']
    DTf, CUf, Yf = (np.asarray(Fd[v], 'f8') for v in ('dT', 'CUC', 'y1750'))
    ok = (np.asarray(R['mask']).astype(bool) & np.all(np.isfinite(DTf), 0) & np.all(np.isfinite(CUf), 0)
          & np.all(np.isfinite(Yf), 0))
    iw = np.where(ok.ravel())[0]
    w = oa_config.area_weights(np.asarray(R['lat'], 'f8'), np.asarray(R['lon'], 'f8')).ravel()[iw]
    sel = lambda A: A.reshape(n, -1)[:, iw]
    out = tl.loo_cells(sel(Yf), [sel(DTf), sel(CUf)], w)
    ST_loo[q] = dict(g_c=float(np.sqrt(np.mean(ec ** 2))), g_u=float(np.sqrt(np.mean(eu ** 2))),
                     g_s=float(np.sqrt(np.mean(sc ** 2))), g_in=int(np.sum(np.abs(ec) < sc)), n=n,
                     c_c=out[5], c_u=out[6], c_s=out[7], c_in=float(out[1]))
rc.savefig(fig, 'loo')
plt.show()
print('leave-one-out, both predictors (global: area of each product; cells: area-weighted RMS):')
for q in REFS_loo:
    s_ = ST_loo[q]
    print(f'  {q:10s} global: error {s_["g_c"]:.4f} vs {s_["g_u"]:.4f}, sigma {s_["g_s"]:.4f}, within {s_["g_in"]}/{s_["n"]};'
          f'  cells: error {s_["c_c"]:.4f} vs {s_["c_u"]:.4f}, sigma {s_["c_s"]:.4f}, within {s_["c_in"]:.2f}')


## Supplementary Fig. 23: the feedback pattern of each ESM

The pattern of the alkalinity rise in each of the four ESMs with Ωarag-dependent calcification and their mean, the pattern used (feedback_pattern.py), on one colour scale. Panel titles: the shift of the decline from 1750 to 2024 on the common area that each pattern gives, each scaled to the observed rate of Barrett et al. (structural_adjustments.py and its runs with the single-model patterns). In OA-Track, their spread enters the 1σ of the alkalinity adjustment in every cell.

In [ ]:
# Notebook cell: Supplementary Fig. 23, the feedback pattern of each of the four ESMs with Omega-dependent
# calcification and their mean (the pattern used), from feedback_pattern.npz, on one colour scale; panel titles: the shift
# of the decline 1750-2024 on the common area that the pattern gives (structural_adjustments.npz and the runs with the
# single-model patterns, structural_adjustments_<key>.npz, each scaled to Barrett et al. with its own m(t)). Reads only.
import pickle
import numpy as np
import oa_config
rc.FIGNAMES.setdefault('patterns', 'FigS32_feedback_patterns')
_sd = oa_config.DATA / 'structural'
_SA2, _FP = np.load(_sd / 'structural_adjustments.npz'), np.load(_sd / 'feedback_pattern.npz')
_lat, _lon = np.asarray(_FP['lat'], 'f8'), np.asarray(_FP['lon'], 'f8')
assert np.allclose(_lat, np.asarray(_SA2['lat'], 'f8')) and np.allclose(_lon, np.asarray(_SA2['lon'], 'f8')), 'grids differ'
_ocean = rc.ocean_on(_lat, _lon).astype(bool)
with open(f'{rc.ROOT}/constraint_results.pkl', 'rb') as f_:
    _cmask = np.asarray(pickle.load(f_)['results_common'][str(_SA2['products'][0])]['mask'], bool)
_w = oa_config.area_weights(_lat, _lon)
_src = str(_SA2['alk_series_source'])


def _shift_of(key_):
    """decline shift on the common area (cell by cell) of the run with one pattern; checks it is from the same pattern file"""
    z_ = _SA2 if key_ == 'pattern' else np.load(_sd / f'structural_adjustments_{key_}.npz')
    assert str(z_['alk_series_source']) == _src, f'{key_}: run from another feedback_pattern.npz: rerun run_structural.sh structural'
    return float(z_['alk_cell_common'])


_PANELS = [('pattern', 'Mean of the four ESMs (used)', _shift_of('pattern')),
           ('pattern_GFDL-CM4', 'GFDL-CM4', _shift_of('pattern_GFDL-CM4')),
           ('pattern_GFDL-ESM4', 'GFDL-ESM4', _shift_of('pattern_GFDL-ESM4')),
           ('pattern_UKESM1-0-LL', 'UKESM1-0-LL', _shift_of('pattern_UKESM1-0-LL')),
           ('pattern_esm2m', 'GFDL-ESM2M-COBALTv2', _shift_of('pattern_esm2m'))]  # mean of five members (legend)
_ALL = {k_: np.asarray(_FP[k_], 'f8') for k_, _, _ in _PANELS}
_sg = lambda v_: format(v_, '+.2f').replace('-', '−')


def _band(a_, s_, n_):
    k_ = _ocean & (_lat[:, None] >= s_) & (_lat[:, None] < n_) & np.isfinite(a_)
    return float(np.sum((a_ * _w)[k_]) / np.sum(_w[k_]))


# layout: the pattern used (a) large on the left, the four single-model patterns (b-e) on the right,
# one colour bar below; the regional numbers are printed instead of written on the maps
fig = plt.figure(figsize=(13.5, 5.8))
_gs = fig.add_gridspec(2, 20, hspace=0.3, wspace=0.1, left=0.01, right=0.99, top=0.93, bottom=0.17)
axes = [fig.add_subplot(_gs[:, :10])] + [fig.add_subplot(_gs[i_, j_:j_ + 5]) for i_ in (0, 1) for j_ in (10, 15)]
_cax_pos = [0.3, 0.08, 0.4, 0.026]
# as Extended Data Fig. 4a: diverging, centred on zero, lower end from the data of all panels (0.5th percentile, steps of 0.5)
# the full range of all panels, rounded outward to 0.25: continuous red-blue colours, none saturated
_lo = min(-0.25, min(float(np.floor(4 * np.nanmin(a_[_ocean])) / 4) for a_ in _ALL.values()))
_hi = max(float(np.ceil(4 * np.nanmax(a_[_ocean])) / 4) for a_ in _ALL.values())
print(f'colour range {_lo:g} to {_hi:g} (full range of the five panels on the ocean)')
# colours on an arcsinh scale (linear near zero, logarithmic for the few extreme cells): the full range without
# saturation, and still contrast between 0 and 2, where most of the ocean lies
ASINH_C = 0.25                    # width of the linear part of the arcsinh scale
_norm = plt.matplotlib.colors.TwoSlopeNorm(vmin=np.arcsinh(_lo / ASINH_C), vcenter=0.0, vmax=np.arcsinh(_hi / ASINH_C))
_TICKS = [t for t in (-15, -5, -2, -1, -0.5, 0, 0.5, 1, 2, 5) if _lo <= t <= _hi]
for k_, (ax, (key_, name_, sh_)) in enumerate(zip(axes, _PANELS)):
    pat_ = _ALL[key_]
    rc.map_mesh(ax, _lon, _lat, np.where(~_ocean, 1.0, np.nan), cmap=plt.matplotlib.colors.ListedColormap(['0.85']), vmin=0, vmax=1)
    pc = rc.map_mesh(ax, _lon, _lat, np.where(_ocean, np.arcsinh(pat_ / ASINH_C), np.nan), cmap='RdBu_r', norm=_norm)
    rc.map_contour(ax, _lon, _lat, _cmask.astype('f8'), levels=[0.5], colors='k', linewidths=0.6)
    neg_ = 100 * np.sum(_w[_ocean & (pat_ < 0)]) / np.sum(_w[_ocean & np.isfinite(pat_)])
    ax.set_title(name_ + (f': {_sg(sh_)} points' if sh_ is not None else ''), fontsize=11 if k_ == 0 else (9.5 if k_ < 5 else 8.5))
    ax.text(0.0, 1.02, 'abcdefghij'[k_], transform=ax.transAxes, fontsize=16 if k_ < 5 else 13, fontweight='bold', va='bottom')
    print(f'{name_}: south of 60S {_band(pat_, -90, -60):.2f}, north of 60N {_band(pat_, 60, 90):.2f}; '
          f'negative on {neg_:.1f} % of the ocean')
cax = fig.add_axes(_cax_pos)
cb = fig.colorbar(pc, cax=cax, orientation='horizontal')
cb.set_ticks(np.arcsinh(np.asarray(_TICKS) / ASINH_C))
cb.set_ticklabels([f'{t:g}'.replace('-', '−') for t in _TICKS])
cb.set_label('Pattern of the alkalinity rise (common-area mean = 1; blue: negative; arcsinh scale)', fontsize=9)
print('decline shift on the common area: ' + ', '.join(f'{n_} {s_:+.3f}' for _, n_, s_ in _PANELS if s_ is not None))
rc.savefig(fig, 'patterns')
plt.show()

## Export: the display items with the names of the manuscript

Copies the png and pdf of every figure and table of the paper to `data/<run>/figures/manuscript/` under the name the manuscript uses (Fig1 to Fig4, EDFig1 to EDFig7, EDTable1 to EDTable3, SuppFig1 to SuppFig23, SuppTable1), so that the build of the manuscript reads one folder with one set of names. The mapping is the single place that ties the names of the notebook to those of the manuscript.

In [ ]:
# Export of the display items with the names of the manuscript: notebook name -> manuscript name.
# Extended Data (10 items): EDFig1 trend fits, EDFig2 Jiang et al. chain, EDFig3 state constraint (mechanism), EDFig4
# the two adjustments to the global mean, EDFig5 from Findlay et al. to this study, EDFig6 crossing against CO2, EDFig7
# states 1750 and 2024 (maps); EDTable1 summary, EDTable2 uncertainty, EDTable3 steps from the PHC. Supplement:
# SuppFig1 products in 2024, 2 coverage, 3 global per product, 4 trends of products and ESMs, 5 constraint per product,
# 6 dSST bias, 7 correction terms, 8 local constraint, 9 regional decline per product, 10-21 the indicators of OA-Track,
# 22 leave-one-out test; SuppTable1 the carbonate system of the common area in 1750 and 2024 (Supplementary Table 1).
# Numbers follow the order of first citation in the manuscript.
import os, shutil
MANUSCRIPT_NAMES = {
    'Fig1_best_estimate': 'Fig1', 'Fig2_constraint': 'Fig2', 'Fig3_crossing': 'Fig4', 'Fig4_coverage': 'Fig3',
    'TableS1_summary': 'EDTable1', 'TableS3_uncertainty': 'EDTable2', 'TableS2_steps': 'EDTable3',
    'FigS11_trend_fits': 'EDFig1', 'FigS15_jiang_chain': 'EDFig2', 'FigS12_state_constraint': 'EDFig3',
    'FigS28_adjustments': 'EDFig4', 'FigS13_findlay': 'EDFig5', 'FigS8_crossing_co2': 'EDFig6',
    'FigS14_states_1750_2024': 'EDFig7',
    'FigS1_products_2024': 'SuppFig1', 'FigS10_coverage_map': 'SuppFig2', 'FigS2_global_per_product': 'SuppFig3',
    'FigS29_trends': 'SuppFig4', 'FigS3_constraint': 'SuppFig5', 'FigS4_dSST_bias': 'SuppFig6',
    'FigS5_correction_terms': 'SuppFig7', 'FigS6_local_constraint': 'SuppFig8',
    'FigS9_regional_decline_per_product': 'SuppFig9', 'FigS30_leave_one_out': 'SuppFig22',
    'FigS32_feedback_patterns': 'SuppFig23',
    'TableS4_product_indicators': 'SuppTable1',
}
for k_, q_ in enumerate(('omega_arag', 'omega_calc', 'ph', 'hplus', 'hplus_free', 'pco2', 'fco2', 'co2', 'hco3', 'co3', 'revelle', 'dic')):
    MANUSCRIPT_NAMES[f'FigS{16 + k_}_indicator_{q_}'] = f'SuppFig{10 + k_}'
_dest = f'{rc.FIGDIR}/manuscript'
os.makedirs(_dest, exist_ok=True)
_missing = []
for nb_name, ms_name in MANUSCRIPT_NAMES.items():
    for ext in ('png', 'pdf'):
        p_ = f'{rc.FIGDIR}/{nb_name}.{ext}'
        if os.path.exists(p_):
            shutil.copyfile(p_, f'{_dest}/{ms_name}.{ext}')
        else:
            _missing.append(f'{nb_name}.{ext}')
print(f'{2 * len(MANUSCRIPT_NAMES) - len(_missing)} files copied to {_dest}' + (f'; missing: {", ".join(_missing)}' if _missing else ''))
